# V-MD3 FMCW Range, Angle, and Rail-SAR Processing

This notebook processes the V-MD3 data collected during Runs A–D of the rail-SAR proof-of-concept experiment.

The notebook is organized into the following major sections:

1. **Notebook setup**
   - Imports
   - Repository and data directories
   - Experiment constants
   - Binary-file discovery
   - Binary decoding functions
   - Cube construction and validation functions
   - Common plotting and processing functions

2. **Run A: Conventional radar processing and validation**
   - Decode RADC and FPGA RFFT binary data
   - Validate array dimensions and axis meanings
   - Construct one-dimensional range profiles
   - Construct two-dimensional range–azimuth heatmaps
   - Compare offline RADC processing with the FPGA RFFT output
   - Evaluate frame-to-frame phase stability

3. **Runs B and C: Centered-target SAR**
   - Use Run B as the background dataset
   - Use Run C as the centered-sphere dataset
   - Construct the coherent SAR image
   - Verify the reconstructed target location

4. **Runs B and D: Offset-target SAR**
   - Use Run B as the background dataset
   - Use Run D as the offset-sphere dataset
   - Reconstruct the SAR image
   - Verify that the target shifts in the expected cross-range direction

All reusable decoding, validation, signal-processing, and plotting operations should be defined as functions in the setup section rather than duplicated throughout the notebook.

## Experiment Data Organization

The experiment contains four acquisition runs.

### Run A — Fixed-position phase test

- The radar remains at the center rail position.
- The sphere is present near boresight.
- Approximately 20 repeated frames were recorded.
- Run A does **not** contain a 17-position synthetic aperture.
- Run A is used to validate:
  - binary decoding;
  - offline range processing;
  - FPGA RFFT interpretation;
  - range–azimuth processing;
  - frame-to-frame phase stability.

### Run B — Background rail scan

- The sphere is removed.
- Data are recorded at 17 mechanical positions.
- Run B characterizes static clutter from the rail, mounting structure, absorber, floor, cables, and surrounding environment.

### Run C — Centered-sphere rail scan

- The sphere is located near the center of the synthetic-aperture scene.
- Data are recorded at 17 mechanical positions.
- Position index 8 is the center rail position.

### Run D — Offset-sphere rail scan

- The sphere is displaced in cross-range relative to Run C.
- Data are recorded at the same 17 mechanical positions.
- Run D is used to verify that the reconstructed target moves in the expected direction.

The SAR position indices are

$$p = 0,1,\ldots,16,$$

with

$$p_{\mathrm{center}} = 8.$$

## 1.0 Notebook Setup

This section establishes the reusable infrastructure used throughout the notebook.

The setup section contains:

1. **Python imports, repository paths, and experiment definitions**
   - repository and data directories;
   - Run A–D binary-data directories;
   - SAR position indices;
   - center-position index;
   - Galil encoder counts.

2. **Binary-file discovery**
   - locate all `.bin` files for Runs A–D;
   - verify the expected number of files in each run.

3. **Reusable function definitions**
   - filename and position-index parsing;
   - binary-record parsing;
   - payload reading;
   - RADC decoding;
   - FPGA RFFT decoding;
   - DONE decoding;
   - stream selection and stack loading;
   - array-shape and data-integrity validation.

4. **File-level manifest construction and validation**

   `FILE_MANIFEST` contains one row per acquisition file, including:

   - run identifier;
   - acquisition type;
   - position index, where applicable;
   - Galil encoder count, where applicable;
   - filename;
   - file path;
   - file size.

5. **Record-level manifest construction and validation**

   `RECORD_MANIFEST` contains one row per stored V-MD3 record, including:

   - run identifier;
   - acquisition file;
   - position index and encoder count, where applicable;
   - record index within the file;
   - stream type: `RADC`, `RFFT`, or `DONE`;
   - frame index within each stream;
   - header and payload byte offsets;
   - payload length;
   - decoded DONE value, where applicable.

6. **Run A loading and validation**
   - load all Run A RADC frames;
   - load all Run A FPGA RFFT frames;
   - load the Run A DONE values;
   - verify frame counts, cube shapes, finite values, and DONE-counter continuity.

The binary files are indexed during setup, but the complete Runs B–D datasets will generally be decoded only when required by their respective processing sections. Run A is loaded during setup because it is used immediately in the next section.

This keeps the notebook self-contained while avoiding unnecessary memory use and long startup times.

The intended data flow is:

```text
binary files
    ↓
file-level manifest
    ↓
record-level manifest
    ↓
stream-specific payload decoder
    ↓
validated NumPy stack
    ↓
range, angle, phase, and SAR processing
```

### Canonical Array Shapes

One decoded RADC frame has shape:

```text
RADC frame: (128, 64, 4)
```

with axis meanings:

```text
axis 0 = fast-time ADC sample
axis 1 = chirp / slow-time index
axis 2 = receive-channel index
```

Therefore:

```python
radc_cube[sample, chirp, channel]
```

A stack of RADC frames has shape:

```text
RADC stack: (number_of_frames, 128, 64, 4)
```

with representation:

```python
radc_stack[frame, sample, chirp, channel]
```

One decoded FPGA RFFT frame also has shape:

```text
RFFT frame: (128, 64, 4)
```

and is provisionally represented as:

```python
rfft_cube[range_bin, slow_time_index, channel]
```

A stack of FPGA RFFT frames has shape:

```text
RFFT stack: (number_of_frames, 128, 64, 4)
```

with representation:

```python
rfft_stack[frame, range_bin, slow_time_index, channel]
```

The term `slow_time_index` is provisional. Run A processing will determine whether this axis represents:

- chirp-indexed range-FFT outputs;
- Doppler-bin outputs;
- or another FPGA-specific ordering.

After its meaning is confirmed, the corresponding variable names, axis labels, comments, and docstrings should be updated throughout the notebook.

### 1.1 Python Imports and Repository and data-directory definitions

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

In [ ]:
# ---------------------------------------------------------------------
# Repository and data locations
# ---------------------------------------------------------------------

REPO_ROOT = Path(r"D:\Research\repos\vmd3_projects")
DATA_ROOT = REPO_ROOT / "data"

RUN_DIRS = {
    letter: DATA_ROOT / f"20260709_run_{letter}" / "bins"
    for letter in "abcd"
}

# ---------------------------------------------------------------------
# SAR acquisition positions: Runs B, C, and D
# ---------------------------------------------------------------------

SAR_POSITION_INDICES = np.arange(17)
CENTER_POSITION_INDEX = 8

GALIL_ENCODER_COUNTS = np.arange(
    0,
    800_000 + 50_000,
    50_000,
)

assert len(SAR_POSITION_INDICES) == 17
assert len(GALIL_ENCODER_COUNTS) == 17
assert SAR_POSITION_INDICES[CENTER_POSITION_INDEX] == 8
assert GALIL_ENCODER_COUNTS[CENTER_POSITION_INDEX] == 400_000

print("Repository:", REPO_ROOT)
print()

for run, folder in RUN_DIRS.items():
    print(f"Run {run.upper()}: {folder}")
    print(f"  Exists: {folder.exists()}")

### 1.2 Binary-file discover to `BIN_FILES`


In [ ]:
# ---------------------------------------------------------------------
# Inspect available binary files
# ---------------------------------------------------------------------

BIN_FILES = {}

for run, folder in RUN_DIRS.items():
    files = sorted(folder.glob("*.bin"))
    BIN_FILES[run] = files

    print(f"Run {run.upper()}")
    print(f"  Folder: {folder}")
    print(f"  Number of .bin files: {len(files)}")

    if not files:
        print("  No binary files found.")
    else:
        for path in files:
            size_bytes = path.stat().st_size
            print(f"  {path.name:<60} {size_bytes:>12,} bytes")

    print()

### 1.3 Function Definitions 
All binary parsing, decoding, loading, and validation functions

In [ ]:
# ---------------------------------------------------------------------
# Build and validate the binary-file manifest
# ---------------------------------------------------------------------

_POSITION_PATTERN = re.compile(
    r"^scan_(?:pos)?(?P<position>\d+)cm\.bin$",
    flags=re.IGNORECASE,
)


def parse_sar_position_index(filename: str) -> int:
    """
    Extract the acquisition position index from a Run B, C, or D filename.

    Examples
    --------
    scan_0cm.bin     -> 0
    scan_pos8cm.bin  -> 8
    scan_pos16cm.bin -> 16

    The extracted number is treated as a position index, not as a
    trustworthy physical distance in centimeters.
    """
    match = _POSITION_PATTERN.fullmatch(filename)

    if match is None:
        raise ValueError(
            f"Could not determine position index from filename: {filename}"
        )

    return int(match.group("position"))

# ---------------------------------------------------------------------
# V-MD3 binary parsing, decoding, loading, and validation functions
# ---------------------------------------------------------------------

FRAME_HEADER_BYTES = 8

STREAM_CODE_TO_NAME = {
    b"RADC": "radc",
    b"RFFT": "rfft",
    b"DONE": "done",
}

EXPECTED_PAYLOAD_BYTES = {
    "radc": 131_072,
    "rfft": 131_072,
    "done": 4,
}

RADC_FRAME_SHAPE = (128, 64, 4)
RFFT_FRAME_SHAPE = (128, 64, 4)


def scan_vmd3_bin(path: Path) -> pd.DataFrame:
    """
    Scan one acquisition .bin file and return one row per stored frame.

    Stored frame format
    -------------------
    bytes 0:4   Stream code: RADC, RFFT, or DONE
    bytes 4:8   Payload length, little-endian unsigned integer
    bytes 8:    Payload
    """
    path = Path(path)
    file_size = path.stat().st_size

    rows = []
    stream_counts = {
        stream_name: 0
        for stream_name in STREAM_CODE_TO_NAME.values()
    }

    offset = 0
    record_index = 0

    with path.open("rb") as file:
        while offset < file_size:
            file.seek(offset)
            header = file.read(FRAME_HEADER_BYTES)

            if len(header) != FRAME_HEADER_BYTES:
                raise EOFError(
                    f"{path.name}: incomplete frame header "
                    f"at byte offset {offset:,}."
                )

            stream_code = header[:4]

            if stream_code not in STREAM_CODE_TO_NAME:
                raise ValueError(
                    f"{path.name}: unknown stream code "
                    f"{stream_code!r} at byte offset {offset:,}."
                )

            stream = STREAM_CODE_TO_NAME[stream_code]

            payload_length = int.from_bytes(
                header[4:8],
                byteorder="little",
                signed=False,
            )

            expected_length = EXPECTED_PAYLOAD_BYTES[stream]

            if payload_length != expected_length:
                raise ValueError(
                    f"{path.name}: {stream.upper()} record "
                    f"{record_index} has payload length "
                    f"{payload_length:,}; expected "
                    f"{expected_length:,}."
                )

            payload_offset = offset + FRAME_HEADER_BYTES
            next_offset = payload_offset + payload_length

            if next_offset > file_size:
                raise EOFError(
                    f"{path.name}: record {record_index} extends "
                    "beyond the end of the file."
                )

            stream_frame_index = stream_counts[stream]
            stream_counts[stream] += 1

            rows.append(
                {
                    "record_index": record_index,
                    "stream": stream,
                    "stream_frame_index": stream_frame_index,
                    "header_offset": offset,
                    "payload_offset": payload_offset,
                    "payload_length": payload_length,
                    "frame_length": (
                        FRAME_HEADER_BYTES + payload_length
                    ),
                }
            )

            offset = next_offset
            record_index += 1

    if offset != file_size:
        raise ValueError(
            f"{path.name}: parser stopped at byte {offset:,}, "
            f"but the file size is {file_size:,}."
        )

    return pd.DataFrame(rows)


def read_record_payload(
    path: Path,
    payload_offset: int,
    payload_length: int,
) -> bytes:
    """Read one payload from a parsed V-MD3 record."""
    path = Path(path)

    with path.open("rb") as file:
        file.seek(int(payload_offset))
        payload = file.read(int(payload_length))

    if len(payload) != int(payload_length):
        raise EOFError(
            f"{path.name}: requested {payload_length:,} bytes "
            f"at offset {payload_offset:,}, but read "
            f"{len(payload):,} bytes."
        )

    return payload


def decode_radc_2d(payload: bytes) -> np.ndarray:
    """
    Decode one mode-0 RADC payload.

    Returns
    -------
    cube
        Shape: (sample, chirp, channel) = (128, 64, 4)
    """
    expected_bytes = EXPECTED_PAYLOAD_BYTES["radc"]

    if len(payload) != expected_bytes:
        raise ValueError(
            f"Expected {expected_bytes:,} RADC bytes, "
            f"received {len(payload):,}."
        )

    # Stored as:
    # chirp, channel, interleaved [Q0, I0, Q1, I1, ...]
    raw = np.frombuffer(
        payload,
        dtype="<i2",
    ).reshape(64, 4, 256)

    q = raw[:, :, 0::2].astype(np.float64)
    i = raw[:, :, 1::2].astype(np.float64)

    # Convert from (chirp, channel, sample)
    # to (sample, chirp, channel).
    cube = np.transpose(
        i + 1j * q,
        (2, 0, 1),
    )

    if cube.shape != RADC_FRAME_SHAPE:
        raise ValueError(
            f"Decoded RADC shape is {cube.shape}; "
            f"expected {RADC_FRAME_SHAPE}."
        )

    return cube


def decode_rfft_2d(payload: bytes) -> np.ndarray:
    """
    Decode one mode-0 FPGA RFFT payload.

    Returns
    -------
    cube
        Provisional shape:
        (range_bin, slow_time_index, channel) = (128, 64, 4)

    Notes
    -----
    The meaning of slow_time_index is not yet confirmed. It may represent
    chirp/slow time, Doppler bins, or another FPGA-specific ordering.
    """
    expected_bytes = EXPECTED_PAYLOAD_BYTES["rfft"]

    if len(payload) != expected_bytes:
        raise ValueError(
            f"Expected {expected_bytes:,} RFFT bytes, "
            f"received {len(payload):,}."
        )

    # Provisional FPGA storage arrangement:
    # channel, range_bin, axis1_index, [Q, I]
    raw = np.frombuffer(
        payload,
        dtype="<i2",
    ).reshape(4, 128, 64, 2)

    q = raw[:, :, :, 0].astype(np.float64)
    i = raw[:, :, :, 1].astype(np.float64)

    # Convert to:
    # range_bin, axis1_index, channel
    cube = np.transpose(
        i + 1j * q,
        (1, 2, 0),
    )

    if cube.shape != RFFT_FRAME_SHAPE:
        raise ValueError(
            f"Decoded RFFT shape is {cube.shape}; "
            f"expected {RFFT_FRAME_SHAPE}."
        )

    return cube


def decode_done(payload: bytes) -> int:
    """
    Decode one DONE payload as a little-endian unsigned integer.

    The values appear to behave as radar-side frame counters.
    """
    expected_bytes = EXPECTED_PAYLOAD_BYTES["done"]

    if len(payload) != expected_bytes:
        raise ValueError(
            f"Expected {expected_bytes} DONE bytes, "
            f"received {len(payload)}."
        )

    return int.from_bytes(
        payload,
        byteorder="little",
        signed=False,
    )


def _select_stream_records(
    record_manifest: pd.DataFrame,
    run: str,
    stream: str,
    position_index: int | None = None,
) -> pd.DataFrame:
    """Select and order records for one run, stream, and position."""
    run = run.upper()
    stream = stream.lower()

    if run not in {"A", "B", "C", "D"}:
        raise ValueError(
            f"Unknown run {run!r}. Expected A, B, C, or D."
        )

    if stream not in {"radc", "rfft", "done"}:
        raise ValueError(
            f"Unknown stream {stream!r}."
        )

    records = record_manifest.loc[
        (record_manifest["run"] == run)
        & (record_manifest["stream"] == stream)
    ].copy()

    if run == "A":
        if position_index is not None:
            raise ValueError(
                "Run A is fixed-position and does not use "
                "a SAR position index."
            )
    else:
        if position_index is None:
            raise ValueError(
                f"position_index is required for Run {run}."
            )

        position_index = int(position_index)

        if position_index not in SAR_POSITION_INDICES:
            raise ValueError(
                f"Position index {position_index} is outside "
                "the expected range 0 through 16."
            )

        records = records.loc[
            records["position_index"] == position_index
        ]

    records = records.sort_values("stream_frame_index")

    if records.empty:
        raise ValueError(
            f"No {stream.upper()} records found for Run {run}, "
            f"position {position_index}."
        )

    return records


def load_radc_stack(
    record_manifest: pd.DataFrame,
    run: str,
    position_index: int | None = None,
) -> np.ndarray:
    """
    Load RADC data with shape:

        (frame, sample, chirp, channel)
    """
    records = _select_stream_records(
        record_manifest,
        run,
        "radc",
        position_index,
    )

    cubes = []

    for record in records.itertuples(index=False):
        payload = read_record_payload(
            record.path,
            record.payload_offset,
            record.payload_length,
        )

        cubes.append(decode_radc_2d(payload))

    return np.stack(cubes, axis=0)


def load_rfft_stack(
    record_manifest: pd.DataFrame,
    run: str,
    position_index: int | None = None,
) -> np.ndarray:
    """
    Load FPGA RFFT data with provisional shape:

        (frame, range_bin, slow_time_index, channel)
    """
    records = _select_stream_records(
        record_manifest,
        run,
        "rfft",
        position_index,
    )

    cubes = []

    for record in records.itertuples(index=False):
        payload = read_record_payload(
            record.path,
            record.payload_offset,
            record.payload_length,
        )

        cubes.append(decode_rfft_2d(payload))

    return np.stack(cubes, axis=0)


def load_done_values(
    record_manifest: pd.DataFrame,
    run: str,
    position_index: int | None = None,
) -> np.ndarray:
    """Load the DONE values for one run or SAR position."""
    records = _select_stream_records(
        record_manifest,
        run,
        "done",
        position_index,
    )

    values = []

    for record in records.itertuples(index=False):
        payload = read_record_payload(
            record.path,
            record.payload_offset,
            record.payload_length,
        )

        values.append(decode_done(payload))

    return np.asarray(values, dtype=np.uint32)


def validate_complex_stack(
    stack: np.ndarray,
    expected_frame_shape: tuple[int, int, int],
    expected_frames: int | None = None,
    label: str = "stack",
) -> None:
    """Validate one decoded complex-data stack."""
    if not isinstance(stack, np.ndarray):
        raise TypeError(f"{label} must be a NumPy array.")

    if stack.ndim != 4:
        raise ValueError(
            f"{label} must have four dimensions; "
            f"received shape {stack.shape}."
        )

    if stack.shape[1:] != expected_frame_shape:
        raise ValueError(
            f"{label} has per-frame shape {stack.shape[1:]}; "
            f"expected {expected_frame_shape}."
        )

    if (
        expected_frames is not None
        and stack.shape[0] != expected_frames
    ):
        raise ValueError(
            f"{label}: expected {expected_frames} frames, "
            f"received {stack.shape[0]}."
        )

    if not np.all(np.isfinite(stack)):
        raise ValueError(
            f"{label} contains NaN or infinite values."
        )

    print(f"{label} validation passed.")
    print(f"  Shape:               {stack.shape}")
    print(f"  Data type:           {stack.dtype}")
    print(f"  I minimum:           {stack.real.min():.0f}")
    print(f"  I maximum:           {stack.real.max():.0f}")
    print(f"  Q minimum:           {stack.imag.min():.0f}")
    print(f"  Q maximum:           {stack.imag.max():.0f}")
    print(
        "  Exact-zero fraction: "
        f"{np.mean(stack == 0):.6f}"
    )


def validate_record_manifest(
    record_manifest: pd.DataFrame,
) -> None:
    """
    Validate the stored records in every acquisition file.

    The acquisition code saves valid RADC, RFFT, and DONE frames in
    arrival order. Therefore, this function does not require repeating
    RADC-RFFT-DONE triplets.
    """
    expected_frames_per_file = {
        "A": 5,
        "B": 20,
        "C": 20,
        "D": 20,
    }

    expected_streams = ("radc", "rfft", "done")

    for (run, filename), group in record_manifest.groupby(
        ["run", "filename"],
        sort=False,
    ):
        group = group.sort_values("record_index").reset_index(drop=True)

        expected_count = expected_frames_per_file[run]
        expected_total_records = expected_count * len(expected_streams)

        # Check total number of stored records.
        if len(group) != expected_total_records:
            raise ValueError(
                f"Run {run}, {filename}: expected "
                f"{expected_total_records} total records, "
                f"found {len(group)}."
            )

        # Check that record indices are sequential.
        actual_record_indices = group["record_index"].astype(int).to_numpy()
        expected_record_indices = np.arange(expected_total_records)

        if not np.array_equal(
            actual_record_indices,
            expected_record_indices,
        ):
            raise ValueError(
                f"Run {run}, {filename}: record indices are "
                "not consecutive."
            )

        # Validate each stream independently.
        for stream in expected_streams:
            stream_group = (
                group.loc[group["stream"] == stream]
                .sort_values("stream_frame_index")
            )

            if len(stream_group) != expected_count:
                raise ValueError(
                    f"Run {run}, {filename}: expected "
                    f"{expected_count} {stream.upper()} records, "
                    f"found {len(stream_group)}."
                )

            actual_stream_indices = (
                stream_group["stream_frame_index"]
                .astype(int)
                .to_numpy()
            )

            expected_stream_indices = np.arange(expected_count)

            if not np.array_equal(
                actual_stream_indices,
                expected_stream_indices,
            ):
                raise ValueError(
                    f"Run {run}, {filename}: "
                    f"{stream.upper()} frame indices are "
                    "not consecutive."
                )

            expected_payload_length = EXPECTED_PAYLOAD_BYTES[stream]

            if not np.all(
                stream_group["payload_length"].to_numpy()
                == expected_payload_length
            ):
                raise ValueError(
                    f"Run {run}, {filename}: one or more "
                    f"{stream.upper()} payload lengths are incorrect."
                )

        # DONE should behave as a consecutive radar-side counter
        # within an acquisition file.
        done_values = (
            group.loc[group["stream"] == "done", "done_value"]
            .dropna()
            .astype(np.int64)
            .to_numpy()
        )

        if len(done_values) != expected_count:
            raise ValueError(
                f"Run {run}, {filename}: expected "
                f"{expected_count} decoded DONE values, "
                f"found {len(done_values)}."
            )

        if (
            len(done_values) > 1
            and not np.all(np.diff(done_values) == 1)
        ):
            raise ValueError(
                f"Run {run}, {filename}: DONE values are not "
                f"consecutive: {done_values.tolist()}"
            )

    print("Record-manifest validation passed for all files.")
    print(
        "Note: RADC, RFFT, and DONE arrival order was not "
        "required to form repeating triplets."
    )

#### Shared Processing Functions

These functions consolidate the range, Doppler, normalization, and beamforming operations used throughout the Run A analysis.

Keeping these operations in reusable functions ensures that the same axis conventions, window definitions, FFT ordering, and normalization are applied consistently. The functions will also be reused when processing Runs B, C, and D.

In [ ]:
# Define the supported spectral windows.
WINDOW_FUNCTIONS = {
    "rectangular": lambda length: np.ones(length),
    "hann": np.hanning,
    "hamming": np.hamming,
    "blackman": np.blackman,
}


def create_window(
    window_name,
    length,
):
    """
    Create one supported spectral window.
    """

    normalized_name = window_name.lower()

    if normalized_name not in WINDOW_FUNCTIONS:
        raise ValueError(
            f"Unsupported window '{window_name}'. "
            f"Choose from {list(WINDOW_FUNCTIONS)}."
        )

    return WINDOW_FUNCTIONS[
        normalized_name
    ](
        length
    )


def compute_radc_range_fft(
    radc_frame,
    range_window="hann",
    remove_fast_time_mean=False,
):
    """
    Compute the range FFT for one RADC frame.

    Input shape:
        (fast-time sample, chirp, RX channel)

    Output shape:
        (range bin, chirp, RX channel)
    """

    if radc_frame.ndim != 3:
        raise ValueError(
            "radc_frame must have shape "
            "(fast-time sample, chirp, RX channel)."
        )

    processed_radc = np.asarray(
        radc_frame,
        dtype=np.complex128,
    ).copy()

    # Optionally remove the fast-time mean from every
    # chirp and RX channel.
    if remove_fast_time_mean:
        processed_radc -= np.mean(
            processed_radc,
            axis=0,
            keepdims=True,
        )

    # Apply the selected window across fast time.
    window = create_window(
        range_window,
        processed_radc.shape[0],
    )

    processed_radc *= (
        window[:, np.newaxis, np.newaxis]
    )

    # Transform fast-time sample index into range-bin index.
    return np.fft.fft(
        processed_radc,
        axis=0,
    )


def compute_radc_range_doppler(
    radc_frame,
    range_window="hann",
    doppler_window="hann",
    remove_fast_time_mean=False,
    center_doppler=True,
):
    """
    Compute the range–Doppler cube for one RADC frame.

    Input shape:
        (fast-time sample, chirp, RX channel)

    Output shape:
        (range bin, Doppler bin, RX channel)
    """

    # Compute the windowed range FFT.
    range_fft = compute_radc_range_fft(
        radc_frame=radc_frame,
        range_window=range_window,
        remove_fast_time_mean=remove_fast_time_mean,
    )

    # Apply the selected window across chirp index.
    window = create_window(
        doppler_window,
        range_fft.shape[1],
    )

    windowed_range_fft = (
        range_fft
        * window[np.newaxis, :, np.newaxis]
    )

    # Transform chirp index into Doppler-bin index.
    range_doppler_cube = np.fft.fft(
        windowed_range_fft,
        axis=1,
    )

    # Center zero Doppler at the middle Doppler bin.
    if center_doppler:
        range_doppler_cube = np.fft.fftshift(
            range_doppler_cube,
            axes=1,
        )

    return range_doppler_cube


def extract_zero_doppler(
    range_doppler_data,
    doppler_axis=-2,
):
    """
    Extract the centered zero-Doppler bin.

    This works with either:
        (range, Doppler, RX)

    or:
        (frame, range, Doppler, RX)
    """

    zero_doppler_bin = (
        range_doppler_data.shape[doppler_axis]
        // 2
    )

    zero_doppler_data = np.take(
        range_doppler_data,
        indices=zero_doppler_bin,
        axis=doppler_axis,
    )

    return (
        zero_doppler_data,
        zero_doppler_bin,
    )


def normalized_magnitude_db(
    complex_data,
    minimum_db=-120,
):
    """
    Normalize complex magnitude to 0 dB and apply a lower floor.
    """

    magnitude = np.abs(
        complex_data
    )

    normalized_magnitude = (
        magnitude
        / np.maximum(
            magnitude.max(),
            np.finfo(float).tiny,
        )
    )

    magnitude_db = 20 * np.log10(
        np.maximum(
            normalized_magnitude,
            np.finfo(float).tiny,
        )
    )

    return np.maximum(
        magnitude_db,
        minimum_db,
    )


def conventional_range_azimuth(
    range_rx_data,
    azimuth_angle_deg,
    rx_positions_m,
    carrier_frequency_hz,
):
    """
    Apply conventional beamforming across RX channels.

    Input shape:
        (range bin, RX channel)

    Output shape:
        (range bin, azimuth angle)
    """

    wavelength_m = (
        299_792_458.0
        / carrier_frequency_hz
    )

    wavenumber_rad_per_m = (
        2 * np.pi
        / wavelength_m
    )

    azimuth_angle_rad = np.deg2rad(
        azimuth_angle_deg
    )

    # Construct the expected RX phase progression
    # for every trial azimuth angle.
    steering_matrix = np.exp(
        -1j
        * wavenumber_rad_per_m
        * rx_positions_m[:, np.newaxis]
        * np.sin(
            azimuth_angle_rad
        )[np.newaxis, :]
    )

    # Coherently combine the RX channels.
    return (
        range_rx_data
        @ np.conj(
            steering_matrix
        )
    )

### 1.4 Build and validate `FILE_MANIFEST`

In [ ]:
# ---------------------------------------------------------------------
# Build and validate the binary-file manifest
# ---------------------------------------------------------------------

manifest_rows = []

for run, files in BIN_FILES.items():
    for path in files:

        # Run A is fixed-position and does not use a SAR position index.
        if run == "a":
            position_index = pd.NA
            encoder_count = pd.NA
            acquisition_type = "fixed_position"

        else:
            position_index = parse_sar_position_index(path.name)

            if position_index not in SAR_POSITION_INDICES:
                raise ValueError(
                    f"Run {run.upper()} contains unexpected position "
                    f"index {position_index}: {path.name}"
                )

            encoder_count = int(
                GALIL_ENCODER_COUNTS[position_index]
            )

            acquisition_type = "sar_scan"

        manifest_rows.append(
            {
                "run": run.upper(),
                "acquisition_type": acquisition_type,
                "position_index": position_index,
                "encoder_count": encoder_count,
                "filename": path.name,
                "file_size_bytes": path.stat().st_size,
                "path": path,
            }
        )


FILE_MANIFEST = pd.DataFrame(manifest_rows)

FILE_MANIFEST["position_index"] = (
    FILE_MANIFEST["position_index"].astype("Int64")
)

FILE_MANIFEST["encoder_count"] = (
    FILE_MANIFEST["encoder_count"].astype("Int64")
)

FILE_MANIFEST = (
    FILE_MANIFEST
    .sort_values(
        ["run", "position_index"],
        na_position="first",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Validate the file-level manifest
# ---------------------------------------------------------------------

expected_file_counts = {
    "A": 1,
    "B": 17,
    "C": 17,
    "D": 17,
}

actual_file_counts = (
    FILE_MANIFEST.groupby("run")
    .size()
    .to_dict()
)

if actual_file_counts != expected_file_counts:
    raise ValueError(
        f"Unexpected file counts: {actual_file_counts}"
    )


expected_positions = set(
    np.asarray(SAR_POSITION_INDICES, dtype=int)
)

for run in ["B", "C", "D"]:
    run_rows = FILE_MANIFEST.loc[
        FILE_MANIFEST["run"] == run
    ]

    run_positions = set(
        run_rows["position_index"]
        .dropna()
        .astype(int)
    )

    if run_positions != expected_positions:
        raise ValueError(
            f"Run {run} positions are incorrect. "
            f"Found: {sorted(run_positions)}"
        )

    if run_rows["position_index"].duplicated().any():
        raise ValueError(
            f"Run {run} contains duplicate position indices."
        )


print("File manifest validation passed.")
print()
print("Files per run:")
print(FILE_MANIFEST.groupby("run").size())

print()
print("File sizes by run:")
print(
    FILE_MANIFEST.groupby("run")["file_size_bytes"]
    .agg(["count", "min", "max", "nunique"])
)

### 1.5 Build `RECORD_MANIFEST`
Build the permanent record-level manifest

In [ ]:
# ---------------------------------------------------------------------
# Build the record-level manifest
# ---------------------------------------------------------------------

record_manifest_parts = []

for file_row in FILE_MANIFEST.itertuples(index=False):
    records = scan_vmd3_bin(file_row.path)

    records.insert(0, "run", file_row.run)
    records.insert(
        1,
        "acquisition_type",
        file_row.acquisition_type,
    )
    records.insert(
        2,
        "position_index",
        file_row.position_index,
    )
    records.insert(
        3,
        "encoder_count",
        file_row.encoder_count,
    )
    records.insert(4, "filename", file_row.filename)
    records.insert(5, "path", file_row.path)

    records["done_value"] = pd.Series(
        pd.NA,
        index=records.index,
        dtype="Int64",
    )

    for index in records.index[
        records["stream"] == "done"
    ]:
        payload = read_record_payload(
            records.at[index, "path"],
            records.at[index, "payload_offset"],
            records.at[index, "payload_length"],
        )

        records.at[index, "done_value"] = decode_done(
            payload
        )

    record_manifest_parts.append(records)


RECORD_MANIFEST = pd.concat(
    record_manifest_parts,
    ignore_index=True,
)

RECORD_MANIFEST["position_index"] = (
    RECORD_MANIFEST["position_index"].astype("Int64")
)

RECORD_MANIFEST["encoder_count"] = (
    RECORD_MANIFEST["encoder_count"].astype("Int64")
)

RECORD_MANIFEST["done_value"] = (
    RECORD_MANIFEST["done_value"].astype("Int64")
)

print("Record manifest created successfully.")
print(f"Total stored records: {len(RECORD_MANIFEST):,}")

Validate and summarize `RECORD_MANIFEST`

In [ ]:
# ---------------------------------------------------------------------
# Validate and summarize the record-level manifest
# ---------------------------------------------------------------------

validate_record_manifest(RECORD_MANIFEST)

print()
print("Frames by run and stream:")

display(
    RECORD_MANIFEST.groupby(["run", "stream"])
    .size()
    .unstack(fill_value=0)
)

### 1.6 Load and Validate Run A Data

In [ ]:
# ---------------------------------------------------------------------
# Load and validate Run A data
# ---------------------------------------------------------------------

run_a_radc = load_radc_stack(
    RECORD_MANIFEST,
    run="A",
)

run_a_rfft = load_rfft_stack(
    RECORD_MANIFEST,
    run="A",
)

run_a_done = load_done_values(
    RECORD_MANIFEST,
    run="A",
)


# Confirm that all three streams contain the same number of frames.
n_radc_frames = run_a_radc.shape[0]
n_rfft_frames = run_a_rfft.shape[0]
n_done_frames = len(run_a_done)

if not (
    n_radc_frames
    == n_rfft_frames
    == n_done_frames
):
    raise ValueError(
        "Run A stream counts do not match:\n"
        f"  RADC: {n_radc_frames}\n"
        f"  RFFT: {n_rfft_frames}\n"
        f"  DONE: {n_done_frames}"
    )


# Validate the decoded complex-data stacks.
validate_complex_stack(
    run_a_radc,
    expected_frame_shape=RADC_FRAME_SHAPE,
    expected_frames=n_radc_frames,
    label="Run A RADC",
)

print()

validate_complex_stack(
    run_a_rfft,
    expected_frame_shape=RFFT_FRAME_SHAPE,
    expected_frames=n_rfft_frames,
    label="Run A FPGA RFFT",
)


# Confirm continuity of the radar-side DONE counter.
done_differences = np.diff(
    run_a_done.astype(np.int64)
)

if not np.all(done_differences == 1):
    raise ValueError(
        "Run A DONE values are not consecutive: "
        f"{run_a_done.tolist()}"
    )


print()
print("Run A stream pairing checks passed.")
print(f"  Number of captures: {n_radc_frames}")
print(f"  DONE values:        {run_a_done.tolist()}")
print()
print("Run A array conventions:")
print(
    "  run_a_radc:",
    run_a_radc.shape,
    "= (frame, sample, chirp, channel)",
)
print(
    "  run_a_rfft:",
    run_a_rfft.shape,
    "= (frame, range_bin, slow_time_index, channel)",
)

# 2.0 Run A — 1D Range and 2D Range–Azimuth Processing

This section validates the fixed-position Run A acquisition before SAR processing. We first inspect the raw complex ADC samples, form offline range profiles, compare them with the FPGA RFFT output, investigate the provisional RFFT axis convention, construct range–azimuth images, and evaluate frame-to-frame phase stability.

## 2.1 — Inspect One Raw RADC Chirp

This cell inspects the complex fast-time samples from one Run A frame, one chirp, and one RX channel before any averaging or additional processing.

The plots show:

- the raw I and Q ADC samples;
- the magnitude of the complex ADC samples;
- the fast-time IQ trajectory;
- a Python-computed range FFT using a rectangular window.

The purpose is to verify that the decoded RADC data contain a plausible complex FMCW beat signal and to identify the dominant FFT bins without yet applying windowing, DC removal, channel combining, or physical range calibration. The initially expected sphere region at bins 17–19 is highlighted for reference.

In [ ]:
# Select one fixed-position Run A measurement for initial inspection.
# Start with the first frame, first chirp, and first RX channel so that
# averaging or combining data does not hide any acquisition problems.
frame_index = 0
chirp_index = 0
channel_index = 0


# Extract all 128 complex fast-time ADC samples from the selected chirp.
# Fast-time samples contain the beat-frequency information used to estimate range.
x = run_a_radc[
    frame_index,
    :,
    chirp_index,
    channel_index,
]

adc_sample_index = np.arange(x.size)


# Compute the initial offline range FFT.
# The FFT converts the fast-time beat signal into range-bin data.
#
# Begin with the simplest possible processing:
#   - no window;
#   - no mean/DC subtraction;
#   - no averaging;
#   - no reordering of the FFT bins.
#
# These processing options will be tested later when comparing against
# the FPGA-generated RFFT data.
X = np.fft.fft(x)
range_bin_index = np.arange(X.size)


# Convert the FFT magnitude to decibels and normalize its largest value to 0 dB.
# Normalization makes the relative strengths of the peaks easier to compare.
X_magnitude_db = 20 * np.log10(
    np.maximum(np.abs(X), np.finfo(float).tiny)
)
X_magnitude_db -= X_magnitude_db.max()


# Create a four-panel diagnostic figure showing:
#   1. I and Q versus fast-time sample;
#   2. complex-sample magnitude versus fast-time sample;
#   3. the trajectory of the samples in the IQ plane;
#   4. the initial offline range FFT.
fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 8),
    constrained_layout=True,
)


# Plot the real (I) and imaginary (Q) parts of the raw ADC samples.
# This shows the sampled complex beat signal before range processing.
axes[0, 0].plot(
    adc_sample_index,
    x.real,
    label="I",
    linewidth=1.2,
)
axes[0, 0].plot(
    adc_sample_index,
    x.imag,
    label="Q",
    linewidth=1.2,
)
axes[0, 0].set_title("RADC I and Q Samples")
axes[0, 0].set_xlabel("ADC sample index")
axes[0, 0].set_ylabel("ADC counts")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()


# Plot the magnitude of each complex ADC sample.
# This helps reveal amplitude variation, clipping, dropouts, or other
# irregularities in the raw beat signal.
axes[0, 1].plot(
    adc_sample_index,
    np.abs(x),
    color="tab:purple",
    linewidth=1.2,
)
axes[0, 1].set_title("RADC Sample Magnitude")
axes[0, 1].set_xlabel("ADC sample index")
axes[0, 1].set_ylabel(r"$|I+jQ|$ (ADC counts)")
axes[0, 1].grid(True, alpha=0.3)


# Plot Q versus I to inspect the trajectory of the complex beat signal.
# A rotating complex sinusoid generally traces a circular or elliptical path.
# The first ADC sample is marked to show where the trajectory begins.
axes[1, 0].plot(
    x.real,
    x.imag,
    marker=".",
    markersize=4,
    linewidth=0.8,
)
axes[1, 0].scatter(
    x.real[0],
    x.imag[0],
    color="tab:red",
    s=45,
    label="First sample",
    zorder=3,
)
axes[1, 0].set_title("RADC IQ Plane")
axes[1, 0].set_xlabel("I (ADC counts)")
axes[1, 0].set_ylabel("Q (ADC counts)")
axes[1, 0].axis("equal")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()


# Plot the magnitude of the offline range FFT.
# Peaks indicate beat-frequency components and therefore candidate target
# or clutter ranges under the provisional range-bin mapping.
axes[1, 1].plot(
    range_bin_index,
    X_magnitude_db,
    color="tab:green",
    linewidth=1.2,
)

# Highlight bins 17–19, where the sphere is initially expected based on
# the measured slant range and the approximate vendor range-bin spacing.
axes[1, 1].axvspan(
    17,
    19,
    color="tab:orange",
    alpha=0.2,
    label="Expected sphere region",
)
axes[1, 1].set_title(
    "Python-Computed RADC Range FFT — Rectangular Window"
)
axes[1, 1].set_xlabel("FFT bin index")
axes[1, 1].set_ylabel("Normalized magnitude (dB)")
axes[1, 1].set_xlim(0, x.size - 1)
axes[1, 1].set_ylim(-80, 3)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()


# Add a figure title identifying the exact frame, chirp, and channel used.
fig.suptitle(
    f"Run A: Frame {frame_index}, Chirp {chirp_index}, "
    f"RX Channel {channel_index}",
    fontsize=14,
)

plt.show()


## 2.2 — Form Chirp-Averaged 1D Range Profiles

This cell computes a Python-based range FFT for all 64 chirps and all four RX channels in one Run A frame.

The FFT power is averaged across chirps to produce a more stable range profile for each RX channel. The four channel powers are then averaged noncoherently to form a combined four-RX profile without requiring phase calibration between the antennas.

The plots are used to examine:

- whether the dominant range peaks are consistent among the four RX channels;
- whether averaging across chirps clarifies the candidate target response;
- whether a strong response appears in or near the expected sphere region;
- whether strong close-range coupling or clutter is present.

The physical range values remain provisional because the Python FFT convention and V-MD3 range mapping have not yet been validated.

In [ ]:
# Select one Run A frame for the initial averaged range profile.
frame_index = 0

# Extract the complete RADC cube for this frame.
# Shape:
#   (fast-time ADC sample, chirp, RX channel)
radc_frame = run_a_radc[frame_index]


# Compute a Python-based range FFT along the fast-time sample axis.
# This produces one complex range spectrum for every chirp and RX channel.
#
# No windowing or fast-time mean subtraction is applied yet so that this
# remains consistent with the first diagnostic FFT.
radc_range_fft = np.fft.fft(
    radc_frame,
    axis=0,
)


# Convert the complex FFT values to power.
# Power is used because averaging complex FFT values directly could cause
# phase cancellation between chirps or channels.
radc_range_power = np.abs(radc_range_fft) ** 2


# Average the range-bin power across the 64 chirps while preserving
# the four RX channels.
#
# Resulting shape:
#   (range bin, RX channel)
range_power_by_channel = np.mean(
    radc_range_power,
    axis=1,
)


# Form a noncoherent four-channel range profile by averaging power
# across the RX channels.
#
# "Noncoherent" means that channel powers are averaged without combining
# their complex phases.
range_power_four_rx = np.mean(
    range_power_by_channel,
    axis=1,
)


# Convert each RX-channel profile to decibels.
range_power_by_channel_db = 10 * np.log10(
    np.maximum(
        range_power_by_channel,
        np.finfo(float).tiny,
    )
)


# Convert the noncoherent four-RX profile to decibels.
range_power_four_rx_db = 10 * np.log10(
    np.maximum(
        range_power_four_rx,
        np.finfo(float).tiny,
    )
)


# Normalize all profiles to the maximum of the four-RX averaged profile.
# Using one common reference preserves the relative levels of the RX channels.
normalization_db = range_power_four_rx_db.max()

range_power_by_channel_db -= normalization_db
range_power_four_rx_db -= normalization_db


# Create the provisional range-bin and range axes.
# The physical range uses the approximate vendor spacing of 4.6875 cm/bin.
range_bin_index = np.arange(radc_frame.shape[0])
candidate_range_m = range_bin_index * 0.046875


# Plot the chirp-averaged range profile for each RX channel.
fig, ax = plt.subplots(
    figsize=(12, 6),
    constrained_layout=True,
)

for channel_index in range(range_power_by_channel_db.shape[1]):
    ax.plot(
        range_bin_index,
        range_power_by_channel_db[:, channel_index],
        linewidth=1.0,
        alpha=0.45,
        label=f"RX {channel_index}",
    )


# Plot the noncoherent four-RX average using a thicker black line.
ax.plot(
    range_bin_index,
    range_power_four_rx_db,
    color="black",
    linewidth=2.0,
    linestyle='--',
    label="Four-RX noncoherent average",
    alpha=0.75,
)


# Highlight the initially expected sphere range-bin region.
ax.axvspan(
    17,
    19,
    color="tab:orange",
    alpha=0.2,
    label="Expected sphere region",
)

ax.set_title(
    f"Run A Frame {frame_index}: "
    "Python-Computed RADC Range Profiles"
)
ax.set_xlabel("Range-bin index")
ax.set_ylabel("Power relative to four-RX maximum (dB)")
ax.set_xlim(0, radc_frame.shape[0] - 1)
ax.set_ylim(-80, 5)
ax.grid(True, alpha=0.3)
ax.legend(
    loc="upper right",
    ncols=2,
)

plt.show()


# Identify the ten strongest bins in the four-RX averaged profile.
# This summarizes the dominant range responses after averaging over all
# chirps and channels.
strongest_range_bins = np.argsort(
    range_power_four_rx
)[::-1][:10]


# Display the strongest range bins and their provisional physical ranges.
range_peak_table = pd.DataFrame(
    {
        "range_bin": strongest_range_bins,
        "candidate_range_m": candidate_range_m[
            strongest_range_bins
        ],
        "relative_power_db": range_power_four_rx_db[
            strongest_range_bins
        ],
    }
).sort_values(
    "relative_power_db",
    ascending=False,
).reset_index(drop=True)

display(range_peak_table)

## 2.3 — Compare the RADC Range–Chirp Map with the FPGA Range–Doppler Map

This cell displays two different processing stages from the same Run A frame.

Let \(s[i,n]\) denote the complex RADC samples, where \(i\) is fast-time sample index and \(n\) is chirp index. A range FFT over \(i\) produces

$$
s[i,n]
\longrightarrow
S[q,n],
$$

where \(q\) is the discrete range-bin index. The symbol \(k\) is reserved for Harrison’s physical wavenumber notation.

The left plot displays $\lvert S[q,n]\rvert$, which is a range–chirp intensity map. Each column is one 1D range profile. This is not a range–azimuth image or a SAR image because chirp index is not a spatial coordinate.

The V-MD3 FPGA performs an additional Doppler FFT across chirp index:

$$
S[q,n]
\longrightarrow
D[q,p],
$$

where \(p\) is Doppler-bin index. The right plot displays the complex `RFFT` stream as the magnitude $\lvert D[q,p]\rvert$. For 64 centered Doppler bins, approximately zero Doppler occurs at \(p=32\). Stationary reflections should therefore concentrate near the center Doppler bin.

On the left, the red box spans the expected range bins across all chirps. On the right, the red box encloses the expected range bins near zero Doppler, where the stationary sphere response should appear.

The two plots are not expected to look identical because the left data remain in the chirp domain while the right data have been transformed into the Doppler domain.

In [ ]:
from matplotlib.patches import Rectangle


# Select one Run A frame and one RX channel for the comparison.
frame_index = 0
channel_index = 0


# Extract the complete RADC cube for the selected frame.
# Shape:
#   (fast-time ADC sample, chirp, RX channel)
radc_frame = run_a_radc[frame_index]


# Apply a Python-based range FFT along the fast-time sample axis.
# This transforms fast-time sample index i into range-bin index q
# while leaving chirp index n and RX channel unchanged.
#
# Resulting shape:
#   (range bin, chirp, RX channel)
radc_range_fft = np.fft.fft(
    radc_frame,
    axis=0,
)


# Select one RX channel from the Python-computed range data.
# This produces S[q,n], a range profile for every chirp.
#
# Shape:
#   (range bin, chirp)
radc_range_chirp_map = radc_range_fft[
    :,
    :,
    channel_index,
]


# Extract the V-MD3 FPGA RFFT output for the same frame and RX channel.
# The FPGA has already performed:
#   1. the range FFT over fast time;
#   2. the Doppler FFT across chirps.
#
# This produces D[q,p].
#
# Shape:
#   (range bin, Doppler bin)
fpga_range_doppler_map = run_a_rfft[
    frame_index,
    :,
    :,
    channel_index,
]


# Convert the Python-computed range–chirp magnitude to decibels.
radc_range_chirp_map_db = 20 * np.log10(
    np.maximum(
        np.abs(radc_range_chirp_map),
        np.finfo(float).tiny,
    )
)


# Normalize the range–chirp map to its own maximum.
radc_range_chirp_map_db -= radc_range_chirp_map_db.max()


# Convert the V-MD3 FPGA range–Doppler magnitude to decibels.
fpga_range_doppler_map_db = 20 * np.log10(
    np.maximum(
        np.abs(fpga_range_doppler_map),
        np.finfo(float).tiny,
    )
)


# Normalize the FPGA range–Doppler map to its own maximum.
# Independent normalization is used because the Python and FPGA
# processing paths currently have different numerical scaling.
fpga_range_doppler_map_db -= fpga_range_doppler_map_db.max()


# Determine the center Doppler bin.
# For 64 centered Doppler bins, this gives bin 32.
zero_doppler_bin = (
    fpga_range_doppler_map.shape[1] // 2
)


# Define the initially expected sphere range-bin region.
expected_range_bin_min = 17
expected_range_bin_max = 19

expected_range_box_bottom = (
    expected_range_bin_min - 0.5
)
expected_range_box_height = (
    expected_range_bin_max
    - expected_range_bin_min
    + 1
)


# Create the side-by-side diagnostic plots.
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 7),
    sharey=True,
    constrained_layout=True,
)


# Plot the Python-computed range data across all chirps.
# Horizontal bands indicate reflections that remain at a consistent
# range throughout the frame.
radc_image = axes[0].imshow(
    radc_range_chirp_map_db,
    origin="lower",
    aspect="auto",
    extent=[
        -0.5,
        radc_range_chirp_map.shape[1] - 0.5,
        -0.5,
        radc_range_chirp_map.shape[0] - 0.5,
    ],
    vmin=-60,
    vmax=0,
    cmap="viridis",
)

axes[0].set_title(
    "Python-Computed RADC Range–Chirp Map"
)
axes[0].set_xlabel(
    "Chirp index, $n$"
)
axes[0].set_ylabel(
    "Range-bin index, $q$"
)


# Draw a red box around the expected range bins across all chirps.
# A stationary target should remain in approximately the same range
# region throughout the frame.
radc_expected_region_box = Rectangle(
    xy=(
        -0.5,
        expected_range_box_bottom,
    ),
    width=radc_range_chirp_map.shape[1],
    height=expected_range_box_height,
    fill=False,
    edgecolor="red",
    linewidth=2.0,
    label="Expected sphere range: bins 17–19",
)

axes[0].add_patch(
    radc_expected_region_box
)


# Plot the V-MD3 FPGA range–Doppler map.
# Stationary reflections should concentrate near the center Doppler bin.
fpga_image = axes[1].imshow(
    fpga_range_doppler_map_db,
    origin="lower",
    aspect="auto",
    extent=[
        -0.5,
        fpga_range_doppler_map.shape[1] - 0.5,
        -0.5,
        fpga_range_doppler_map.shape[0] - 0.5,
    ],
    vmin=-60,
    vmax=0,
    cmap="viridis",
)

axes[1].set_title(
    "V-MD3 FPGA Range–Doppler Map"
)
axes[1].set_xlabel(
    "Doppler-bin index, $p$"
)


# Mark the center Doppler bin corresponding approximately to zero speed.
axes[1].axvline(
    zero_doppler_bin,
    color="white",
    linestyle="--",
    linewidth=1.2,
    alpha=0.8,
    label=f"Zero Doppler: bin {zero_doppler_bin}",
)


# Draw a red box around the expected range bins near zero Doppler.
# The box includes the center Doppler bin and one neighboring bin
# on either side to allow for spectral spreading.
fpga_expected_region_box = Rectangle(
    xy=(
        zero_doppler_bin - 1.5,
        expected_range_box_bottom,
    ),
    width=3.0,
    height=expected_range_box_height,
    fill=False,
    edgecolor="red",
    linewidth=2.0,
    label="Expected stationary-sphere region",
)

axes[1].add_patch(
    fpga_expected_region_box
)


# Limit the displayed range so that the close-range responses are clear.
for ax in axes:
    ax.set_ylim(-0.5, 50)
    ax.legend(
        loc="upper right",
    )


# Add one shared colorbar because both heatmaps use the same
# normalized decibel display limits.
colorbar = fig.colorbar(
    fpga_image,
    ax=axes,
    shrink=0.88,
    pad=0.02,
)

colorbar.set_label(
    "Normalized magnitude (dB)"
)


# Add a figure title that reflects the two different processing domains.
fig.suptitle(
    f"Run A Frame {frame_index}, RX Channel {channel_index}: "
    "Range–Chirp and FPGA Range–Doppler Data",
    fontsize=14,
)

plt.show()

### 2.3 Results — Interpretation of the V-MD3 RFFT Axes

The RADC-derived plot and the V-MD3 `RFFT` plot represent two different processing domains.

For the RADC path, the range FFT transforms fast-time sample index $i$ into range-bin index $q$:

$$
s[i,n]
\overset{\text{range FFT}}{\longrightarrow}
S[q,n].
$$

The resulting left-hand plot is a range–chirp intensity map. Its horizontal axis remains chirp index $n$, and its horizontal bands show stationary reflections remaining in consistent range bins throughout the frame.

The V-MD3 `RFFT` output does not contain an untransformed chirp axis. The FPGA has already applied a Doppler FFT across chirps:

$$
S[q,n]
\overset{\text{Doppler FFT}}{\longrightarrow}
D[q,p].
$$

The decoded V-MD3 RFFT shape is therefore interpreted as:

```text
(range bin q, Doppler bin p, RX channel)

## 2.4 — Compare Python-Computed and FPGA Range–Doppler Maps

This cell completes the range–Doppler processing of the RADC data and compares the result directly with the V-MD3 FPGA `RFFT` output.

Let $s[i,n]$ denote the complex RADC samples, where $i$ is fast-time sample index and $n$ is chirp index. The Python processing applies two Fourier transforms:

$$
s[i,n]
\overset{\text{range FFT}}{\longrightarrow}
S[q,n]
\overset{\text{Doppler FFT}}{\longrightarrow}
D_{\text{Python}}[q,p],
$$

where $q$ is range-bin index and $p$ is Doppler-bin index. The Doppler spectrum is shifted so that zero Doppler appears at the center bin, $p=32$.

The left plot shows the Python-computed range–Doppler map. The right plot shows the V-MD3 FPGA range–Doppler map from the `RFFT` stream. These maps are now in the same processing domain and can be compared directly by response location and overall structure.

This initial comparison uses rectangular windows and no mean subtraction, clutter removal, or channel combination. The FPGA may use different windowing, scaling, thresholding, or fixed-point processing, so exact magnitude agreement is not yet expected.

The red box encloses the initially expected stationary-sphere region at range bins 17–19 and near-zero Doppler bins 31–33.

In [ ]:
from matplotlib.patches import Rectangle


# Select one Run A frame and one RX channel.
frame_index = 0
channel_index = 0


# Extract the complete RADC cube for the selected frame.
# Shape:
#   (fast-time sample i, chirp n, RX channel)
radc_frame = run_a_radc[
    frame_index
]


# Compute the Python range–Doppler cube using rectangular
# range and Doppler windows.
#
# The shared function performs:
#   1. range windowing across fast time;
#   2. range FFT along fast time;
#   3. Doppler windowing across chirps;
#   4. Doppler FFT across chirps;
#   5. Doppler centering with fftshift.
#
# Output shape:
#   (range bin q, Doppler bin p, RX channel)
python_range_doppler_cube = (
    compute_radc_range_doppler(
        radc_frame=radc_frame,
        range_window="rectangular",
        doppler_window="rectangular",
    )
)


# Select one RX channel from the Python-computed cube.
# Shape:
#   (range bin q, Doppler bin p)
python_range_doppler_map = (
    python_range_doppler_cube[
        :,
        :,
        channel_index,
    ]
)


# Extract the corresponding V-MD3 FPGA range–Doppler map.
# Shape:
#   (range bin q, Doppler bin p)
fpga_range_doppler_map = run_a_rfft[
    frame_index,
    :,
    :,
    channel_index,
]


# Confirm that the Python and FPGA maps have matching dimensions.
assert (
    python_range_doppler_map.shape
    == fpga_range_doppler_map.shape
), (
    "Python and FPGA range–Doppler map shapes do not match: "
    f"{python_range_doppler_map.shape} versus "
    f"{fpga_range_doppler_map.shape}."
)


# Normalize both maps independently to 0 dB using the
# shared normalization function.
python_range_doppler_map_db = (
    normalized_magnitude_db(
        python_range_doppler_map
    )
)

fpga_range_doppler_map_db = (
    normalized_magnitude_db(
        fpga_range_doppler_map
    )
)


# Determine the centered zero-Doppler bin.
zero_doppler_bin = (
    python_range_doppler_map.shape[1] // 2
)

number_of_doppler_bins = (
    python_range_doppler_map.shape[1]
)


# Define the expected stationary-sphere region.
# Range bins 17–19 are the provisional expected range.
# Doppler bins 31–33 include zero Doppler and its neighbors.
expected_range_bin_min = 17
expected_range_bin_max = 19

expected_doppler_bin_min = (
    zero_doppler_bin - 1
)
expected_doppler_bin_max = (
    zero_doppler_bin + 1
)


# Create the side-by-side range–Doppler comparison.
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 7),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)


# Plot the rectangular-window Python range–Doppler map.
python_image = axes[0].imshow(
    python_range_doppler_map_db,
    origin="lower",
    aspect="auto",
    extent=[
        -0.5,
        python_range_doppler_map.shape[1] - 0.5,
        -0.5,
        python_range_doppler_map.shape[0] - 0.5,
    ],
    vmin=-60,
    vmax=0,
    cmap="viridis",
)

axes[0].set_title(
    "Python Range–Doppler Map — Rectangular Windows"
)
axes[0].set_xlabel(
    "Doppler-bin index, $p$"
)
axes[0].set_ylabel(
    "Range-bin index, $q$"
)


# Plot the V-MD3 FPGA range–Doppler map.
fpga_image = axes[1].imshow(
    fpga_range_doppler_map_db,
    origin="lower",
    aspect="auto",
    extent=[
        -0.5,
        fpga_range_doppler_map.shape[1] - 0.5,
        -0.5,
        fpga_range_doppler_map.shape[0] - 0.5,
    ],
    vmin=-60,
    vmax=0,
    cmap="viridis",
)

axes[1].set_title(
    "V-MD3 FPGA Range–Doppler Map"
)
axes[1].set_xlabel(
    "Doppler-bin index, $p$"
)


# Mark the expected sphere region and zero-Doppler bin
# on both maps.
for ax in axes:

    expected_region_box = Rectangle(
        xy=(
            expected_doppler_bin_min - 0.5,
            expected_range_bin_min - 0.5,
        ),
        width=(
            expected_doppler_bin_max
            - expected_doppler_bin_min
            + 1
        ),
        height=(
            expected_range_bin_max
            - expected_range_bin_min
            + 1
        ),
        fill=False,
        edgecolor="red",
        linewidth=2.0,
        label="Expected stationary-sphere region",
    )

    ax.add_patch(
        expected_region_box
    )

    ax.axvline(
        zero_doppler_bin,
        color="white",
        linestyle="--",
        linewidth=1.1,
        alpha=0.8,
        label=f"Zero Doppler: bin {zero_doppler_bin}",
    )

    ax.set_xlim(
        -0.5,
        number_of_doppler_bins - 0.5,
    )
    ax.set_ylim(
        -0.5,
        50,
    )

    ax.legend(
        loc="upper right",
    )


# Add one shared colorbar because both maps use the same
# normalized decibel display limits.
colorbar = fig.colorbar(
    fpga_image,
    ax=axes,
    shrink=0.88,
    pad=0.02,
)

colorbar.set_label(
    "Normalized magnitude (dB)"
)


# Add a title identifying the selected data and processing.
fig.suptitle(
    f"Run A Frame {frame_index}, RX Channel {channel_index}: "
    "Rectangular-Window Python versus FPGA Processing",
    fontsize=14,
)

plt.show()

### 2.4 Results — Rectangular-Window Python and FPGA Comparison

The Python-computed and V-MD3 FPGA range–Doppler maps show strong agreement in their principal response locations.

Both processing paths concentrate the stationary-scene energy at approximately:

$$
p=32,
$$

confirming the centered zero-Doppler convention. They also contain corresponding responses near several of the same range bins, including approximately:

$$
q\approx1,\ 22,\ 27,\ 31,\ \text{and}\ 41.
$$

This agreement supports the following processing and decoding conventions:

- the RADC fast-time axis is decoded correctly;
- the Python range FFT is applied along the correct axis;
- the RFFT payload reshape is substantially correct;
- RFFT axis 0 represents range-bin index;
- RFFT axis 1 represents Doppler-bin index;
- the FPGA Doppler spectrum is centered at bin 32;
- provisional Run A pairing of RADC and RFFT records by stream-frame index is reasonable.

The Python map uses rectangular range and Doppler windows. Its stationary responses are consequently narrow and concentrated primarily in a single Doppler bin. The FPGA responses are broader across neighboring range and Doppler bins and have smoother sidelobe behavior.

This difference suggests that the FPGA applies nonrectangular windowing before one or both FFTs. Additional differences may result from fixed-point arithmetic, numerical scaling, quantization, and zero suppression within the FPGA processing chain.

The strong response near:

$$
(q,p)\approx(22,32)
$$

appears in both maps and lies outside the initially expected sphere region at range bins 17–19. Because both processing paths place the response at the same range bin, the discrepancy is not caused by the Python FFT or the provisional FPGA reshape.

Run A alone does not determine whether the response near bin 22 is the sphere with a fixed range offset or a stronger stationary clutter reflection. Background comparison with the sphere-removed measurements will later be required to identify the sphere conclusively.

Overall, the rectangular-window result validates the basic range–Doppler processing sequence and data organization, but it does not reproduce the FPGA peak widths and sidelobe structure. This motivates comparing Hann, Hamming, and Blackman windows against the FPGA output.

## 2.5 — Compare Range–Doppler Window Functions

This cell illustrates how the selected window changes the Python-computed range–Doppler map.

The same RADC frame and RX channel are processed using three window functions:

- Hann;
- Hamming;
- Blackman.

For each case, the selected window is applied across fast time before the range FFT and across chirps before the Doppler FFT. 

The windowed data are $s_w[i,n]=s[i,n]w_r[i]w_D[n]$.

The two FFTs then produce the range–Doppler map. 

The processing sequence is $s_w[i,n]\rightarrow S_w[q,n]\rightarrow D_w[q,p]$, where the first transform is the range FFT and the second is the Doppler FFT.

The V-MD3 FPGA `RFFT` output is included as a reference. All four maps are normalized independently to 0 dB and displayed using the same decibel limits.

The comparison demonstrates the windowing tradeoff:

- weaker tapering produces narrower main lobes but higher sidelobes;
- stronger tapering reduces sidelobes but broadens the main lobes;
- the best match to the FPGA is the window whose peak widths, sidelobes, and overall response structure most closely resemble the FPGA output.

A normalized-magnitude correlation is also calculated over the displayed range region. This metric provides an initial quantitative comparison, although fixed-point scaling, thresholding, and other undisclosed FPGA processing can prevent an exact match.

In [ ]:
from matplotlib.patches import Rectangle


# Select one Run A frame and one RX channel.
# Every window processes exactly the same input data.
frame_index = 0
channel_index = 0

radc_frame = run_a_radc[
    frame_index
]

number_of_chirps = radc_frame.shape[1]


# Define the window combinations to compare.
# The same window is used for both the range and Doppler FFTs.
window_configurations = {
    "Hann": "hann",
    "Hamming": "hamming",
    "Blackman": "blackman",
}


# Process the RADC frame using each candidate window.
#
# The shared function returns the complete cube with shape:
#   (range bin q, Doppler bin p, RX channel)
python_windowed_cubes = {}

for display_name, window_name in window_configurations.items():

    python_windowed_cubes[display_name] = (
        compute_radc_range_doppler(
            radc_frame=radc_frame,
            range_window=window_name,
            doppler_window=window_name,
        )
    )


# Select one RX channel from each Python-computed cube.
# Each resulting map has shape:
#   (range bin q, Doppler bin p)
python_windowed_maps = {
    display_name: range_doppler_cube[
        :,
        :,
        channel_index,
    ]
    for display_name, range_doppler_cube
    in python_windowed_cubes.items()
}


# Extract the V-MD3 FPGA range–Doppler map for the same
# frame and RX channel.
fpga_range_doppler_map = run_a_rfft[
    frame_index,
    :,
    :,
    channel_index,
]


# Normalize the FPGA reference map using the shared function.
fpga_map_db = normalized_magnitude_db(
    fpga_range_doppler_map
)


# Normalize each Python-computed map.
python_windowed_maps_db = {
    display_name: normalized_magnitude_db(
        windowed_map
    )
    for display_name, windowed_map
    in python_windowed_maps.items()
}


# Arrange the maps in the desired plotting order.
comparison_maps = {
    "V-MD3 FPGA Reference": fpga_map_db,
    "Python — Hann": python_windowed_maps_db["Hann"],
    "Python — Hamming": python_windowed_maps_db["Hamming"],
    "Python — Blackman": python_windowed_maps_db["Blackman"],
}


# Determine the centered zero-Doppler bin.
zero_doppler_bin = (
    fpga_range_doppler_map.shape[1] // 2
)


# Define the provisional expected stationary-sphere region.
expected_range_bin_min = 17
expected_range_bin_max = 19

expected_doppler_bin_min = zero_doppler_bin - 1
expected_doppler_bin_max = zero_doppler_bin + 1


# Create the four-panel range–Doppler comparison.
fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 10),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

axes_flat = axes.ravel()


# Plot the FPGA reference and the three Python results
# with identical axes and color limits.
for plot_index, (
    ax,
    (map_title, map_db),
) in enumerate(
    zip(
        axes_flat,
        comparison_maps.items(),
    )
):
    image = ax.imshow(
        map_db,
        origin="lower",
        aspect="auto",
        extent=[
            -0.5,
            map_db.shape[1] - 0.5,
            -0.5,
            map_db.shape[0] - 0.5,
        ],
        vmin=-60,
        vmax=0,
        cmap="viridis",
    )

    ax.set_title(
        map_title
    )

    # Mark the centered zero-Doppler bin.
    ax.axvline(
        zero_doppler_bin,
        color="white",
        linestyle="--",
        linewidth=1.1,
        alpha=0.8,
        label=f"Zero Doppler: bin {zero_doppler_bin}",
    )

    # Draw a red box around the expected sphere region.
    expected_region_box = Rectangle(
        xy=(
            expected_doppler_bin_min - 0.5,
            expected_range_bin_min - 0.5,
        ),
        width=(
            expected_doppler_bin_max
            - expected_doppler_bin_min
            + 1
        ),
        height=(
            expected_range_bin_max
            - expected_range_bin_min
            + 1
        ),
        fill=False,
        edgecolor="red",
        linewidth=2.0,
        label="Expected stationary-sphere region",
    )

    ax.add_patch(
        expected_region_box
    )

    # Display the close-range region relevant to Run A.
    ax.set_xlim(
        -0.5,
        number_of_chirps - 0.5,
    )
    ax.set_ylim(
        -0.5,
        50,
    )

    # Show the legend only once to reduce visual clutter.
    if plot_index == 0:
        ax.legend(
            loc="upper right",
        )


# Label the shared plot axes.
for ax in axes[1, :]:
    ax.set_xlabel(
        "Doppler-bin index, $p$"
    )

for ax in axes[:, 0]:
    ax.set_ylabel(
        "Range-bin index, $q$"
    )


# Add one shared colorbar because all four maps use the
# same normalized decibel limits.
colorbar = fig.colorbar(
    image,
    ax=axes,
    shrink=0.90,
    pad=0.02,
)

colorbar.set_label(
    "Normalized magnitude (dB)"
)


# Add a figure title identifying the selected data.
fig.suptitle(
    f"Run A Frame {frame_index}, RX Channel {channel_index}: "
    "Range–Doppler Window Comparison",
    fontsize=15,
)

plt.show()


# Compare the Python and FPGA maps numerically over the displayed
# range region. Convert the normalized dB maps back to normalized
# linear magnitude before calculating correlation.
comparison_range_slice = slice(
    0,
    51,
)

fpga_normalized_magnitude = (
    10 ** (
        fpga_map_db / 20
    )
)

fpga_comparison_values = (
    fpga_normalized_magnitude[
        comparison_range_slice,
        :,
    ].ravel()
)

window_comparison_results = []


for window_name, map_db in python_windowed_maps_db.items():

    python_normalized_magnitude = (
        10 ** (
            map_db / 20
        )
    )

    python_comparison_values = (
        python_normalized_magnitude[
            comparison_range_slice,
            :,
        ].ravel()
    )

    # Calculate the Pearson correlation between the normalized
    # Python and FPGA magnitudes.
    magnitude_correlation = np.corrcoef(
        fpga_comparison_values,
        python_comparison_values,
    )[0, 1]

    window_comparison_results.append(
        {
            "window": window_name,
            "normalized_magnitude_correlation": (
                magnitude_correlation
            ),
        }
    )


# Display the windows from highest to lowest FPGA correlation.
window_comparison_table = pd.DataFrame(
    window_comparison_results
).sort_values(
    "normalized_magnitude_correlation",
    ascending=False,
).reset_index(
    drop=True
)

display(
    window_comparison_table
)

### 2.5 Results — Window-Function Comparison

The Hann, Hamming, and Blackman windows all preserve the principal response locations observed in the V-MD3 FPGA range–Doppler map. The dominant responses remain centered near zero Doppler at:

$$
p=32,
$$

and the same major range responses remain visible near approximately:

$$
q\approx1,\ 22,\ 27,\ 31,\ \text{and}\ 41.
$$

This demonstrates that windowing changes the widths and sidelobes of the responses but does not substantially change their underlying range- or Doppler-bin locations.

The rectangular-window result from the preceding section produced responses that were narrower than those in the FPGA map. Applying a tapered window broadens the main lobes and reduces the sidelobes, producing maps that more closely resemble the FPGA output.

Of the tested windows, the Hann result provides the closest visual match to the FPGA reference. In particular:

- the response widths around zero Doppler are similar;
- the close-range response near $q\approx1$ has a comparable two-dimensional shape;
- the responses near $q\approx22$, $q\approx27$, and the higher range bins have similar spreading;
- low-level sidelobe energy is reduced relative to the rectangular-window result.

The Hamming result is also similar to the FPGA map. Its differences from Hann are comparatively subtle at the displayed 60 dB dynamic range. Hamming provides greater suppression of the first sidelobe, but its response shape does not appear to match the FPGA as closely as Hann in this frame.

The Blackman window produces the broadest responses in both dimensions. This is particularly visible around the close-range response and the group of responses between approximately $q=30$ and $q=45$. Although Blackman provides stronger sidelobe suppression, its main-lobe broadening exceeds that observed in the FPGA output.

The visual comparison therefore supports using Hann windows for the Python range–Doppler processing:

$$
w_r[i]=w_{\mathrm{Hann}}[i]
$$

across fast time and

$$
w_D[n]=w_{\mathrm{Hann}}[n]
$$

across chirp index.

The selected processing sequence is consequently:

$$
s[i,n]w_r[i]
\overset{\text{range FFT}}{\longrightarrow}
S_w[q,n],
$$

followed by:

$$
S_w[q,n]w_D[n]
\overset{\text{Doppler FFT}}{\longrightarrow}
D_w[q,p].
$$

The strong stationary response near $(q,p)\approx(22,32)$ remains outside the initially expected sphere region at range bins 17–19. Because its position is unchanged across all tested windows and agrees with the FPGA output, window selection does not explain the range discrepancy. Determining whether this response is the sphere or stationary clutter will require comparison with the sphere-removed background measurements.

Based on the visual agreement, Hann is adopted as the default range and Doppler window for subsequent Python processing. This choice provides a practical match to the FPGA response while maintaining a reasonable balance between main-lobe width and sidelobe suppression.

## 2.6 — Form Preliminary Range–Azimuth Maps

This cell uses the complex data from the four RX antennas to estimate the azimuth direction of reflections at each range bin.

The Python path uses Hann-windowed range–Doppler processing of the RADC data. The FPGA path uses the V-MD3 `RFFT` range–Doppler output. Because the Run A scene is stationary, the zero-Doppler slice at $p=32$ is extracted from each processing path.

For every range bin $q$, the four complex RX values are coherently combined over a trial azimuth grid. Conventional beamforming reinforces signals whose phase progression across the antenna array matches the steering vector for a particular angle.

The resulting maps display:

- range-bin index $q$ on the vertical axis;
- estimated azimuth angle on the horizontal axis;
- normalized coherent beamformer magnitude as color.

This is a preliminary, uncalibrated range–azimuth map. The calculation assumes that the four RX channels form a uniformly spaced linear array with 2.464 mm element spacing. The angle sign and absolute angle accuracy remain provisional until the RX-channel order and phase calibration are independently verified.

The red box marks the initially expected sphere region near range bins 17–19 and boresight. The stronger response near range bin 22 remains visible for comparison.

In [ ]:
# Select one Run A frame.
# All four RX channels are retained for coherent beamforming.
frame_index = 0


# Define the approximate V-MD3 carrier frequency and RX spacing.
carrier_frequency_hz = 61.0e9
rx_element_spacing_m = 2.464e-3


# Extract the complete RADC cube for the selected frame.
# Shape:
#   (fast-time sample i, chirp n, RX channel m)
radc_frame = run_a_radc[
    frame_index
]

number_of_rx_channels = radc_frame.shape[2]


# Compute the Hann-windowed Python range–Doppler cube.
# Output shape:
#   (range bin q, Doppler bin p, RX channel m)
python_range_doppler_cube = (
    compute_radc_range_doppler(
        radc_frame=radc_frame,
        range_window="hann",
        doppler_window="hann",
    )
)


# Extract the centered zero-Doppler slice from the
# Python-computed cube.
#
# Output shape:
#   (range bin q, RX channel m)
python_zero_doppler_rx, zero_doppler_bin = (
    extract_zero_doppler(
        python_range_doppler_cube,
        doppler_axis=1,
    )
)


# Extract the centered zero-Doppler slice from the
# V-MD3 FPGA RFFT cube.
fpga_zero_doppler_rx, fpga_zero_doppler_bin = (
    extract_zero_doppler(
        run_a_rfft[
            frame_index
        ],
        doppler_axis=1,
    )
)


# Confirm that both processing paths use the same
# centered zero-Doppler bin.
assert (
    zero_doppler_bin
    == fpga_zero_doppler_bin
), (
    "Python and FPGA zero-Doppler bins do not match: "
    f"{zero_doppler_bin} versus "
    f"{fpga_zero_doppler_bin}."
)


# Define the assumed positions of the four RX antennas.
# Centering the positions around zero makes the array geometry
# symmetric about its midpoint.
rx_positions_m = (
    np.arange(number_of_rx_channels)
    - (number_of_rx_channels - 1) / 2
) * rx_element_spacing_m


# Define the trial azimuth-angle grid.
# The sign remains provisional until the physical RX-channel
# ordering is independently confirmed.
azimuth_angle_deg = np.linspace(
    -70.0,
    70.0,
    281,
)


# Apply conventional beamforming to the Python zero-Doppler data.
#
# Output shape:
#   (range bin q, trial azimuth angle)
python_range_azimuth_map = (
    conventional_range_azimuth(
        range_rx_data=python_zero_doppler_rx,
        azimuth_angle_deg=azimuth_angle_deg,
        rx_positions_m=rx_positions_m,
        carrier_frequency_hz=carrier_frequency_hz,
    )
)


# Apply the same conventional beamforming operation to the
# V-MD3 FPGA zero-Doppler data.
fpga_range_azimuth_map = (
    conventional_range_azimuth(
        range_rx_data=fpga_zero_doppler_rx,
        azimuth_angle_deg=azimuth_angle_deg,
        rx_positions_m=rx_positions_m,
        carrier_frequency_hz=carrier_frequency_hz,
    )
)


# Normalize the Python and FPGA maps independently to 0 dB.
python_range_azimuth_map_db = (
    normalized_magnitude_db(
        python_range_azimuth_map
    )
)

fpga_range_azimuth_map_db = (
    normalized_magnitude_db(
        fpga_range_azimuth_map
    )
)


# Confirm that the two beamformed maps have matching dimensions.
assert (
    python_range_azimuth_map.shape
    == fpga_range_azimuth_map.shape
), (
    "Python and FPGA range–azimuth map shapes do not match: "
    f"{python_range_azimuth_map.shape} versus "
    f"{fpga_range_azimuth_map.shape}."
)


# Create the side-by-side range–azimuth comparison.
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 7),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)


# Plot the Hann-windowed Python range–azimuth map.
python_image = axes[0].imshow(
    python_range_azimuth_map_db,
    origin="lower",
    aspect="auto",
    extent=[
        azimuth_angle_deg[0],
        azimuth_angle_deg[-1],
        -0.5,
        python_range_azimuth_map.shape[0] - 0.5,
    ],
    vmin=-40,
    vmax=0,
    cmap="viridis",
)

axes[0].set_title(
    "Python RADC-Derived Range–Azimuth Map"
)
axes[0].set_xlabel(
    "Azimuth angle (degrees, provisional sign)"
)
axes[0].set_ylabel(
    "Range-bin index, $q$"
)


# Plot the FPGA-derived range–azimuth map.
fpga_image = axes[1].imshow(
    fpga_range_azimuth_map_db,
    origin="lower",
    aspect="auto",
    extent=[
        azimuth_angle_deg[0],
        azimuth_angle_deg[-1],
        -0.5,
        fpga_range_azimuth_map.shape[0] - 0.5,
    ],
    vmin=-40,
    vmax=0,
    cmap="viridis",
)

axes[1].set_title(
    "V-MD3 FPGA RFFT-Derived Range–Azimuth Map"
)
axes[1].set_xlabel(
    "Azimuth angle (degrees, provisional sign)"
)


# Mark the expected sphere region near boresight.
# The angular interval is wider than a point target because
# the sphere is extended and the four-element array has
# limited angular resolution.
for ax in axes:

    expected_sphere_box = Rectangle(
        xy=(-10.0, 16.5),
        width=20.0,
        height=3.0,
        fill=False,
        edgecolor="red",
        linewidth=2.0,
        label="Expected sphere region",
    )

    ax.add_patch(
        expected_sphere_box
    )

    # Mark the assumed boresight direction.
    ax.axvline(
        0.0,
        color="white",
        linestyle="--",
        linewidth=1.1,
        alpha=0.8,
        label="Boresight",
    )

    ax.set_xlim(
        -70,
        70,
    )
    ax.set_ylim(
        -0.5,
        50,
    )

    ax.legend(
        loc="upper right",
    )


# Add a shared normalized-magnitude colorbar.
colorbar = fig.colorbar(
    fpga_image,
    ax=axes,
    shrink=0.88,
    pad=0.02,
)

colorbar.set_label(
    "Normalized coherent magnitude (dB)"
)


# Add a figure title identifying the selected frame.
fig.suptitle(
    f"Run A Frame {frame_index}: "
    "Preliminary Four-RX Range–Azimuth Comparison",
    fontsize=14,
)

plt.show()

### 2.6 Results — Preliminary Range–Azimuth Comparison

The Python RADC-derived and V-MD3 FPGA-derived range–azimuth maps show strong visual agreement. The principal range responses, angular main lobes, nulls, and sidelobe patterns occur at approximately the same locations in both maps.

This agreement supports several conclusions:

- the Hann-windowed Python range–Doppler processing reproduces the FPGA output well;
- the V-MD3 RFFT range and Doppler axes are being interpreted correctly;
- the four complex RX channels are decoded in a mutually coherent form;
- the same conventional beamforming operation can be applied to both processing paths;
- the provisional RFFT channel ordering is consistent with the RADC channel ordering.

The strongest close-range response near $q\approx1$ is likely dominated by direct coupling or other near-field effects. Far-field plane-wave steering is not expected to produce a physically reliable angle estimate for this response.

Several stationary reflections between approximately $q=20$ and $q=31$ produce broad angular responses. The dominant response near $q\approx22$ focuses relatively close to boresight, although its apparent center is slightly offset from $0^\circ$.

The broad main lobes and repeated responses at larger positive and negative angles are expected consequences of conventional beamforming with only four uniformly weighted RX elements. These features should not automatically be interpreted as separate physical targets.

The initially expected sphere region at range bins 17–19 does not contain the dominant response. The stronger response near $q\approx22$ appears in both the Python and FPGA maps, confirming that its location is not caused by differences between the two processing paths. Background comparison will be required to determine whether this response is the sphere with a fixed range bias or a stationary clutter reflector.

This result is a range–azimuth map formed at one fixed radar position. It is not yet a SAR image. SAR processing will later use coherent measurements from the 17 mechanical aperture positions to improve cross-range resolution.

The absolute azimuth calibration and the sign of the angle axis remain provisional. A known off-boresight target or documented physical RX-channel ordering will be needed to verify them.

## 2.7 — Frame-to-Frame Complex Stability

This cell evaluates whether the complex range data remain stable across the five Run A frames.

Phase stability is required for coherent averaging and SAR processing. If the phase changes unpredictably between frames, complex measurements from different frames or aperture positions can cancel when combined.

The analysis compares two candidate range bins:

- range bin 18, inside the geometrically expected sphere region;
- range bin 22, containing the stronger observed stationary response.

For each frame, Hann-windowed range and Doppler FFTs are applied to the RADC data. The complex value at zero Doppler, $p=32$, is then extracted for RX channel 0. The corresponding complex value is also extracted directly from the V-MD3 FPGA `RFFT` output.

The plots show:

- magnitude variation relative to the mean magnitude;
- phase change relative to the first frame.

The results table also reports a coherence factor. A coherence factor near 1 indicates that the complex samples have nearly constant phase and can be combined coherently. A lower value indicates phase variation or insufficient signal-to-noise ratio.

In [ ]:
# Define the candidate range bins to test.
# Bin 18 is inside the geometrically expected sphere region.
# Bin 22 contains the stronger observed stationary response.
range_bins_to_test = [
    18,
    22,
]

channel_index = 0

number_of_frames = run_a_radc.shape[0]

frame_indices = np.arange(
    number_of_frames
)


# Process every Run A RADC frame using the shared
# Hann-windowed range–Doppler function.
#
# Each individual cube has shape:
#   (range bin q, Doppler bin p, RX channel)
#
# The stacked result has shape:
#   (frame, range bin q, Doppler bin p, RX channel)
python_range_doppler_stack = np.stack(
    [
        compute_radc_range_doppler(
            radc_frame=run_a_radc[frame_index],
            range_window="hann",
            doppler_window="hann",
        )
        for frame_index in frame_indices
    ],
    axis=0,
)


# Extract the centered zero-Doppler data from all
# Python-computed frames.
#
# Output shape:
#   (frame, range bin q, RX channel)
python_zero_doppler_stack, zero_doppler_bin = (
    extract_zero_doppler(
        python_range_doppler_stack,
        doppler_axis=2,
    )
)


# Extract the centered zero-Doppler data from all
# V-MD3 FPGA RFFT frames.
fpga_zero_doppler_stack, fpga_zero_doppler_bin = (
    extract_zero_doppler(
        run_a_rfft,
        doppler_axis=2,
    )
)


# Confirm that the Python and FPGA data use the same
# centered zero-Doppler bin.
assert (
    zero_doppler_bin
    == fpga_zero_doppler_bin
), (
    "Python and FPGA zero-Doppler bins do not match: "
    f"{zero_doppler_bin} versus "
    f"{fpga_zero_doppler_bin}."
)


# Convert a complex sequence into magnitude variation
# relative to its mean magnitude.
def relative_magnitude_db(
    complex_sequence,
):
    """
    Return magnitude in dB relative to the sequence's mean.
    """

    magnitude = np.abs(
        complex_sequence
    )

    mean_magnitude = np.mean(
        magnitude
    )

    return 20 * np.log10(
        np.maximum(
            magnitude,
            np.finfo(float).tiny,
        )
        / np.maximum(
            mean_magnitude,
            np.finfo(float).tiny,
        )
    )


# Convert a complex sequence into unwrapped phase
# relative to its first sample.
def relative_phase_deg(
    complex_sequence,
):
    """
    Return unwrapped phase in degrees relative to frame 0.
    """

    phase_rad = np.unwrap(
        np.angle(
            complex_sequence
        )
    )

    phase_rad -= phase_rad[0]

    return np.rad2deg(
        phase_rad
    )


# Create magnitude and phase stability plots for both
# processing paths.
fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 9),
    sharex=True,
    constrained_layout=True,
)


# Plot the Python-computed magnitude stability.
for range_bin in range_bins_to_test:

    complex_sequence = (
        python_zero_doppler_stack[
            :,
            range_bin,
            channel_index,
        ]
    )

    axes[0, 0].plot(
        frame_indices,
        relative_magnitude_db(
            complex_sequence
        ),
        marker="o",
        linewidth=1.5,
        label=f"Range bin {range_bin}",
    )

axes[0, 0].set_title(
    "Python RADC-Derived Magnitude Stability"
)
axes[0, 0].set_ylabel(
    "Magnitude relative to mean (dB)"
)
axes[0, 0].grid(
    True,
    alpha=0.3,
)
axes[0, 0].legend()


# Plot the Python-computed phase stability.
for range_bin in range_bins_to_test:

    complex_sequence = (
        python_zero_doppler_stack[
            :,
            range_bin,
            channel_index,
        ]
    )

    axes[0, 1].plot(
        frame_indices,
        relative_phase_deg(
            complex_sequence
        ),
        marker="o",
        linewidth=1.5,
        label=f"Range bin {range_bin}",
    )

axes[0, 1].set_title(
    "Python RADC-Derived Phase Stability"
)
axes[0, 1].set_ylabel(
    "Phase relative to frame 0 (degrees)"
)
axes[0, 1].grid(
    True,
    alpha=0.3,
)
axes[0, 1].legend()


# Plot the FPGA magnitude stability.
for range_bin in range_bins_to_test:

    complex_sequence = (
        fpga_zero_doppler_stack[
            :,
            range_bin,
            channel_index,
        ]
    )

    axes[1, 0].plot(
        frame_indices,
        relative_magnitude_db(
            complex_sequence
        ),
        marker="o",
        linewidth=1.5,
        label=f"Range bin {range_bin}",
    )

axes[1, 0].set_title(
    "V-MD3 FPGA Magnitude Stability"
)
axes[1, 0].set_xlabel(
    "Run A frame index"
)
axes[1, 0].set_ylabel(
    "Magnitude relative to mean (dB)"
)
axes[1, 0].grid(
    True,
    alpha=0.3,
)
axes[1, 0].legend()


# Plot the FPGA phase stability.
for range_bin in range_bins_to_test:

    complex_sequence = (
        fpga_zero_doppler_stack[
            :,
            range_bin,
            channel_index,
        ]
    )

    axes[1, 1].plot(
        frame_indices,
        relative_phase_deg(
            complex_sequence
        ),
        marker="o",
        linewidth=1.5,
        label=f"Range bin {range_bin}",
    )

axes[1, 1].set_title(
    "V-MD3 FPGA Phase Stability"
)
axes[1, 1].set_xlabel(
    "Run A frame index"
)
axes[1, 1].set_ylabel(
    "Phase relative to frame 0 (degrees)"
)
axes[1, 1].grid(
    True,
    alpha=0.3,
)
axes[1, 1].legend()


# Add a title identifying the selected Doppler bin and RX channel.
fig.suptitle(
    f"Run A Frame-to-Frame Stability: "
    f"Zero Doppler Bin {zero_doppler_bin}, "
    f"RX Channel {channel_index}",
    fontsize=14,
)

plt.show()


# Calculate numerical magnitude and phase stability metrics.
stability_results = []

processing_paths = {
    "Python RADC-derived": python_zero_doppler_stack,
    "V-MD3 FPGA": fpga_zero_doppler_stack,
}


for processing_name, zero_doppler_stack in processing_paths.items():

    for range_bin in range_bins_to_test:

        complex_sequence = zero_doppler_stack[
            :,
            range_bin,
            channel_index,
        ]

        magnitude_variation_db = (
            relative_magnitude_db(
                complex_sequence
            )
        )

        # Convert the complex samples to unit phasors so the
        # phase analysis is not weighted by magnitude.
        unit_phasors = (
            complex_sequence
            / np.maximum(
                np.abs(
                    complex_sequence
                ),
                np.finfo(float).tiny,
            )
        )

        # Calculate the mean circular phase direction.
        mean_phase_rad = np.angle(
            np.sum(
                unit_phasors
            )
        )

        # Calculate each frame's phase deviation from the
        # mean phase direction.
        phase_deviation_deg = np.rad2deg(
            np.angle(
                unit_phasors
                * np.exp(
                    -1j * mean_phase_rad
                )
            )
        )

        # Calculate the coherent-sum efficiency.
        # A value of 1 indicates perfectly aligned phases.
        coherence_factor = (
            np.abs(
                np.sum(
                    complex_sequence
                )
            )
            / np.maximum(
                np.sum(
                    np.abs(
                        complex_sequence
                    )
                ),
                np.finfo(float).tiny,
            )
        )

        stability_results.append(
            {
                "processing_path": processing_name,
                "range_bin": range_bin,
                "magnitude_std_db": np.std(
                    magnitude_variation_db,
                    ddof=1,
                ),
                "phase_std_deg": np.std(
                    phase_deviation_deg,
                    ddof=1,
                ),
                "coherence_factor": coherence_factor,
            }
        )


# Display the numerical stability metrics.
stability_results_table = pd.DataFrame(
    stability_results
)

display(
    stability_results_table
)

### Results - Frame-to-Frame Complex Stability

The Run A measurements exhibit strong frame-to-frame complex stability at both tested range bins.

Range bin 22, which contains the stronger stationary response, is exceptionally stable. Its magnitude changes by only approximately $0.01$ dB across the five frames, while its phase remains within approximately $0.3^\circ$ of the first frame. The Python RADC-derived and V-MD3 FPGA results show nearly identical behavior.

Range bin 18 is also reasonably stable, but it exhibits greater variation than range bin 22. The largest change occurs at frame index 2, where the Python result changes by approximately $-0.4$ dB and $-5^\circ$. The FPGA output contains a corresponding change of approximately $-0.2$ dB and $-3.5^\circ$.

Because the same frame-2 variation appears in both processing paths, it is likely present in the measured radar data rather than being introduced by the Python FFT processing.

The greater stability of range bin 22 is consistent with its stronger signal magnitude. Range bin 18 lies in a weaker portion of the range spectrum and is consequently more sensitive to noise and small changes in the received signal.

The results demonstrate that the V-MD3 maintains strong complex coherence across the five fixed-position Run A frames. Range bin 22 is the more reliable phase reference for evaluating coherent processing.

This result supports coherent averaging of the five Run A frames. Coherent averaging should reinforce stationary reflections while reducing uncorrelated noise without destroying their phase information.

The test confirms coherence across repeated frames at one fixed radar position. It does not by itself confirm phase coherence across the 17 mechanical aperture positions. Position-to-position phase consistency must be evaluated separately before SAR reconstruction.

## 2.8 — Matched Complex Background Subtraction

### 2.8.1 — Load the Matched Run B Background

This cell loads the Run B measurements acquired at position index 8, which is the same physical antenna position used for Run A.

Run B contains no sphere and therefore provides a matched measurement of the stationary background and system coupling. At position index 8, the file contains 20 RADC frames, 20 FPGA RFFT frames, and 20 DONE records.

The RADC and RFFT streams are loaded and processed independently. For Runs B–D, equal stream-frame indices are not assumed to represent exact hardware pairing because the UDP records were saved in arrival order.

The immediate purpose of this cell is to confirm that the matched background data have the expected dimensions and valid complex values before evaluating their phase stability and performing background subtraction.

In [ ]:
# Identify the matched no-target background measurement.
background_run = "B"
background_position_index = 8
expected_background_frames = 20


# Load all RADC frames from Run B position index 8.
# Expected shape:
#   (frame, fast-time sample, chirp, RX channel)
run_b_center_radc = load_radc_stack(
    RECORD_MANIFEST,
    run=background_run,
    position_index=background_position_index,
)


# Load all FPGA RFFT frames from the same position.
# Expected shape:
#   (frame, range bin, Doppler bin, RX channel)
run_b_center_rfft = load_rfft_stack(
    RECORD_MANIFEST,
    run=background_run,
    position_index=background_position_index,
)


# Load the corresponding radar-side DONE counters.
run_b_center_done = load_done_values(
    RECORD_MANIFEST,
    run=background_run,
    position_index=background_position_index,
)


# Define the expected data-cube dimensions.
expected_radc_shape = (
    expected_background_frames,
    128,
    64,
    4,
)

expected_rfft_shape = (
    expected_background_frames,
    128,
    64,
    4,
)


# Confirm that the loaded stacks have the expected dimensions.
assert (
    run_b_center_radc.shape
    == expected_radc_shape
), (
    "Unexpected Run B center RADC shape: "
    f"{run_b_center_radc.shape}"
)

assert (
    run_b_center_rfft.shape
    == expected_rfft_shape
), (
    "Unexpected Run B center RFFT shape: "
    f"{run_b_center_rfft.shape}"
)

assert (
    run_b_center_done.shape[0]
    == expected_background_frames
), (
    "Unexpected number of Run B center DONE records: "
    f"{run_b_center_done.shape[0]}"
)


# Confirm that both radar data stacks contain complex values.
assert np.iscomplexobj(
    run_b_center_radc
), "Run B center RADC data are not complex."

assert np.iscomplexobj(
    run_b_center_rfft
), "Run B center RFFT data are not complex."


# Confirm that all complex values are finite.
assert np.all(
    np.isfinite(
        run_b_center_radc
    )
), "Run B center RADC contains nonfinite values."

assert np.all(
    np.isfinite(
        run_b_center_rfft
    )
), "Run B center RFFT contains nonfinite values."


# Confirm continuity of the radar-side DONE counter.
assert np.all(
    np.diff(
        run_b_center_done
    )
    == 1
), (
    "Run B center DONE values are not consecutive: "
    f"{run_b_center_done}"
)


# Calculate exact-zero fractions as an additional diagnostic.
radc_zero_fraction = np.mean(
    run_b_center_radc == 0
)

rfft_zero_fraction = np.mean(
    run_b_center_rfft == 0
)


# Print a compact validation summary.
print("Run B matched-background validation")
print(
    f"Position index:          "
    f"{background_position_index}"
)
print(
    f"RADC shape:              "
    f"{run_b_center_radc.shape}"
)
print(
    f"FPGA RFFT shape:         "
    f"{run_b_center_rfft.shape}"
)
print(
    f"DONE values:             "
    f"{run_b_center_done.tolist()}"
)
print(
    f"RADC exact-zero fraction:"
    f" {radc_zero_fraction:.6f}"
)
print(
    f"RFFT exact-zero fraction:"
    f" {rfft_zero_fraction:.6f}"
)

### 2.8.2 — Verify Background Coherence and Cross-Run Alignment

This cell evaluates whether the 20 matched Run B background frames can be coherently averaged and whether the Run A and Run B measurements share a consistent complex phase reference.

The Run B RADC frames are processed using the same Hann-windowed range–Doppler functions used for Run A. The zero-Doppler complex data are extracted for all four RX channels.

For each range bin and RX channel, a coherence factor is calculated across the 20 background frames. A value near 1 indicates that the background measurements can be coherently averaged.

The coherently averaged Run A and Run B data are then compared over reference range bins that exclude the possible sphere region. For each RX channel, a complex scale factor is estimated that maps the Run B background to the Run A measurement.

A nearly constant magnitude and phase offset indicates that complex background subtraction is appropriate. If the phase relationship varies strongly across range, direct subtraction may require additional registration or calibration.

In [ ]:
# Process all 20 matched Run B RADC frames using the
# shared Hann-windowed range–Doppler function.
#
# Output shape:
#   (frame, range bin q, Doppler bin p, RX channel)
run_b_range_doppler_stack = np.stack(
    [
        compute_radc_range_doppler(
            radc_frame=run_b_center_radc[frame_index],
            range_window="hann",
            doppler_window="hann",
        )
        for frame_index in range(
            run_b_center_radc.shape[0]
        )
    ],
    axis=0,
)


# Extract the centered zero-Doppler data from Run B.
#
# Output shape:
#   (frame, range bin q, RX channel)
run_b_zero_doppler_stack, run_b_zero_doppler_bin = (
    extract_zero_doppler(
        run_b_range_doppler_stack,
        doppler_axis=2,
    )
)


# Confirm that Run A and Run B use the same zero-Doppler bin.
assert (
    zero_doppler_bin
    == run_b_zero_doppler_bin
), (
    "Run A and Run B zero-Doppler bins do not match: "
    f"{zero_doppler_bin} versus "
    f"{run_b_zero_doppler_bin}."
)


# Calculate the Run B coherence factor at every range bin
# and RX channel.
#
# A value of 1 means that the frame phases are perfectly aligned.
run_b_coherence_by_range = (
    np.abs(
        np.sum(
            run_b_zero_doppler_stack,
            axis=0,
        )
    )
    / np.maximum(
        np.sum(
            np.abs(
                run_b_zero_doppler_stack
            ),
            axis=0,
        ),
        np.finfo(float).tiny,
    )
)


# Coherently average the Run A zero-Doppler data across
# its five frames.
#
# Shape:
#   (range bin q, RX channel)
run_a_zero_doppler_mean = np.mean(
    python_zero_doppler_stack,
    axis=0,
)


# Coherently average the Run B zero-Doppler data across
# its 20 frames.
run_b_zero_doppler_mean = np.mean(
    run_b_zero_doppler_stack,
    axis=0,
)


# Define range bins used to estimate the cross-run alignment.
#
# Exclude:
#   - bin 0, which is dominated by DC or direct coupling;
#   - bins 15–25, which include both the geometrically expected
#     sphere region and the stronger response near bin 22.
number_of_range_bins = (
    run_a_zero_doppler_mean.shape[0]
)

reference_range_mask = np.zeros(
    number_of_range_bins,
    dtype=bool,
)

reference_range_mask[1:51] = True
reference_range_mask[15:26] = False


# Retain only reference samples that are strong enough in both
# runs to provide a meaningful phase comparison.
minimum_reference_level_db = -35.0
minimum_reference_level_linear = (
    10 ** (
        minimum_reference_level_db / 20
    )
)

run_a_normalized_magnitude = (
    np.abs(
        run_a_zero_doppler_mean
    )
    / np.maximum(
        np.abs(
            run_a_zero_doppler_mean
        ).max(),
        np.finfo(float).tiny,
    )
)

run_b_normalized_magnitude = (
    np.abs(
        run_b_zero_doppler_mean
    )
    / np.maximum(
        np.abs(
            run_b_zero_doppler_mean
        ).max(),
        np.finfo(float).tiny,
    )
)


# Estimate one complex Run B-to-Run A alignment coefficient
# independently for each RX channel.
alignment_results = []
channel_alignment_coefficients = np.zeros(
    run_a_zero_doppler_mean.shape[1],
    dtype=np.complex128,
)


for channel_index in range(
    run_a_zero_doppler_mean.shape[1]
):

    channel_reference_mask = (
        reference_range_mask
        & (
            run_a_normalized_magnitude[
                :,
                channel_index,
            ]
            >= minimum_reference_level_linear
        )
        & (
            run_b_normalized_magnitude[
                :,
                channel_index,
            ]
            >= minimum_reference_level_linear
        )
    )

    run_a_reference = (
        run_a_zero_doppler_mean[
            channel_reference_mask,
            channel_index,
        ]
    )

    run_b_reference = (
        run_b_zero_doppler_mean[
            channel_reference_mask,
            channel_index,
        ]
    )

    if run_a_reference.size == 0:
        raise ValueError(
            f"No valid reference bins were found for "
            f"RX channel {channel_index}."
        )

    # Estimate the least-squares complex coefficient alpha
    # such that Run A is approximately alpha times Run B.
    alignment_coefficient = (
        np.sum(
            run_a_reference
            * np.conj(
                run_b_reference
            )
        )
        / np.maximum(
            np.sum(
                np.abs(
                    run_b_reference
                ) ** 2
            ),
            np.finfo(float).tiny,
        )
    )

    channel_alignment_coefficients[
        channel_index
    ] = alignment_coefficient

    # Calculate normalized complex correlation over the
    # selected reference bins.
    complex_correlation = (
        np.abs(
            np.sum(
                run_a_reference
                * np.conj(
                    run_b_reference
                )
            )
        )
        / np.maximum(
            np.sqrt(
                np.sum(
                    np.abs(
                        run_a_reference
                    ) ** 2
                )
                * np.sum(
                    np.abs(
                        run_b_reference
                    ) ** 2
                )
            ),
            np.finfo(float).tiny,
        )
    )

    alignment_results.append(
        {
            "rx_channel": channel_index,
            "reference_bin_count": (
                run_a_reference.size
            ),
            "alignment_magnitude": np.abs(
                alignment_coefficient
            ),
            "alignment_phase_deg": np.rad2deg(
                np.angle(
                    alignment_coefficient
                )
            ),
            "complex_correlation": (
                complex_correlation
            ),
            "median_background_coherence": np.median(
                run_b_coherence_by_range[
                    reference_range_mask,
                    channel_index,
                ]
            ),
        }
    )


# Plot Run B frame coherence and the Run A-to-Run B
# phase difference across range.
range_bin_index = np.arange(
    number_of_range_bins
)

fig, axes = plt.subplots(
    2,
    1,
    figsize=(12, 9),
    sharex=True,
    constrained_layout=True,
)


# Plot Run B coherence across range for all four RX channels.
for channel_index in range(
    run_b_coherence_by_range.shape[1]
):
    axes[0].plot(
        range_bin_index,
        run_b_coherence_by_range[
            :,
            channel_index,
        ],
        linewidth=1.2,
        label=f"RX {channel_index}",
    )

axes[0].set_title(
    "Run B Background Frame-to-Frame Coherence"
)
axes[0].set_ylabel(
    "Coherence factor"
)
axes[0].set_ylim(
    0,
    1.02,
)
axes[0].set_xlim(
    0,
    50,
)
axes[0].grid(
    True,
    alpha=0.3,
)
axes[0].legend(
    ncols=4,
)


# Plot the Run A-to-Run B phase difference across range.
for channel_index in range(
    run_a_zero_doppler_mean.shape[1]
):

    valid_phase_mask = (
        reference_range_mask
        & (
            run_a_normalized_magnitude[
                :,
                channel_index,
            ]
            >= minimum_reference_level_linear
        )
        & (
            run_b_normalized_magnitude[
                :,
                channel_index,
            ]
            >= minimum_reference_level_linear
        )
    )

    phase_difference_deg = np.full(
        number_of_range_bins,
        np.nan,
    )

    phase_difference_deg[
        valid_phase_mask
    ] = np.rad2deg(
        np.angle(
            run_a_zero_doppler_mean[
                valid_phase_mask,
                channel_index,
            ]
            * np.conj(
                run_b_zero_doppler_mean[
                    valid_phase_mask,
                    channel_index,
                ]
            )
        )
    )

    axes[1].plot(
        range_bin_index,
        phase_difference_deg,
        marker="o",
        markersize=4,
        linewidth=1.0,
        label=f"RX {channel_index}",
    )

axes[1].set_title(
    "Run A-to-Run B Phase Difference in Reference Bins"
)
axes[1].set_xlabel(
    "Range-bin index, $q$"
)
axes[1].set_ylabel(
    "Phase difference (degrees)"
)
axes[1].set_xlim(
    0,
    50,
)
axes[1].set_ylim(
    -180,
    180,
)
axes[1].grid(
    True,
    alpha=0.3,
)
axes[1].legend(
    ncols=4,
)

plt.show()


# Display the estimated cross-run alignment parameters.
alignment_results_table = pd.DataFrame(
    alignment_results
)

display(
    alignment_results_table
)

#### Results

The matched Run B background measurements exhibit excellent frame-to-frame complex coherence.

Across the selected reference range bins, the median background coherence factor is greater than approximately $0.9997$ for every RX channel. This confirms that the 20 Run B frames can be coherently averaged without significant phase cancellation.

The lower coherence values around range bins 12–16 occur in a relatively weak portion of the range spectrum. These localized reductions are therefore more likely caused by low signal-to-noise ratio than by general radar phase instability.

The coherently averaged Run A and Run B measurements also exhibit strong cross-run agreement. The normalized complex correlations range from approximately $0.996$ to $0.9999$ across the four RX channels.

The estimated Run B-to-Run A magnitude corrections range from approximately 1.025 to 1.036, corresponding to only a few percent difference in complex amplitude. The estimated phase corrections range from approximately $-0.9^\circ$ to $-1.4^\circ$.

The similarity of the alignment coefficients among the four RX channels indicates that the radar retained a consistent complex reference between Runs A and B. The small per-channel corrections can therefore be applied to the coherently averaged Run B background before subtraction.

These results justify aligned complex background subtraction. The subtraction must be performed on the complex RX data before magnitude calculation and beamforming.

### 2.8.3 — Apply Aligned Complex Background Subtraction

This cell applies the estimated per-channel complex alignment coefficients to the coherently averaged Run B background.

The aligned background is subtracted from the coherently averaged Run A zero-Doppler data before beamforming. Performing subtraction at the complex RX-data stage preserves the phase information required for range–azimuth processing and later SAR reconstruction.

Three range–azimuth maps are compared:

- the coherently averaged Run A scene;
- the aligned Run B background;
- the complex background-subtracted residual.

All maps use the maximum magnitude of the raw Run A map as a common 0 dB reference. The shared reference allows the reduction in stationary clutter to be compared directly rather than renormalizing every map independently.

In [ ]:
# Apply the estimated complex alignment coefficient independently
# to each Run B RX channel.
#
# Shape:
#   (range bin q, RX channel)
aligned_run_b_zero_doppler_mean = (
    run_b_zero_doppler_mean
    * channel_alignment_coefficients[
        np.newaxis,
        :,
    ]
)


# Subtract the aligned complex background from the coherently
# averaged Run A data.
#
# Subtraction is performed before magnitude calculation and
# beamforming so that complex phase information is preserved.
background_subtracted_zero_doppler = (
    run_a_zero_doppler_mean
    - aligned_run_b_zero_doppler_mean
)


# Define the same four-RX array geometry and azimuth grid
# used in the preliminary range–azimuth comparison.
carrier_frequency_hz = 61.0e9
rx_element_spacing_m = 2.464e-3

number_of_rx_channels = (
    run_a_zero_doppler_mean.shape[1]
)

rx_positions_m = (
    np.arange(number_of_rx_channels)
    - (number_of_rx_channels - 1) / 2
) * rx_element_spacing_m

azimuth_angle_deg = np.linspace(
    -70.0,
    70.0,
    281,
)


# Beamform the coherently averaged Run A data.
run_a_range_azimuth_map = (
    conventional_range_azimuth(
        range_rx_data=run_a_zero_doppler_mean,
        azimuth_angle_deg=azimuth_angle_deg,
        rx_positions_m=rx_positions_m,
        carrier_frequency_hz=carrier_frequency_hz,
    )
)


# Beamform the aligned Run B background.
run_b_range_azimuth_map = (
    conventional_range_azimuth(
        range_rx_data=aligned_run_b_zero_doppler_mean,
        azimuth_angle_deg=azimuth_angle_deg,
        rx_positions_m=rx_positions_m,
        carrier_frequency_hz=carrier_frequency_hz,
    )
)


# Beamform the complex background-subtracted residual.
subtracted_range_azimuth_map = (
    conventional_range_azimuth(
        range_rx_data=background_subtracted_zero_doppler,
        azimuth_angle_deg=azimuth_angle_deg,
        rx_positions_m=rx_positions_m,
        carrier_frequency_hz=carrier_frequency_hz,
    )
)


# Use the raw coherently averaged Run A map as the common
# magnitude reference for all three images.
common_reference_magnitude = np.maximum(
    np.abs(
        run_a_range_azimuth_map
    ).max(),
    np.finfo(float).tiny,
)


# Convert all maps to dB using the same Run A reference.
def magnitude_db_relative_to_run_a(
    complex_map,
):
    """
    Convert magnitude to dB relative to the raw Run A maximum.
    """

    return 20 * np.log10(
        np.maximum(
            np.abs(
                complex_map
            ),
            np.finfo(float).tiny,
        )
        / common_reference_magnitude
    )


run_a_range_azimuth_map_db = (
    magnitude_db_relative_to_run_a(
        run_a_range_azimuth_map
    )
)

run_b_range_azimuth_map_db = (
    magnitude_db_relative_to_run_a(
        run_b_range_azimuth_map
    )
)

subtracted_range_azimuth_map_db = (
    magnitude_db_relative_to_run_a(
        subtracted_range_azimuth_map
    )
)


# Arrange the maps in processing order.
comparison_maps = {
    "Coherently Averaged Run A": (
        run_a_range_azimuth_map_db
    ),
    "Aligned Run B Background": (
        run_b_range_azimuth_map_db
    ),
    "Complex Background-Subtracted": (
        subtracted_range_azimuth_map_db
    ),
}


# Create the three-panel comparison.
fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 7),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)


for plot_index, (
    ax,
    (map_title, map_db),
) in enumerate(
    zip(
        axes,
        comparison_maps.items(),
    )
):
    image = ax.imshow(
        map_db,
        origin="lower",
        aspect="auto",
        extent=[
            azimuth_angle_deg[0],
            azimuth_angle_deg[-1],
            -0.5,
            map_db.shape[0] - 0.5,
        ],
        vmin=-50,
        vmax=0,
        cmap="viridis",
    )

    ax.set_title(
        map_title
    )
    ax.set_xlabel(
        "Azimuth angle (degrees, provisional sign)"
    )

    # Mark the expected sphere region.
    expected_sphere_box = Rectangle(
        xy=(-10.0, 16.5),
        width=20.0,
        height=3.0,
        fill=False,
        edgecolor="red",
        linewidth=2.0,
        label="Expected sphere region",
    )

    ax.add_patch(
        expected_sphere_box
    )

    # Mark the assumed boresight direction.
    ax.axvline(
        0.0,
        color="white",
        linestyle="--",
        linewidth=1.1,
        alpha=0.8,
        label="Boresight",
    )

    ax.set_xlim(
        -70,
        70,
    )
    ax.set_ylim(
        -0.5,
        50,
    )

    if plot_index == 0:
        ax.legend(
            loc="upper right",
        )


axes[0].set_ylabel(
    "Range-bin index, $q$"
)


# Add a shared colorbar using the raw Run A maximum as 0 dB.
colorbar = fig.colorbar(
    image,
    ax=axes,
    shrink=0.88,
    pad=0.02,
)

colorbar.set_label(
    "Magnitude relative to raw Run A maximum (dB)"
)


fig.suptitle(
    "Run A Matched Complex Background Subtraction",
    fontsize=15,
)

plt.show()


# Quantify background suppression over the same reference
# range bins used for cross-run alignment.
suppression_results = []

for channel_index in range(
    number_of_rx_channels
):

    run_a_reference = (
        run_a_zero_doppler_mean[
            reference_range_mask,
            channel_index,
        ]
    )

    residual_reference = (
        background_subtracted_zero_doppler[
            reference_range_mask,
            channel_index,
        ]
    )

    background_suppression_db = 10 * np.log10(
        np.maximum(
            np.sum(
                np.abs(
                    run_a_reference
                ) ** 2
            ),
            np.finfo(float).tiny,
        )
        / np.maximum(
            np.sum(
                np.abs(
                    residual_reference
                ) ** 2
            ),
            np.finfo(float).tiny,
        )
    )

    suppression_results.append(
        {
            "rx_channel": channel_index,
            "background_suppression_db": (
                background_suppression_db
            ),
        }
    )


suppression_results_table = pd.DataFrame(
    suppression_results
)

display(
    suppression_results_table
)

#### Results

The matched Run B background measurements were coherently aligned to Run A and subtracted independently for each RX channel before angle beamforming.

The subtraction provides substantial suppression of the repeatable stationary scene:

| RX channel | Background suppression |
|---:|---:|
| 0 | $19.5\ \text{dB}$ |
| 1 | $23.9\ \text{dB}$ |
| 2 | $27.3\ \text{dB}$ |
| 3 | $28.0\ \text{dB}$ |

The strong close-range response near $q\approx0$ is almost completely removed in the background-subtracted map. This confirms that the complex Run B-to-Run A alignment and subtraction are functioning correctly.

A strong residual remains near:

$$ q\approx22. $$

Under the nominal V-MD3 range mapping, this corresponds to:

$$ R_{\text{nominal}}=22\left(\frac{6}{128}\right)\approx1.03\ \text{m}. $$

The independently measured sphere slant range is approximately $0.84\ \text{m}$, which nominally corresponds to:

$$ q_{\text{expected}}=\frac{0.84}{6/128}\approx18. $$

The expected region near $q=17$–$19$ is not the dominant target-associated response. Because the feature near $q\approx22$ remains after subtraction of the matched no-target background, it is associated with the sphere or with multipath introduced by the sphere rather than ordinary stationary clutter.

The angle beamformer cannot cause the four-bin range displacement because it combines RX channels independently at each range bin:

$$ Y[q,\theta]=\sum_{m=0}^{M-1}X[q,m]w_m^*(\theta). $$

No operation in the beamformer combines different values of $q$. Therefore, the response near $q\approx22$ was already present in the complex range data before beamforming.

Earlier comparisons also established that:

- the Python RADC and V-MD3 FPGA RFFT products have consistent range–Doppler structure;
- both place zero Doppler at $p=32$;
- Hann-windowed Python processing most closely resembles the FPGA output;
- changing the spectral window alters main-lobe width and sidelobe level but does not explain a displacement of approximately four range bins.

The current measurement cannot determine whether the discrepancy is caused by:

- a fixed internal delay or range-bin offset;
- an inaccurate range-scale assumption;
- sphere-induced multipath;
- or a stronger delayed scattering path.

Distinguishing these possibilities requires measurements of the sphere at two or more known slant ranges. A one-point correction should therefore not be applied to the current data.

The background-subtracted target response remains broad in azimuth. This is expected because subtraction removes repeatable clutter but does not narrow the four-RX array point-spread function. The remaining angular width and structure must be addressed through verification of the RX ordering, steering-vector convention, array geometry, and channel calibration.

For the remainder of Run A processing, $q\approx22$ will be treated as the observed sphere-associated range bin, while its physical range remains uncalibrated.

## 2.9 — Verify and Improve the Four-RX Angle Beamforming

The purpose of this section is to evaluate and improve the angular processing applied to the complex background-subtracted Run A data.

Conventional beamforming evaluates the coherent RX sum:

$$ Y[q,\theta]=\sum_{m=0}^{M-1}X[q,m]w_m^*(\theta), $$

where the ideal steering-vector element is:

$$ w_m(\theta)=\exp\left(-jkx_m\sin\theta\right). $$

Here:

- $q$ is range-bin index;
- $m$ is RX-channel index;
- $x_m$ is the physical position of RX element $m$;
- $k=2\pi/\lambda$ is the physical wavenumber;
- $\theta$ is the trial azimuth angle.

For an ideal uniform linear array, the received phase should vary approximately linearly with RX-element position:

$$ \phi_m\approx\phi_0-kx_m\sin\theta_{\text{target}}. $$

The slope of this phase progression provides an independent estimate of the target angle:

$$ \theta_{\text{target}}\approx\sin^{-1}\left(-\frac{1}{k}\frac{d\phi}{dx}\right). $$

This section first examines the uncalibrated RX response at the observed sphere-associated range bin $q=22$. It reports:

- relative magnitude across the four RX channels;
- unwrapped phase progression across the physical RX positions;
- the angle inferred from the measured phase slope;
- the residual error from an ideal linear phase progression;
- spatial coherence after steering to the estimated angle;
- the peak angle and angular width of the conventional beamformed response.

No RX calibration is applied in the first diagnostic. The results will determine whether channel ordering, phase calibration, or steering-vector corrections are justified.

### 2.9.1 — Four-RX Beamforming Diagnostic Functions

The following functions analyze the RX-channel vector at one target-associated range bin and quantify the resulting angular response.

They do not modify the Run A data or apply channel calibration.

In [ ]:
def analyze_rx_phase_progression(
    target_rx_vector,
    rx_positions_m,
    carrier_frequency_hz,
):
    """
    Fit a linear phase progression across the RX elements.

    Parameters
    ----------
    target_rx_vector : array-like, complex
        Complex target response with shape (RX channel,).

    rx_positions_m : array-like, float
        Physical RX-element positions in meters.

    carrier_frequency_hz : float
        Radar carrier frequency in hertz.

    Returns
    -------
    dict
        Magnitudes, measured and fitted phases, inferred angle,
        phase-fit error, and spatial coherence.
    """

    target_rx_vector = np.asarray(
        target_rx_vector,
        dtype=np.complex128,
    )

    rx_positions_m = np.asarray(
        rx_positions_m,
        dtype=float,
    )

    if target_rx_vector.ndim != 1:
        raise ValueError(
            "target_rx_vector must be one-dimensional."
        )

    if target_rx_vector.shape != rx_positions_m.shape:
        raise ValueError(
            "target_rx_vector and rx_positions_m must "
            "have the same shape."
        )

    wavelength_m = (
        299_792_458.0
        / carrier_frequency_hz
    )

    wavenumber_rad_per_m = (
        2 * np.pi
        / wavelength_m
    )

    # Calculate the measured magnitude and unwrap the
    # inter-element phase progression.
    magnitude = np.abs(
        target_rx_vector
    )

    unwrapped_phase_rad = np.unwrap(
        np.angle(
            target_rx_vector
        )
    )

    # Fit:
    #
    #     phase(x) = phase_slope*x + phase_intercept
    #
    phase_slope_rad_per_m, phase_intercept_rad = (
        np.polyfit(
            rx_positions_m,
            unwrapped_phase_rad,
            deg=1,
        )
    )

    fitted_phase_rad = (
        phase_slope_rad_per_m
        * rx_positions_m
        + phase_intercept_rad
    )

    phase_fit_error_rad = (
        unwrapped_phase_rad
        - fitted_phase_rad
    )

    phase_fit_rms_deg = np.rad2deg(
        np.sqrt(
            np.mean(
                phase_fit_error_rad ** 2
            )
        )
    )

    # Under the steering convention used by
    # conventional_range_azimuth():
    #
    #     phase slope = -k*sin(theta)
    #
    estimated_sine = np.clip(
        -phase_slope_rad_per_m
        / wavenumber_rad_per_m,
        -1.0,
        1.0,
    )

    estimated_angle_deg = np.rad2deg(
        np.arcsin(
            estimated_sine
        )
    )

    estimated_steering_vector = np.exp(
        -1j
        * wavenumber_rad_per_m
        * rx_positions_m
        * estimated_sine
    )

    # Normalize the coherent sum by the sum of the channel
    # magnitudes. A value near 1 indicates strong spatial
    # coherence at the estimated angle.
    spatial_coherence = (
        np.abs(
            np.sum(
                target_rx_vector
                * np.conj(
                    estimated_steering_vector
                )
            )
        )
        / np.maximum(
            np.sum(
                np.abs(
                    target_rx_vector
                )
            ),
            np.finfo(float).tiny,
        )
    )

    return {
        "magnitude": magnitude,
        "unwrapped_phase_rad": unwrapped_phase_rad,
        "fitted_phase_rad": fitted_phase_rad,
        "phase_fit_error_rad": phase_fit_error_rad,
        "phase_slope_rad_per_m": phase_slope_rad_per_m,
        "phase_intercept_rad": phase_intercept_rad,
        "estimated_angle_deg": estimated_angle_deg,
        "phase_fit_rms_deg": phase_fit_rms_deg,
        "spatial_coherence": spatial_coherence,
        "wavelength_m": wavelength_m,
        "wavenumber_rad_per_m": wavenumber_rad_per_m,
    }


def form_angular_power_profile(
    range_azimuth_map,
    selected_range_bins,
):
    """
    Form an angular power profile by noncoherently combining
    selected range bins.

    Parameters
    ----------
    range_azimuth_map : array-like, complex
        Complex map with shape (range bin, azimuth angle).

    selected_range_bins : array-like, int
        Range bins containing the target response.

    Returns
    -------
    angular_power : ndarray
        Linear angular power.

    angular_power_db : ndarray
        Angular power normalized to 0 dB.
    """

    selected_range_bins = np.asarray(
        selected_range_bins,
        dtype=int,
    )

    selected_data = range_azimuth_map[
        selected_range_bins,
        :,
    ]

    angular_power = np.sum(
        np.abs(
            selected_data
        ) ** 2,
        axis=0,
    )

    reference_power = np.maximum(
        angular_power.max(),
        np.finfo(float).tiny,
    )

    angular_power_db = 10 * np.log10(
        np.maximum(
            angular_power
            / reference_power,
            np.finfo(float).tiny,
        )
    )

    return (
        angular_power,
        angular_power_db,
    )


def measure_angular_mainlobe(
    azimuth_angle_deg,
    angular_power_db,
    threshold_db=-3.0,
):
    """
    Measure the peak angle and contiguous main-lobe width at
    a specified level below the peak.

    The angular profile is expected to be normalized to 0 dB.
    """

    azimuth_angle_deg = np.asarray(
        azimuth_angle_deg,
        dtype=float,
    )

    angular_power_db = np.asarray(
        angular_power_db,
        dtype=float,
    )

    if azimuth_angle_deg.shape != angular_power_db.shape:
        raise ValueError(
            "The angle and angular-profile arrays must "
            "have the same shape."
        )

    peak_index = int(
        np.argmax(
            angular_power_db
        )
    )

    peak_angle_deg = float(
        azimuth_angle_deg[
            peak_index
        ]
    )

    threshold_level_db = (
        angular_power_db[
            peak_index
        ]
        + threshold_db
    )

    left_index = peak_index

    while (
        left_index > 0
        and angular_power_db[
            left_index - 1
        ]
        >= threshold_level_db
    ):
        left_index -= 1

    right_index = peak_index

    while (
        right_index
        < angular_power_db.size - 1
        and angular_power_db[
            right_index + 1
        ]
        >= threshold_level_db
    ):
        right_index += 1

    left_angle_deg = float(
        azimuth_angle_deg[
            left_index
        ]
    )

    right_angle_deg = float(
        azimuth_angle_deg[
            right_index
        ]
    )

    mainlobe_width_deg = (
        right_angle_deg
        - left_angle_deg
    )

    return {
        "peak_index": peak_index,
        "peak_angle_deg": peak_angle_deg,
        "threshold_db": threshold_db,
        "left_angle_deg": left_angle_deg,
        "right_angle_deg": right_angle_deg,
        "mainlobe_width_deg": mainlobe_width_deg,
    }

### 2.9.2 — Inspect the Uncalibrated Four-RX Spatial Response

This cell analyzes the complex background-subtracted RX vector at the observed sphere-associated range bin $q=22$.

The measured inter-element phase is fitted to the ideal linear phase progression of a uniform linear array. The resulting phase slope provides an estimated target angle that can be compared with the peak of the conventional beamformed response.

For the angular profile, power is combined noncoherently across range bins 20–24. This includes the broadened target-associated range response without allowing complex phase changes across neighboring range bins to cancel.

The image and angular profile remain uncalibrated. Their purpose is to determine whether the existing RX ordering and steering-vector convention already produce a physically consistent response.

In [ ]:
# ---------------------------------------------------------------------
# Select the sphere-associated range response.
# ---------------------------------------------------------------------

observed_target_range_bin = 22

# Use a small range interval around q = 22 when measuring
# the angular profile. The range bins are combined in power,
# not complex amplitude.
target_range_bins = np.arange(
    20,
    25,
)


# Extract the complex four-RX target vector before beamforming.
target_rx_vector = (
    background_subtracted_zero_doppler[
        observed_target_range_bin,
        :,
    ]
)


# Confirm that the existing array description matches the data.
assert (
    target_rx_vector.shape[0]
    == rx_positions_m.size
), (
    "The target RX vector and physical RX-position array "
    "have different lengths."
)


# ---------------------------------------------------------------------
# Analyze the uncalibrated RX phase progression.
# ---------------------------------------------------------------------

rx_phase_diagnostic = (
    analyze_rx_phase_progression(
        target_rx_vector=target_rx_vector,
        rx_positions_m=rx_positions_m,
        carrier_frequency_hz=carrier_frequency_hz,
    )
)


# Normalize RX magnitudes to the strongest channel.
relative_rx_magnitude_db = 20 * np.log10(
    np.maximum(
        rx_phase_diagnostic["magnitude"]
        / np.maximum(
            rx_phase_diagnostic["magnitude"].max(),
            np.finfo(float).tiny,
        ),
        np.finfo(float).tiny,
    )
)


# Express phase relative to RX 0 for easier interpretation.
measured_relative_phase_deg = np.rad2deg(
    rx_phase_diagnostic[
        "unwrapped_phase_rad"
    ]
    - rx_phase_diagnostic[
        "unwrapped_phase_rad"
    ][0]
)

fitted_relative_phase_deg = np.rad2deg(
    rx_phase_diagnostic[
        "fitted_phase_rad"
    ]
    - rx_phase_diagnostic[
        "fitted_phase_rad"
    ][0]
)


# Build a table describing the complex RX vector.
rx_spatial_response_table = pd.DataFrame(
    {
        "rx_channel": np.arange(
            target_rx_vector.size
        ),
        "rx_position_mm": (
            rx_positions_m
            * 1e3
        ),
        "relative_magnitude_db": (
            relative_rx_magnitude_db
        ),
        "measured_relative_phase_deg": (
            measured_relative_phase_deg
        ),
        "fitted_relative_phase_deg": (
            fitted_relative_phase_deg
        ),
        "phase_fit_error_deg": np.rad2deg(
            rx_phase_diagnostic[
                "phase_fit_error_rad"
            ]
        ),
    }
)


# ---------------------------------------------------------------------
# Measure the angular response of the existing conventional beamformer.
# ---------------------------------------------------------------------

angular_power, angular_power_db = (
    form_angular_power_profile(
        range_azimuth_map=(
            subtracted_range_azimuth_map
        ),
        selected_range_bins=(
            target_range_bins
        ),
    )
)

angular_mainlobe_results = (
    measure_angular_mainlobe(
        azimuth_angle_deg=azimuth_angle_deg,
        angular_power_db=angular_power_db,
        threshold_db=-3.0,
    )
)


# Normalize the residual range–azimuth map to its own maximum
# for this angular diagnostic.
subtracted_map_normalized_db = (
    normalized_magnitude_db(
        subtracted_range_azimuth_map,
        minimum_db=-50,
    )
)


# ---------------------------------------------------------------------
# Plot the RX response and existing beamformed result.
# ---------------------------------------------------------------------

fig, axes = plt.subplots(
    2,
    2,
    figsize=(15, 11),
    constrained_layout=True,
)


# Plot relative RX-channel magnitude.
axes[0, 0].bar(
    np.arange(
        target_rx_vector.size
    ),
    relative_rx_magnitude_db,
    color="tab:blue",
    alpha=0.8,
)

axes[0, 0].set_title(
    f"RX Magnitude at $q={observed_target_range_bin}$"
)

axes[0, 0].set_xlabel(
    "RX channel"
)

axes[0, 0].set_ylabel(
    "Magnitude relative to strongest RX (dB)"
)

axes[0, 0].set_xticks(
    np.arange(
        target_rx_vector.size
    )
)

axes[0, 0].grid(
    True,
    axis="y",
    alpha=0.3,
)


# Plot the measured and fitted spatial phase progression.
axes[0, 1].plot(
    rx_positions_m * 1e3,
    measured_relative_phase_deg,
    marker="o",
    linewidth=2.0,
    label="Measured phase",
)

axes[0, 1].plot(
    rx_positions_m * 1e3,
    fitted_relative_phase_deg,
    marker="s",
    linestyle="--",
    linewidth=1.5,
    label="Linear phase fit",
)

axes[0, 1].set_title(
    "Inter-Element Phase Progression"
)

axes[0, 1].set_xlabel(
    "RX-element position (mm)"
)

axes[0, 1].set_ylabel(
    "Unwrapped phase relative to RX 0 (degrees)"
)

axes[0, 1].grid(
    True,
    alpha=0.3,
)

axes[0, 1].legend()


# Plot the background-subtracted range–azimuth map.
image = axes[1, 0].imshow(
    subtracted_map_normalized_db,
    origin="lower",
    aspect="auto",
    extent=[
        azimuth_angle_deg[0],
        azimuth_angle_deg[-1],
        -0.5,
        subtracted_map_normalized_db.shape[0] - 0.5,
    ],
    vmin=-50,
    vmax=0,
    cmap="viridis",
)

axes[1, 0].axhline(
    observed_target_range_bin,
    color="tab:red",
    linestyle="-.",
    linewidth=1.8,
    label=r"Observed $q=22$",
)

# Draw a red rectangle around the expected near-boresight
# sphere region.
expected_sphere_box = Rectangle(
    xy=(-10.0, 16.5),
    width=20.0,
    height=3.0,
    fill=False,
    edgecolor="red",
    linewidth=2.0,
    label=r"Expected $q=17$–$19$",
)

axes[1, 0].add_patch(
    expected_sphere_box
)

axes[1, 0].axvline(
    0.0,
    color="white",
    linestyle="--",
    linewidth=1.2,
    alpha=0.8,
    label="Boresight",
)

axes[1, 0].set_title(
    "Uncalibrated Background-Subtracted Map"
)

axes[1, 0].set_xlabel(
    "Azimuth angle (degrees, provisional sign)"
)

axes[1, 0].set_ylabel(
    "Range-bin index, $q$"
)

axes[1, 0].set_xlim(
    -70,
    70,
)

axes[1, 0].set_ylim(
    -0.5,
    50,
)

axes[1, 0].legend(
    loc="upper right",
)

colorbar = fig.colorbar(
    image,
    ax=axes[1, 0],
    pad=0.02,
)

colorbar.set_label(
    "Magnitude relative to residual-map maximum (dB)"
)


# Plot the target-associated angular power profile.
axes[1, 1].plot(
    azimuth_angle_deg,
    angular_power_db,
    color="black",
    linewidth=2.0,
)

axes[1, 1].axvline(
    angular_mainlobe_results[
        "peak_angle_deg"
    ],
    color="tab:red",
    linestyle="-.",
    linewidth=1.8,
    label=(
        "Beamformed peak: "
        f"{angular_mainlobe_results['peak_angle_deg']:.1f}°"
    ),
)

axes[1, 1].axvline(
    rx_phase_diagnostic[
        "estimated_angle_deg"
    ],
    color="tab:blue",
    linestyle=":",
    linewidth=1.8,
    label=(
        "Phase-slope estimate: "
        f"{rx_phase_diagnostic['estimated_angle_deg']:.1f}°"
    ),
)

axes[1, 1].axhline(
    -3.0,
    color="tab:gray",
    linestyle="--",
    linewidth=1.2,
    label="-3 dB level",
)

axes[1, 1].axvspan(
    angular_mainlobe_results[
        "left_angle_deg"
    ],
    angular_mainlobe_results[
        "right_angle_deg"
    ],
    color="tab:green",
    alpha=0.15,
    label=(
        "-3 dB width: "
        f"{angular_mainlobe_results['mainlobe_width_deg']:.1f}°"
    ),
)

axes[1, 1].set_title(
    "Target-Associated Angular Power Profile"
)

axes[1, 1].set_xlabel(
    "Azimuth angle (degrees, provisional sign)"
)

axes[1, 1].set_ylabel(
    "Normalized power (dB)"
)

axes[1, 1].set_xlim(
    -70,
    70,
)

axes[1, 1].set_ylim(
    -30,
    3,
)

axes[1, 1].grid(
    True,
    alpha=0.3,
)

axes[1, 1].legend(
    loc="lower right",
    fontsize=8,
)


fig.suptitle(
    "Run A Uncalibrated Four-RX Spatial Response",
    fontsize=15,
)

plt.show()


# ---------------------------------------------------------------------
# Display numerical results.
# ---------------------------------------------------------------------

display(
    rx_spatial_response_table
)


beamforming_diagnostic_summary = pd.DataFrame(
    [
        {
            "observed_range_bin": (
                observed_target_range_bin
            ),
            "phase_slope_angle_deg": (
                rx_phase_diagnostic[
                    "estimated_angle_deg"
                ]
            ),
            "beamformed_peak_angle_deg": (
                angular_mainlobe_results[
                    "peak_angle_deg"
                ]
            ),
            "phase_fit_rms_deg": (
                rx_phase_diagnostic[
                    "phase_fit_rms_deg"
                ]
            ),
            "spatial_coherence": (
                rx_phase_diagnostic[
                    "spatial_coherence"
                ]
            ),
            "minus_3db_width_deg": (
                angular_mainlobe_results[
                    "mainlobe_width_deg"
                ]
            ),
        }
    ]
)

display(
    beamforming_diagnostic_summary
)

### 2.9 Results — Uncalibrated Four-RX Beamforming

#### Purpose of the diagnostic

The background-subtracted zero-Doppler data have the form:

$$ X[q,m], $$

where $q$ is range-bin index and $m$ is RX-channel index.

At the observed sphere-associated range bin $q=22$, the four RX channels form the complex spatial vector:

$$ \mathbf{x}[22]=\begin{bmatrix}X[22,0]&X[22,1]&X[22,2]&X[22,3]\end{bmatrix}. $$

The relative phases of these four measurements contain angle information. For an ideal uniform linear array, the phase should vary approximately linearly with physical RX-element position:

$$ \phi_m\approx\phi_0-kx_m\sin\theta. $$

The diagnostic therefore performs two related angle estimates:

1. fit a straight line to the measured phase versus RX position and calculate the corresponding angle;
2. scan a conventional beamformer across trial angles and find the angle that produces the largest coherent sum.

The diagnostic also measures the RX amplitude balance, phase-fit error, spatial coherence, and angular main-lobe width.

No RX amplitude or phase calibration was applied.

---

#### RX-channel magnitude balance

The measured channel magnitudes at $q=22$ are:

| RX channel | Relative magnitude | Approximate linear amplitude |
|---:|---:|---:|
| 0 | $-4.27\ \text{dB}$ | 0.61 |
| 1 | $-3.58\ \text{dB}$ | 0.66 |
| 2 | $-3.44\ \text{dB}$ | 0.67 |
| 3 | $0.00\ \text{dB}$ | 1.00 |

RX channel 3 contains the strongest target-associated response. The other channels have amplitudes approximately 61–67% of the RX 3 amplitude.

This represents a noticeable amplitude imbalance, but it does not prevent coherent beamforming. The imbalance may be caused by:

- different RX-channel gains;
- antenna-element pattern differences;
- target or multipath coupling;
- residual differences remaining after background subtraction.

Unequal channel amplitudes change the effective aperture weighting. They can make the sidelobes asymmetric and slightly reduce the effective use of the complete array.

Amplitude equalization should not yet be derived from this sphere measurement because the response at $q=22$ may include multipath. A full amplitude calibration would be better obtained from an independent reference target at a known range and angle.

---

#### Measured phase progression

The measured phase relative to RX 0 is:

| RX channel | Measured relative phase |
|---:|---:|
| 0 | $0.0^\circ$ |
| 1 | $-19.5^\circ$ |
| 2 | $-16.8^\circ$ |
| 3 | $-21.0^\circ$ |

The measured phases are not perfectly linear, but they are reasonably close to a linear spatial progression.

A least-squares straight-line fit gives an inferred target angle of:

$$ \theta_{\text{phase}}\approx1.92^\circ. $$

The RMS difference between the measured phases and the fitted linear progression is:

$$ \epsilon_{\phi,\text{RMS}}\approx5.00^\circ. $$

A phase-fit error of approximately $5^\circ$ is small. It indicates that the four RX measurements are reasonably consistent with a single plane wave arriving from near boresight.

The individual phase-fit residuals are approximately:

| RX channel | Phase-fit residual |
|---:|---:|
| 0 | $+5.26^\circ$ |
| 1 | $-8.17^\circ$ |
| 2 | $+0.56^\circ$ |
| 3 | $+2.35^\circ$ |

RX 1 has the largest phase deviation, but the error remains modest.

Possible contributors to these small deviations include channel phase mismatch, noise, the finite size of the sphere, and multipath. The deviations are not large enough to demonstrate a serious RX-ordering or array-geometry problem.

---

#### Phase-slope angle and beamformed angle

The conventional beamformer independently reaches its maximum at:

$$ \theta_{\text{BF}}=2.0^\circ. $$

The phase-slope estimate is:

$$ \theta_{\text{phase}}\approx1.92^\circ. $$

The two results differ by only:

$$ |\theta_{\text{BF}}-\theta_{\text{phase}}|\approx0.08^\circ. $$

This close agreement confirms that the measured phase progression and the steering-vector implementation are internally consistent.

The phase-slope estimate and beamformed peak are mathematically related because both use the spatial phase progression. Their agreement is therefore primarily a validation of the implementation, not two completely independent physical measurements.

The result supports the following conclusions:

- the assumed RX ordering is locally plausible;
- the element spacing and carrier frequency produce a consistent angular mapping;
- the steering-vector sign used in the beamformer is internally consistent;
- the sphere-associated response is nearly at boresight.

Because the target is only approximately $2^\circ$ from boresight, this measurement cannot conclusively determine the absolute left/right angle sign. A known off-boresight target is required to verify whether positive angle corresponds to the intended physical direction.

---

#### Spatial coherence

The measured spatial coherence is:

$$ C=0.9966. $$

Spatial coherence is calculated as:

$$ C=\frac{\left|\sum_mX_mw_m^*(\widehat{\theta})\right|}{\sum_m|X_m|}. $$

The denominator represents the largest coherent magnitude that could be obtained if all measured RX phasors were perfectly aligned. A value near 1 means that the steering vector nearly achieves this ideal alignment.

The measured value of 0.9966 means that the four channels add almost perfectly after steering to the estimated target angle. This is strong evidence that:

- the complex phase information is valid;
- the Run A frame averaging preserved coherence;
- the Run B subtraction preserved the target phase;
- the existing four-RX beamformer is focusing the target appropriately.

A phase-only calibration derived from this same target could increase the coherence only marginally. Such a correction would risk fitting the calibration to this particular target and its multipath rather than correcting a general channel error.

Therefore, no phase correction is justified from this result.

---

#### Angular main-lobe width

Power from range bins 20–24 was combined noncoherently to measure the angular response:

$$ P(\theta)=\sum_{q=20}^{24}|Y[q,\theta]|^2. $$

The measured $-3\ \text{dB}$ angular width is:

$$ \Delta\theta_{3\text{dB}}\approx25.5^\circ. $$

For a four-element array with approximately half-wavelength element spacing, a broadside uniform-weight main-lobe width of roughly $25^\circ$–$30^\circ$ is expected.

The measured width is therefore consistent with the physical four-element aperture. The broad horizontal appearance of the target is not primarily evidence of failed phase calibration. It is largely the expected angular point-spread function of the short four-RX array.

The smaller peaks away from boresight can contain contributions from:

- the array sidelobes;
- the unequal RX-channel amplitudes;
- residual clutter;
- sphere-associated multipath;
- other responses included in range bins 20–24.

The angular profile is consequently the response of the complete measured scene in the selected range interval, not a pure theoretical array factor.

---

#### Interpretation of the range–azimuth image

The dominant sphere-associated response remains near $q=22$ and peaks near an azimuth angle of $2^\circ$.

The target response extends over a broad range of displayed angles because the four-element array has limited angular resolution. Increasing the number of plotted angle samples would make the response appear smoother, but it would not narrow the physical main lobe.

Background subtraction improved target-to-clutter contrast, while beamforming identified the target’s approximate arrival direction. Neither operation can overcome the physical angular-resolution limit of the four-RX aperture.

The final Run A image should therefore be interpreted as the range–angle point-spread response of the sphere and associated propagation paths, not as a resolved geometric image of the sphere.

---

#### Conclusions

The uncalibrated four-RX beamformer is performing well.

The principal findings are:

- the phase-slope angle is approximately $1.92^\circ$;
- the beamformed peak is $2.0^\circ$;
- the phase-fit RMS error is approximately $5.0^\circ$;
- spatial coherence is 0.9966;
- the measured $-3\ \text{dB}$ width is $25.5^\circ$;
- the angular width agrees with the expected four-element array response;
- the RX amplitudes differ by approximately 3.4–4.3 dB;
- no phase calibration is presently justified;
- absolute left/right angle sign remains provisional;
- amplitude calibration should require an independent calibration measurement.

The existing RX ordering, geometry, and steering convention are therefore adequate for the Run A image near boresight.

The broad angular response is predominantly a physical aperture limitation rather than a processing failure. The longer mechanical aperture used for SAR is expected to provide substantially better cross-range localization.

## 2.10 — Aperture Weighting and the Resolution–Sidelobe Tradeoff

The four RX channels are spatial samples taken at four different antenna positions. Before angle beamforming, each spatial sample may be multiplied by a real amplitude coefficient:

$$ X_w[q,m]=a_mX[q,m], $$

where $a_m$ is the aperture-window coefficient applied to RX element $m$.

The complete beamforming operation is then:

$$ Y[q,\theta]=\sum_{m=0}^{M-1}a_mX[q,m]\exp\left(+jkx_m\sin\theta\right). $$

The two parts of this expression have different purposes:

- the complex exponential performs angle-dependent phase steering;
- the real coefficient $a_m$ controls the amplitude contribution from each RX position.

Aperture weighting does not intentionally change the measured RX phases. The steering phase remains the same as in Section 2.9.

### Why aperture windowing changes the beam

Uniform weighting uses:

$$ a_m=1. $$

All four elements contribute equally, including the two elements at the ends of the array. This uses the largest effective aperture and therefore produces the narrowest angular main lobe. The abrupt termination at the ends of the finite array also produces sidelobes.

A tapered window gradually reduces the contributions from the outer elements. This makes the spatial aperture smoother and can reduce sidelobes. However, down-weighting the outer elements shortens the effective aperture and therefore broadens the main lobe.

The tradeoff is:

- larger effective aperture $\rightarrow$ narrower main lobe and better angular resolution;
- smoother aperture taper $\rightarrow$ lower sidelobes but poorer angular resolution.

This is the same mathematical tradeoff encountered when applying a window before a range FFT. The difference is the dimension being windowed:

- a fast-time window controls the range point-spread function;
- a chirp-index window controls the Doppler point-spread function;
- an RX-aperture window controls the angular point-spread function.

### Four-element window coefficients

For four RX elements, the unnormalized window coefficients are approximately:

| Window | RX coefficients |
|---|---|
| Uniform | $[1,\ 1,\ 1,\ 1]$ |
| Hamming | $[0.08,\ 0.77,\ 0.77,\ 0.08]$ |
| Hann | $[0,\ 0.75,\ 0.75,\ 0]$ |

The Hann window completely removes the two outer elements for a four-element array. It therefore reduces the effective array to approximately two contributing elements. This is expected to substantially broaden the angular response.

For a fair comparison, each window is normalized so that:

$$ \sum_m a_m=1. $$

This gives every ideal boresight beamformer the same coherent signal gain. The comparison then isolates changes in beam shape rather than arbitrary changes in overall scale.

### Noise penalty

After unit-sum normalization, the relative output-noise factor is:

$$ G_{\text{noise}}=M\sum_{m=0}^{M-1}a_m^2. $$

Uniform weighting gives the minimum noise factor:

$$ G_{\text{noise}}=1. $$

A tapered window places more weight on fewer spatial samples, increasing the output-noise variance relative to the uniformly averaged array. The corresponding penalty is:

$$ L_{\text{noise}}=10\log_{10}\left(G_{\text{noise}}\right). $$

### Purpose of this comparison

The theoretical array factor will first be calculated using an ideal target at boresight. This isolates the effect of the aperture weights from clutter, multipath, and channel imbalance.

The same weights will then be applied to the measured background-subtracted Run A data.

For each window, the comparison reports:

- normalized aperture coefficients;
- theoretical $-3\ \text{dB}$ angular width;
- measured peak angle;
- measured $-3\ \text{dB}$ width;
- relative noise penalty;
- changes in the measured range–azimuth map.

The goal is not to select the visually smoothest image. The goal is to understand the physical tradeoff and select the weighting that best preserves useful angular resolution with only four RX elements.

### 2.10.1 — Aperture-Weighting Functions

These functions create normalized spatial-window coefficients, apply them before conventional beamforming, calculate the ideal array response, and quantify the noise penalty.

The existing physical steering-vector function remains responsible for phase alignment.

In [ ]:
def create_normalized_aperture_weights(
    window_name,
    number_of_elements,
):
    """
    Create real aperture-window coefficients and normalize
    them to unit sum.

    Unit-sum normalization preserves the ideal coherent gain
    of a target at the steering angle.
    """

    raw_weights = np.asarray(
        create_window(
            window_name=window_name,
            length=number_of_elements,
        ),
        dtype=float,
    )

    weight_sum = np.sum(
        raw_weights
    )

    if weight_sum <= 0:
        raise ValueError(
            "The aperture weights must have a positive sum."
        )

    normalized_weights = (
        raw_weights
        / weight_sum
    )

    return (
        raw_weights,
        normalized_weights,
    )


def weighted_conventional_range_azimuth(
    range_rx_data,
    azimuth_angle_deg,
    rx_positions_m,
    carrier_frequency_hz,
    aperture_weights,
):
    """
    Apply real aperture-amplitude weights before conventional
    angle beamforming.

    Input shape:
        (range bin, RX channel)

    Output shape:
        (range bin, azimuth angle)
    """

    range_rx_data = np.asarray(
        range_rx_data,
        dtype=np.complex128,
    )

    aperture_weights = np.asarray(
        aperture_weights,
        dtype=float,
    )

    if range_rx_data.ndim != 2:
        raise ValueError(
            "range_rx_data must have shape "
            "(range bin, RX channel)."
        )

    if aperture_weights.shape != (
        range_rx_data.shape[1],
    ):
        raise ValueError(
            "aperture_weights must contain one coefficient "
            "for each RX channel."
        )

    if np.any(
        aperture_weights < 0
    ):
        raise ValueError(
            "The aperture weights must be nonnegative."
        )

    # Apply the real spatial-amplitude taper.
    weighted_range_rx_data = (
        range_rx_data
        * aperture_weights[
            np.newaxis,
            :,
        ]
    )

    # Reuse the existing physical steering-vector beamformer.
    return conventional_range_azimuth(
        range_rx_data=weighted_range_rx_data,
        azimuth_angle_deg=azimuth_angle_deg,
        rx_positions_m=rx_positions_m,
        carrier_frequency_hz=carrier_frequency_hz,
    )


def form_ideal_array_power_profile(
    target_angle_deg,
    evaluation_angle_deg,
    rx_positions_m,
    carrier_frequency_hz,
    aperture_weights,
):
    """
    Form the ideal normalized power response of the weighted
    RX array for one target angle.
    """

    wavelength_m = (
        299_792_458.0
        / carrier_frequency_hz
    )

    wavenumber_rad_per_m = (
        2 * np.pi
        / wavelength_m
    )

    target_angle_rad = np.deg2rad(
        target_angle_deg
    )

    # Create the ideal complex RX vector produced by a target
    # at the specified angle.
    ideal_target_rx_vector = np.exp(
        -1j
        * wavenumber_rad_per_m
        * rx_positions_m
        * np.sin(
            target_angle_rad
        )
    )

    ideal_response = (
        weighted_conventional_range_azimuth(
            range_rx_data=(
                ideal_target_rx_vector[
                    np.newaxis,
                    :,
                ]
            ),
            azimuth_angle_deg=(
                evaluation_angle_deg
            ),
            rx_positions_m=rx_positions_m,
            carrier_frequency_hz=(
                carrier_frequency_hz
            ),
            aperture_weights=aperture_weights,
        )[0]
    )

    ideal_power = np.abs(
        ideal_response
    ) ** 2

    ideal_power_db = 10 * np.log10(
        np.maximum(
            ideal_power
            / np.maximum(
                ideal_power.max(),
                np.finfo(float).tiny,
            ),
            np.finfo(float).tiny,
        )
    )

    return (
        ideal_power,
        ideal_power_db,
    )


def calculate_aperture_noise_penalty(
    normalized_weights,
):
    """
    Calculate the noise-power penalty relative to a
    unit-sum uniform aperture.

    For M elements:
        noise factor = M * sum(a_m^2)
    """

    normalized_weights = np.asarray(
        normalized_weights,
        dtype=float,
    )

    number_of_elements = (
        normalized_weights.size
    )

    noise_factor = (
        number_of_elements
        * np.sum(
            normalized_weights ** 2
        )
    )

    noise_penalty_db = 10 * np.log10(
        np.maximum(
            noise_factor,
            np.finfo(float).tiny,
        )
    )

    return (
        noise_factor,
        noise_penalty_db,
    )

### 2.10.2 — Compare Uniform, Hamming, and Hann Aperture Weighting

This cell compares three spatial-amplitude windows.

The ideal array-factor comparison uses a point target at boresight. The measured comparison uses the complex background-subtracted Run A data and combines power from range bins 20–24 when calculating the target-associated angular profile.

All aperture windows are normalized to unit sum. Therefore, differences in the plots represent changes in angular response rather than arbitrary differences in window scale.

The range–azimuth maps use the uniform-weighted map maximum as a common decibel reference.

In [ ]:
# ---------------------------------------------------------------------
# Define the aperture windows to compare.
# ---------------------------------------------------------------------

aperture_window_definitions = {
    "Uniform": "rectangular",
    "Hamming": "hamming",
    "Hann": "hann",
}

number_of_rx_elements = (
    background_subtracted_zero_doppler.shape[1]
)

ideal_target_angle_deg = 0.0

# Use a fine grid for measuring the theoretical array factor.
ideal_evaluation_angle_deg = np.linspace(
    -89.5,
    89.5,
    1791,
)

# Use the same measured target-associated range interval as
# the Section 2.9 diagnostic.
target_range_bins = np.arange(
    20,
    25,
)


# ---------------------------------------------------------------------
# Create and characterize each aperture window.
# ---------------------------------------------------------------------

aperture_weight_results = {}
aperture_summary_rows = []

for window_label, window_name in (
    aperture_window_definitions.items()
):
    raw_weights, normalized_weights = (
        create_normalized_aperture_weights(
            window_name=window_name,
            number_of_elements=(
                number_of_rx_elements
            ),
        )
    )

    noise_factor, noise_penalty_db = (
        calculate_aperture_noise_penalty(
            normalized_weights
        )
    )

    # Calculate the ideal array response.
    ideal_power, ideal_power_db = (
        form_ideal_array_power_profile(
            target_angle_deg=(
                ideal_target_angle_deg
            ),
            evaluation_angle_deg=(
                ideal_evaluation_angle_deg
            ),
            rx_positions_m=rx_positions_m,
            carrier_frequency_hz=(
                carrier_frequency_hz
            ),
            aperture_weights=(
                normalized_weights
            ),
        )
    )

    ideal_mainlobe_results = (
        measure_angular_mainlobe(
            azimuth_angle_deg=(
                ideal_evaluation_angle_deg
            ),
            angular_power_db=(
                ideal_power_db
            ),
            threshold_db=-3.0,
        )
    )

    # Beamform the measured complex residual using the same
    # aperture-amplitude coefficients.
    measured_range_azimuth_map = (
        weighted_conventional_range_azimuth(
            range_rx_data=(
                background_subtracted_zero_doppler
            ),
            azimuth_angle_deg=(
                azimuth_angle_deg
            ),
            rx_positions_m=rx_positions_m,
            carrier_frequency_hz=(
                carrier_frequency_hz
            ),
            aperture_weights=(
                normalized_weights
            ),
        )
    )

    measured_angular_power, measured_angular_power_db = (
        form_angular_power_profile(
            range_azimuth_map=(
                measured_range_azimuth_map
            ),
            selected_range_bins=(
                target_range_bins
            ),
        )
    )

    measured_mainlobe_results = (
        measure_angular_mainlobe(
            azimuth_angle_deg=(
                azimuth_angle_deg
            ),
            angular_power_db=(
                measured_angular_power_db
            ),
            threshold_db=-3.0,
        )
    )

    aperture_weight_results[
        window_label
    ] = {
        "window_name": window_name,
        "raw_weights": raw_weights,
        "normalized_weights": normalized_weights,
        "noise_factor": noise_factor,
        "noise_penalty_db": noise_penalty_db,
        "ideal_power_db": ideal_power_db,
        "ideal_mainlobe": (
            ideal_mainlobe_results
        ),
        "measured_map": (
            measured_range_azimuth_map
        ),
        "measured_angular_power_db": (
            measured_angular_power_db
        ),
        "measured_mainlobe": (
            measured_mainlobe_results
        ),
    }

    aperture_summary_rows.append(
        {
            "window": window_label,
            "normalized_weights": (
                np.array2string(
                    normalized_weights,
                    precision=3,
                    separator=", ",
                )
            ),
            "noise_penalty_db": (
                noise_penalty_db
            ),
            "ideal_peak_angle_deg": (
                ideal_mainlobe_results[
                    "peak_angle_deg"
                ]
            ),
            "ideal_minus_3db_width_deg": (
                ideal_mainlobe_results[
                    "mainlobe_width_deg"
                ]
            ),
            "measured_peak_angle_deg": (
                measured_mainlobe_results[
                    "peak_angle_deg"
                ]
            ),
            "measured_minus_3db_width_deg": (
                measured_mainlobe_results[
                    "mainlobe_width_deg"
                ]
            ),
        }
    )


# ---------------------------------------------------------------------
# Plot the spatial weights and angular profiles.
# ---------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5.5),
    constrained_layout=True,
)

window_colors = {
    "Uniform": "black",
    "Hamming": "tab:blue",
    "Hann": "tab:orange",
}


# Plot the normalized spatial-amplitude coefficients.
for window_label, result in (
    aperture_weight_results.items()
):
    axes[0].plot(
        np.arange(
            number_of_rx_elements
        ),
        result["normalized_weights"],
        marker="o",
        linewidth=2.0,
        color=window_colors[
            window_label
        ],
        label=window_label,
    )

axes[0].set_title(
    "Normalized RX-Aperture Weights"
)

axes[0].set_xlabel(
    "RX channel"
)

axes[0].set_ylabel(
    "Normalized amplitude coefficient"
)

axes[0].set_xticks(
    np.arange(
        number_of_rx_elements
    )
)

axes[0].grid(
    True,
    alpha=0.3,
)

axes[0].legend()


# Plot the theoretical array power responses.
for window_label, result in (
    aperture_weight_results.items()
):
    axes[1].plot(
        ideal_evaluation_angle_deg,
        result["ideal_power_db"],
        linewidth=2.0,
        color=window_colors[
            window_label
        ],
        label=(
            f"{window_label}: "
            f"{result['ideal_mainlobe']['mainlobe_width_deg']:.1f}°"
        ),
    )

axes[1].axhline(
    -3.0,
    color="tab:gray",
    linestyle="--",
    linewidth=1.0,
)

axes[1].set_title(
    "Ideal Four-RX Array Power Response"
)

axes[1].set_xlabel(
    "Azimuth angle (degrees)"
)

axes[1].set_ylabel(
    "Normalized power (dB)"
)

axes[1].set_xlim(
    -70,
    70,
)

axes[1].set_ylim(
    -40,
    3,
)

axes[1].grid(
    True,
    alpha=0.3,
)

axes[1].legend()


# Plot the measured target-associated angular responses.
for window_label, result in (
    aperture_weight_results.items()
):
    axes[2].plot(
        azimuth_angle_deg,
        result[
            "measured_angular_power_db"
        ],
        linewidth=2.0,
        color=window_colors[
            window_label
        ],
        label=(
            f"{window_label}: "
            f"{result['measured_mainlobe']['mainlobe_width_deg']:.1f}°"
        ),
    )

axes[2].axhline(
    -3.0,
    color="tab:gray",
    linestyle="--",
    linewidth=1.0,
)

axes[2].set_title(
    "Measured Target-Associated Angular Response"
)

axes[2].set_xlabel(
    "Azimuth angle (degrees, provisional sign)"
)

axes[2].set_ylabel(
    "Normalized power (dB)"
)

axes[2].set_xlim(
    -70,
    70,
)

axes[2].set_ylim(
    -30,
    3,
)

axes[2].grid(
    True,
    alpha=0.3,
)

axes[2].legend()


fig.suptitle(
    "Four-RX Aperture-Weighting Tradeoff",
    fontsize=15,
)

plt.show()


# ---------------------------------------------------------------------
# Compare the measured range–azimuth maps using one common reference.
# ---------------------------------------------------------------------

uniform_map = aperture_weight_results[
    "Uniform"
]["measured_map"]

common_map_reference = np.maximum(
    np.abs(
        uniform_map
    ).max(),
    np.finfo(float).tiny,
)


fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 6.5),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)


for ax, (
    window_label,
    result,
) in zip(
    axes,
    aperture_weight_results.items(),
):
    measured_map = result[
        "measured_map"
    ]

    measured_map_db = 20 * np.log10(
        np.maximum(
            np.abs(
                measured_map
            )
            / common_map_reference,
            np.finfo(float).tiny,
        )
    )

    measured_map_db = np.maximum(
        measured_map_db,
        -50.0,
    )

    image = ax.imshow(
        measured_map_db,
        origin="lower",
        aspect="auto",
        extent=[
            azimuth_angle_deg[0],
            azimuth_angle_deg[-1],
            -0.5,
            measured_map_db.shape[0] - 0.5,
        ],
        vmin=-50,
        vmax=0,
        cmap="viridis",
    )

    # Draw the expected sphere region as a red rectangle.
    expected_sphere_box = Rectangle(
        xy=(-10.0, 16.5),
        width=20.0,
        height=3.0,
        fill=False,
        edgecolor="red",
        linewidth=2.0,
        label=r"Expected $q=17$–$19$",
    )

    ax.add_patch(
        expected_sphere_box
    )

    # Mark the observed sphere-associated range bin.
    ax.axhline(
        22,
        color="cyan",
        linestyle="-.",
        linewidth=1.7,
        label=r"Observed $q=22$",
    )

    ax.axvline(
        0.0,
        color="white",
        linestyle="--",
        linewidth=1.1,
        alpha=0.8,
        label="Boresight",
    )

    ax.set_title(
        f"{window_label} Aperture Weighting"
    )

    ax.set_xlabel(
        "Azimuth angle (degrees, provisional sign)"
    )

    ax.set_xlim(
        -70,
        70,
    )

    ax.set_ylim(
        -0.5,
        50,
    )


axes[0].set_ylabel(
    "Range-bin index, $q$"
)

axes[0].legend(
    loc="upper right",
    fontsize=8,
)

colorbar = fig.colorbar(
    image,
    ax=axes,
    shrink=0.88,
    pad=0.02,
)

colorbar.set_label(
    "Magnitude relative to uniform-map maximum (dB)"
)

fig.suptitle(
    "Measured Run A Range–Azimuth Maps: "
    "Aperture-Weighting Comparison",
    fontsize=15,
)

plt.show()


# ---------------------------------------------------------------------
# Display the numerical comparison.
# ---------------------------------------------------------------------

aperture_weighting_summary = pd.DataFrame(
    aperture_summary_rows
)

display(
    aperture_weighting_summary
)

### 2.10 Results — Aperture-Weighting Tradeoff

#### What was changed

The aperture-weighting comparison changed only the real amplitude coefficient applied to each RX channel:

$$ X_w[q,m]=a_mX[q,m]. $$

The angle-dependent steering phases were unchanged. Therefore, the experiment isolates the effect of spatial-amplitude tapering on the angular point-spread function.

Each window was normalized so that:

$$ \sum_m a_m=1. $$

The normalized coefficients were:

| Window | Normalized RX coefficients |
|---|---|
| Uniform | $[0.25,\ 0.25,\ 0.25,\ 0.25]$ |
| Hamming | $[0.047,\ 0.453,\ 0.453,\ 0.047]$ |
| Hann | $[0,\ 0.5,\ 0.5,\ 0]$ |

Uniform weighting uses all four elements equally.

Hamming weighting places approximately 91% of the total weight on the two center elements. The two outer elements together contribute only approximately 9%.

Hann weighting completely removes the outer elements. The effective array therefore contains only RX 1 and RX 2.

---

#### Theoretical angular widths

The theoretical $-3\ \text{dB}$ widths are:

| Window | Ideal $-3\ \text{dB}$ width |
|---|---:|
| Uniform | $26.2^\circ$ |
| Hamming | $46.6^\circ$ |
| Hann | $59.6^\circ$ |

Uniform weighting produces the narrowest main lobe because it uses the complete four-element aperture.

Hamming weighting reduces the abrupt spatial discontinuity at the aperture edges, but it substantially reduces the contribution of the outer elements. The effective aperture becomes shorter and the main lobe broadens by approximately:

$$ 46.6^\circ-26.2^\circ=20.4^\circ. $$

Hann weighting removes both outer elements and produces the broadest response. Its angular width is more than twice the uniform width:

$$ \frac{59.6^\circ}{26.2^\circ}\approx2.28. $$

The uniform array factor exhibits distinct nulls and sidelobes. Hamming and Hann weighting suppress the distinct sidelobe structure, but the energy is redistributed into a much broader main lobe.

Therefore, sidelobe suppression does not come without a cost. A smoother-looking response can represent poorer angular resolution.

---

#### Measured angular widths

The measured target-associated widths are:

| Window | Measured peak angle | Measured $-3\ \text{dB}$ width |
|---|---:|---:|
| Uniform | $2.0^\circ$ | $25.5^\circ$ |
| Hamming | $1.0^\circ$ | $45.0^\circ$ |
| Hann | $-0.5^\circ$ | $60.0^\circ$ |

The measured and theoretical widths agree closely:

| Window | Ideal width | Measured width | Difference |
|---|---:|---:|---:|
| Uniform | $26.2^\circ$ | $25.5^\circ$ | $-0.7^\circ$ |
| Hamming | $46.6^\circ$ | $45.0^\circ$ | $-1.6^\circ$ |
| Hann | $59.6^\circ$ | $60.0^\circ$ | $+0.4^\circ$ |

This agreement is strong evidence that:

- the physical RX-element spacing is represented correctly;
- the steering-vector implementation is correct;
- the measured angular width is dominated by the expected array point-spread function;
- the broad response is not primarily caused by a software error or failed phase calibration.

---

#### Changes in measured peak angle

The measured peak changes from $2.0^\circ$ with uniform weighting to $1.0^\circ$ with Hamming weighting and $-0.5^\circ$ with Hann weighting.

This does not indicate that the physical target moved. Each window uses a different combination of the measured RX channels:

- uniform weighting uses all four channels equally;
- Hamming weighting is dominated by RX 1 and RX 2;
- Hann weighting uses only RX 1 and RX 2.

Because the channels contain small phase differences, amplitude differences, clutter, and possible multipath, changing their relative weights changes the angle that maximizes the coherent sum.

The Hamming and Hann profiles are also much broader. Their peak locations are consequently more sensitive to small changes in the measured channel vector.

The uniform-weight estimate remains the most appropriate angle estimate because it uses the complete physical aperture.

---

#### Noise penalty

For unit-sum aperture weights, the relative output-noise factor is:

$$ G_{\text{noise}}=M\sum_m a_m^2. $$

The measured window coefficients give:

| Window | Noise penalty |
|---|---:|
| Uniform | $0.00\ \text{dB}$ |
| Hamming | $2.20\ \text{dB}$ |
| Hann | $3.01\ \text{dB}$ |

Uniform weighting averages four independent RX noise contributions efficiently.

Hamming and Hann weighting concentrate most or all of the output on only two channels. After normalizing the coherent target gain, this increases the relative output-noise variance.

The Hann result has a $3\ \text{dB}$ penalty because it effectively averages two elements instead of four.

---

#### Interpretation of the measured maps

All three maps retain the dominant sphere-associated response near $q=22$. Aperture weighting does not change the target range because the weights operate only across RX channel index.

The uniform map provides the strongest angular localization. It produces a narrower bright region around boresight, along with visible sidelobe and null structure.

The Hamming map appears smoother, but the target response is distributed across a much larger angular interval.

The Hann map is smoothest but also most horizontally smeared. Because the outer RX elements receive zero weight, much of the available angular-resolution information has been discarded.

These results demonstrate that a visually smoother heatmap is not necessarily a better-resolved image.

---

#### Selection for the final Run A image

Uniform aperture weighting is selected for the final Run A range–azimuth image because it provides:

- the narrowest theoretical and measured angular main lobe;
- use of the complete four-element physical aperture;
- the smallest noise penalty;
- the most stable target-angle estimate;
- measured behavior that agrees closely with the theoretical array response.

Hamming and Hann weighting successfully demonstrate spatial sidelobe suppression, but their resolution and noise penalties are too large for this four-element array.

This conclusion does not mean that aperture tapering is generally undesirable. The V-MD3 physical array contains only four elements, so tapering quickly reduces the effective aperture to approximately two elements.

The SAR dataset contains 17 mechanical aperture positions. With more spatial samples, aperture weighting can reduce cross-range sidelobes without discarding such a large fraction of the effective aperture. Uniform, Hamming, and other tapers should therefore be reconsidered after forming the initial uniformly weighted SAR image.

---

#### Conclusion

The close agreement between theoretical and measured angular widths confirms that the Run A angular response is physically reasonable.

The broad horizontal target response is primarily the expected point-spread function of the four-RX aperture. It is not evidence that the beamformer requires additional phase calibration.

Uniform weighting will be retained for the final Run A image.

## 2.11 — Final Run A Range–Azimuth Image

The preceding sections established that:

- the Run A frames are phase coherent;
- the matched Run B background can be coherently aligned and subtracted;
- the sphere-associated response is near $q=22$;
- the four-RX phase progression is physically consistent;
- the conventional beamformer produces the expected angular point-spread function;
- uniform aperture weighting provides the best angular resolution for the four-element array.

The final Run A processing chain is:

$$ \text{RADC}\rightarrow\text{Hann range–Doppler processing}\rightarrow\text{zero-Doppler extraction}\rightarrow\text{coherent frame averaging}\rightarrow\text{aligned complex background subtraction}\rightarrow\text{uniform four-RX beamforming}. $$

The final image remains a range–azimuth point-spread response rather than a resolved geometric image of the sphere.

This section measures:

- the detected target-associated range bin;
- the nominal, uncalibrated range;
- the $-3\ \text{dB}$ range width;
- the beamformed peak angle;
- the $-3\ \text{dB}$ angular width;
- the spatial coherence;
- the RX amplitude spread;
- the peak-to-background contrast.

Because the range discrepancy remains unresolved, physical range values are labeled nominal and uncalibrated.

### 2.11.1 — Final-Image Measurement Functions

The following functions measure the target range width and calculate a conservative peak-to-background contrast.

The background reference is the 95th percentile of the image power outside the target-associated range interval. Using a percentile rather than the absolute background maximum prevents one isolated residual pixel from defining the complete background level.

In [ ]:
def measure_range_mainlobe(
    range_bin_index,
    range_power_db,
    target_search_bins,
    threshold_db=-3.0,
):
    """
    Find the target peak and estimate its contiguous range
    width at the selected level below the peak.

    Linear interpolation is used at the threshold crossings.
    """

    range_bin_index = np.asarray(
        range_bin_index,
        dtype=float,
    )

    range_power_db = np.asarray(
        range_power_db,
        dtype=float,
    )

    target_search_bins = np.asarray(
        target_search_bins,
        dtype=int,
    )

    peak_index = int(
        target_search_bins[
            np.argmax(
                range_power_db[
                    target_search_bins
                ]
            )
        ]
    )

    peak_level_db = (
        range_power_db[
            peak_index
        ]
    )

    threshold_level_db = (
        peak_level_db
        + threshold_db
    )

    left_inside_index = peak_index

    while (
        left_inside_index > 0
        and range_power_db[
            left_inside_index - 1
        ]
        >= threshold_level_db
    ):
        left_inside_index -= 1

    right_inside_index = peak_index

    while (
        right_inside_index
        < range_power_db.size - 1
        and range_power_db[
            right_inside_index + 1
        ]
        >= threshold_level_db
    ):
        right_inside_index += 1

    # Interpolate the left threshold crossing.
    if left_inside_index > 0:
        x0 = range_bin_index[
            left_inside_index - 1
        ]
        x1 = range_bin_index[
            left_inside_index
        ]

        y0 = range_power_db[
            left_inside_index - 1
        ]
        y1 = range_power_db[
            left_inside_index
        ]

        left_crossing = (
            x0
            + (
                threshold_level_db - y0
            )
            * (x1 - x0)
            / np.maximum(
                y1 - y0,
                np.finfo(float).tiny,
            )
        )
    else:
        left_crossing = range_bin_index[0]

    # Interpolate the right threshold crossing.
    if right_inside_index < range_power_db.size - 1:
        x0 = range_bin_index[
            right_inside_index
        ]
        x1 = range_bin_index[
            right_inside_index + 1
        ]

        y0 = range_power_db[
            right_inside_index
        ]
        y1 = range_power_db[
            right_inside_index + 1
        ]

        right_crossing = (
            x0
            + (
                threshold_level_db - y0
            )
            * (x1 - x0)
            / np.minimum(
                y1 - y0,
                -np.finfo(float).tiny,
            )
        )
    else:
        right_crossing = range_bin_index[-1]

    return {
        "peak_index": peak_index,
        "peak_bin": range_bin_index[
            peak_index
        ],
        "threshold_db": threshold_db,
        "left_crossing_bin": left_crossing,
        "right_crossing_bin": right_crossing,
        "width_bins": (
            right_crossing
            - left_crossing
        ),
    }


def calculate_map_peak_to_background_contrast(
    complex_map,
    target_range_bins,
    background_range_mask,
    background_percentile=95.0,
):
    """
    Calculate target peak power relative to a specified
    percentile of the background-region power.
    """

    target_range_bins = np.asarray(
        target_range_bins,
        dtype=int,
    )

    background_range_mask = np.asarray(
        background_range_mask,
        dtype=bool,
    )

    map_power = np.abs(
        complex_map
    ) ** 2

    target_peak_power = np.max(
        map_power[
            target_range_bins,
            :,
        ]
    )

    background_power = map_power[
        background_range_mask,
        :,
    ]

    background_reference_power = np.percentile(
        background_power,
        background_percentile,
    )

    contrast_db = 10 * np.log10(
        np.maximum(
            target_peak_power,
            np.finfo(float).tiny,
        )
        / np.maximum(
            background_reference_power,
            np.finfo(float).tiny,
        )
    )

    return {
        "target_peak_power": target_peak_power,
        "background_reference_power": (
            background_reference_power
        ),
        "background_percentile": (
            background_percentile
        ),
        "contrast_db": contrast_db,
    }

### 2.11.2 — Compare Minimal and Final Run A Processing

The minimally processed baseline uses:

- one Run A frame;
- rectangular range and Doppler windows;
- zero-Doppler extraction;
- uniform four-RX beamforming;
- no coherent frame averaging;
- no background subtraction.

The final processing uses:

- Hann range and Doppler windows;
- coherent averaging of all Run A frames;
- aligned complex Run B background subtraction;
- uniform four-RX beamforming.

Both images are beamformed. A range–angle image requires coherent combination across the RX channels to create the angle dimension. The comparison therefore isolates the improvements produced by spectral windowing, coherent frame averaging, and complex background subtraction.

Rectangular and Hann windows have different coherent gains. For a stationary signal located at the evaluated FFT bins, the two-dimensional coherent window gain is proportional to:

$$ G_w=\left(\sum_iw_r[i]\right)\left(\sum_nw_D[n]\right). $$

Each complex map is divided by its corresponding window gain before comparison. This prevents the lower coherent gain of the Hann window from being incorrectly interpreted as clutter suppression.

Both images use the minimally processed map maximum as the common $0\ \text{dB}$ reference.

In [ ]:
# ---------------------------------------------------------------------
# Define the nominal range mapping.
# These variables are also used by the following subsection.
# ---------------------------------------------------------------------

nominal_maximum_range_m = 6.0

number_of_range_bins = (
    run_a_radc.shape[1]
)

nominal_range_bin_spacing_m = (
    nominal_maximum_range_m
    / number_of_range_bins
)

range_bin_index = np.arange(
    number_of_range_bins
)


# ---------------------------------------------------------------------
# Select the final uniformly weighted Run A map.
# ---------------------------------------------------------------------

final_range_azimuth_map = (
    aperture_weight_results[
        "Uniform"
    ][
        "measured_map"
    ]
)

uniform_aperture_weights = (
    aperture_weight_results[
        "Uniform"
    ][
        "normalized_weights"
    ]
)


# ---------------------------------------------------------------------
# Recreate the minimally processed baseline.
# ---------------------------------------------------------------------

baseline_frame_index = 0

# Process one Run A frame using rectangular range and
# Doppler windows.
baseline_range_doppler_cube = (
    compute_radc_range_doppler(
        radc_frame=run_a_radc[
            baseline_frame_index
        ],
        range_window="rectangular",
        doppler_window="rectangular",
    )
)

# Extract the centered zero-Doppler complex RX data.
baseline_zero_doppler_rx, baseline_zero_doppler_bin = (
    extract_zero_doppler(
        baseline_range_doppler_cube,
        doppler_axis=1,
    )
)

# Apply the same uniformly weighted physical beamformer
# used for the final image.
baseline_range_azimuth_map = (
    weighted_conventional_range_azimuth(
        range_rx_data=(
            baseline_zero_doppler_rx
        ),
        azimuth_angle_deg=(
            azimuth_angle_deg
        ),
        rx_positions_m=rx_positions_m,
        carrier_frequency_hz=(
            carrier_frequency_hz
        ),
        aperture_weights=(
            uniform_aperture_weights
        ),
    )
)


# ---------------------------------------------------------------------
# Calculate the rectangular and Hann coherent window gains.
# ---------------------------------------------------------------------

number_of_fast_time_samples = (
    run_a_radc.shape[1]
)

number_of_chirps = (
    run_a_radc.shape[2]
)

rectangular_range_window = create_window(
    window_name="rectangular",
    length=number_of_fast_time_samples,
)

rectangular_doppler_window = create_window(
    window_name="rectangular",
    length=number_of_chirps,
)

hann_range_window = create_window(
    window_name="hann",
    length=number_of_fast_time_samples,
)

hann_doppler_window = create_window(
    window_name="hann",
    length=number_of_chirps,
)

rectangular_window_gain = (
    np.sum(
        rectangular_range_window
    )
    * np.sum(
        rectangular_doppler_window
    )
)

hann_window_gain = (
    np.sum(
        hann_range_window
    )
    * np.sum(
        hann_doppler_window
    )
)


# Correct both complex maps for their respective coherent
# window gains before comparing their magnitudes.
baseline_map_gain_corrected = (
    baseline_range_azimuth_map
    / rectangular_window_gain
)

final_map_gain_corrected = (
    final_range_azimuth_map
    / hann_window_gain
)


# ---------------------------------------------------------------------
# Convert both maps to dB using one common baseline reference.
# ---------------------------------------------------------------------

comparison_reference_magnitude = np.maximum(
    np.abs(
        baseline_map_gain_corrected
    ).max(),
    np.finfo(float).tiny,
)

baseline_comparison_map_db = 20 * np.log10(
    np.maximum(
        np.abs(
            baseline_map_gain_corrected
        )
        / comparison_reference_magnitude,
        np.finfo(float).tiny,
    )
)

final_comparison_map_db = 20 * np.log10(
    np.maximum(
        np.abs(
            final_map_gain_corrected
        )
        / comparison_reference_magnitude,
        np.finfo(float).tiny,
    )
)

baseline_comparison_map_db = np.maximum(
    baseline_comparison_map_db,
    -50.0,
)

final_comparison_map_db = np.maximum(
    final_comparison_map_db,
    -50.0,
)


# ---------------------------------------------------------------------
# Plot the minimally processed and final maps side by side.
# ---------------------------------------------------------------------

comparison_maps = {
    "Minimal Processing": (
        baseline_comparison_map_db
    ),
    "Final Processing": (
        final_comparison_map_db
    ),
}

comparison_subtitles = {
    "Minimal Processing": (
        "1 frame, rectangular windows,\n"
        "no background subtraction"
    ),
    "Final Processing": (
        "Hann windows, coherent averaging,\n"
        "aligned complex background subtraction"
    ),
}


fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 7),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)


for plot_index, (
    ax,
    (processing_label, map_db),
) in enumerate(
    zip(
        axes,
        comparison_maps.items(),
    )
):
    image = ax.imshow(
        map_db,
        origin="lower",
        aspect="auto",
        extent=[
            azimuth_angle_deg[0],
            azimuth_angle_deg[-1],
            -0.5,
            map_db.shape[0] - 0.5,
        ],
        vmin=-50,
        vmax=0,
        cmap="viridis",
    )

    expected_sphere_box = Rectangle(
        xy=(-10.0, 16.5),
        width=20.0,
        height=3.0,
        fill=False,
        edgecolor="red",
        linewidth=2.0,
        label=r"Expected $q=17$–$19$",
    )

    ax.add_patch(
        expected_sphere_box
    )

    ax.axhline(
        22,
        color="cyan",
        linestyle="-.",
        linewidth=1.7,
        label=r"Observed $q=22$",
    )

    ax.axvline(
        0.0,
        color="white",
        linestyle="--",
        linewidth=1.2,
        alpha=0.8,
        label="Boresight",
    )

    ax.set_title(
        f"{processing_label}\n"
        f"{comparison_subtitles[processing_label]}"
    )

    ax.set_xlabel(
        "Azimuth angle (degrees, provisional sign)"
    )

    ax.set_xlim(
        -70,
        70,
    )

    ax.set_ylim(
        -0.5,
        50,
    )

    if plot_index == 0:
        ax.legend(
            loc="upper right",
            fontsize=8,
        )


axes[0].set_ylabel(
    "Range-bin index, $q$"
)

colorbar = fig.colorbar(
    image,
    ax=axes,
    shrink=0.88,
    pad=0.02,
)

colorbar.set_label(
    "Window-gain-corrected magnitude relative to "
    "minimal-processing maximum (dB)"
)

fig.suptitle(
    "Run A Processing Improvement: "
    "Minimal Baseline versus Final Result",
    fontsize=15,
)

plt.show()


# ---------------------------------------------------------------------
# Display the processing differences.
# ---------------------------------------------------------------------

run_a_processing_comparison = pd.DataFrame(
    [
        {
            "processing": "Minimal",
            "frames": 1,
            "range_window": "Rectangular",
            "doppler_window": "Rectangular",
            "complex_background_subtraction": False,
            "rx_aperture_weighting": "Uniform",
        },
        {
            "processing": "Final",
            "frames": run_a_radc.shape[0],
            "range_window": "Hann",
            "doppler_window": "Hann",
            "complex_background_subtraction": True,
            "rx_aperture_weighting": "Uniform",
        },
    ]
)

display(
    run_a_processing_comparison
)


print(
    "Centered zero-Doppler bin:",
    baseline_zero_doppler_bin,
)

print(
    "Rectangular two-dimensional coherent window gain:",
    f"{rectangular_window_gain:.1f}",
)

print(
    "Hann two-dimensional coherent window gain:",
    f"{hann_window_gain:.1f}",
)

print(
    "Hann-to-rectangular gain correction:",
    f"{rectangular_window_gain / hann_window_gain:.3f}",
)

### 2.11.3 — Measure the Final Run A Image

The final uniformly weighted image is now measured quantitatively.

The range profile is extracted at the measured beamformed peak angle. Its $-3\ \text{dB}$ width is measured around the strongest response within range bins 15–26.

The angular profile combines power across target-associated range bins 20–24.

Peak-to-background contrast is calculated using the 95th-percentile image power from range bins 5–50 after excluding bins 15–26. This removes both the expected and observed target intervals from the background estimate while avoiding the strong close-range coupling region.

Because the range discrepancy remains unresolved, the reported physical range and range width use the nominal, uncalibrated V-MD3 mapping.

In [ ]:
# ---------------------------------------------------------------------
# Retrieve the final angular profile and its measured main lobe.
# ---------------------------------------------------------------------

final_angular_power_db = (
    aperture_weight_results[
        "Uniform"
    ][
        "measured_angular_power_db"
    ]
)

final_angular_mainlobe = (
    aperture_weight_results[
        "Uniform"
    ][
        "measured_mainlobe"
    ]
)


# ---------------------------------------------------------------------
# Determine the final target angle.
# ---------------------------------------------------------------------

final_peak_angle_index = int(
    np.argmax(
        final_angular_power_db
    )
)

final_peak_angle_deg = float(
    azimuth_angle_deg[
        final_peak_angle_index
    ]
)


# ---------------------------------------------------------------------
# Extract the range profile at the target-associated angle.
# ---------------------------------------------------------------------

final_range_power = np.abs(
    final_range_azimuth_map[
        :,
        final_peak_angle_index,
    ]
) ** 2

target_search_bins = np.arange(
    15,
    27,
)

target_range_reference_power = np.maximum(
    final_range_power[
        target_search_bins
    ].max(),
    np.finfo(float).tiny,
)

final_range_power_db = 10 * np.log10(
    np.maximum(
        final_range_power
        / target_range_reference_power,
        np.finfo(float).tiny,
    )
)


# Measure the target-associated range main lobe.
final_range_mainlobe = (
    measure_range_mainlobe(
        range_bin_index=range_bin_index,
        range_power_db=final_range_power_db,
        target_search_bins=target_search_bins,
        threshold_db=-3.0,
    )
)

final_peak_range_bin = int(
    final_range_mainlobe[
        "peak_bin"
    ]
)

final_nominal_peak_range_m = (
    final_peak_range_bin
    * nominal_range_bin_spacing_m
)

final_range_width_bins = (
    final_range_mainlobe[
        "width_bins"
    ]
)

final_nominal_range_width_m = (
    final_range_width_bins
    * nominal_range_bin_spacing_m
)


# ---------------------------------------------------------------------
# Calculate a conservative peak-to-background contrast.
# ---------------------------------------------------------------------

background_range_mask = np.zeros(
    final_range_azimuth_map.shape[0],
    dtype=bool,
)

# Include range bins 5–50.
background_range_mask[5:51] = True

# Exclude both the expected and observed target intervals.
background_range_mask[15:27] = False

final_contrast_result = (
    calculate_map_peak_to_background_contrast(
        complex_map=final_range_azimuth_map,
        target_range_bins=np.arange(
            20,
            25,
        ),
        background_range_mask=(
            background_range_mask
        ),
        background_percentile=95.0,
    )
)


# ---------------------------------------------------------------------
# Retrieve the remaining validated measurements.
# ---------------------------------------------------------------------

rx_amplitude_spread_db = (
    relative_rx_magnitude_db.max()
    - relative_rx_magnitude_db.min()
)

final_spatial_coherence = (
    rx_phase_diagnostic[
        "spatial_coherence"
    ]
)

final_angular_width_deg = (
    final_angular_mainlobe[
        "mainlobe_width_deg"
    ]
)


# Normalize the final map to its own maximum for display.
final_map_db = normalized_magnitude_db(
    final_range_azimuth_map,
    minimum_db=-50,
)


# ---------------------------------------------------------------------
# Plot the final map and its range and angular profiles.
# ---------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(19, 6.5),
    constrained_layout=True,
)


# Final range–azimuth map.
image = axes[0].imshow(
    final_map_db,
    origin="lower",
    aspect="auto",
    extent=[
        azimuth_angle_deg[0],
        azimuth_angle_deg[-1],
        -0.5,
        final_map_db.shape[0] - 0.5,
    ],
    vmin=-50,
    vmax=0,
    cmap="viridis",
)

expected_sphere_box = Rectangle(
    xy=(-10.0, 16.5),
    width=20.0,
    height=3.0,
    fill=False,
    edgecolor="red",
    linewidth=2.0,
    label=r"Expected $q=17$–$19$",
)

axes[0].add_patch(
    expected_sphere_box
)

axes[0].axhline(
    final_peak_range_bin,
    color="cyan",
    linestyle="-.",
    linewidth=1.7,
    label=(
        f"Measured $q={final_peak_range_bin}$"
    ),
)

axes[0].axvline(
    final_peak_angle_deg,
    color="white",
    linestyle="--",
    linewidth=1.2,
    label=(
        f"Peak angle: "
        f"{final_peak_angle_deg:.1f}°"
    ),
)

axes[0].scatter(
    final_peak_angle_deg,
    final_peak_range_bin,
    marker="o",
    s=80,
    facecolors="none",
    edgecolors="white",
    linewidths=1.8,
    zorder=4,
)

axes[0].set_title(
    "Final Run A Range–Azimuth Image"
)

axes[0].set_xlabel(
    "Azimuth angle (degrees, provisional sign)"
)

axes[0].set_ylabel(
    "Range-bin index, $q$"
)

axes[0].set_xlim(
    -70,
    70,
)

axes[0].set_ylim(
    -0.5,
    50,
)

axes[0].legend(
    loc="upper right",
    fontsize=8,
)

colorbar = fig.colorbar(
    image,
    ax=axes[0],
    pad=0.02,
)

colorbar.set_label(
    "Magnitude relative to final-map maximum (dB)"
)


# Range profile at the measured target angle.
axes[1].plot(
    range_bin_index,
    final_range_power_db,
    color="black",
    linewidth=2.0,
)

axes[1].axvline(
    final_peak_range_bin,
    color="cyan",
    linestyle="-.",
    linewidth=1.7,
    label=(
        f"Peak: $q={final_peak_range_bin}$"
    ),
)

axes[1].axvline(
    final_range_mainlobe[
        "left_crossing_bin"
    ],
    color="tab:green",
    linestyle=":",
    linewidth=1.5,
)

axes[1].axvline(
    final_range_mainlobe[
        "right_crossing_bin"
    ],
    color="tab:green",
    linestyle=":",
    linewidth=1.5,
)

axes[1].axhline(
    -3.0,
    color="tab:gray",
    linestyle="--",
    linewidth=1.2,
    label="-3 dB level",
)

axes[1].axvspan(
    final_range_mainlobe[
        "left_crossing_bin"
    ],
    final_range_mainlobe[
        "right_crossing_bin"
    ],
    color="tab:green",
    alpha=0.15,
    label=(
        f"Width: "
        f"{final_range_width_bins:.2f} bins"
    ),
)

axes[1].set_title(
    f"Range Profile at "
    f"{final_peak_angle_deg:.1f}°"
)

axes[1].set_xlabel(
    "Range-bin index, $q$"
)

axes[1].set_ylabel(
    "Normalized power (dB)"
)

axes[1].set_xlim(
    0,
    50,
)

axes[1].set_ylim(
    -50,
    3,
)

axes[1].grid(
    True,
    alpha=0.3,
)

axes[1].legend(
    loc="lower right",
    fontsize=8,
)


# Target-associated angular profile.
axes[2].plot(
    azimuth_angle_deg,
    final_angular_power_db,
    color="black",
    linewidth=2.0,
)

axes[2].axvline(
    final_peak_angle_deg,
    color="tab:red",
    linestyle="-.",
    linewidth=1.7,
    label=(
        f"Peak: "
        f"{final_peak_angle_deg:.1f}°"
    ),
)

axes[2].axhline(
    -3.0,
    color="tab:gray",
    linestyle="--",
    linewidth=1.2,
    label="-3 dB level",
)

axes[2].axvspan(
    final_angular_mainlobe[
        "left_angle_deg"
    ],
    final_angular_mainlobe[
        "right_angle_deg"
    ],
    color="tab:green",
    alpha=0.15,
    label=(
        f"Width: "
        f"{final_angular_width_deg:.1f}°"
    ),
)

axes[2].set_title(
    "Target-Associated Angular Profile"
)

axes[2].set_xlabel(
    "Azimuth angle (degrees, provisional sign)"
)

axes[2].set_ylabel(
    "Normalized power (dB)"
)

axes[2].set_xlim(
    -70,
    70,
)

axes[2].set_ylim(
    -30,
    3,
)

axes[2].grid(
    True,
    alpha=0.3,
)

axes[2].legend(
    loc="lower right",
    fontsize=8,
)


fig.suptitle(
    "Final Run A Four-RX Processing Result",
    fontsize=15,
)

plt.show()


# ---------------------------------------------------------------------
# Display the final Run A measurement summary.
# ---------------------------------------------------------------------

final_run_a_summary = pd.DataFrame(
    [
        {
            "detected_range_bin": (
                final_peak_range_bin
            ),
            "nominal_uncalibrated_range_m": (
                final_nominal_peak_range_m
            ),
            "minus_3db_range_width_bins": (
                final_range_width_bins
            ),
            "nominal_range_width_m": (
                final_nominal_range_width_m
            ),
            "peak_angle_deg": (
                final_peak_angle_deg
            ),
            "minus_3db_angular_width_deg": (
                final_angular_width_deg
            ),
            "spatial_coherence": (
                final_spatial_coherence
            ),
            "rx_amplitude_spread_db": (
                rx_amplitude_spread_db
            ),
            "peak_to_95th_percentile_background_db": (
                final_contrast_result[
                    "contrast_db"
                ]
            ),
        }
    ]
)

display(
    final_run_a_summary
)

### 2.11.4 — Effect of Display Scaling on Apparent Image Contrast

The same final complex range–azimuth map is displayed using three visualization choices:

1. a $50\ \text{dB}$ logarithmic range;
2. a tighter $25\ \text{dB}$ logarithmic range;
3. normalized linear magnitude similar to the live-script input.

No radar processing changes between the panels. Differences in apparent background cleanliness are caused only by magnitude scaling, clipping, and colormap selection.

A logarithmic display reveals weak responses and is preferred for technical analysis. A linear display suppresses low-level structure visually and can produce a cleaner-looking image, but it does not improve target-to-background contrast.

In [ ]:
# Convert the final complex map to normalized linear magnitude.
final_display_magnitude = np.abs(
    final_range_azimuth_map
)

final_display_magnitude /= np.maximum(
    final_display_magnitude.max(),
    np.finfo(float).tiny,
)

# Convert the same normalized magnitude to decibels.
final_display_db = 20 * np.log10(
    np.maximum(
        final_display_magnitude,
        np.finfo(float).tiny,
    )
)


fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 6.5),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)


display_configurations = [
    {
        "title": "Logarithmic Display\n50 dB Dynamic Range",
        "data": final_display_db,
        "vmin": -50,
        "vmax": 0,
        "cmap": "viridis",
        "colorbar_label": "Normalized magnitude (dB)",
    },
    {
        "title": "Logarithmic Display\n25 dB Dynamic Range",
        "data": final_display_db,
        "vmin": -25,
        "vmax": 0,
        "cmap": "inferno",
        "colorbar_label": "Normalized magnitude (dB)",
    },
    {
        "title": "Linear-Magnitude Display\nLive-Script Style",
        "data": final_display_magnitude,
        "vmin": 0,
        "vmax": 1,
        "cmap": "inferno",
        "colorbar_label": "Normalized linear magnitude",
    },
]


for ax, display_configuration in zip(
    axes,
    display_configurations,
):
    image = ax.imshow(
        display_configuration[
            "data"
        ],
        origin="lower",
        aspect="auto",
        extent=[
            azimuth_angle_deg[0],
            azimuth_angle_deg[-1],
            -0.5,
            final_display_magnitude.shape[0] - 0.5,
        ],
        vmin=display_configuration[
            "vmin"
        ],
        vmax=display_configuration[
            "vmax"
        ],
        cmap=display_configuration[
            "cmap"
        ],
    )

    expected_sphere_box = Rectangle(
        xy=(-10.0, 16.5),
        width=20.0,
        height=3.0,
        fill=False,
        edgecolor="red",
        linewidth=2.0,
        label=r"Expected $q=17$–$19$",
    )

    ax.add_patch(
        expected_sphere_box
    )

    ax.axhline(
        22,
        color="cyan",
        linestyle="-.",
        linewidth=1.5,
        label=r"Observed $q=22$",
    )

    ax.axvline(
        0.0,
        color="white",
        linestyle="--",
        linewidth=1.0,
        alpha=0.8,
    )

    ax.set_title(
        display_configuration[
            "title"
        ]
    )

    ax.set_xlabel(
        "Azimuth angle (degrees, provisional sign)"
    )

    ax.set_xlim(
        -70,
        70,
    )

    ax.set_ylim(
        -0.5,
        50,
    )

    colorbar = fig.colorbar(
        image,
        ax=ax,
        pad=0.02,
    )

    colorbar.set_label(
        display_configuration[
            "colorbar_label"
        ]
    )


axes[0].set_ylabel(
    "Range-bin index, $q$"
)

fig.suptitle(
    "Identical Final Run A Data with Different Display Scaling",
    fontsize=15,
)

plt.show()

# 3.0 — Run C Rail-SAR Processing

## Objective

Run C contains the metal sphere near the center of the scene while the radar was measured at 17 positions along the Galil rail.

Run B contains the corresponding no-target background measurements at the same 17 rail positions.

The first SAR image will use:

- Run C as the target-present measurement;
- Run B as the matched background;
- all 17 mechanical aperture positions;
- one RX channel;
- Hann range and Doppler processing;
- coherent frame averaging;
- matched complex background subtraction;
- uniformly weighted backprojection.

The initial objective is to determine whether the centered sphere focuses near $x=0$ with substantially better cross-range localization than the four-RX range–azimuth image.

---

## Physical array beamforming versus SAR

The Run A range–azimuth image used four simultaneous RX channels. For each range bin, the channels were phase-aligned and summed over trial angles:

$$ Y[q,\theta]=\sum_{\ell=0}^{3}X[q,\ell]w_\ell^*(\theta), $$

where $\ell$ is RX-channel index.

SAR applies the same fundamental coherent-summation idea across mechanical rail positions rather than only across the four RX elements.

For SAR, let:

- $m$ denote rail-position index;
- $x_m$ denote the physical rail position;
- $q$ denote range-bin index;
- $\ell$ denote RX-channel index;
- $G[m,q,\ell]$ denote the complex background-subtracted range data.

The first reconstruction will select one RX channel $\ell_0$, producing:

$$ G[m,q]=G[m,q,\ell_0]. $$

The 17 values of $m$ then form the synthetic aperture.

---

## Rail geometry

Runs B, C, and D contain 17 position indices:

$$ m=0,1,\ldots,16. $$

The corresponding Galil encoder counts range from 0 to 800,000 in increments of 50,000. The center position is:

$$ m=8,\qquad \text{encoder count}=400{,}000. $$

The complete physical aperture length is approximately:

$$ L=0.10\ \text{m}. $$

Taking the center position as $x=0$, the rail coordinate is:

$$ x_m=-0.05+\frac{\text{encoder count}_m}{800{,}000}(0.10). $$

Because there are 16 intervals across the $10\ \text{cm}$ aperture, the physical position increment is:

$$ \Delta x=\frac{0.10}{16}=0.00625\ \text{m}=6.25\ \text{mm}. $$

The resulting rail positions extend from:

$$ -5\ \text{cm}\leq x_m\leq+5\ \text{cm}. $$

---

## SAR image coordinates

The first SAR image will use coordinates $(x,z)$:

- $x$ is cross-range, parallel to the Galil rail;
- $z$ is downrange in the two-dimensional imaging plane;
- the radar phase center at the middle rail position is the coordinate origin;
- the centered sphere is expected near $x=0$.

Here, $z$ represents the slant-range direction in the selected imaging plane. It should not be interpreted as ground range or vertical height.

The initial image region will cover approximately:

$$ -0.05\ \text{m}\leq x\leq+0.05\ \text{m}, $$

and:

$$ 0.75\ \text{m}\leq z\leq1.10\ \text{m}. $$

The downrange interval includes both the physically measured sphere range near $0.84\ \text{m}$ and the nominal radar response near $1.03\ \text{m}$.

---

## Complex range profile at each rail position

At every Run C position, each RADC frame is processed using:

1. a Hann window across fast time;
2. a range FFT;
3. a Hann window across chirp index;
4. a Doppler FFT;
5. centered zero-Doppler extraction.

For rail position $m$, frame $f$, range bin $q$, and RX channel $\ell$, let the resulting complex data be:

$$ C_f[m,q,\ell]. $$

Because the repeated frames are phase stable, they can be coherently averaged:

$$ \overline{C}[m,q,\ell]=\frac{1}{N_C}\sum_{f=0}^{N_C-1}C_f[m,q,\ell]. $$

The matched Run B background is processed identically:

$$ \overline{B}[m,q,\ell]=\frac{1}{N_B}\sum_{f=0}^{N_B-1}B_f[m,q,\ell]. $$

A complex coefficient $\alpha_{m,\ell}$ aligns Run B to Run C at each rail position and RX channel. The target-associated residual is:

$$ G[m,q,\ell]=\overline{C}[m,q,\ell]-\alpha_{m,\ell}\overline{B}[m,q,\ell]. $$

Subtraction is performed before magnitude calculation so that the phase required for SAR is preserved.

---

## Why the first image uses one RX channel

The four RX channels have slightly different physical phase centers and measured amplitude responses.

Combining all four channels in SAR requires either:

- modeling each RX phase center separately;
- modeling the transmitter and receiver locations as a bistatic geometry;
- or independently reconstructing each RX channel before coherent combination.

The first reconstruction therefore uses one RX channel:

$$ G[m,q]=G[m,q,\ell_0]. $$

This isolates the phase progression created by mechanical radar motion and provides the clearest validation of the basic backprojection geometry.

After the one-channel image is validated, the remaining RX channels can be added using their appropriate phase-center geometry.

---

## Expected range-bin behavior across the aperture

For a centered target at physical range $R_0$, the distance from rail position $x_m$ is approximately:

$$ R_m=\sqrt{R_0^2+x_m^2}. $$

Using:

$$ R_0=0.84\ \text{m},\qquad |x_m|\leq0.05\ \text{m}, $$

the center-to-edge range change is only:

$$ \Delta R=\sqrt{0.84^2+0.05^2}-0.84\approx1.5\ \text{mm}. $$

This is much smaller than the nominal range-bin spacing:

$$ \Delta R_{\text{bin}}=46.875\ \text{mm}. $$

Therefore, the sphere may remain in approximately the same range bin at every rail position. A visibly curved range-migration track is not required for successful focusing.

Although the range change is small relative to one range bin, it is significant relative to the wavelength. The corresponding round-trip phase change is approximately:

$$ \Delta\phi=\frac{4\pi\Delta R}{\lambda}. $$

Using $\lambda\approx4.92\ \text{mm}$ gives:

$$ \Delta\phi\approx218^\circ. $$

The useful SAR information is therefore contained primarily in the complex phase progression across the rail, even when the magnitude peak does not visibly migrate between range bins.

---

## Backprojection geometry

For a candidate image point $(x,z)$ and rail position $x_m$, the monostatic propagation distance is:

$$ R_m(x,z)=\sqrt{(x-x_m)^2+z^2}. $$

A monostatic radar signal travels from the radar to the target and back. The corresponding propagation phase is approximately:

$$ \phi_m(x,z)=-\frac{4\pi}{\lambda}R_m(x,z). $$

To focus a candidate pixel, backprojection applies the opposite phase:

$$ \exp\left(+j\frac{4\pi}{\lambda}R_m(x,z)\right). $$

The uniformly weighted backprojection image is:

$$ I_{\text{BP}}(x,z)=\sum_{m=0}^{M-1}G_m\!\left(R_m(x,z)\right)\exp\left(+j\frac{4\pi}{\lambda}R_m(x,z)\right). $$

Here, $G_m(R_m)$ is the complex range profile from rail position $m$, interpolated at the range corresponding to the candidate image point.

At the correct pixel location:

- the sampled ranges correspond to the target path;
- the phase corrections align the 17 complex measurements;
- the measurements add coherently.

At an incorrect location:

- the predicted path lengths are incorrect;
- residual phase differences remain;
- the measurements partially cancel.

Backprojection is therefore a spatial matched filter over candidate image locations.

---

## Phase-sign convention

The required compensation sign depends on the stored complex-IQ convention and the direction of the Fourier transforms.

The physical model predicts a compensation term proportional to:

$$ \exp\left(+j\frac{4\pi}{\lambda}R_m\right). $$

However, an opposite stored-IQ convention can reverse the measured phase sign.

The first Run C reconstruction will therefore evaluate both phase-compensation signs. The correct convention should produce:

- a sharper centered response;
- greater coherent gain;
- smaller cross-range width;
- a peak near the known target position.

After selecting the sign using centered Run C, that sign will be locked and independently tested using the shifted-target Run D measurement.

---

## Provisional treatment of the range discrepancy

Run A established that the dominant sphere-associated response occurs near:

$$ q_{\text{observed}}\approx22, $$

while the physical sphere range near $0.84\ \text{m}$ nominally corresponds to:

$$ q_{\text{expected}}\approx18. $$

A one-point measurement cannot determine whether this discrepancy is caused by a fixed offset, range-scale error, or multipath.

Before backprojection, Run C will be inspected to determine its observed sphere-associated range bin.

If Run C also places the centered sphere near $q=22$, the first reconstruction may use the local registration:

$$ q_m(x,z)=q_{\text{ref}}+\frac{R_m(x,z)-R_{\text{ref}}}{\Delta R_{\text{bin}}}, $$

with:

$$ q_{\text{ref}}=22,\qquad R_{\text{ref}}=0.84\ \text{m}. $$

This registration associates the observed range bin with the independently measured center-target distance while retaining the nominal local range-bin spacing.

It is only a provisional image-formation registration. It is not a demonstrated calibration of the complete radar range axis.

---

## Expected SAR resolution

For a broadside synthetic aperture of length $L$, the approximate cross-range resolution is:

$$ \delta_x\approx\frac{\lambda R_0}{2L}. $$

Using:

$$ \lambda\approx4.92\ \text{mm},\qquad R_0\approx0.84\ \text{m},\qquad L=0.10\ \text{m}, $$

gives:

$$ \delta_x\approx2.1\ \text{cm}. $$

The approximate range resolution remains:

$$ \delta_R\approx4.7\ \text{cm}. $$

SAR is therefore expected to improve cross-range localization substantially, but it does not create additional range bandwidth.

---

## Expected image appearance

A successful centered-target Run C image should show:

- a dominant response near $x=0$;
- a cross-range width on the order of $2$–$3\ \text{cm}$ for a point-like response;
- a range width on the order of the existing FMCW range resolution or larger;
- cross-range sidelobes from the uniformly weighted finite aperture;
- substantially less horizontal smearing than the four-RX range–azimuth image.

The sphere should not necessarily appear as a filled circle. A radar image represents scattering strength convolved with the SAR point-spread function. The sphere may appear as one bright blob, a surface-associated response, or multiple features if multipath is present.

---

## Run C processing sequence

The Run C SAR workflow will proceed in the following order:

1. Load and validate all Run B and Run C RADC measurements.
2. Convert encoder counts to physical rail coordinates.
3. Form Hann-windowed complex zero-Doppler range profiles.
4. Verify frame coherence at every rail position.
5. Coherently average repeated frames.
6. Align and subtract the matched Run B background at each position.
7. Select one RX channel for the first reconstruction.
8. Plot the aperture–range magnitude data.
9. Plot the target-bin magnitude and unwrapped phase across the rail.
10. Compare the measured aperture phase with the expected centered-target phase.
11. Define the SAR image grid.
12. Interpolate the complex range profile for each candidate pixel.
13. Apply round-trip phase compensation.
14. Sum coherently across the 17 rail positions.
15. Compare both candidate phase signs.
16. Measure the focused peak location and cross-range width.
17. Lock the validated processing convention for Run D.

The magnitude-only range–angle heatmap is not used as the SAR input. SAR requires the complex range profiles so that phase alignment can occur across the mechanical aperture.

## 3.1 — Load and Validate Runs B and C Across the Rail

Run B and Run C each contain 17 acquisition files, one for every Galil position.

For this first SAR reconstruction:

- Run B provides the matched no-target background;
- Run C provides the centered-sphere scene;
- only the RADC and DONE streams are loaded;
- the V-MD3 FPGA RFFT output is retained as a later cross-check but is not required for the first reconstruction.

The loaded RADC arrays use the convention:

$$ s[m,f,i,n,\ell], $$

where:

- $m$ is rail-position index;
- $f$ is repeated-frame index;
- $i$ is fast-time sample index;
- $n$ is chirp index;
- $\ell$ is RX-channel index.

The expected array shape for each run is:

```text
(17 rail positions, 20 frames, 128 samples, 64 chirps, 4 RX channels)

### 3.1.1 — Multi-Position Loading and Validation Functions

The following functions build the physical rail-position table, load one complete SAR run, and validate its multi-position data arrays.

The data remain complex throughout loading and validation.

In [ ]:
def build_sar_position_table(
    file_manifest,
    runs=("B", "C"),
    aperture_length_m=0.10,
    center_position_index=8,
):
    """
    Build and validate the physical rail-position table.

    Encoder counts from every requested run must agree at
    every position index.
    """

    run_position_tables = {}

    for run in runs:
        run = run.upper()

        run_rows = (
            file_manifest.loc[
                file_manifest["run"] == run,
                [
                    "position_index",
                    "encoder_count",
                ],
            ]
            .drop_duplicates()
            .sort_values(
                "position_index"
            )
            .reset_index(
                drop=True
            )
        )

        if run_rows.empty:
            raise ValueError(
                f"No position records were found for Run {run}."
            )

        run_rows[
            "position_index"
        ] = run_rows[
            "position_index"
        ].astype(int)

        run_rows[
            "encoder_count"
        ] = run_rows[
            "encoder_count"
        ].astype(int)

        run_position_tables[
            run
        ] = run_rows

    # Use the first requested run as the reference position table.
    reference_run = runs[0].upper()

    reference_table = (
        run_position_tables[
            reference_run
        ].copy()
    )

    expected_position_indices = np.asarray(
        SAR_POSITION_INDICES,
        dtype=int,
    )

    if not np.array_equal(
        reference_table[
            "position_index"
        ].to_numpy(),
        expected_position_indices,
    ):
        raise ValueError(
            f"Run {reference_run} does not contain the "
            "expected ordered SAR position indices."
        )

    # Confirm that all requested runs use identical encoder counts.
    for run in runs[1:]:
        run = run.upper()

        comparison_table = (
            run_position_tables[
                run
            ]
        )

        if not np.array_equal(
            comparison_table[
                "position_index"
            ].to_numpy(),
            reference_table[
                "position_index"
            ].to_numpy(),
        ):
            raise ValueError(
                f"Run {run} position indices do not match "
                f"Run {reference_run}."
            )

        if not np.array_equal(
            comparison_table[
                "encoder_count"
            ].to_numpy(),
            reference_table[
                "encoder_count"
            ].to_numpy(),
        ):
            raise ValueError(
                f"Run {run} encoder counts do not match "
                f"Run {reference_run}."
            )

    center_rows = reference_table.loc[
        reference_table[
            "position_index"
        ]
        == int(
            center_position_index
        )
    ]

    if len(center_rows) != 1:
        raise ValueError(
            "The center position index does not identify "
            "exactly one encoder count."
        )

    center_encoder_count = int(
        center_rows.iloc[0][
            "encoder_count"
        ]
    )

    minimum_encoder_count = int(
        reference_table[
            "encoder_count"
        ].min()
    )

    maximum_encoder_count = int(
        reference_table[
            "encoder_count"
        ].max()
    )

    encoder_span = (
        maximum_encoder_count
        - minimum_encoder_count
    )

    if encoder_span <= 0:
        raise ValueError(
            "The encoder-count span must be positive."
        )

    meters_per_encoder_count = (
        aperture_length_m
        / encoder_span
    )

    reference_table[
        "rail_position_m"
    ] = (
        reference_table[
            "encoder_count"
        ]
        - center_encoder_count
    ) * meters_per_encoder_count

    reference_table[
        "rail_position_cm"
    ] = (
        100
        * reference_table[
            "rail_position_m"
        ]
    )

    rail_position_differences_m = np.diff(
        reference_table[
            "rail_position_m"
        ].to_numpy()
    )

    if not np.allclose(
        rail_position_differences_m,
        rail_position_differences_m[0],
    ):
        raise ValueError(
            "The calculated rail positions are not uniformly spaced."
        )

    reference_table.attrs[
        "aperture_length_m"
    ] = aperture_length_m

    reference_table.attrs[
        "center_position_index"
    ] = center_position_index

    reference_table.attrs[
        "center_encoder_count"
    ] = center_encoder_count

    reference_table.attrs[
        "meters_per_encoder_count"
    ] = meters_per_encoder_count

    return reference_table


def load_sar_radc_run(
    record_manifest,
    run,
    position_table,
):
    """
    Load all RADC and DONE measurements for one SAR run.

    Returns
    -------
    radc_stack
        Shape:
        (position, frame, sample, chirp, RX channel)

    done_stack
        Shape:
        (position, frame)
    """

    run = run.upper()

    radc_by_position = []
    done_by_position = []

    expected_frame_count = None

    for position_index in position_table[
        "position_index"
    ].astype(int):
        position_radc = load_radc_stack(
            record_manifest,
            run=run,
            position_index=position_index,
        )

        position_done = load_done_values(
            record_manifest,
            run=run,
            position_index=position_index,
        )

        if expected_frame_count is None:
            expected_frame_count = (
                position_radc.shape[0]
            )

        if (
            position_radc.shape[0]
            != expected_frame_count
        ):
            raise ValueError(
                f"Run {run}, position {position_index}: "
                "RADC frame count differs from the "
                "other positions."
            )

        if (
            position_done.shape[0]
            != expected_frame_count
        ):
            raise ValueError(
                f"Run {run}, position {position_index}: "
                "DONE count does not match the RADC count."
            )

        radc_by_position.append(
            position_radc
        )

        done_by_position.append(
            position_done
        )

    radc_stack = np.stack(
        radc_by_position,
        axis=0,
    )

    done_stack = np.stack(
        done_by_position,
        axis=0,
    )

    return (
        radc_stack,
        done_stack,
    )


def validate_sar_radc_run(
    radc_stack,
    done_stack,
    position_table,
    label,
):
    """
    Validate one complete multi-position RADC run and return
    a position-level summary table.
    """

    expected_number_of_positions = len(
        position_table
    )

    if radc_stack.ndim != 5:
        raise ValueError(
            f"{label} RADC must have shape "
            "(position, frame, sample, chirp, RX)."
        )

    if (
        radc_stack.shape[0]
        != expected_number_of_positions
    ):
        raise ValueError(
            f"{label} contains {radc_stack.shape[0]} "
            "positions; expected "
            f"{expected_number_of_positions}."
        )

    if (
        radc_stack.shape[2:]
        != RADC_FRAME_SHAPE
    ):
        raise ValueError(
            f"{label} has per-frame shape "
            f"{radc_stack.shape[2:]}; expected "
            f"{RADC_FRAME_SHAPE}."
        )

    expected_done_shape = (
        radc_stack.shape[0],
        radc_stack.shape[1],
    )

    if done_stack.shape != expected_done_shape:
        raise ValueError(
            f"{label} DONE shape is {done_stack.shape}; "
            f"expected {expected_done_shape}."
        )

    if not np.iscomplexobj(
        radc_stack
    ):
        raise ValueError(
            f"{label} RADC data are not complex."
        )

    if not np.all(
        np.isfinite(
            radc_stack
        )
    ):
        raise ValueError(
            f"{label} RADC data contain nonfinite values."
        )

    # DONE values must be consecutive within every file.
    done_differences = np.diff(
        done_stack.astype(
            np.int64
        ),
        axis=1,
    )

    if not np.all(
        done_differences == 1
    ):
        raise ValueError(
            f"{label} contains nonconsecutive DONE values."
        )

    validation_rows = []

    for position_array_index, position_row in enumerate(
        position_table.itertuples(
            index=False
        )
    ):
        position_radc = radc_stack[
            position_array_index
        ]

        position_rms = np.sqrt(
            np.mean(
                np.abs(
                    position_radc
                ) ** 2
            )
        )

        validation_rows.append(
            {
                "run": label,
                "position_index": int(
                    position_row.position_index
                ),
                "encoder_count": int(
                    position_row.encoder_count
                ),
                "rail_position_cm": float(
                    position_row.rail_position_cm
                ),
                "frame_count": int(
                    position_radc.shape[0]
                ),
                "done_first": int(
                    done_stack[
                        position_array_index,
                        0,
                    ]
                ),
                "done_last": int(
                    done_stack[
                        position_array_index,
                        -1,
                    ]
                ),
                "radc_rms": float(
                    position_rms
                ),
                "exact_zero_fraction": float(
                    np.mean(
                        position_radc == 0
                    )
                ),
            }
        )

    return pd.DataFrame(
        validation_rows
    )

### 3.1.2 — Load and Validate the Run B and Run C RADC Data

This cell constructs the physical rail coordinates and loads the complete Run B and Run C RADC datasets.

The arrays are retained in complex form for subsequent range processing, background subtraction, aperture-phase analysis, and backprojection.

In [ ]:
# ---------------------------------------------------------------------
# Build the common Run B/Run C physical rail-position table.
# ---------------------------------------------------------------------

sar_aperture_length_m = 0.10

sar_position_table = (
    build_sar_position_table(
        file_manifest=FILE_MANIFEST,
        runs=("B", "C"),
        aperture_length_m=(
            sar_aperture_length_m
        ),
        center_position_index=(
            CENTER_POSITION_INDEX
        ),
    )
)

rail_positions_m = (
    sar_position_table[
        "rail_position_m"
    ].to_numpy(
        dtype=float
    )
)

rail_position_step_m = float(
    np.diff(
        rail_positions_m
    )[0]
)


print("SAR rail geometry")
print(
    f"  Number of positions: "
    f"{rail_positions_m.size}"
)
print(
    f"  First position:       "
    f"{rail_positions_m[0]:+.5f} m"
)
print(
    f"  Center position:      "
    f"{rail_positions_m[CENTER_POSITION_INDEX]:+.5f} m"
)
print(
    f"  Final position:       "
    f"{rail_positions_m[-1]:+.5f} m"
)
print(
    f"  Position increment:   "
    f"{rail_position_step_m:.5f} m"
)
print(
    f"  Aperture length:      "
    f"{rail_positions_m[-1] - rail_positions_m[0]:.5f} m"
)

display(
    sar_position_table
)


# ---------------------------------------------------------------------
# Load Run B: matched no-target background at all positions.
# ---------------------------------------------------------------------

print("Loading Run B RADC data...")

run_b_sar_radc, run_b_sar_done = (
    load_sar_radc_run(
        record_manifest=RECORD_MANIFEST,
        run="B",
        position_table=(
            sar_position_table
        ),
    )
)

print(
    "Run B loading complete:",
    run_b_sar_radc.shape,
)


# ---------------------------------------------------------------------
# Load Run C: centered sphere at all positions.
# ---------------------------------------------------------------------

print("Loading Run C RADC data...")

run_c_sar_radc, run_c_sar_done = (
    load_sar_radc_run(
        record_manifest=RECORD_MANIFEST,
        run="C",
        position_table=(
            sar_position_table
        ),
    )
)

print(
    "Run C loading complete:",
    run_c_sar_radc.shape,
)


# ---------------------------------------------------------------------
# Validate the complete Run B and Run C arrays.
# ---------------------------------------------------------------------

run_b_sar_validation = (
    validate_sar_radc_run(
        radc_stack=run_b_sar_radc,
        done_stack=run_b_sar_done,
        position_table=(
            sar_position_table
        ),
        label="B",
    )
)

run_c_sar_validation = (
    validate_sar_radc_run(
        radc_stack=run_c_sar_radc,
        done_stack=run_c_sar_done,
        position_table=(
            sar_position_table
        ),
        label="C",
    )
)


# Confirm that both runs have identical array dimensions.
assert (
    run_b_sar_radc.shape
    == run_c_sar_radc.shape
), (
    "Run B and Run C RADC shapes do not match: "
    f"{run_b_sar_radc.shape} versus "
    f"{run_c_sar_radc.shape}."
)

assert (
    run_b_sar_done.shape
    == run_c_sar_done.shape
), (
    "Run B and Run C DONE shapes do not match."
)


# Confirm the intended physical geometry.
assert np.isclose(
    rail_positions_m[
        CENTER_POSITION_INDEX
    ],
    0.0,
)

assert np.isclose(
    rail_positions_m[0],
    -0.05,
)

assert np.isclose(
    rail_positions_m[-1],
    0.05,
)

assert np.isclose(
    rail_position_step_m,
    0.00625,
)


# Combine the two validation tables for display.
sar_run_validation_summary = pd.concat(
    [
        run_b_sar_validation,
        run_c_sar_validation,
    ],
    ignore_index=True,
)

display(
    sar_run_validation_summary
)


# Report memory use because both complete RADC runs are
# retained for the next processing stage.
run_b_memory_mib = (
    run_b_sar_radc.nbytes
    / 1024 ** 2
)

run_c_memory_mib = (
    run_c_sar_radc.nbytes
    / 1024 ** 2
)

print("Multi-position RADC validation passed.")
print(
    f"  Run B shape: "
    f"{run_b_sar_radc.shape}"
)
print(
    f"  Run C shape: "
    f"{run_c_sar_radc.shape}"
)
print(
    f"  Run B memory: "
    f"{run_b_memory_mib:.1f} MiB"
)
print(
    f"  Run C memory: "
    f"{run_c_memory_mib:.1f} MiB"
)
print(
    f"  Combined memory: "
    f"{run_b_memory_mib + run_c_memory_mib:.1f} MiB"
)

## 3.2 — Form the Complex Background-Subtracted Synthetic-Aperture Data

The raw Run B and Run C measurements are now converted into the complex range data that will be used for SAR focusing.

This section does **not** form a SAR image yet. It prepares the synthetic-aperture measurements while preserving the phase information needed for image formation.

At each rail position $m$, each RADC frame is processed using the same Hann-windowed range–Doppler processing developed in Section 2:

1. apply a Hann window across fast time;
2. compute the range FFT;
3. apply a Hann window across chirp index;
4. compute the Doppler FFT;
5. extract the centered zero-Doppler bin.

Let $c_{m,f}[i,n,\ell]$ represent Run C, where:

- $m$ is the rail-position index;
- $f$ is the repeated-frame index;
- $i$ is the fast-time sample index;
- $n$ is the chirp index;
- $\ell$ is the RX-channel index.

The resulting range–Doppler data are

$$C_{m,f}[q,p,\ell]=\operatorname{DFT}_n\left\{w_D[n]\operatorname{DFT}_i\left\{w_R[i]c_{m,f}[i,n,\ell]\right\}\right\}.$$

Because the sphere and laboratory background are stationary during each measurement, the centered zero-Doppler bin is retained:

$$C^{(0)}_{m,f}[q,\ell]=C_{m,f}[q,p_0,\ell],$$

where $p_0=32$ for the 64-point centered Doppler FFT.

<div style="font-size:2.0em; font-weight:bold;">
Coherent frame averaging
</div>

The 20 repeated frames at each rail position are averaged as complex values:

$$\overline{C}_m[q,\ell]=\frac{1}{F_C}\sum_{f=0}^{F_C-1}C^{(0)}_{m,f}[q,\ell].$$

Run B is processed in the same way:

$$\overline{B}_m[q,\ell]=\frac{1}{F_B}\sum_{f=0}^{F_B-1}B^{(0)}_{m,f}[q,\ell].$$

This averaging occurs only among repeated frames acquired at the **same rail position**. Measurements from different rail positions are not averaged together because their position-dependent phase differences are the information used for SAR focusing.

A frame-coherence factor is also calculated:

$$\gamma_m[q,\ell]=\frac{\left|\sum_f C^{(0)}_{m,f}[q,\ell]\right|}{\sum_f\left|C^{(0)}_{m,f}[q,\ell]\right|}.$$

A value near 1 indicates that the repeated measurements have nearly constant phase and can be coherently averaged. Low coherence in a weak noise-dominated range bin is not necessarily a problem; coherence is most meaningful where measurable signals are present.

<div style="font-size:2.0em; font-weight:bold;">
Matched complex background subtraction
</div>

Run B contains the same static laboratory scene without the sphere. Before subtraction, one complex Run B-to-Run C alignment coefficient is estimated independently for every rail position and RX channel.

For a set of reference range bins $\mathcal{Q}_{\mathrm{ref}}$, the least-squares coefficient is

$$\alpha_{m,\ell}=\frac{\sum_{q\in\mathcal{Q}_{\mathrm{ref}}}\overline{C}_m[q,\ell]\overline{B}_m^*[q,\ell]}{\sum_{q\in\mathcal{Q}_{\mathrm{ref}}}\left|\overline{B}_m[q,\ell]\right|^2}.$$

The reference bins exclude:

- range bin 0, which is dominated by direct coupling and DC;
- bins 15–26, which contain both the geometrically expected sphere region and the observed response near $q=22$.

The aligned background is subtracted as complex data:

$$G_m[q,\ell]=\overline{C}_m[q,\ell]-\alpha_{m,\ell}\overline{B}_m[q,\ell].$$

The residual $G_m[q,\ell]$ is the input to the later SAR image-formation step.

The coefficient $\alpha_{m,\ell}$ is applied only to the Run B background. It does not rotate or otherwise phase-correct the Run C target data. Therefore, the position-dependent Run C phase required for SAR is preserved.

All four RX channels are retained during this section. We will examine their coherence and target strength before selecting a single RX channel for the first SAR reconstruction.

The aperture–range plots produced below are diagnostic displays of the unfocused data. Their horizontal axis is physical rail position, not image cross-range. They are therefore not yet SAR images.

### 3.2.1 — Complex Aperture-Processing Functions

The following functions:

- process every frame into a zero-Doppler complex range profile;
- calculate frame-to-frame coherence;
- coherently average the repeated frames;
- estimate the matched Run B-to-Run C alignment;
- perform complex background subtraction.

The functions retain separate position, range-bin, and RX-channel axes so that no synthetic-aperture phase information is discarded.

In [ ]:
def process_sar_radc_to_zero_doppler(
    sar_radc_stack,
    range_window="hann",
    doppler_window="hann",
):
    """
    Process a multi-position RADC stack into complex
    zero-Doppler range profiles.

    Input shape:
        (
            rail position m,
            frame f,
            fast-time sample i,
            chirp n,
            RX channel l,
        )

    Output shape:
        (
            rail position m,
            frame f,
            range bin q,
            RX channel l,
        )
    """

    if sar_radc_stack.ndim != 5:
        raise ValueError(
            "sar_radc_stack must have shape "
            "(position, frame, sample, chirp, RX)."
        )

    (
        number_of_positions,
        number_of_frames,
        number_of_samples,
        _,
        number_of_rx_channels,
    ) = sar_radc_stack.shape

    zero_doppler_stack = np.empty(
        (
            number_of_positions,
            number_of_frames,
            number_of_samples,
            number_of_rx_channels,
        ),
        dtype=np.complex128,
    )

    zero_doppler_bin = None

    for position_index in range(
        number_of_positions
    ):
        for frame_index in range(
            number_of_frames
        ):
            range_doppler_cube = (
                compute_radc_range_doppler(
                    radc_frame=(
                        sar_radc_stack[
                            position_index,
                            frame_index,
                        ]
                    ),
                    range_window=range_window,
                    doppler_window=doppler_window,
                )
            )

            (
                zero_doppler_profile,
                current_zero_doppler_bin,
            ) = extract_zero_doppler(
                range_doppler_cube,
                doppler_axis=1,
            )

            if zero_doppler_bin is None:
                zero_doppler_bin = (
                    current_zero_doppler_bin
                )

            if (
                current_zero_doppler_bin
                != zero_doppler_bin
            ):
                raise ValueError(
                    "The zero-Doppler bin changed "
                    "during processing."
                )

            zero_doppler_stack[
                position_index,
                frame_index,
            ] = zero_doppler_profile

    return (
        zero_doppler_stack,
        zero_doppler_bin,
    )


def calculate_sar_frame_coherence(
    zero_doppler_stack,
):
    """
    Calculate coherent-sum efficiency across repeated frames.

    Input shape:
        (position, frame, range bin, RX)

    Output shape:
        (position, range bin, RX)
    """

    if zero_doppler_stack.ndim != 4:
        raise ValueError(
            "zero_doppler_stack must have shape "
            "(position, frame, range bin, RX)."
        )

    coherent_sum = np.sum(
        zero_doppler_stack,
        axis=1,
    )

    noncoherent_sum = np.sum(
        np.abs(
            zero_doppler_stack
        ),
        axis=1,
    )

    return (
        np.abs(
            coherent_sum
        )
        / np.maximum(
            noncoherent_sum,
            np.finfo(float).tiny,
        )
    )


def coherently_average_sar_frames(
    zero_doppler_stack,
):
    """
    Coherently average repeated frames at each rail position.

    Input shape:
        (position, frame, range bin, RX)

    Output shape:
        (position, range bin, RX)
    """

    return np.mean(
        zero_doppler_stack,
        axis=1,
    )


def align_and_subtract_sar_background(
    target_mean,
    background_mean,
    target_coherence,
    background_coherence,
    reference_range_mask,
    minimum_reference_level_db=-35.0,
):
    """
    Align the matched background independently at every
    rail position and RX channel, then subtract it.

    Inputs:
        target_mean:
            Run C complex mean, shape (position, range, RX)

        background_mean:
            Run B complex mean, shape (position, range, RX)

        target_coherence, background_coherence:
            Frame-coherence arrays with the same shape

        reference_range_mask:
            Boolean array over range bin

    Returns:
        aligned_background
        background_subtracted
        alignment_coefficients
        alignment_results_table
    """

    if target_mean.shape != background_mean.shape:
        raise ValueError(
            "Target and background means must have "
            "identical shapes."
        )

    if target_mean.ndim != 3:
        raise ValueError(
            "Mean data must have shape "
            "(position, range bin, RX)."
        )

    (
        number_of_positions,
        number_of_range_bins,
        number_of_rx_channels,
    ) = target_mean.shape

    if reference_range_mask.shape != (
        number_of_range_bins,
    ):
        raise ValueError(
            "reference_range_mask has the wrong shape."
        )

    minimum_reference_level_linear = (
        10 ** (
            minimum_reference_level_db / 20
        )
    )

    alignment_coefficients = np.empty(
        (
            number_of_positions,
            number_of_rx_channels,
        ),
        dtype=np.complex128,
    )

    aligned_background = np.empty_like(
        background_mean,
        dtype=np.complex128,
    )

    background_subtracted = np.empty_like(
        target_mean,
        dtype=np.complex128,
    )

    alignment_results = []

    for position_index in range(
        number_of_positions
    ):
        for rx_channel in range(
            number_of_rx_channels
        ):
            target_profile = target_mean[
                position_index,
                :,
                rx_channel,
            ]

            background_profile = background_mean[
                position_index,
                :,
                rx_channel,
            ]

            # Establish a separate reference magnitude for
            # each position, channel, and run. Only bins inside
            # the permitted reference region contribute.
            target_reference_peak = np.max(
                np.abs(
                    target_profile[
                        reference_range_mask
                    ]
                )
            )

            background_reference_peak = np.max(
                np.abs(
                    background_profile[
                        reference_range_mask
                    ]
                )
            )

            target_is_strong = (
                np.abs(
                    target_profile
                )
                >= (
                    minimum_reference_level_linear
                    * target_reference_peak
                )
            )

            background_is_strong = (
                np.abs(
                    background_profile
                )
                >= (
                    minimum_reference_level_linear
                    * background_reference_peak
                )
            )

            valid_reference_mask = (
                reference_range_mask
                & target_is_strong
                & background_is_strong
            )

            target_reference = target_profile[
                valid_reference_mask
            ]

            background_reference = (
                background_profile[
                    valid_reference_mask
                ]
            )

            if target_reference.size < 3:
                raise ValueError(
                    "Too few valid background-alignment "
                    f"bins at position {position_index}, "
                    f"RX {rx_channel}."
                )

            # Least-squares coefficient alpha such that
            # target_reference is approximately
            # alpha times background_reference.
            numerator = np.sum(
                target_reference
                * np.conj(
                    background_reference
                )
            )

            denominator = np.sum(
                np.abs(
                    background_reference
                ) ** 2
            )

            alignment_coefficient = (
                numerator
                / np.maximum(
                    denominator,
                    np.finfo(float).tiny,
                )
            )

            alignment_coefficients[
                position_index,
                rx_channel,
            ] = alignment_coefficient

            aligned_profile = (
                alignment_coefficient
                * background_profile
            )

            residual_profile = (
                target_profile
                - aligned_profile
            )

            aligned_background[
                position_index,
                :,
                rx_channel,
            ] = aligned_profile

            background_subtracted[
                position_index,
                :,
                rx_channel,
            ] = residual_profile

            # Normalized complex correlation in the bins used
            # to estimate the alignment.
            complex_correlation = (
                np.abs(
                    numerator
                )
                / np.maximum(
                    np.sqrt(
                        np.sum(
                            np.abs(
                                target_reference
                            ) ** 2
                        )
                        * denominator
                    ),
                    np.finfo(float).tiny,
                )
            )

            residual_reference = residual_profile[
                valid_reference_mask
            ]

            # Reduction of reference-region power after
            # aligned complex subtraction.
            reference_residual_reduction_db = (
                10
                * np.log10(
                    np.maximum(
                        np.sum(
                            np.abs(
                                target_reference
                            ) ** 2
                        ),
                        np.finfo(float).tiny,
                    )
                    / np.maximum(
                        np.sum(
                            np.abs(
                                residual_reference
                            ) ** 2
                        ),
                        np.finfo(float).tiny,
                    )
                )
            )

            alignment_results.append(
                {
                    "position_index": position_index,
                    "rx_channel": rx_channel,
                    "reference_bin_count": (
                        target_reference.size
                    ),
                    "alignment_magnitude": np.abs(
                        alignment_coefficient
                    ),
                    "alignment_phase_deg": np.rad2deg(
                        np.angle(
                            alignment_coefficient
                        )
                    ),
                    "complex_correlation": (
                        complex_correlation
                    ),
                    "median_target_frame_coherence": (
                        np.median(
                            target_coherence[
                                position_index,
                                valid_reference_mask,
                                rx_channel,
                            ]
                        )
                    ),
                    "median_background_frame_coherence": (
                        np.median(
                            background_coherence[
                                position_index,
                                valid_reference_mask,
                                rx_channel,
                            ]
                        )
                    ),
                    "reference_residual_reduction_db": (
                        reference_residual_reduction_db
                    ),
                }
            )

    alignment_results_table = pd.DataFrame(
        alignment_results
    )

    # Add an unwrapped phase column separately for each RX.
    alignment_results_table[
        "alignment_phase_unwrapped_deg"
    ] = np.nan

    for rx_channel in range(
        number_of_rx_channels
    ):
        channel_rows = (
            alignment_results_table[
                "rx_channel"
            ]
            == rx_channel
        )

        channel_phase_rad = np.angle(
            alignment_coefficients[
                :,
                rx_channel,
            ]
        )

        alignment_results_table.loc[
            channel_rows,
            "alignment_phase_unwrapped_deg",
        ] = np.rad2deg(
            np.unwrap(
                channel_phase_rad
            )
        )

    return (
        aligned_background,
        background_subtracted,
        alignment_coefficients,
        alignment_results_table,
    )

### 3.2.2 — Process Runs B and C Across the Synthetic Aperture

This cell applies the functions above to all 17 rail positions.

The output arrays have shape

$$(\text{rail position},\ \text{range bin},\ \text{RX channel}).$$

The diagnostic aperture–range plots use RX channel 0 initially. They show how the complex range-profile magnitude changes as the radar moves along the rail.

These plots remain **unfocused**. A point target may appear as a horizontal or slightly curved response because its energy has not yet been phase-aligned to a candidate image location.

The final noncoherent range profile combines power over all rail positions and RX channels. It is used only to identify the dominant target-associated range bin; the complex samples themselves remain unchanged for SAR processing.

In [ ]:
# Select the processing windows used for the first SAR result.
sar_range_window = "hann"
sar_doppler_window = "hann"


# Process every Run B frame into a complex zero-Doppler
# range profile.
print("Processing Run B across the synthetic aperture...")

(
    run_b_sar_zero_doppler,
    run_b_sar_zero_doppler_bin,
) = process_sar_radc_to_zero_doppler(
    sar_radc_stack=run_b_sar_radc,
    range_window=sar_range_window,
    doppler_window=sar_doppler_window,
)


# Process every Run C frame using the same processing path.
print("Processing Run C across the synthetic aperture...")

(
    run_c_sar_zero_doppler,
    run_c_sar_zero_doppler_bin,
) = process_sar_radc_to_zero_doppler(
    sar_radc_stack=run_c_sar_radc,
    range_window=sar_range_window,
    doppler_window=sar_doppler_window,
)


assert (
    run_b_sar_zero_doppler_bin
    == run_c_sar_zero_doppler_bin
), "Run B and Run C zero-Doppler bins do not match."

print(
    "Zero-Doppler bin:",
    run_c_sar_zero_doppler_bin,
)


# Calculate frame-to-frame coherence before averaging.
run_b_sar_frame_coherence = (
    calculate_sar_frame_coherence(
        run_b_sar_zero_doppler
    )
)

run_c_sar_frame_coherence = (
    calculate_sar_frame_coherence(
        run_c_sar_zero_doppler
    )
)


# Coherently average the 20 repeated frames separately
# at every rail position.
run_b_sar_zero_doppler_mean = (
    coherently_average_sar_frames(
        run_b_sar_zero_doppler
    )
)

run_c_sar_zero_doppler_mean = (
    coherently_average_sar_frames(
        run_c_sar_zero_doppler
    )
)


# Define the range bins used to align Run B with Run C.
#
# Retain bins 1–50, but exclude bins 15–26 because
# this interval contains both the expected and observed
# sphere-associated regions.
sar_number_of_range_bins = (
    run_c_sar_zero_doppler_mean.shape[1]
)

sar_reference_range_mask = np.zeros(
    sar_number_of_range_bins,
    dtype=bool,
)

sar_reference_range_mask[1:51] = True
sar_reference_range_mask[15:27] = False


# Align and subtract the matched Run B background.
(
    run_b_sar_aligned_mean,
    run_c_sar_background_subtracted,
    sar_background_alignment_coefficients,
    sar_background_alignment_table,
) = align_and_subtract_sar_background(
    target_mean=run_c_sar_zero_doppler_mean,
    background_mean=run_b_sar_zero_doppler_mean,
    target_coherence=run_c_sar_frame_coherence,
    background_coherence=run_b_sar_frame_coherence,
    reference_range_mask=(
        sar_reference_range_mask
    ),
    minimum_reference_level_db=-35.0,
)


# Confirm the final unfocused complex aperture-data shape.
expected_sar_profile_shape = (
    sar_position_table.shape[0],
    sar_number_of_range_bins,
    run_c_sar_radc.shape[-1],
)

assert (
    run_c_sar_background_subtracted.shape
    == expected_sar_profile_shape
), (
    "Unexpected background-subtracted SAR shape: "
    f"{run_c_sar_background_subtracted.shape}"
)

assert np.all(
    np.isfinite(
        run_c_sar_background_subtracted
    )
), "The background-subtracted SAR data contain nonfinite values."


# Create a compact per-RX validation summary.
sar_alignment_summary = (
    sar_background_alignment_table
    .groupby(
        "rx_channel",
        as_index=False,
    )
    .agg(
        median_reference_bin_count=(
            "reference_bin_count",
            "median",
        ),
        minimum_complex_correlation=(
            "complex_correlation",
            "min",
        ),
        median_complex_correlation=(
            "complex_correlation",
            "median",
        ),
        median_target_frame_coherence=(
            "median_target_frame_coherence",
            "median",
        ),
        median_background_frame_coherence=(
            "median_background_frame_coherence",
            "median",
        ),
        median_residual_reduction_db=(
            "reference_residual_reduction_db",
            "median",
        ),
        minimum_residual_reduction_db=(
            "reference_residual_reduction_db",
            "min",
        ),
        alignment_phase_span_deg=(
            "alignment_phase_unwrapped_deg",
            lambda values: (
                np.max(values)
                - np.min(values)
            ),
        ),
    )
)


# Calculate the noncoherent residual range power by summing
# power over rail position and RX channel.
sar_residual_range_power = np.sum(
    np.abs(
        run_c_sar_background_subtracted
    ) ** 2,
    axis=(0, 2),
)

sar_residual_range_power_db = (
    10
    * np.log10(
        np.maximum(
            sar_residual_range_power,
            np.finfo(float).tiny,
        )
        / np.maximum(
            np.max(
                sar_residual_range_power
            ),
            np.finfo(float).tiny,
        )
    )
)


# Search for the strongest target-associated response over
# the same practical range interval used for Run A.
sar_target_search_bins = np.arange(
    10,
    36,
)

run_c_observed_range_bin = int(
    sar_target_search_bins[
        np.argmax(
            sar_residual_range_power[
                sar_target_search_bins
            ]
        )
    ]
)

print(
    "Background-subtracted SAR data shape:",
    run_c_sar_background_subtracted.shape,
)

print(
    "Strongest Run C residual range bin:",
    run_c_observed_range_bin,
)


# ----------------------------------------------------------
# Plot the unfocused aperture–range data for one RX channel.
# ----------------------------------------------------------

sar_diagnostic_rx_channel = 0
sar_display_maximum_range_bin = 50
sar_display_floor_db = -50.0

sar_common_reference = np.max(
    np.abs(
        run_c_sar_zero_doppler_mean[
            :,
            :sar_display_maximum_range_bin + 1,
            sar_diagnostic_rx_channel,
        ]
    )
)


def magnitude_db_with_reference(
    complex_data,
    reference_magnitude,
    minimum_db=-50.0,
):
    """
    Convert magnitude to dB using a supplied common reference.
    """

    magnitude_db = (
        20
        * np.log10(
            np.maximum(
                np.abs(
                    complex_data
                ),
                np.finfo(float).tiny,
            )
            / np.maximum(
                reference_magnitude,
                np.finfo(float).tiny,
            )
        )
    )

    return np.maximum(
        magnitude_db,
        minimum_db,
    )


aperture_range_datasets = {
    "Coherently Averaged Run C": (
        run_c_sar_zero_doppler_mean
    ),
    "Aligned Run B Background": (
        run_b_sar_aligned_mean
    ),
    "Complex Background-Subtracted": (
        run_c_sar_background_subtracted
    ),
}

position_step_cm = (
    rail_position_step_m * 100
)

image_extent = [
    rail_positions_m[0] * 100
    - position_step_cm / 2,
    rail_positions_m[-1] * 100
    + position_step_cm / 2,
    -0.5,
    sar_display_maximum_range_bin + 0.5,
]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 6),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

image_handle = None

for axis, (
    plot_title,
    aperture_data,
) in zip(
    axes,
    aperture_range_datasets.items(),
):
    aperture_range_db = (
        magnitude_db_with_reference(
            aperture_data[
                :,
                :sar_display_maximum_range_bin + 1,
                sar_diagnostic_rx_channel,
            ],
            reference_magnitude=(
                sar_common_reference
            ),
            minimum_db=sar_display_floor_db,
        )
    )

    image_handle = axis.imshow(
        aperture_range_db.T,
        origin="lower",
        aspect="auto",
        extent=image_extent,
        cmap="viridis",
        vmin=sar_display_floor_db,
        vmax=0,
    )

    # Mark the geometrically expected q = 17–19 region
    # with a visible red rectangle.
    expected_range_rectangle = Rectangle(
        (
            image_extent[0],
            16.5,
        ),
        image_extent[1] - image_extent[0],
        3.0,
        fill=False,
        edgecolor="red",
        linewidth=2,
        label="Expected q = 17–19",
    )

    axis.add_patch(
        expected_range_rectangle
    )

    # Mark the strongest observed Run C residual range bin.
    axis.axhline(
        run_c_observed_range_bin,
        color="cyan",
        linestyle="-.",
        linewidth=1.5,
        label=(
            "Observed "
            f"q = {run_c_observed_range_bin}"
        ),
    )

    axis.axvline(
        0,
        color="white",
        linestyle="--",
        linewidth=1,
        alpha=0.8,
        label="Aperture center",
    )

    axis.set_title(
        plot_title
    )

    axis.set_xlabel(
        "Rail position, $x_m$ (cm)"
    )

    axis.grid(
        False
    )

axes[0].set_ylabel(
    "Range-bin index, $q$"
)

axes[0].legend(
    loc="upper right",
    fontsize=8,
)

fig.colorbar(
    image_handle,
    ax=axes,
    label=(
        "Magnitude relative to Run C "
        "maximum (dB)"
    ),
)

fig.suptitle(
    "Run C Unfocused Synthetic-Aperture Range Data "
    f"— RX {sar_diagnostic_rx_channel}",
    fontsize=14,
)

plt.show()


# ----------------------------------------------------------
# Plot the residual range-power profile.
# ----------------------------------------------------------

fig, axis = plt.subplots(
    1,
    1,
    figsize=(11, 5),
    constrained_layout=True,
)

axis.plot(
    np.arange(
        sar_number_of_range_bins
    ),
    sar_residual_range_power_db,
    color="black",
    linewidth=1.5,
)

axis.axvspan(
    17,
    19,
    facecolor="none",
    edgecolor="red",
    linewidth=2,
    label="Expected q = 17–19",
)

axis.axvline(
    run_c_observed_range_bin,
    color="cyan",
    linestyle="-.",
    linewidth=1.5,
    label=(
        "Observed "
        f"q = {run_c_observed_range_bin}"
    ),
)

axis.set_xlim(
    0,
    50,
)

axis.set_ylim(
    -50,
    2,
)

axis.set_title(
    "Run C Background-Subtracted Residual Range Power"
)

axis.set_xlabel(
    "Range-bin index, $q$"
)

axis.set_ylabel(
    "Power relative to residual maximum (dB)"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()

plt.show()


# ----------------------------------------------------------
# Plot the complex Run B-to-Run C alignment coefficients.
# These diagnose cross-run background matching; they are
# not phase corrections applied to the Run C target.
# ----------------------------------------------------------

fig, axes = plt.subplots(
    2,
    1,
    figsize=(11, 8),
    sharex=True,
    constrained_layout=True,
)

for rx_channel in range(
    run_c_sar_radc.shape[-1]
):
    axes[0].plot(
        rail_positions_m * 100,
        20
        * np.log10(
            np.maximum(
                np.abs(
                    sar_background_alignment_coefficients[
                        :,
                        rx_channel,
                    ]
                ),
                np.finfo(float).tiny,
            )
        ),
        marker="o",
        linewidth=1.2,
        label=f"RX {rx_channel}",
    )

    channel_rows = (
        sar_background_alignment_table[
            "rx_channel"
        ]
        == rx_channel
    )

    axes[1].plot(
        rail_positions_m * 100,
        sar_background_alignment_table.loc[
            channel_rows,
            "alignment_phase_unwrapped_deg",
        ],
        marker="o",
        linewidth=1.2,
        label=f"RX {rx_channel}",
    )

axes[0].set_title(
    "Run B-to-Run C Alignment Magnitude"
)

axes[0].set_ylabel(
    "Alignment magnitude (dB)"
)

axes[0].grid(
    True,
    alpha=0.3,
)

axes[0].legend(
    ncols=4,
)

axes[1].set_title(
    "Run B-to-Run C Unwrapped Alignment Phase"
)

axes[1].set_xlabel(
    "Rail position, $x_m$ (cm)"
)

axes[1].set_ylabel(
    "Alignment phase (degrees)"
)

axes[1].grid(
    True,
    alpha=0.3,
)

axes[1].legend(
    ncols=4,
)

plt.show()


# Display the compact numerical validation summary.
display(
    sar_alignment_summary
)

### 3.2.3 — Results and Interpretation

The multi-position processing produced a complex background-subtracted array with the expected shape

$$(17,\ 128,\ 4),$$

corresponding to 17 rail positions, 128 range bins, and four RX channels. The centered zero-Doppler bin remains $p_0=32$ throughout the processing.

<div style="font-size:2.0em; font-weight:bold;">
What was done
</div>

At every rail position, the 20 repeated Run B and Run C frames were processed into complex zero-Doppler range profiles and coherently averaged. Coherent averaging means that the real and imaginary components were averaged before calculating magnitude. Consequently, stable stationary returns reinforce one another while uncorrelated noise is reduced without discarding the phase required for SAR.

The repeated frames were averaged only within each fixed rail position. The 17 rail positions have not yet been combined. Their position-dependent complex phase histories remain intact for the later SAR-focusing operation.

A separate least-squares complex coefficient was then estimated for every rail position and RX channel to align the matched Run B background with Run C. The aligned background was subtracted from Run C before any magnitude calculation or SAR focusing.

**Frame-to-frame coherence**

The median frame-coherence factors are greater than approximately $0.99999$ for both runs and every RX channel. These values are extremely close to the ideal value of 1.

This indicates that the radar maintained nearly constant complex phase over the 20 repeated frames at each rail position. Coherent averaging is therefore justified and does not produce meaningful phase cancellation.

This result establishes coherence across repeated frames at each position. SAR also requires a physically meaningful phase progression **between** rail positions; that aperture phase history will be examined during image formation.

<div style="font-size:2.0em; font-weight:bold;">
Run B-to-Run C alignment
</div>


The alignment magnitudes remain close to $0$ dB over the complete rail aperture. The largest visible departure is approximately $-0.32$ dB for RX 0 at one position, while most coefficients remain within approximately $\pm0.1$ dB.

The unwrapped alignment phases are also exceptionally stable. Their total spans across the aperture are approximately:

- RX 0: $1.37^\circ$;
- RX 1: $1.19^\circ$;
- RX 2: $0.61^\circ$;
- RX 3: $0.58^\circ$.

The minimum complex correlation at any position is approximately $0.988$, while the median correlations range from approximately $0.993$ to $0.9995$. The similarity of the magnitude and phase coefficients across the rail indicates that Runs B and C retained a highly consistent complex reference.

These alignment phases describe the relationship between the Run B and Run C background measurements. They are not corrections applied to the Run C target phase history.

<div style="font-size:2.0em; font-weight:bold;">
Background-subtraction performance
</div>

The median reduction of the reference-region residual power is:

- RX 0: approximately $18.6$ dB;
- RX 1: approximately $24.3$ dB;
- RX 2: approximately $27.8$ dB;
- RX 3: approximately $29.8$ dB.

RX 2 and RX 3 provide the greatest stationary-background suppression, although all four channels produce substantial reduction. The four RX channels will remain separate until their target strength and aperture-phase behavior are examined.

The strong near-range coupling around $q=0$ is greatly reduced in the complex background-subtracted data. Additional weaker residuals remain at larger range bins. These may contain imperfectly cancelled clutter, multipath, sidelobes, noise, or other scene changes between Runs B and C.

<div style="font-size:2.0em; font-weight:bold;">
Target-associated range location
</div>

After background subtraction, the strongest residual response occurs at

$$q_{\mathrm{observed}}=22.$$

Using the nominal V-MD3 range-bin spacing, this corresponds to

$$R_{\mathrm{nominal}}=22(0.046875\ \text{m})=1.03125\ \text{m}.$$

The geometrically expected sphere region remains at $q=17$–$19$, corresponding approximately to the measured $0.84$ m slant range. No comparably strong residual is observed in that region.

The response at $q=22$ extends across the complete 10 cm rail aperture and remains after matched complex background subtraction. This agrees with the target-associated response previously observed in fixed-position Run A. The repeated appearance of the response in two target-present runs supports identifying it as sphere-associated rather than stationary laboratory clutter.

The result also confirms that the unexpected range location is already present before SAR focusing. It is not introduced by four-RX angle beamforming, aperture weighting, or synthetic-aperture image formation.

No general range-scale or range-offset correction has been established. For the first SAR reconstruction, the known sphere geometry can be used as a provisional local registration: the measured target response near $q=22$ will be associated with the approximately $0.84$ m reference slant range. This is a one-point registration for image formation, not a complete radar range calibration.

<div style="font-size:2.0em; font-weight:bold;">
Why this is not yet a SAR image
</div>

The aperture–range plot displays magnitude as a function of rail position and range bin. It does not compensate the phase accumulated as the radar moves relative to a candidate target location.

The bright horizontal response near $q=22$ therefore represents unfocused target-associated energy. Its horizontal extent does not indicate the final cross-range resolution.

During backprojection, each image pixel will predict a different propagation distance to every rail position. The corresponding complex measurements will be phase-corrected and coherently summed across the aperture. Energy consistent with the sphere geometry should then reinforce near the correct image location, while spatially inconsistent residual clutter should remain defocused.

These results validate the complex aperture dataset for the first SAR backprojection experiment.

<div style="font-size:3.0em; font-weight:bold;">
Interpreting Coherence and Correlation
</div>

Coherence and correlation are both normalized between 0 and 1 in this analysis, but they answer different questions.

<div style="font-size:2.0em; font-weight:bold;">
Frame-to-frame coherence
</div>

Frame coherence determines whether repeated complex measurements can be coherently integrated without significant phase cancellation.

At one rail position $m$, range bin $q$, and RX channel $\ell$, let $x_f$ denote the complex measurement from repeated frame $f$. The frame-coherence factor is

$$\gamma_m[q,\ell]=\frac{\left|\sum_f x_f\right|}{\sum_f|x_f|}.$$

The denominator is the magnitude that would be obtained if every frame added perfectly in phase. The numerator is the magnitude produced by the actual complex sum.

Therefore:

- $\gamma=1$ means that all frames have the same phase and reinforce perfectly;
- $\gamma\approx0$ means that the frame phases largely cancel;
- an intermediate value indicates partial phase variation.

The frame magnitudes do not need to be identical. For example, complex samples with magnitudes 1, 2, and 5 will still produce $\gamma=1$ if they all have the same phase.

This makes $\gamma$ primarily a measure of phase consistency, although stronger frames contribute more heavily than weaker frames.

A high frame-coherence factor justifies coherent averaging:

$$\overline{x}=\frac{1}{F}\sum_{f=0}^{F-1}x_f.$$

The phase-consistent signal remains after averaging, while uncorrelated noise is reduced.

In this notebook, frame coherence is calculated across the 20 repeated frames acquired at each fixed rail position. It does not yet test coherence between the 17 different rail positions.

<div style="font-size:2.0em; font-weight:bold;">
Complex correlation
</div>

Complex correlation compares the shape of two complex vectors. In this analysis, it compares the coherently averaged Run C and Run B range profiles over the selected reference range bins.

The normalized complex-correlation magnitude is

$$\rho=\frac{\left|\sum_{q\in\mathcal{Q}_{\mathrm{ref}}}\overline{C}[q]\overline{B}^{*}[q]\right|}{\sqrt{\sum_{q\in\mathcal{Q}_{\mathrm{ref}}}|\overline{C}[q]|^2\sum_{q\in\mathcal{Q}_{\mathrm{ref}}}|\overline{B}[q]|^2}}.$$

Therefore:

- $\rho=1$ means that the two complex range profiles have the same shape, apart from one overall complex scale factor;
- a lower value means that their relative magnitudes, phases, or both differ across the reference range bins.

A high correlation means that Run B is a good matched-background template for Run C. One complex coefficient can then align its overall magnitude and phase before subtraction.

Correlation does not establish that the individual Run B or Run C frames are stable. That is the role of the frame-coherence factor.

<div style="font-size:2.0em; font-weight:bold;">
How the two metrics differ
</div>

| Metric | Samples being compared | Independent-variable axis | Question answered |
|---|---|---|---|
| Frame coherence, $\gamma$ | Repeated measurements from one run | Frame index $f$ | Will repeated frames reinforce during coherent averaging? |
| Complex correlation, $\rho$ | Run C profile versus Run B profile | Reference range bin $q$ | Is Run B a good complex background template for Run C? |
| SAR coherent summation | Run C target measurements from different rail positions | Rail-position index $m$ | Do the measurements reinforce after geometric phase compensation? |

For example, Run C could have excellent frame coherence but poor correlation with Run B. This would mean that Run C is internally stable, but the background scene changed between the two runs.

Conversely, Run B and Run C could have similar average range profiles but poor frame coherence. Their averaged profiles might appear correlated, but coherent frame integration would suffer phase cancellation and become unreliable.

The present results show both near-perfect frame coherence and high Run B-to-Run C correlation. This supports coherent frame averaging and matched complex background subtraction before SAR focusing.

The frame-coherence factor used here is also called coherent-sum efficiency or phase coherence. It should not be confused with the frequency-dependent magnitude-squared coherence function used in spectral analysis.

## 3.3 — Validate the Synthetic-Aperture Phase History

Frame coherence established that the 20 repeated measurements at each rail position can be coherently averaged. SAR requires an additional form of phase consistency: the target phase must change predictably as the radar moves across the 17 rail positions.

For the background-subtracted Run C data, define the target-associated complex sample

$$g_{m,\ell}=G_m[q_t,\ell],$$

where $m$ is the rail-position index, $\ell$ is the RX-channel index, and $q_t=22$ is the observed target-associated range bin.

<div style="font-size:2.0em; font-weight:bold;">
Expected geometric phase history
</div>

For a point target at cross-range position $x_t$ and downrange coordinate $z_t$, the approximate distance from RX channel $\ell$ at rail position $m$ is

$$R_{m,\ell}=\sqrt{\left(x_t-x_{m,\ell}\right)^2+z_t^2},$$

where

$$x_{m,\ell}=x_m+x_{\ell}$$

is the rail position plus the fixed RX-element offset.

This uses a monostatic phase-center approximation. The exact V-MD3 propagation path is transmitter-to-target plus target-to-receiver, but the transmitter location within the module is not presently available. Using one RX channel and the translated module position provides an appropriate first model, which can later be refined if the antenna geometry becomes available.

Relative to the center aperture position, the expected two-way propagation-phase change has magnitude

$$\Delta\psi_{m,\ell}=\frac{4\pi}{\lambda}\left(R_{m,\ell}-R_{\mathrm{center},\ell}\right).$$

The sign of the phase measured by the RADC processing depends on the radar's IQ, mixer, and FFT conventions. Therefore, both candidate models are tested:

$$\psi_{m,\ell}^{(+)}=+\frac{4\pi}{\lambda}\left(R_{m,\ell}-R_{\mathrm{center},\ell}\right),$$

$$\psi_{m,\ell}^{(-)}=-\frac{4\pi}{\lambda}\left(R_{m,\ell}-R_{\mathrm{center},\ell}\right).$$

The sign that produces the strongest coherent sum will be retained for backprojection and later checked using the offset-target Run D dataset.

<div style="font-size:2.0em; font-weight:bold;">
Why phase changes even without visible range migration
</div>

At the edge of the 10 cm aperture, the path-length change for a centered target at approximately $0.84$ m is only

$$\Delta R_{\mathrm{edge}}\approx\sqrt{(0.05\ \text{m})^2+(0.84\ \text{m})^2}-0.84\ \text{m}\approx1.49\ \text{mm}.$$

This is much smaller than the nominal range-bin spacing of $46.875$ mm, so the response is not expected to move visibly into another range bin.

However, the corresponding two-way phase change at approximately 61 GHz is

$$\Delta\psi_{\mathrm{edge}}\approx\frac{4\pi(1.49\ \text{mm})}{4.91\ \text{mm}}\approx218^\circ.$$

The target can therefore remain in the same range bin while accumulating more than half a cycle of measurable phase change. SAR obtains its improved cross-range resolution from this phase history, not from visible movement between range bins.

The fixed range offset also does not prevent this test. Only the relative distance change $R_{m,\ell}-R_{\mathrm{center},\ell}$ enters the phase model, so a constant absolute range offset cancels.

<div style="font-size:2.0em; font-weight:bold;">
Aperture focusing coherence
</div>

For each candidate phase sign, the predicted phase is removed and the rail-position samples are coherently summed. The resulting aperture-focusing coherence is

$$\eta_{\ell}=\frac{\left|\sum_m g_{m,\ell}\exp\left(-j\psi_{m,\ell}\right)\right|}{\sum_m|g_{m,\ell}|}.$$

A value near 1 means that the target samples reinforce after geometric phase compensation. A low value means that the assumed location, phase sign, range bin, or propagation model does not describe the measured phase history.

Unlike the earlier frame-coherence factor, $\eta_\ell$ is calculated across rail positions rather than repeated frames.

The sphere is an electrically large extended target rather than an ideal point scatterer, so a perfect match is not required. Nevertheless, a smooth measured phase history and a clear preference for one phase sign would demonstrate that the data contain usable synthetic-aperture information.

### 3.3.1 — Aperture Phase-History Functions

The following function extracts the target-associated complex sample from every rail position and RX channel. It then compares the measured phase history with both possible two-way propagation-phase signs.

The function also reports:

- aperture-focusing coherence;
- circular RMS phase-model error;
- target-magnitude variation across the rail;
- target power relative to the 95th-percentile residual reference level.

No phase correction is permanently applied to the data in this section.

In [ ]:
def evaluate_sar_aperture_phase_history(
    aperture_range_data,
    target_range_bin,
    rail_positions_m,
    rx_positions_m,
    carrier_frequency_hz,
    target_cross_range_m,
    target_downrange_m,
    center_position_index,
    reference_range_mask,
):
    """
    Compare measured target phase across the rail with
    positive and negative two-way propagation models.

    aperture_range_data shape:
        (rail position, range bin, RX channel)

    Returns:
        target_samples
        measured_relative_phase_deg
        expected_unsigned_phase_rad
        target_magnitude_db
        phase_history_results_table
    """

    if aperture_range_data.ndim != 3:
        raise ValueError(
            "aperture_range_data must have shape "
            "(position, range bin, RX)."
        )

    (
        number_of_positions,
        number_of_range_bins,
        number_of_rx_channels,
    ) = aperture_range_data.shape

    if rail_positions_m.shape != (
        number_of_positions,
    ):
        raise ValueError(
            "rail_positions_m has the wrong shape."
        )

    if rx_positions_m.shape != (
        number_of_rx_channels,
    ):
        raise ValueError(
            "rx_positions_m has the wrong shape."
        )

    if reference_range_mask.shape != (
        number_of_range_bins,
    ):
        raise ValueError(
            "reference_range_mask has the wrong shape."
        )

    if not (
        0
        <= target_range_bin
        < number_of_range_bins
    ):
        raise ValueError(
            "target_range_bin is outside the data."
        )

    speed_of_light_m_per_s = 299_792_458.0

    wavelength_m = (
        speed_of_light_m_per_s
        / carrier_frequency_hz
    )

    # Extract the target-associated complex sample at every
    # rail position and RX channel.
    #
    # Shape:
    #   (rail position, RX channel)
    target_samples = aperture_range_data[
        :,
        target_range_bin,
        :,
    ]

    # Approximate the physical position of each translated
    # RX element.
    #
    # Shape:
    #   (rail position, RX channel)
    effective_rx_positions_m = (
        rail_positions_m[:, np.newaxis]
        + rx_positions_m[np.newaxis, :]
    )

    # Calculate the one-way geometric range from each
    # translated RX location to the provisional target point.
    geometric_range_m = np.sqrt(
        (
            target_cross_range_m
            - effective_rx_positions_m
        ) ** 2
        + target_downrange_m ** 2
    )

    center_geometric_range_m = (
        geometric_range_m[
            center_position_index,
            :,
        ]
    )

    relative_geometric_range_m = (
        geometric_range_m
        - center_geometric_range_m[
            np.newaxis,
            :
        ]
    )

    # Unsigned two-way phase change. Both positive and
    # negative signs will be tested below.
    expected_unsigned_phase_rad = (
        4
        * np.pi
        / wavelength_m
        * relative_geometric_range_m
    )

    # Unwrap each RX-channel phase history and reference it
    # to the center rail position.
    measured_phase_rad = np.unwrap(
        np.angle(
            target_samples
        ),
        axis=0,
    )

    measured_relative_phase_rad = (
        measured_phase_rad
        - measured_phase_rad[
            center_position_index,
            :
        ][
            np.newaxis,
            :
        ]
    )

    measured_relative_phase_deg = np.rad2deg(
        measured_relative_phase_rad
    )

    # Express each channel's target magnitude relative to
    # its own maximum over the rail.
    target_magnitude = np.abs(
        target_samples
    )

    target_magnitude_db = (
        20
        * np.log10(
            np.maximum(
                target_magnitude,
                np.finfo(float).tiny,
            )
            / np.maximum(
                np.max(
                    target_magnitude,
                    axis=0,
                    keepdims=True,
                ),
                np.finfo(float).tiny,
            )
        )
    )

    phase_history_results = []

    for rx_channel in range(
        number_of_rx_channels
    ):
        channel_samples = target_samples[
            :,
            rx_channel,
        ]

        # Measure the target power relative to a strong
        # residual-background reference. The 95th percentile
        # is more conservative than the median noise floor.
        target_mean_power = np.mean(
            np.abs(
                channel_samples
            ) ** 2
        )

        reference_residual_power = (
            np.abs(
                aperture_range_data[
                    :,
                    reference_range_mask,
                    rx_channel,
                ]
            ) ** 2
        )

        reference_95th_power = np.percentile(
            reference_residual_power,
            95,
        )

        target_to_95th_reference_db = (
            10
            * np.log10(
                np.maximum(
                    target_mean_power,
                    np.finfo(float).tiny,
                )
                / np.maximum(
                    reference_95th_power,
                    np.finfo(float).tiny,
                )
            )
        )

        aperture_magnitude_spread_db = (
            np.max(
                target_magnitude_db[
                    :,
                    rx_channel,
                ]
            )
            - np.min(
                target_magnitude_db[
                    :,
                    rx_channel,
                ]
            )
        )

        # Reference the measured complex phase to the
        # center-position sample. Its magnitude is irrelevant
        # for the circular phase-error calculation.
        measured_relative_complex = (
            channel_samples
            * np.conj(
                channel_samples[
                    center_position_index
                ]
            )
        )

        for phase_sign in (
            -1,
            1,
        ):
            predicted_relative_phase_rad = (
                phase_sign
                * expected_unsigned_phase_rad[
                    :,
                    rx_channel,
                ]
            )

            # Remove the predicted phase before summing
            # across the rail positions.
            phase_compensation = np.exp(
                -1j
                * predicted_relative_phase_rad
            )

            compensated_samples = (
                channel_samples
                * phase_compensation
            )

            focusing_coherence = (
                np.abs(
                    np.sum(
                        compensated_samples
                    )
                )
                / np.maximum(
                    np.sum(
                        np.abs(
                            channel_samples
                        )
                    ),
                    np.finfo(float).tiny,
                )
            )

            # Calculate circular phase error so that errors
            # near +180 and -180 degrees are handled properly.
            circular_phase_error_rad = np.angle(
                measured_relative_complex
                * np.exp(
                    -1j
                    * predicted_relative_phase_rad
                )
            )

            circular_phase_rms_deg = np.sqrt(
                np.mean(
                    np.rad2deg(
                        circular_phase_error_rad
                    ) ** 2
                )
            )

            phase_history_results.append(
                {
                    "rx_channel": rx_channel,
                    "phase_sign": phase_sign,
                    "phase_model": (
                        f"{phase_sign:+d}"
                        " x 4*pi*delta_R/lambda"
                    ),
                    "focusing_coherence": (
                        focusing_coherence
                    ),
                    "circular_phase_rms_deg": (
                        circular_phase_rms_deg
                    ),
                    "target_to_95th_reference_db": (
                        target_to_95th_reference_db
                    ),
                    "aperture_magnitude_spread_db": (
                        aperture_magnitude_spread_db
                    ),
                }
            )

    phase_history_results_table = pd.DataFrame(
        phase_history_results
    )

    return (
        target_samples,
        measured_relative_phase_deg,
        expected_unsigned_phase_rad,
        target_magnitude_db,
        phase_history_results_table,
    )

### 3.3.2 — Compare the Measured and Predicted Aperture Phase

Run C was acquired with the sphere near the center of the synthetic aperture, so the provisional target coordinates are

$$x_t=0\ \text{m},$$

$$z_t=0.84\ \text{m}.$$

The test uses the measured sphere-associated range bin $q_t=22$. Because the predicted rail-induced range change is much smaller than one range bin, the fixed-bin phase history is sufficient for this initial validation.

Both candidate propagation-phase signs are evaluated. The preferred sign is the one that produces the greatest median focusing coherence across the four RX channels.

A provisional single RX channel is also identified from the largest focusing coherence under the selected sign. This selection will be reviewed after examining the phase plots and numerical results.

In [ ]:
# Define the known Run C target geometry.
run_c_target_cross_range_m = 0.0
run_c_target_downrange_m = 0.84

# Use the measured sphere-associated response.
sar_phase_test_range_bin = (
    run_c_observed_range_bin
)

# Confirm that the array geometry matches the SAR data.
assert (
    rx_positions_m.shape[0]
    == run_c_sar_background_subtracted.shape[2]
), "RX geometry does not match the SAR data."


# Evaluate both possible propagation-phase signs.
(
    run_c_target_aperture_samples,
    run_c_measured_relative_phase_deg,
    run_c_expected_unsigned_phase_rad,
    run_c_target_magnitude_db,
    sar_phase_history_results,
) = evaluate_sar_aperture_phase_history(
    aperture_range_data=(
        run_c_sar_background_subtracted
    ),
    target_range_bin=sar_phase_test_range_bin,
    rail_positions_m=rail_positions_m,
    rx_positions_m=rx_positions_m,
    carrier_frequency_hz=carrier_frequency_hz,
    target_cross_range_m=(
        run_c_target_cross_range_m
    ),
    target_downrange_m=(
        run_c_target_downrange_m
    ),
    center_position_index=(
        CENTER_POSITION_INDEX
    ),
    reference_range_mask=(
        sar_reference_range_mask
    ),
)


# Summarize each phase-sign hypothesis across RX channels.
sar_phase_sign_summary = (
    sar_phase_history_results
    .groupby(
        [
            "phase_sign",
            "phase_model",
        ],
        as_index=False,
    )
    .agg(
        median_focusing_coherence=(
            "focusing_coherence",
            "median",
        ),
        minimum_focusing_coherence=(
            "focusing_coherence",
            "min",
        ),
        median_phase_rms_deg=(
            "circular_phase_rms_deg",
            "median",
        ),
        maximum_phase_rms_deg=(
            "circular_phase_rms_deg",
            "max",
        ),
    )
)


# Select one phase sign globally for all RX channels.
best_sign_row = sar_phase_sign_summary.loc[
    sar_phase_sign_summary[
        "median_focusing_coherence"
    ].idxmax()
]

selected_sar_phase_sign = int(
    best_sign_row[
        "phase_sign"
    ]
)


# Identify the strongest-coherence RX channel under the
# selected global phase-sign model.
selected_sign_results = (
    sar_phase_history_results[
        sar_phase_history_results[
            "phase_sign"
        ]
        == selected_sar_phase_sign
    ]
    .sort_values(
        by=[
            "focusing_coherence",
            "target_to_95th_reference_db",
        ],
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

provisional_sar_rx_channel = int(
    selected_sign_results.loc[
        0,
        "rx_channel",
    ]
)


# Calculate the largest predicted phase excursion.
maximum_expected_phase_excursion_deg = (
    np.max(
        np.abs(
            np.rad2deg(
                run_c_expected_unsigned_phase_rad
            )
        )
    )
)


print(
    "Target-associated range bin:",
    sar_phase_test_range_bin,
)

print(
    "Maximum predicted phase excursion:",
    f"{maximum_expected_phase_excursion_deg:.1f} degrees",
)

print(
    "Selected SAR phase sign:",
    f"{selected_sar_phase_sign:+d}",
)

print(
    "Provisional single-RX channel:",
    provisional_sar_rx_channel,
)


# ----------------------------------------------------------
# Plot measured and predicted phase histories.
# ----------------------------------------------------------

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 9),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for rx_channel, axis in enumerate(
    axes.flat
):
    predicted_phase_deg = np.rad2deg(
        selected_sar_phase_sign
        * run_c_expected_unsigned_phase_rad[
            :,
            rx_channel,
        ]
    )

    axis.plot(
        rail_positions_m * 100,
        run_c_measured_relative_phase_deg[
            :,
            rx_channel,
        ],
        marker="o",
        linewidth=1.5,
        label="Measured phase",
    )

    axis.plot(
        rail_positions_m * 100,
        predicted_phase_deg,
        marker="s",
        linestyle="--",
        linewidth=1.3,
        label="Geometric prediction",
    )

    channel_result = selected_sign_results[
        selected_sign_results[
            "rx_channel"
        ]
        == rx_channel
    ].iloc[0]

    axis.set_title(
        f"RX {rx_channel}: "
        f"coherence = "
        f"{channel_result['focusing_coherence']:.4f}, "
        f"RMS error = "
        f"{channel_result['circular_phase_rms_deg']:.1f}°"
    )

    axis.set_xlabel(
        "Rail position, $x_m$ (cm)"
    )

    axis.set_ylabel(
        "Phase relative to center (degrees)"
    )

    axis.grid(
        True,
        alpha=0.3,
    )

    axis.legend()

fig.suptitle(
    "Run C Measured and Predicted "
    f"Aperture Phase at q = {sar_phase_test_range_bin}",
    fontsize=14,
)

plt.show()


# ----------------------------------------------------------
# Plot target magnitude and phase-sign coherence.
# ----------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5),
    constrained_layout=True,
)


# Target magnitude variation over the rail.
for rx_channel in range(
    run_c_target_magnitude_db.shape[1]
):
    axes[0].plot(
        rail_positions_m * 100,
        run_c_target_magnitude_db[
            :,
            rx_channel,
        ],
        marker="o",
        linewidth=1.3,
        label=f"RX {rx_channel}",
    )

axes[0].set_title(
    f"Target Magnitude Across the Rail at "
    f"q = {sar_phase_test_range_bin}"
)

axes[0].set_xlabel(
    "Rail position, $x_m$ (cm)"
)

axes[0].set_ylabel(
    "Magnitude relative to channel maximum (dB)"
)

axes[0].grid(
    True,
    alpha=0.3,
)

axes[0].legend(
    ncols=2,
)


# Compare focusing coherence for both phase signs.
rx_channel_index = np.arange(
    run_c_target_magnitude_db.shape[1]
)

bar_width = 0.35

for sign_plot_index, phase_sign in enumerate(
    (-1, 1)
):
    sign_rows = (
        sar_phase_history_results[
            sar_phase_history_results[
                "phase_sign"
            ]
            == phase_sign
        ]
        .sort_values(
            "rx_channel"
        )
    )

    bar_offset = (
        sign_plot_index - 0.5
    ) * bar_width

    axes[1].bar(
        rx_channel_index + bar_offset,
        sign_rows[
            "focusing_coherence"
        ],
        width=bar_width,
        label=(
            f"Phase sign {phase_sign:+d}"
        ),
    )

axes[1].set_title(
    "Aperture-Focusing Coherence"
)

axes[1].set_xlabel(
    "RX channel"
)

axes[1].set_ylabel(
    "Focusing coherence"
)

axes[1].set_xticks(
    rx_channel_index
)

axes[1].set_ylim(
    0,
    1.02,
)

axes[1].grid(
    True,
    axis="y",
    alpha=0.3,
)

axes[1].legend()

plt.show()


# Display the sign-level and channel-level results.
display(
    sar_phase_sign_summary
)

display(
    sar_phase_history_results.sort_values(
        by=[
            "phase_sign",
            "rx_channel",
        ]
    )
)

### 3.3.3 — Results and Interpretation

The phase-history test extracted the complex Run C response at

$$q_t=22$$

and compared its phase across the 17 rail positions with the phase predicted for a target initially assumed to be at

$$x_t=0\ \text{m},\qquad z_t=0.84\ \text{m}.$$

The largest predicted phase excursion is approximately $250^\circ$. This value includes the offsets of the individual RX elements, causing the outer RX phase centers to extend slightly beyond the nominal $\pm5$ cm rail aperture.

<div style="font-size:2.0em; font-weight:bold;">
Sub-bin range change
</div>

The measured phase curvature is produced by the same changing slant-range geometry that can cause range migration. However, the target does not visibly migrate between integer range bins.

The rail-induced range change is only approximately $1.5$–$1.7$ mm, while the nominal range-bin spacing is

$$\Delta R_{\mathrm{bin}}=46.875\ \text{mm}.$$

The target-associated response therefore remains in $q=22$. Although the distance change is much smaller than one range bin, it produces more than half a cycle of phase change at 61 GHz.

The present measurement is best described as a **sub-bin range change observed through phase**, rather than resolved range migration.

The unexpected absolute range location remains visible in the earlier range-power result: the strongest sphere-associated response occurs at $q=22$ rather than the geometrically expected $q=17$–$19$.

The phase histories are referenced to the center aperture position:

$$\Delta\phi_m=\phi_m-\phi_{\mathrm{center}}=\frac{4\pi}{\lambda}\left(R_m-R_{\mathrm{center}}\right).$$

Consequently, constant absolute range and system-phase terms cancel. The phase plots test the change in distance across the aperture, while the range-bin plots contain the FMCW absolute-range information.

<div style="font-size:2.0em; font-weight:bold;">
Measured aperture phase
</div>

All four RX channels exhibit a curved phase history rather than random position-to-position phase.

The phase generally increases toward the aperture edges, as expected because the propagation distance increases away from the aperture position closest to the target. RX 2 and RX 3 agree particularly well with the geometric prediction over the positive half of the rail.

The measured phase curves are not symmetric about $x_m=0$. Their minima appear toward the negative side of the rail, suggesting that the dominant effective scattering center is not located exactly at the assumed $x_t=0$ position.

For a point target,

$$R_m=\sqrt{(x_m-x_t)^2+z_t^2},$$

and near its closest aperture position,

$$R_m-R_{\min}\approx\frac{(x_m-x_t)^2}{2z_t}.$$

The phase-history shape therefore contains two kinds of spatial information:

- the horizontal location of the phase minimum contains cross-range information about $x_t$;
- the curvature contains downrange information about $z_t$.

The asymmetry may also be influenced by the electrically large sphere, multipath, uncertainty in the V-MD3 antenna phase centers, or multiple scattering contributions within $q=22$.

<div style="font-size:2.0em; font-weight:bold;">
Magnitude across the aperture
</div>

The magnitude plot shows the strength of the $q=22$ complex response at each rail position. Each RX channel is normalized to its own maximum, so the plot shows position-to-position variation within each channel rather than absolute gain differences between channels.

| RX channel | Aperture-magnitude spread |
|---:|---:|
| 0 | approximately $9.17$ dB |
| 1 | approximately $5.94$ dB |
| 2 | approximately $5.08$ dB |
| 3 | approximately $5.98$ dB |

The target-associated response remains measurable across the complete aperture, but its amplitude is not constant. RX 0 contains the greatest variation, including an isolated reduction near $x_m=-2.5$ cm.

The variation may result from the antenna pattern, aspect-dependent sphere scattering, multipath, imperfect background subtraction, or fractional redistribution of energy among neighboring range bins.

These measured amplitudes will be retained during SAR processing. Independently normalizing every rail position could artificially amplify weak or noise-dominated measurements.

<div style="font-size:2.0em; font-weight:bold;">
Propagation-phase sign
</div>

The two tested measured-phase models were

$$\psi_m^{(+)}=+\frac{4\pi}{\lambda}\Delta R_m,$$

and

$$\psi_m^{(-)}=-\frac{4\pi}{\lambda}\Delta R_m.$$

Their aperture-focusing coherences are:

| RX channel | Phase sign $-1$ | Phase sign $+1$ |
|---:|---:|---:|
| 0 | $0.356$ | $0.604$ |
| 1 | $0.340$ | $0.653$ |
| 2 | $0.316$ | $0.764$ |
| 3 | $0.277$ | $0.792$ |

The $+1$ model produces substantially greater focusing coherence for every RX channel. Its median coherence is approximately $0.709$, compared with approximately $0.328$ for the $-1$ model.

The measured propagation phase is therefore modeled as

$$\psi_m=+\frac{4\pi}{\lambda}\Delta R_m.$$

To focus the measured data, the conjugate phase correction must be applied:

$$\exp\left(-j\frac{4\pi}{\lambda}\Delta R_m\right).$$

The consistent preference for the same sign across all four channels demonstrates that the $q=22$ residual contains a physical position-dependent phase history rather than random subtraction residue.

<div style="font-size:2.0em; font-weight:bold;">
Aperture focusing
</div>

Geometric phase compensation followed by coherent summation across the synthetic aperture is called **SAR focusing**.

Each complex rail sample can be interpreted as an arrow whose length is its magnitude and whose direction is its phase. Before focusing, these arrows point in different directions because the target-to-radar distance changes with rail position.

For a candidate target location, the predicted phase is removed:

$$\widetilde{g}_m=g_m\exp(-j\psi_m).$$

The corrected measurements are coherently summed, and their focusing coherence is

$$\eta=\frac{\left|\sum_m\widetilde{g}_m\right|}{\sum_m|g_m|}.$$

The denominator represents the largest possible sum magnitude for the measured sample amplitudes.

Therefore:

- $\eta=1$ represents perfect phase alignment;
- $\eta\approx0$ represents strong phase cancellation;
- an intermediate value represents partial alignment.

RX 3 produces the largest focusing coherence:

$$\eta_{\mathrm{RX3}}\approx0.792.$$

This means that focusing RX 3 at the assumed location $x=0$, $z=0.84$ m produces approximately $79.2\%$ of the maximum coherent-sum magnitude possible from those 17 samples.

This is not an image-accuracy percentage. It measures coherent-addition efficiency for only the tested candidate location.

The present calculation is a **single-point focusing test**. Full backprojection repeats this operation for every candidate image pixel. A pixel whose predicted phase history matches the measured data will produce a large coherent sum, while an incorrect pixel will produce greater cancellation.

This is also synthetic-aperture beamforming. The earlier Run A beamforming combined four simultaneous physical RX channels. SAR focusing combines measurements acquired at the 17 translated radar positions.

<div style="font-size:2.0em; font-weight:bold;">
Provisional RX selection
</div>

Under the selected $+1$ phase convention:

| RX | Focusing coherence | RMS phase error | Target-to-reference level | Magnitude spread |
|---:|---:|---:|---:|---:|
| 0 | $0.604$ | $53.5^\circ$ | $12.34$ dB | $9.17$ dB |
| 1 | $0.653$ | $55.0^\circ$ | $12.52$ dB | $5.94$ dB |
| 2 | $0.764$ | $52.1^\circ$ | $12.16$ dB | $5.08$ dB |
| 3 | $0.792$ | $50.2^\circ$ | $14.96$ dB | $5.98$ dB |

RX 3 provides the greatest focusing coherence, lowest RMS phase error, and greatest target-to-reference level. It is therefore selected provisionally for the first single-RX backprojection image.

The approximately $50^\circ$ residual phase error indicates that the initial $x=0$, $z=0.84$ m point model is not an exact description of the measured sphere response. Backprojection will search over both cross-range and downrange rather than forcing the target to this initial test location.

These results validate proceeding to two-dimensional SAR backprojection using RX 3 and the selected $+1$ measured-phase convention.

## 3.4 — Form the First Single-RX SAR Image by Backprojection

This section forms the first two-dimensional SAR image from the complex background-subtracted Run C data.

The image coordinates are:

- $x$: cross-range, parallel to the mechanical rail;
- $z$: downrange coordinate within the two-dimensional slant-range imaging plane.

For the first reconstruction, RX 3 is treated as a translated single-element radar. This avoids combining the four RX channels before their complete bistatic phase-center geometry is known.

<div style="font-size:2.0em; font-weight:bold;">
Backprojection geometry
</div>

For a candidate image pixel at $(x,z)$, the approximate distance from aperture position $m$ is

$$R_m(x,z)=\sqrt{\left(x-x_m\right)^2+z^2}.$$

Here, $x_m$ includes the mechanical rail position and the fixed offset of the selected RX element.

This is a monostatic phase-center approximation. The exact V-MD3 path is transmitter-to-pixel plus pixel-to-receiver, but the transmitter coordinate inside the module is not presently available.

<div style="font-size:2.0em; font-weight:bold;">
Local range registration
</div>

The measured sphere-associated return occurs at $q_{\mathrm{ref}}=22$, while its known physical slant range is approximately $0.84$ m. The first image therefore uses a provisional one-point range registration.

Let $R_{\mathrm{reg}}$ be the distance from the center-aperture RX position to the known reference point $(0,0.84\ \text{m})$. The fractional measured range-bin coordinate associated with a candidate pixel is

$$\widehat{q}_m(x,z)=q_{\mathrm{ref}}+\frac{R_m(x,z)-R_{\mathrm{reg}}}{\Delta R_{\mathrm{bin}}}.$$

This maps the known target reference point to $q=22$ while preserving geometric range changes across the image and aperture.

It is not a complete range calibration. It does not establish whether the discrepancy is a constant offset, scale error, timing delay, or target-associated propagation effect.

Because $\widehat{q}_m$ is generally not an integer, the complex range profile is linearly interpolated between neighboring bins. Interpolation is applied to the complex values before magnitude calculation.

<div style="font-size:2.0em; font-weight:bold;">
Coherent backprojection
</div>

Section 3.3 established that the measured aperture phase follows the $+1$ convention. The focusing correction is therefore

$$\exp\left[-j\frac{4\pi}{\lambda}\left(R_m(x,z)-R_{\mathrm{center}}(x,z)\right)\right].$$

Subtracting the center-position range removes only a phase term common to all aperture samples for a given pixel. It does not change the focused magnitude.

With aperture weights $a_m$, the backprojected image is

$$I_{\mathrm{BP}}(x,z)=\sum_m a_m G_m\left[\widehat{q}_m(x,z)\right]\exp\left[-j\frac{4\pi}{\lambda}\left(R_m(x,z)-R_{\mathrm{center}}(x,z)\right)\right].$$

The first reconstruction uses uniform aperture weights. Therefore, every measured rail position is admitted equally, although the measured target amplitudes remain naturally unequal.

No additional Hann or Hamming aperture window is applied yet. Uniform weighting preserves the greatest available cross-range resolution and provides the clearest baseline for evaluating the synthetic aperture.

<div style="font-size:2.0em; font-weight:bold;">
Noncoherent comparison
</div>

For comparison, a noncoherent geometric accumulation is also calculated:

$$I_{\mathrm{NC}}(x,z)=\sum_m a_m\left|G_m\left[\widehat{q}_m(x,z)\right]\right|.$$

This applies the same range geometry but discards the aperture phase before summation. It cannot provide true SAR cross-range focusing.

Comparing $I_{\mathrm{NC}}$ with $I_{\mathrm{BP}}$ demonstrates how geometric phase compensation changes a broad unfocused response into a spatially concentrated SAR response.

<div style="font-size:2.0em; font-weight:bold;">
Pixel-by-pixel focusing coherence
</div>

A focusing-coherence map is calculated as

$$\eta(x,z)=\frac{\left|I_{\mathrm{BP}}(x,z)\right|}{I_{\mathrm{NC}}(x,z)}.$$

A large value means that the phase correction for that pixel causes the aperture measurements to reinforce efficiently. A small value means that the corrected measurements continue to cancel.

A weak noise-dominated pixel can occasionally have high normalized coherence by chance. The coherence display is therefore masked where the focused image is more than 25 dB below its peak.

<div style="font-size:2.0em; font-weight:bold;">
Expected resolution
</div>

The approximate ideal cross-range resolution for a target at range $R_0$ is

$$\delta_x\approx\frac{\lambda R_0}{2L}.$$

For $\lambda\approx4.91$ mm, $R_0\approx0.84$ m, and $L=0.10$ m,

$$\delta_x\approx2.1\ \text{cm}.$$

The SAR processing should therefore provide substantially narrower cross-range focusing than the approximately $25.5^\circ$ four-RX beamformed response from Run A.

The range resolution is still set primarily by the FMCW bandwidth and range-window main lobe. SAR focusing does not create additional range bandwidth.

The sphere is an electrically large extended scatterer, so its measured image width is not necessarily equal to the point-target resolution. The image may contain one dominant scattering center, multiple scattering centers, or a broadened scattering region rather than an optical outline of the sphere.

### 3.4.1 — Backprojection and Image-Measurement Functions

The following functions:

- interpolate the complex range profile at fractional range-bin coordinates;
- apply geometric phase compensation;
- coherently sum the 17 aperture positions;
- calculate a noncoherent comparison image;
- calculate pixel-by-pixel focusing coherence;
- measure contiguous $-3$ dB image widths.

The backprojection operates on one RX channel at a time.

In [ ]:
def backproject_single_rx(
    aperture_range_profiles,
    aperture_positions_m,
    image_cross_range_m,
    image_downrange_m,
    carrier_frequency_hz,
    measured_phase_sign,
    reference_range_bin,
    registration_range_m,
    range_bin_spacing_m,
    reference_aperture_index,
    aperture_weights=None,
):
    """
    Form a two-dimensional single-RX SAR image using
    time-domain backprojection.

    The algorithm treats every image pixel as a possible
    scatterer location. For each candidate pixel, it:

        1. calculates the distance from every aperture
           position to the pixel;

        2. converts that distance into a fractional measured
           range-bin coordinate;

        3. interpolates the measured complex range profile;

        4. removes the predicted propagation phase;

        5. coherently sums the corrected measurements across
           the synthetic aperture.

    Inputs
    ------
    aperture_range_profiles:
        Complex range profiles with shape
        (aperture position, range bin).

    aperture_positions_m:
        Physical position of the selected RX phase center at
        every rail position.

    image_cross_range_m:
        One-dimensional array of candidate x coordinates.

    image_downrange_m:
        One-dimensional array of candidate z coordinates.

    measured_phase_sign:
        Sign of the measured propagation phase identified in
        Section 3.3.

    reference_range_bin:
        Measured range bin assigned to the known physical
        registration point.

    registration_range_m:
        Physical distance from the center-aperture phase
        center to the known registration point.

    range_bin_spacing_m:
        Nominal physical distance represented by one range bin.

    Returns
    -------
    coherent_image:
        Complex focused SAR image.

    noncoherent_image:
        Magnitude-only accumulation using the same range
        geometry but no phase focusing.

    focusing_coherence:
        Coherent-sum efficiency calculated for every pixel.

    valid_support_count:
        Number of aperture positions contributing a valid
        interpolated range sample to every pixel.
    """

    # Backprojection expects one complex range profile from
    # every physical aperture position. The first dimension is
    # rail position, and the second dimension is range bin.
    if aperture_range_profiles.ndim != 2:
        raise ValueError(
            "aperture_range_profiles must have shape "
            "(aperture position, range bin)."
        )

    (
        number_of_aperture_positions,
        number_of_range_bins,
    ) = aperture_range_profiles.shape

    # Every measured range profile must have a corresponding
    # physical aperture coordinate. Otherwise, the propagation
    # distance and phase cannot be calculated correctly.
    if aperture_positions_m.shape != (
        number_of_aperture_positions,
    ):
        raise ValueError(
            "aperture_positions_m has the wrong shape."
        )

    # If no deliberate synthetic-aperture window is supplied,
    # use uniform weighting. This gives every rail position the
    # same additional processing coefficient and provides the
    # narrowest baseline main lobe.
    if aperture_weights is None:
        aperture_weights = np.ones(
            number_of_aperture_positions,
            dtype=float,
        )
    else:
        aperture_weights = np.asarray(
            aperture_weights,
            dtype=float,
        )

    if aperture_weights.shape != (
        number_of_aperture_positions,
    ):
        raise ValueError(
            "aperture_weights has the wrong shape."
        )

    # Negative amplitude weights are not used in the present
    # aperture-window comparison. The measured complex phase
    # remains inside aperture_range_profiles.
    if np.any(
        aperture_weights < 0
    ):
        raise ValueError(
            "Aperture weights must be nonnegative."
        )

    if not np.any(
        aperture_weights > 0
    ):
        raise ValueError(
            "At least one aperture weight must be positive."
        )

    # Normalize the weights so that they sum to one.
    #
    # This makes the coherent result a weighted average instead
    # of a weighted sum. It changes only the overall image scale,
    # not the focused peak position, resolution, or normalized
    # image shape.
    aperture_weights = (
        aperture_weights
        / np.sum(
            aperture_weights
        )
    )

    # Convert the carrier frequency into wavelength.
    #
    # Wavelength determines how much propagation phase changes
    # for a given change in target distance.
    speed_of_light_m_per_s = 299_792_458.0

    wavelength_m = (
        speed_of_light_m_per_s
        / carrier_frequency_hz
    )

    # Construct a two-dimensional grid containing the x and z
    # coordinate of every candidate image pixel.
    #
    # Both output grids have shape:
    #   (number of downrange pixels, number of cross-range pixels)
    (
        image_cross_range_grid_m,
        image_downrange_grid_m,
    ) = np.meshgrid(
        image_cross_range_m,
        image_downrange_m,
    )

    # Initialize the complex focused-image accumulator.
    #
    # Each aperture position will contribute one interpolated,
    # phase-corrected complex value to every candidate pixel.
    coherent_image = np.zeros(
        image_cross_range_grid_m.shape,
        dtype=np.complex128,
    )

    # Initialize a second accumulator that adds only magnitudes.
    #
    # This produces a noncoherent comparison that uses the same
    # range geometry but discards aperture phase.
    noncoherent_image = np.zeros(
        image_cross_range_grid_m.shape,
        dtype=float,
    )

    # Count how many aperture positions provide a valid
    # fractional range-bin sample for every image pixel.
    #
    # Complete support means all 17 rail positions contributed
    # to the pixel.
    valid_support_count = np.zeros(
        image_cross_range_grid_m.shape,
        dtype=np.int32,
    )

    # Select the center rail position as the relative-phase
    # reference.
    #
    # SAR focusing depends on phase differences across the
    # aperture. Subtracting the center-position range removes a
    # phase common to all aperture measurements for one pixel.
    # That common phase does not affect the final image magnitude.
    center_aperture_position_m = (
        aperture_positions_m[
            reference_aperture_index
        ]
    )

    # Calculate the distance from the center aperture position
    # to every candidate image pixel.
    #
    # This will later be subtracted from the range calculated
    # at each other aperture position.
    center_pixel_range_m = np.sqrt(
        (
            image_cross_range_grid_m
            - center_aperture_position_m
        ) ** 2
        + image_downrange_grid_m ** 2
    )

    # Process one physical aperture position at a time.
    #
    # For efficiency, each loop iteration processes the entire
    # two-dimensional image grid simultaneously using NumPy.
    for aperture_index in range(
        number_of_aperture_positions
    ):
        aperture_position_m = (
            aperture_positions_m[
                aperture_index
            ]
        )

        # STEP 1: Calculate geometric distance.
        #
        # For this aperture position, calculate the distance to
        # every candidate image pixel.
        pixel_range_m = np.sqrt(
            (
                image_cross_range_grid_m
                - aperture_position_m
            ) ** 2
            + image_downrange_grid_m ** 2
        )

        # STEP 2: Convert physical distance into the measured
        # fractional range-bin coordinate.
        #
        # The local registration states that registration_range_m
        # corresponds to reference_range_bin. Distances greater
        # than the reference move toward larger bins, while
        # shorter distances move toward smaller bins.
        #
        # Example:
        #   fractional_range_bin = 22.35
        #
        # means that this candidate pixel should be evaluated
        # between measured range bins 22 and 23.
        fractional_range_bin = (
            reference_range_bin
            + (
                pixel_range_m
                - registration_range_m
            )
            / range_bin_spacing_m
        )

        # Identify the integer bin immediately below each
        # fractional bin coordinate.
        lower_range_bin = np.floor(
            fractional_range_bin
        ).astype(
            np.int64
        )

        # Save the fractional distance from the lower bin.
        #
        # For q = 22.35, this value is 0.35.
        interpolation_fraction = (
            fractional_range_bin
            - lower_range_bin
        )

        # Linear interpolation requires both a lower and an
        # upper bin. Therefore, valid coordinates must lie
        # between bin 0 and the second-to-last range bin.
        valid_range_mask = (
            (lower_range_bin >= 0)
            & (
                lower_range_bin
                < number_of_range_bins - 1
            )
        )

        # Clip the array indices only to prevent unsafe indexing.
        #
        # Clipping does not make an out-of-range pixel valid.
        # Invalid pixels will be set to zero after interpolation.
        lower_range_bin_clipped = np.clip(
            lower_range_bin,
            0,
            number_of_range_bins - 2,
        )

        upper_range_bin_clipped = (
            lower_range_bin_clipped + 1
        )

        # Extract the measured complex data from the two
        # neighboring range bins.
        lower_complex_sample = (
            aperture_range_profiles[
                aperture_index,
                lower_range_bin_clipped,
            ]
        )

        upper_complex_sample = (
            aperture_range_profiles[
                aperture_index,
                upper_range_bin_clipped,
            ]
        )

        # STEP 3: Interpolate the complex range profile.
        #
        # For q = 22.35:
        #
        #   G(22.35) = 0.65 G[22] + 0.35 G[23]
        #
        # Interpolation is performed on complex samples so that
        # both magnitude and phase are retained. Interpolating
        # magnitude alone would destroy the information required
        # for coherent SAR focusing.
        interpolated_complex_sample = (
            (
                1
                - interpolation_fraction
            )
            * lower_complex_sample
            + interpolation_fraction
            * upper_complex_sample
        )

        # Pixels whose predicted ranges fall outside the
        # measured range-profile support do not receive a
        # contribution from this aperture position.
        interpolated_complex_sample = np.where(
            valid_range_mask,
            interpolated_complex_sample,
            0.0 + 0.0j,
        )

        # STEP 4: Calculate the rail-induced range difference.
        #
        # The difference between pixel_range_m and the center
        # aperture range determines how much the measured phase
        # should change as the radar moves along the rail.
        relative_pixel_range_m = (
            pixel_range_m
            - center_pixel_range_m
        )

        # STEP 5: Remove the predicted propagation phase.
        #
        # Section 3.3 found that the measured data follow the
        # +4*pi*delta_R/lambda phase convention. Multiplication
        # by the negative exponential removes that phase.
        #
        # If the candidate pixel is correct, the compensated
        # measurements from the 17 rail positions should point
        # in approximately the same complex direction.
        phase_correction = np.exp(
            -1j
            * measured_phase_sign
            * 4
            * np.pi
            / wavelength_m
            * relative_pixel_range_m
        )

        aperture_weight = aperture_weights[
            aperture_index
        ]

        # STEP 6: Add the phase-corrected complex contribution.
        #
        # This is the coherent synthetic-aperture sum. Correct
        # candidate pixels reinforce; incorrect pixels undergo
        # partial phase cancellation.
        coherent_image += (
            aperture_weight
            * interpolated_complex_sample
            * phase_correction
        )

        # Form the noncoherent comparison by adding magnitude
        # before phase correction.
        #
        # This retains range-dependent amplitude but cannot
        # produce true synthetic-aperture cross-range focusing.
        noncoherent_image += (
            aperture_weight
            * np.abs(
                interpolated_complex_sample
            )
        )

        # Record that this aperture position supplied a valid
        # range sample to each supported pixel.
        valid_support_count += (
            valid_range_mask.astype(
                np.int32
            )
        )

    # Calculate the coherent-sum efficiency for each pixel.
    #
    # The denominator is the largest coherent magnitude that
    # could be obtained from the measured sample amplitudes if
    # all compensated samples aligned perfectly.
    focusing_coherence = (
        np.abs(
            coherent_image
        )
        / np.maximum(
            noncoherent_image,
            np.finfo(float).tiny,
        )
    )

    return (
        coherent_image,
        noncoherent_image,
        focusing_coherence,
        valid_support_count,
        image_cross_range_grid_m,
        image_downrange_grid_m,
    )


def normalized_power_db(
    complex_or_magnitude_data,
    minimum_db=-80.0,
):
    """
    Convert a complex image or a nonnegative magnitude image
    into normalized power in decibels.

    The result is normalized to its own maximum, so the
    strongest displayed pixel is 0 dB.
    """

    # abs() returns magnitude for complex data and leaves a
    # nonnegative magnitude image unchanged.
    magnitude = np.abs(
        complex_or_magnitude_data
    )

    # Radar images are displayed here as normalized power.
    power = magnitude ** 2

    normalized_power = (
        power
        / np.maximum(
            np.max(
                power
            ),
            np.finfo(float).tiny,
        )
    )

    power_db = (
        10
        * np.log10(
            np.maximum(
                normalized_power,
                np.finfo(float).tiny,
            )
        )
    )

    # Apply a lower display floor so extremely small numerical
    # values do not dominate the plot's color scale.
    return np.maximum(
        power_db,
        minimum_db,
    )


def measure_contiguous_minus_3db_width(
    coordinate,
    power_profile,
):
    """
    Measure the contiguous -3 dB width surrounding the
    strongest peak in a one-dimensional power profile.

    This measures the width of the observed image response.
    For an extended sphere, it should not automatically be
    interpreted as the point-target resolution.
    """

    coordinate = np.asarray(
        coordinate,
        dtype=float,
    )

    power_profile = np.asarray(
        power_profile,
        dtype=float,
    )

    if coordinate.ndim != 1:
        raise ValueError(
            "coordinate must be one-dimensional."
        )

    if power_profile.shape != coordinate.shape:
        raise ValueError(
            "power_profile must match coordinate."
        )

    # Normalize the profile so that its strongest point is
    # exactly 0 dB.
    normalized_power = (
        power_profile
        / np.maximum(
            np.max(
                power_profile
            ),
            np.finfo(float).tiny,
        )
    )

    power_profile_db = (
        10
        * np.log10(
            np.maximum(
                normalized_power,
                np.finfo(float).tiny,
            )
        )
    )

    # Identify the strongest point in the profile.
    peak_index = int(
        np.argmax(
            power_profile
        )
    )

    threshold_db = -3.0

    # Starting at the peak, walk left until the profile falls
    # below -3 dB. This keeps the measurement attached to the
    # main peak instead of accidentally including a separated
    # sidelobe above -3 dB.
    left_inside_index = peak_index

    while (
        left_inside_index > 0
        and power_profile_db[
            left_inside_index - 1
        ]
        >= threshold_db
    ):
        left_inside_index -= 1

    # Repeat the same search toward increasing coordinate.
    right_inside_index = peak_index

    while (
        right_inside_index
        < coordinate.size - 1
        and power_profile_db[
            right_inside_index + 1
        ]
        >= threshold_db
    ):
        right_inside_index += 1

    def interpolate_threshold_crossing(
        coordinate_1,
        level_1,
        coordinate_2,
        level_2,
        threshold,
    ):
        """
        Estimate the coordinate where two adjacent samples
        cross the selected dB threshold.
        """

        if np.isclose(
            level_1,
            level_2,
        ):
            return (
                coordinate_1
                + coordinate_2
            ) / 2

        return (
            coordinate_1
            + (
                threshold
                - level_1
            )
            * (
                coordinate_2
                - coordinate_1
            )
            / (
                level_2
                - level_1
            )
        )

    # Interpolate the left -3 dB crossing. If the profile
    # remains above -3 dB at the grid boundary, the width
    # cannot be measured completely.
    if left_inside_index == 0:
        left_crossing = np.nan
    else:
        left_crossing = (
            interpolate_threshold_crossing(
                coordinate[
                    left_inside_index - 1
                ],
                power_profile_db[
                    left_inside_index - 1
                ],
                coordinate[
                    left_inside_index
                ],
                power_profile_db[
                    left_inside_index
                ],
                threshold_db,
            )
        )

    # Interpolate the right -3 dB crossing.
    if (
        right_inside_index
        == coordinate.size - 1
    ):
        right_crossing = np.nan
    else:
        right_crossing = (
            interpolate_threshold_crossing(
                coordinate[
                    right_inside_index
                ],
                power_profile_db[
                    right_inside_index
                ],
                coordinate[
                    right_inside_index + 1
                ],
                power_profile_db[
                    right_inside_index + 1
                ],
                threshold_db,
            )
        )

    # A valid width requires both threshold crossings.
    if (
        np.isfinite(
            left_crossing
        )
        and np.isfinite(
            right_crossing
        )
    ):
        width = (
            right_crossing
            - left_crossing
        )
    else:
        width = np.nan

    return {
        "peak_index": peak_index,
        "peak_coordinate": coordinate[
            peak_index
        ],
        "left_crossing": left_crossing,
        "right_crossing": right_crossing,
        "width": width,
        "power_profile_db": power_profile_db,
    }

### 3.4.2 — Run the First Single-RX Backprojection

The first image uses:

- the complex background-subtracted Run C data;
- provisional RX channel 3;
- the measured $+1$ propagation-phase convention;
- uniform aperture weighting;
- the known $0.84$ m sphere range registered to measured bin $q=22$;
- linear interpolation of the complex range profiles.

The reconstruction grid covers

$$-0.10\ \text{m}\leq x\leq+0.10\ \text{m},$$

$$0.70\ \text{m}\leq z\leq1.05\ \text{m}.$$

The cross-range grid spacing is 1 mm, and the downrange grid spacing is 2 mm. These spacings oversample the expected physical resolution to produce a smooth image display. They do not create additional radar resolution.

<div style="font-size:2.0em; font-weight:bold;">
Backprojection as a pixel-by-pixel hypothesis test
</div>

Backprojection treats every candidate image pixel $(x,z)$ as a hypothesis:

> If a scatterer were located at this pixel, what complex measurement should it have produced at every rail position?

The algorithm calculates the propagation distance from the candidate pixel to every aperture position. It then extracts the measured complex range sample associated with that distance, removes the predicted propagation phase, and coherently adds the corrected measurements.

If the candidate pixel represents a real scatterer, the corrected measurements should align and reinforce. If the candidate location is incorrect, the measurements generally remain misaligned and partially cancel.

The process is repeated for every pixel to form the complete image.

<div style="font-size:2.0em; font-weight:bold;">
Step 1 — Select one complex aperture dataset
</div>

For the first reconstruction, the code selects RX channel 3 from the background-subtracted data:

$$G_m[q]=G_m[q,\ell=3].$$

The resulting array has dimensions

$$(17\ \text{aperture positions},\ 128\ \text{range bins}).$$

Each row is the complex range profile measured at one rail position. The values remain complex so that both amplitude and phase are available during focusing.

The physical position of RX 3 is approximated as

$$x_m^{(\mathrm{RX3})}=x_m+x_{\mathrm{RX3}},$$

where $x_m$ is the mechanical rail coordinate and $x_{\mathrm{RX3}}$ is the fixed RX-element offset.

<div style="font-size:2.0em; font-weight:bold;">
Step 2 — Construct the candidate image grid
</div>

The code creates arrays of candidate cross-range and downrange coordinates. A two-dimensional mesh grid represents every candidate pixel:

$$(x_i,z_j).$$

No measurement is directly assigned to these pixels yet. At this stage, the grid is only a collection of possible scatterer locations that will be tested against the measured data.

<div style="font-size:2.0em; font-weight:bold;">
Step 3 — Calculate the distance to each candidate pixel
</div>

For rail position $m$, the distance to every candidate pixel is

$$R_m(x,z)=\sqrt{\left(x-x_m\right)^2+z^2}.$$

This produces a complete range surface for one aperture position.

The function repeats this calculation for all 17 rail positions. Therefore, every image pixel is associated with 17 predicted propagation distances.

<div style="font-size:2.0em; font-weight:bold;">
Step 4 — Convert the predicted distance into a measured range-bin coordinate
</div>

The known reference target at approximately $0.84$ m appears in measured range bin $q_{\mathrm{ref}}=22$. The algorithm uses this as a provisional local registration.

For every pixel and aperture position, the predicted fractional range-bin coordinate is

$$\widehat{q}_m(x,z)=q_{\mathrm{ref}}+\frac{R_m(x,z)-R_{\mathrm{reg}}}{\Delta R_{\mathrm{bin}}}.$$

For example, a candidate location slightly farther than the reference location produces a fractional bin slightly greater than 22. A closer candidate produces a fractional bin below 22.

This operation tells the algorithm where in the measured complex range profile it should look for the hypothesized pixel.

<div style="font-size:2.0em; font-weight:bold;">
Step 5 — Interpolate the complex range data
</div>

The predicted coordinate $\widehat{q}_m(x,z)$ is generally not an integer. Suppose, for example, that

$$\widehat{q}_m=22.35.$$

The code linearly interpolates between complex bins 22 and 23:

$$G_m(22.35)=0.65G_m[22]+0.35G_m[23].$$

Interpolation is performed directly on the complex values, not on magnitude or power. This is necessary because SAR focusing depends on the phase of the interpolated measurement.

Interpolating magnitude first would discard the phase information and prevent coherent aperture focusing.

<div style="font-size:2.0em; font-weight:bold;">
Step 6 — Predict and remove the aperture phase
</div>

Even after selecting the correct range sample, the target measurement has a different phase at every rail position because its propagation distance changes.

Section 3.3 established the measured phase convention

$$\psi_m=+\frac{4\pi}{\lambda}\Delta R_m.$$

The code removes this predicted phase using

$$\exp\left[-j\frac{4\pi}{\lambda}\left(R_m(x,z)-R_{\mathrm{center}}(x,z)\right)\right].$$

The focused complex contribution from aperture position $m$ is therefore

$$\widetilde{G}_m(x,z)=G_m\left[\widehat{q}_m(x,z)\right]\exp\left[-j\frac{4\pi}{\lambda}\left(R_m(x,z)-R_{\mathrm{center}}(x,z)\right)\right].$$

The center-aperture range is subtracted only to remove a phase term common to all aperture positions for that pixel. A common phase rotation does not change the final image magnitude.

<div style="font-size:2.0em; font-weight:bold;">
Step 7 — Coherently sum the aperture positions
</div>

After phase compensation, the 17 complex contributions are added:

$$I_{\mathrm{BP}}(x,z)=\sum_m a_m\widetilde{G}_m(x,z).$$

The first image uses uniform aperture weights:

$$a_m=\frac{1}{17}.$$

Uniform weights do not force the measured target amplitudes to be equal. They simply apply the same additional processing coefficient to every rail position.

If the pixel location is correct, the phase-compensated complex values point in approximately the same direction and produce a large sum. If the pixel is incorrect, they point in different directions and partially cancel.

This coherent summation is the operation that **focuses the synthetic aperture**.

<div style="font-size:2.0em; font-weight:bold;">
Step 8 — Calculate the displayed image power
</div>

After every pixel has been tested, the complex backprojection result is converted to power:

$$P_{\mathrm{BP}}(x,z)=\left|I_{\mathrm{BP}}(x,z)\right|^2.$$

The image is normalized to its maximum and displayed in decibels:

$$P_{\mathrm{BP,dB}}(x,z)=10\log_{10}\left(\frac{P_{\mathrm{BP}}(x,z)}{\max P_{\mathrm{BP}}}\right).$$

Magnitude or power is calculated only after coherent aperture summation. Calculating magnitude before summation would destroy the phase relationships that produce SAR cross-range focusing.

<div style="font-size:2.0em; font-weight:bold;">
Noncoherent comparison
</div>

The code also forms a noncoherent geometric accumulation:

$$I_{\mathrm{NC}}(x,z)=\sum_m a_m\left|G_m\left[\widehat{q}_m(x,z)\right]\right|.$$

This uses the same candidate ranges and complex interpolation but takes magnitude before adding the aperture positions. It therefore discards the relative aperture phase.

The noncoherent result can place energy at plausible ranges, but it cannot provide true synthetic-aperture cross-range focusing.

Comparing the noncoherent and coherent images demonstrates how much spatial concentration is produced specifically by phase compensation and coherent integration.

<div style="font-size:2.0em; font-weight:bold;">
Focusing coherence
</div>

For every pixel, the code also calculates

$$\eta(x,z)=\frac{\left|I_{\mathrm{BP}}(x,z)\right|}{I_{\mathrm{NC}}(x,z)}.$$

This compares the magnitude of the actual coherent sum with the largest sum magnitude possible for the measured amplitudes.

A large value indicates that the candidate pixel’s phase model aligns the aperture measurements efficiently. A small value indicates continued phase cancellation.

A noise-dominated pixel can occasionally have high normalized coherence by chance. The coherence map is therefore displayed only where the focused image is within 25 dB of its peak.

<div style="font-size:2.0em; font-weight:bold;">
Equivalent algorithmic pseudocode
</div>

Conceptually, backprojection performs the following nested calculation:

```text
for every candidate image pixel (x, z):

    coherent_sum = 0
    noncoherent_sum = 0

    for every rail position m:

        calculate the distance R_m(x, z)

        convert R_m into fractional range bin q_hat_m

        interpolate the complex measured range profile
        to obtain G_m(q_hat_m)

        calculate the predicted propagation phase

        remove the predicted phase from G_m(q_hat_m)

        add the corrected complex value to coherent_sum

        add its magnitude to noncoherent_sum

    save coherent_sum as the complex SAR pixel

    calculate focusing coherence as
    abs(coherent_sum) / noncoherent_sum

In [ ]:
# Select the single RX channel identified in Section 3.3.
#
# The first SAR image uses only one RX channel because the
# exact V-MD3 transmitter and receiver phase-center geometry
# is not yet known. Using one channel avoids introducing an
# incorrect inter-RX phase model into the synthetic aperture.
selected_backprojection_rx_channel = (
    provisional_sar_rx_channel
)


# Retain the measured propagation-phase sign determined by the
# Section 3.3 phase-history test.
#
# A value of +1 means that the measured phase increases as
# +4*pi*delta_R/lambda. The backprojection function will apply
# the corresponding negative exponential to remove it.
backprojection_measured_phase_sign = (
    selected_sar_phase_sign
)


# Extract the selected channel's complex background-subtracted
# range profile at every rail position.
#
# Input shape before selection:
#   (17 positions, 128 range bins, 4 RX channels)
#
# Output shape:
#   (17 positions, 128 range bins)
#
# These complex profiles are the measurements that will be
# interpolated and coherently combined during backprojection.
selected_rx_aperture_profiles = (
    run_c_sar_background_subtracted[
        :,
        :,
        selected_backprojection_rx_channel,
    ]
)


# Approximate the physical phase-center position of the
# selected RX channel at every mechanical rail location.
#
# rail_positions_m describes the translated module center.
# rx_positions_m adds the selected element's fixed position
# inside the four-RX array.
#
# This remains a monostatic approximation because the exact
# transmitter coordinate inside the V-MD3 is not yet known.
selected_rx_aperture_positions_m = (
    rail_positions_m
    + rx_positions_m[
        selected_backprojection_rx_channel
    ]
)


# ----------------------------------------------------------
# Define the provisional one-point local range registration.
# ----------------------------------------------------------
#
# WHY THIS IS NEEDED:
#
# The physical sphere range is approximately 0.84 m, but the
# measured sphere-associated response occurs at range bin 22.
# Under the nominal 6 m / 128 mapping, bin 22 would instead be
# interpreted as approximately 1.03 m.
#
# Backprojection must know which measured range-bin sample to
# use for a candidate physical image location. Since a complete
# radar range calibration is not yet available, we provide one
# known correspondence:
#
#       known physical target location <--> measured bin 22
#
# This registration anchors the image's physical range axis.
#
# It does not alter the raw data, shift the range profiles, or
# prove that the radar has a constant 19 cm offset. It is only
# a local mapping used for this first image.
sar_reference_range_bin = (
    run_c_observed_range_bin
)

sar_reference_cross_range_m = 0.0
sar_reference_downrange_m = 0.84


# Determine the physical coordinate of the selected RX element
# at the center mechanical rail position.
selected_center_aperture_position_m = (
    selected_rx_aperture_positions_m[
        CENTER_POSITION_INDEX
    ]
)


# Calculate the exact geometric distance from the selected
# center-aperture RX phase center to the known registration
# point at x = 0 and z = 0.84 m.
#
# The distance is very close to 0.84 m but includes the small
# fixed cross-range offset of RX 3.
#
# The backprojection will enforce:
#
#       sar_registration_range_m <--> range bin 22
sar_registration_range_m = np.sqrt(
    (
        sar_reference_cross_range_m
        - selected_center_aperture_position_m
    ) ** 2
    + sar_reference_downrange_m ** 2
)


# Define the nominal physical spacing between measured range
# bins.
#
# This spacing is used only to convert a physical range change
# relative to the registration point into a fractional bin
# change. The registration above supplies the absolute anchor.
sar_nominal_maximum_range_m = 6.0

sar_number_of_range_bins = (
    selected_rx_aperture_profiles.shape[1]
)

sar_nominal_range_bin_spacing_m = (
    sar_nominal_maximum_range_m
    / sar_number_of_range_bins
)


# Define the physical coordinates to test during image
# formation.
#
# Every combination of x and z is treated as a candidate
# scatterer location. Backprojection will determine how well
# the measured aperture data focus at each candidate.
sar_image_cross_range_m = np.linspace(
    -0.10,
    0.10,
    201,
)

sar_image_downrange_m = np.linspace(
    0.70,
    1.05,
    176,
)


# The 1 mm cross-range and 2 mm downrange grid spacings
# oversample the expected physical resolution.
#
# Fine pixels make the image and peak-location estimate smooth,
# but they do not improve the radar's true resolution.
sar_cross_range_grid_spacing_m = (
    sar_image_cross_range_m[1]
    - sar_image_cross_range_m[0]
)

sar_downrange_grid_spacing_m = (
    sar_image_downrange_m[1]
    - sar_image_downrange_m[0]
)


# Use uniform synthetic-aperture weighting for the first image.
#
# This gives each rail position the same additional processing
# coefficient and preserves the narrowest baseline main lobe.
# The naturally measured amplitude differences between rail
# positions remain present in the complex data.
sar_uniform_aperture_weights = np.ones(
    selected_rx_aperture_profiles.shape[0],
    dtype=float,
)


# Run the backprojection algorithm.
#
# For every candidate image pixel, the function:
#
#   1. calculates range from all 17 aperture positions;
#   2. maps each predicted range to a fractional measured bin;
#   3. interpolates the complex range profiles;
#   4. removes the predicted rail-dependent phase;
#   5. coherently sums the corrected measurements.
#
# It simultaneously creates a magnitude-only noncoherent
# comparison and a focusing-coherence map.
(
    run_c_sar_backprojection_image,
    run_c_sar_noncoherent_image,
    run_c_sar_focusing_coherence,
    run_c_sar_valid_support_count,
    run_c_sar_cross_range_grid_m,
    run_c_sar_downrange_grid_m,
) = backproject_single_rx(
    aperture_range_profiles=(
        selected_rx_aperture_profiles
    ),
    aperture_positions_m=(
        selected_rx_aperture_positions_m
    ),
    image_cross_range_m=(
        sar_image_cross_range_m
    ),
    image_downrange_m=(
        sar_image_downrange_m
    ),
    carrier_frequency_hz=(
        carrier_frequency_hz
    ),
    measured_phase_sign=(
        backprojection_measured_phase_sign
    ),
    reference_range_bin=(
        sar_reference_range_bin
    ),
    registration_range_m=(
        sar_registration_range_m
    ),
    range_bin_spacing_m=(
        sar_nominal_range_bin_spacing_m
    ),
    reference_aperture_index=(
        CENTER_POSITION_INDEX
    ),
    aperture_weights=(
        sar_uniform_aperture_weights
    ),
)


# Verify that every image pixel used all 17 rail positions.
#
# A smaller support count would mean that the predicted range
# for at least one aperture position fell outside the measured
# 128-bin range profile. Different support across the image
# could introduce artificial brightness changes.
assert np.all(
    run_c_sar_valid_support_count
    == selected_rx_aperture_profiles.shape[0]
), (
    "At least one image pixel does not have complete "
    "aperture support."
)


# Convert both image products into normalized power in dB.
#
# They are independently normalized to 0 dB so that their
# spatial shapes can be compared. Their colors do not directly
# compare absolute coherent and noncoherent amplitude.
sar_image_display_floor_db = -25.0

run_c_sar_backprojection_db = normalized_power_db(
    run_c_sar_backprojection_image,
    minimum_db=sar_image_display_floor_db,
)

run_c_sar_noncoherent_db = normalized_power_db(
    run_c_sar_noncoherent_image,
    minimum_db=sar_image_display_floor_db,
)


# Calculate focused-image power before locating its peak.
run_c_sar_image_power = (
    np.abs(
        run_c_sar_backprojection_image
    ) ** 2
)


# Find the pixel that produced the largest coherent sum.
#
# This is the candidate location whose range and phase model
# best concentrate the measured aperture data.
(
    sar_peak_downrange_index,
    sar_peak_cross_range_index,
) = np.unravel_index(
    np.argmax(
        run_c_sar_image_power
    ),
    run_c_sar_image_power.shape,
)


# Convert the strongest pixel's array indices into physical
# cross-range and downrange coordinates.
sar_peak_cross_range_m = (
    sar_image_cross_range_m[
        sar_peak_cross_range_index
    ]
)

sar_peak_downrange_m = (
    sar_image_downrange_m[
        sar_peak_downrange_index
    ]
)


# Read the focusing coherence at the strongest image pixel.
#
# This reports how efficiently the 17 phase-corrected complex
# measurements reinforce at the selected image location.
sar_peak_focusing_coherence = (
    run_c_sar_focusing_coherence[
        sar_peak_downrange_index,
        sar_peak_cross_range_index,
    ]
)


# Determine which fractional measured range bin the focused
# peak uses at the center aperture position.
#
# A value near 22 means the reconstructed peak remains tied to
# the observed sphere-associated response. A substantially
# different value would indicate that another range feature
# became dominant during reconstruction.
sar_peak_center_range_m = np.sqrt(
    (
        sar_peak_cross_range_m
        - selected_center_aperture_position_m
    ) ** 2
    + sar_peak_downrange_m ** 2
)

sar_peak_center_range_bin = (
    sar_reference_range_bin
    + (
        sar_peak_center_range_m
        - sar_registration_range_m
    )
    / sar_nominal_range_bin_spacing_m
)


# Extract one-dimensional image-power profiles through the
# strongest focused pixel.
#
# The cross-range profile measures horizontal focusing at the
# peak downrange coordinate.
#
# The downrange profile measures range extent at the peak
# cross-range coordinate.
sar_cross_range_power_profile = (
    run_c_sar_image_power[
        sar_peak_downrange_index,
        :,
    ]
)

sar_downrange_power_profile = (
    run_c_sar_image_power[
        :,
        sar_peak_cross_range_index,
    ]
)


# Measure the contiguous -3 dB widths around the strongest
# peak in both image directions.
#
# These are observed sphere-response widths. Since the sphere
# is not an ideal point target, they are not automatically the
# system's point-target resolutions.
sar_cross_range_width_result = (
    measure_contiguous_minus_3db_width(
        coordinate=sar_image_cross_range_m,
        power_profile=(
            sar_cross_range_power_profile
        ),
    )
)

sar_downrange_width_result = (
    measure_contiguous_minus_3db_width(
        coordinate=sar_image_downrange_m,
        power_profile=(
            sar_downrange_power_profile
        ),
    )
)


# Calculate the ideal broadside point-target cross-range
# resolution expected from the synthetic aperture:
#
#       delta_x approximately lambda*R / (2*L)
#
# This gives a theoretical reference for the observed focused
# width. An extended sphere can appear wider than this value.
speed_of_light_m_per_s = 299_792_458.0

sar_wavelength_m = (
    speed_of_light_m_per_s
    / carrier_frequency_hz
)

sar_effective_aperture_length_m = (
    selected_rx_aperture_positions_m[-1]
    - selected_rx_aperture_positions_m[0]
)

sar_theoretical_cross_range_resolution_m = (
    sar_wavelength_m
    * sar_peak_downrange_m
    / (
        2
        * sar_effective_aperture_length_m
    )
)


# Collect the most important first-image measurements into one
# table for later interpretation and comparison with alternative
# processing choices.
sar_first_image_metrics = pd.DataFrame(
    [
        {
            "rx_channel": (
                selected_backprojection_rx_channel
            ),
            "measured_phase_sign": (
                backprojection_measured_phase_sign
            ),
            "peak_cross_range_m": (
                sar_peak_cross_range_m
            ),
            "peak_downrange_m": (
                sar_peak_downrange_m
            ),
            "peak_center_range_bin": (
                sar_peak_center_range_bin
            ),
            "peak_focusing_coherence": (
                sar_peak_focusing_coherence
            ),
            "observed_minus3db_cross_range_width_m": (
                sar_cross_range_width_result[
                    "width"
                ]
            ),
            "observed_minus3db_downrange_width_m": (
                sar_downrange_width_result[
                    "width"
                ]
            ),
            "ideal_point_target_cross_range_resolution_m": (
                sar_theoretical_cross_range_resolution_m
            ),
            "nominal_range_bin_spacing_m": (
                sar_nominal_range_bin_spacing_m
            ),
        }
    ]
)


# Print the principal reconstruction results before plotting.
print(
    "Selected RX channel:",
    selected_backprojection_rx_channel,
)

print(
    "Measured phase sign:",
    f"{backprojection_measured_phase_sign:+d}",
)

print(
    "Peak cross-range:",
    f"{sar_peak_cross_range_m:.4f} m",
)

print(
    "Peak downrange:",
    f"{sar_peak_downrange_m:.4f} m",
)

print(
    "Peak center-aperture range-bin coordinate:",
    f"{sar_peak_center_range_bin:.3f}",
)

print(
    "Peak focusing coherence:",
    f"{sar_peak_focusing_coherence:.4f}",
)


# ----------------------------------------------------------
# Compare the unfocused and focused spatial results.
# ----------------------------------------------------------

# A weak noise-dominated pixel can sometimes have high
# normalized coherence by chance. Display coherence only where
# focused image power is within 25 dB of the image peak.
coherence_display_mask = (
    run_c_sar_backprojection_db
    >= -25.0
)

masked_focusing_coherence = np.ma.masked_where(
    ~coherence_display_mask,
    run_c_sar_focusing_coherence,
)


# Create a private copy of the magma color map.
#
# Values masked above are assigned the color black. This makes
# it visually clear that these locations were intentionally
# excluded rather than assigned a measured coherence value.
coherence_colormap = plt.colormaps[
    "magma"
].copy()

coherence_colormap.set_bad(
    color="black",
    alpha=1.0,
)


fig, axes = plt.subplots(
    1,
    3,
    figsize=(17, 6),
    constrained_layout=True,
)

# Express both image axes in centimeters so the plot has a
# physically meaningful and equal spatial aspect ratio.
image_extent_cm = [
    sar_image_cross_range_m[0] * 100,
    sar_image_cross_range_m[-1] * 100,
    sar_image_downrange_m[0] * 100,
    sar_image_downrange_m[-1] * 100,
]


# Plot magnitude-only accumulation.
#
# This uses candidate range geometry but does not use aperture
# phase to obtain cross-range focusing.
noncoherent_handle = axes[0].imshow(
    run_c_sar_noncoherent_db,
    origin="lower",
    extent=image_extent_cm,
    aspect="equal",
    cmap="viridis",
    vmin=sar_image_display_floor_db,
    vmax=0,
)

axes[0].set_title(
    "Noncoherent Geometric Accumulation"
)

axes[0].set_xlabel(
    "Cross-range, $x$ (cm)"
)

axes[0].set_ylabel(
    "Downrange, $z$ (cm)"
)

fig.colorbar(
    noncoherent_handle,
    ax=axes[0],
    label="Normalized power (dB)",
)


# Plot the coherently focused SAR image.
#
# Spatial concentration in this panel is produced by removing
# the candidate pixel's predicted aperture phase before summing.
coherent_handle = axes[1].imshow(
    run_c_sar_backprojection_db,
    origin="lower",
    extent=image_extent_cm,
    aspect="equal",
    cmap="viridis",
    vmin=sar_image_display_floor_db,
    vmax=0,
)

axes[1].set_title(
    "Coherent SAR Backprojection"
)

axes[1].set_xlabel(
    "Cross-range, $x$ (cm)"
)

axes[1].set_ylabel(
    "Downrange, $z$ (cm)"
)

fig.colorbar(
    coherent_handle,
    ax=axes[1],
    label="Normalized power (dB)",
)


# Plot coherent-sum efficiency for image regions with
# meaningful focused power.
coherence_handle = axes[2].imshow(
    masked_focusing_coherence,
    origin="lower",
    extent=image_extent_cm,
    aspect="equal",
    cmap=coherence_colormap,
    vmin=0,
    vmax=1,
)

axes[2].set_title(
    "Focusing Coherence "
    "(Masked Where Image < -25 dB)"
)

axes[2].set_xlabel(
    "Cross-range, $x$ (cm)"
)

axes[2].set_ylabel(
    "Downrange, $z$ (cm)"
)

fig.colorbar(
    coherence_handle,
    ax=axes[2],
    label="Focusing coherence",
)


for axis in axes:
    # Mark the physical point used to register q = 22 to
    # approximately 0.84 m. This is an input reference, not a
    # forced cross-range peak location.
    axis.plot(
        sar_reference_cross_range_m * 100,
        sar_reference_downrange_m * 100,
        marker="+",
        color="red",
        markersize=14,
        markeredgewidth=2,
        label="Initial reference",
    )

    # Mark the location that produced the largest coherent
    # backprojection sum.
    axis.plot(
        sar_peak_cross_range_m * 100,
        sar_peak_downrange_m * 100,
        marker="x",
        color="cyan",
        markersize=10,
        markeredgewidth=2,
        label="Focused peak",
    )

axes[1].legend(
    loc="upper right",
)

fig.suptitle(
    "Run C First Single-RX SAR Reconstruction",
    fontsize=15,
)

plt.show()


# ----------------------------------------------------------
# Examine one-dimensional cuts through the focused peak.
# ----------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5),
    constrained_layout=True,
)


# Plot the cross-range response while holding downrange fixed
# at the focused peak. This shows the horizontal concentration
# produced by the synthetic aperture.
axes[0].plot(
    sar_image_cross_range_m * 100,
    sar_cross_range_width_result[
        "power_profile_db"
    ],
    color="black",
    linewidth=1.5,
)

axes[0].axhline(
    -3,
    color="gray",
    linestyle="--",
    linewidth=1,
    label="-3 dB",
)

axes[0].axvline(
    sar_peak_cross_range_m * 100,
    color="cyan",
    linestyle="-.",
    linewidth=1.5,
    label="Focused peak",
)

axes[0].axvline(
    0,
    color="red",
    linestyle=":",
    linewidth=1.5,
    label="Initial reference",
)

if np.isfinite(
    sar_cross_range_width_result[
        "left_crossing"
    ]
):
    axes[0].axvspan(
        sar_cross_range_width_result[
            "left_crossing"
        ] * 100,
        sar_cross_range_width_result[
            "right_crossing"
        ] * 100,
        color="green",
        alpha=0.15,
        label="-3 dB width",
    )

axes[0].set_title(
    "Cross-Range Profile Through Focused Peak"
)

axes[0].set_xlabel(
    "Cross-range, $x$ (cm)"
)

axes[0].set_ylabel(
    "Normalized power (dB)"
)

axes[0].set_ylim(
    -40,
    2,
)

axes[0].grid(
    True,
    alpha=0.3,
)

axes[0].legend()


# Plot the downrange response while holding cross-range fixed
# at the focused peak. Its width is governed mainly by the FMCW
# range response rather than the synthetic-aperture length.
axes[1].plot(
    sar_image_downrange_m,
    sar_downrange_width_result[
        "power_profile_db"
    ],
    color="black",
    linewidth=1.5,
)

axes[1].axhline(
    -3,
    color="gray",
    linestyle="--",
    linewidth=1,
    label="-3 dB",
)

axes[1].axvline(
    sar_peak_downrange_m,
    color="cyan",
    linestyle="-.",
    linewidth=1.5,
    label="Focused peak",
)

axes[1].axvline(
    sar_reference_downrange_m,
    color="red",
    linestyle=":",
    linewidth=1.5,
    label="Initial reference",
)

if np.isfinite(
    sar_downrange_width_result[
        "left_crossing"
    ]
):
    axes[1].axvspan(
        sar_downrange_width_result[
            "left_crossing"
        ],
        sar_downrange_width_result[
            "right_crossing"
        ],
        color="green",
        alpha=0.15,
        label="-3 dB width",
    )

axes[1].set_title(
    "Downrange Profile Through Focused Peak"
)

axes[1].set_xlabel(
    "Downrange, $z$ (m)"
)

axes[1].set_ylabel(
    "Normalized power (dB)"
)

axes[1].set_ylim(
    -40,
    2,
)

axes[1].grid(
    True,
    alpha=0.3,
)

axes[1].legend()

plt.show()


# Display the numerical image measurements for use in the
# Section 3.4 results summary.
display(
    sar_first_image_metrics
)

### 3.4.3 — Results and Interpretation

The first single-RX backprojection successfully produced a spatially focused response from the complex Run C synthetic-aperture data.

The principal reconstruction results are:

| Measurement | Result |
|---|---:|
| Selected RX channel | RX 3 |
| Measured propagation-phase sign | $+1$ |
| Focused peak cross-range | $-0.007$ m |
| Focused peak downrange | $0.840$ m |
| Center-aperture range-bin coordinate | $22.001$ |
| Peak focusing coherence | $0.9607$ |
| Observed cross-range $-3$ dB width | $0.01765$ m |
| Observed downrange $-3$ dB width | $0.01846$ m |
| Approximate $\lambda R/(2L)$ value | $0.02064$ m |
| Nominal range-bin spacing | $0.046875$ m |

<div style="font-size:2.0em; font-weight:bold;">
Which panel is the SAR image?
</div>

The middle panel, labeled `Coherent SAR Backprojection`, is the first SAR image.

The three panels have different purposes:

- the left panel is a noncoherent geometric accumulation;
- the middle panel is the coherently focused SAR image;
- the right panel is a focusing-coherence diagnostic.

The noncoherent image uses the predicted pixel ranges but takes magnitude before summing the aperture positions. It therefore contains range-dependent horizontal bands but little meaningful cross-range localization.

The coherent image retains the complex phase, removes the predicted propagation phase for each candidate pixel, and then sums across the 17 rail positions. The resulting spatial concentration is produced by synthetic-aperture focusing.

<div style="font-size:2.0em; font-weight:bold;">
Focused peak location
</div>

The strongest focused response occurs at

$$x_{\mathrm{peak}}=-0.007\ \text{m},$$

$$z_{\mathrm{peak}}=0.840\ \text{m}.$$

The approximately $-0.7$ cm cross-range location is consistent with the asymmetry observed in the Section 3.3 phase-history plots. Those plots suggested that the effective scattering center was slightly toward the negative side of the rail rather than exactly at $x=0$.

Section 3.3 tested only the assumed location $x=0$, $z=0.84$ m and obtained an RX 3 focusing coherence of approximately

$$\eta\approx0.792.$$

Backprojection searched over cross-range and found a better-matching location at $x=-0.7$ cm. At that location, the focusing coherence increased to

$$\eta_{\mathrm{peak}}\approx0.961.$$

This substantial increase demonstrates that the measured aperture phase contains usable cross-range position information.

The peak center-aperture range-bin coordinate is

$$\widehat q_{\mathrm{peak}}=22.001.$$

The focused image therefore uses the same $q=22$ sphere-associated response identified in the fixed-position and unfocused-aperture analyses.

<div style="font-size:2.0em; font-weight:bold;">
Meaning of the downrange location
</div>

The reconstructed peak appears at $z=0.84$ m because the first image uses the provisional local registration

$$q=22\longleftrightarrow R\approx0.84\ \text{m}.$$

This registration anchors the strongest measured response to the independently measured physical sphere range.

The resulting $z=0.84$ m location is therefore not an independent confirmation that the original nominal range mapping was correct. The measured data still place the sphere-associated response at $q=22$, which nominally corresponds to approximately $1.03$ m.

The registration provides a physical coordinate system for the first SAR image without yet claiming a complete radar range calibration.

In contrast, the $x=-0.7$ cm result was not forced by the range registration. It was selected through coherent aperture-phase focusing.

<div style="font-size:2.0em; font-weight:bold;">
Focusing coherence
</div>

For every candidate pixel, the focusing coherence is

$$\eta(x,z)=\frac{\left|I_{\mathrm{BP}}(x,z)\right|}{I_{\mathrm{NC}}(x,z)}.$$

It compares the magnitude of the coherent aperture sum with the largest sum magnitude possible from the measured sample amplitudes.

The peak value

$$\eta_{\mathrm{peak}}\approx0.961$$

means that the phase-corrected aperture samples at the focused peak produce approximately $96.1\%$ of their maximum possible coherent-sum magnitude.

This is strong evidence that:

- the selected $+1$ measured-phase convention is correct;
- the $q=22$ response contains a physical synthetic-aperture phase history;
- the candidate location near $x=-0.7$ cm provides a good geometric explanation for that phase history.

Focusing coherence is not target power. A weak pixel can sometimes have high normalized coherence by chance. Therefore, the coherence map is displayed only where the focused image power is within 25 dB of its peak.

Pixels below this threshold are masked and displayed in black. Black regions do not represent zero coherence or missing aperture measurements; they represent locations where coherence was intentionally not interpreted because the focused signal was weak.

Bright power in the coherent SAR image together with high focusing coherence provides stronger evidence of a physical scatterer than either metric alone.

<div style="font-size:2.0em; font-weight:bold;">
Cross-range focusing
</div>

The observed cross-range $-3$ dB width is

$$\Delta x_{-3\mathrm{dB}}\approx1.76\ \text{cm}.$$

The commonly used approximate cross-range resolution expression gives

$$\delta_x\approx\frac{\lambda R}{2L}\approx2.06\ \text{cm}.$$

This value and the measured $-3$ dB width use slightly different width conventions.

For an ideal uniformly weighted aperture, the approximate full $-3$ dB width is

$$\Delta x_{-3\mathrm{dB}}\approx0.443\frac{\lambda R}{L}.$$

For the present parameters,

$$\Delta x_{-3\mathrm{dB}}\approx0.443\frac{(4.91\ \text{mm})(0.84\ \text{m})}{0.10\ \text{m}}\approx1.83\ \text{cm}.$$

The measured width of approximately $1.76$ cm is close to this ideal uniform-aperture value. The first reconstruction therefore exhibits physically plausible cross-range focusing.

This width describes the observed sphere-associated scattering response. A point-target measurement would still be required for a rigorous experimental resolution measurement.

<div style="font-size:2.0em; font-weight:bold;">
Cross-range sidelobes
</div>

The cross-range profile contains additional peaks approximately 11–20 dB below the focused response.

These are consistent with the sidelobe structure expected from a finite uniformly weighted synthetic aperture. Uniform weighting provides the narrowest main lobe but does not strongly suppress sidelobes.

Some additional structure may also arise from:

- multipath;
- the sphere's extended scattering behavior;
- residual background mismatch;
- the approximate single-RX phase-center model.

A later synthetic-aperture weighting comparison can test Hann or Hamming weights. These windows should reduce sidelobes at the cost of a broader cross-range main lobe and a sensitivity penalty.

<div style="font-size:2.0em; font-weight:bold;">
Downrange response
</div>

The measured downrange $-3$ dB width is

$$\Delta z_{-3\mathrm{dB}}\approx1.85\ \text{cm}.$$

This value should not be interpreted as the physical range resolution. It is substantially narrower than the nominal range-bin spacing

$$\Delta R_{\mathrm{bin}}=4.6875\ \text{cm}$$

and the previously measured Hann-windowed Run A range response.

The present backprojection linearly interpolates between the 128 complex FFT bins. Neighboring complex bins can have substantially different phases. Linear interpolation between them can pass through sharp cancellation points and artificially narrow the displayed downrange peak.

The repeated strong features above and below the main response are also tied to the sampled range response, residual range sidelobes, multipath, and complex interpolation behavior.

The downrange image structure is therefore provisional.

<div style="font-size:2.0em; font-weight:bold;">
Overall assessment
</div>

The first reconstruction successfully demonstrates synthetic-aperture focusing.

The strongest evidence is:

1. the coherent result is substantially more localized than the noncoherent accumulation;
2. the focused peak remains associated with measured range bin $q=22$;
3. the peak shifts to the cross-range location suggested by the measured phase asymmetry;
4. focusing coherence increases from approximately $0.792$ at the assumed point to approximately $0.961$ at the image peak;
5. the measured cross-range width agrees closely with the ideal uniform-aperture $-3$ dB width.

The result is not yet a final calibrated image. Its main limitations are:

- provisional one-point range registration;
- linear interpolation of sparsely sampled complex range bins;
- uniform-aperture sidelobes;
- approximate single-RX monostatic phase-center geometry;
- an electrically large sphere and possible multipath.

The next processing step should oversample the complex range response before backprojection. A zero-padded range FFT will provide denser samples of the existing range point-spread function without claiming improved physical range resolution. The backprojection can then be repeated to determine which downrange structures are physical and which were caused by linear interpolation of the 128-point range FFT.

## 3.5 — Oversample the Complex Range Response Before Backprojection

The first backprojection used linear interpolation between the 128 complex range-FFT bins. Its cross-range focusing was physically reasonable, but its apparent downrange $-3$ dB width was narrower than the nominal range-bin spacing.

This suggests that linear interpolation between sparsely sampled complex FFT bins may be distorting the displayed range point-spread function.

<div style="font-size:2.0em; font-weight:bold;">
Zero-padding the range FFT
</div>

The original range FFT uses $N=128$ fast-time samples and produces 128 range-frequency samples. A zero-padded FFT evaluates the same finite-duration signal spectrum on a denser frequency grid.

For a zero-padded transform of length $K>N$,

$$X_K[q']=\sum_{i=0}^{N-1}w_R[i]x[i]\exp\left(-j\frac{2\pi q'i}{K}\right).$$

This section uses

$$K=8N=1024.$$

The resulting nominal range-grid spacing becomes

$$\Delta R_{\mathrm{grid}}=\frac{6\ \text{m}}{1024}=5.859375\ \text{mm}.$$

This does **not** improve the physical range resolution. No new samples, bandwidth, or measured information are added. Zero-padding only samples the existing range response more densely.

The distinction is:

- physical resolution describes whether two nearby targets can be separated;
- FFT-grid spacing describes how densely the existing response is represented numerically.

<div style="font-size:2.0em; font-weight:bold;">
Why this helps backprojection
</div>

Backprojection requires the complex range value associated with each candidate pixel. With only 128 range bins, neighboring complex samples are separated by approximately 46.875 mm in nominal range.

Linear interpolation directly between those sparse complex samples can create artificial cancellation if adjacent bins have substantially different phases.

The 1024-point FFT evaluates the range spectrum at eight times as many locations. Backprojection can then interpolate between much more closely spaced complex samples, providing a more faithful representation of the measured range point-spread function.

<div style="font-size:2.0em; font-weight:bold;">
What is being compared
</div>

This section does not compare a Run B image against a Run C image.

Both reconstruction paths image the same physical Run C target-present scene after matched Run B background subtraction. The comparison is:

- background-subtracted Run C using the original 128-point range FFT;
- background-subtracted Run C using the 1024-point zero-padded range FFT.

Run B must be reprocessed with the same FFT size as Run C so that their complex range samples occupy matching range grids before subtraction. Run B is used only to construct the aligned background estimate in each processing path.

<div style="font-size:2.0em; font-weight:bold;">
Processing sequence
</div>

For Run B and Run C, the processing is repeated as follows:

1. retain RX channel 3;
2. apply the same Hann fast-time window;
3. calculate a 1024-point zero-padded range FFT;
4. apply the same Hann Doppler window;
5. extract zero Doppler;
6. coherently average the 20 frames at each rail position;
7. apply the previously estimated Run B-to-Run C complex alignment;
8. subtract the aligned background;
9. locate the target-associated peak on the denser range grid;
10. repeat the same single-RX backprojection.

The previously estimated background-alignment coefficients are reused. This keeps the Run B-to-Run C correction unchanged so that the comparison primarily tests range oversampling rather than a different background-subtraction method.

### 3.5.1 — Oversampled Range-Processing Functions

The following functions calculate a zero-padded range FFT for one selected RX channel across all rail positions and repeated frames.

Only the zero-Doppler value is retained. The centered zero-Doppler FFT bin is mathematically equal to the Hann-weighted complex sum over chirp index, so the function does not need to calculate the other 63 unused Doppler bins.

The output remains complex and preserves the phase required for background subtraction and SAR focusing.

In [ ]:
def process_single_rx_sar_with_oversampled_range_fft(
    sar_radc_stack,
    rx_channel,
    range_fft_size,
    range_window="hann",
    doppler_window="hann",
):
    """
    Process one RX channel from a multi-position RADC stack
    using an oversampled range FFT.

    Input shape:
        (
            aperture position,
            repeated frame,
            fast-time sample,
            chirp,
            RX channel,
        )

    Output shape:
        (
            aperture position,
            repeated frame,
            oversampled range bin,
        )

    Zero-padding increases the density of the FFT range grid.
    It does not add bandwidth or improve physical resolution.
    """

    if sar_radc_stack.ndim != 5:
        raise ValueError(
            "sar_radc_stack must have shape "
            "(position, frame, sample, chirp, RX)."
        )

    (
        number_of_positions,
        number_of_frames,
        number_of_fast_time_samples,
        number_of_chirps,
        number_of_rx_channels,
    ) = sar_radc_stack.shape

    if not (
        0
        <= rx_channel
        < number_of_rx_channels
    ):
        raise ValueError(
            "rx_channel is outside the available channels."
        )

    if range_fft_size < number_of_fast_time_samples:
        raise ValueError(
            "range_fft_size must be at least as large as "
            "the number of fast-time samples."
        )

    # Construct the same fast-time window used in the original
    # 128-point range processing.
    #
    # Keeping the window unchanged ensures that this section
    # tests FFT-grid density rather than a different range
    # sidelobe-versus-main-lobe tradeoff.
    range_window_coefficients = create_window(
        range_window,
        number_of_fast_time_samples,
    )

    # Construct the same slow-time window used before extracting
    # the stationary, zero-Doppler response.
    doppler_window_coefficients = create_window(
        doppler_window,
        number_of_chirps,
    )

    # Allocate storage only for the zero-Doppler range profiles.
    #
    # We do not store a complete 1024 x 64 range-Doppler cube
    # because the other Doppler bins are not used for the
    # stationary sphere reconstruction.
    oversampled_zero_doppler_stack = np.empty(
        (
            number_of_positions,
            number_of_frames,
            range_fft_size,
        ),
        dtype=np.complex128,
    )

    for position_index in range(
        number_of_positions
    ):
        for frame_index in range(
            number_of_frames
        ):
            # Extract one RX channel from one measured RADC
            # frame.
            #
            # Shape:
            #   (fast-time sample, chirp)
            selected_rx_radc = np.asarray(
                sar_radc_stack[
                    position_index,
                    frame_index,
                    :,
                    :,
                    rx_channel,
                ],
                dtype=np.complex128,
            )

            # Apply the Hann window across fast time.
            #
            # This controls the range sidelobe response before
            # evaluating the oversampled range spectrum.
            windowed_fast_time_data = (
                selected_rx_radc
                * range_window_coefficients[
                    :,
                    np.newaxis,
                ]
            )

            # Calculate the zero-padded range FFT.
            #
            # The input still contains only 128 measured
            # fast-time samples. Specifying n=1024 appends
            # implicit zeros and evaluates the same spectrum
            # at 1024 frequency locations.
            #
            # Shape:
            #   (oversampled range bin, chirp)
            oversampled_range_fft = np.fft.fft(
                windowed_fast_time_data,
                n=range_fft_size,
                axis=0,
            )

            # Apply the Hann Doppler window across chirps.
            windowed_slow_time_data = (
                oversampled_range_fft
                * doppler_window_coefficients[
                    np.newaxis,
                    :,
                ]
            )

            # Extract zero Doppler.
            #
            # The unshifted Doppler-FFT bin p=0 is the complex
            # sum over chirp index. After fftshift, this same
            # value would appear at centered Doppler bin 32.
            #
            # Calculating the sum directly is exactly equivalent
            # to calculating the complete Doppler FFT and then
            # retaining only its zero-frequency bin.
            oversampled_zero_doppler_stack[
                position_index,
                frame_index,
                :,
            ] = np.sum(
                windowed_slow_time_data,
                axis=1,
            )

    centered_zero_doppler_bin = (
        number_of_chirps // 2
    )

    return (
        oversampled_zero_doppler_stack,
        centered_zero_doppler_bin,
    )


def calculate_single_rx_frame_coherence(
    zero_doppler_stack,
):
    """
    Calculate repeated-frame coherent-sum efficiency for
    a single-RX oversampled range stack.

    Input shape:
        (position, frame, range bin)

    Output shape:
        (position, range bin)
    """

    if zero_doppler_stack.ndim != 3:
        raise ValueError(
            "zero_doppler_stack must have shape "
            "(position, frame, range bin)."
        )

    # The numerator is the actual magnitude obtained by
    # coherently summing the repeated frames.
    coherent_sum_magnitude = np.abs(
        np.sum(
            zero_doppler_stack,
            axis=1,
        )
    )

    # The denominator is the largest sum magnitude possible
    # if every repeated-frame phase aligned perfectly.
    maximum_possible_sum_magnitude = np.sum(
        np.abs(
            zero_doppler_stack
        ),
        axis=1,
    )

    return (
        coherent_sum_magnitude
        / np.maximum(
            maximum_possible_sum_magnitude,
            np.finfo(float).tiny,
        )
    )

### 3.5.2 — Repeat Backprojection with an Eight-Times Oversampled Range FFT

This cell recomputes the Run B and Run C complex range profiles using a 1024-point range FFT and repeats the single-RX backprojection.

The physical image grid, aperture positions, phase convention, aperture weighting, and background-alignment coefficients remain unchanged.

The comparison therefore asks:

> Does a more densely sampled complex range response remove the suspiciously narrow downrange peak or alter the repeated downrange lobes?

The original and oversampled images are normalized independently for shape comparison. The comparison does not imply that zero-padding provides processing gain or additional physical resolution.

In [ ]:
# Use the same single RX channel selected for the first
# backprojection. Keeping the channel fixed isolates the effect
# of range oversampling.
oversampled_sar_rx_channel = (
    selected_backprojection_rx_channel
)


# Increase the range FFT from 128 to 1024 points.
#
# The eight-times-denser grid provides smoother sampling of the
# existing range response but does not increase radar bandwidth.
range_oversampling_factor = 8

oversampled_range_fft_size = (
    run_c_sar_radc.shape[2]
    * range_oversampling_factor
)


# Reprocess every Run B frame for the selected RX channel.
print(
    "Processing oversampled Run B range profiles..."
)

(
    run_b_oversampled_zero_doppler,
    run_b_oversampled_zero_doppler_bin,
) = process_single_rx_sar_with_oversampled_range_fft(
    sar_radc_stack=run_b_sar_radc,
    rx_channel=oversampled_sar_rx_channel,
    range_fft_size=oversampled_range_fft_size,
    range_window="hann",
    doppler_window="hann",
)


# Reprocess every Run C frame using exactly the same windows
# and range-FFT size.
print(
    "Processing oversampled Run C range profiles..."
)

(
    run_c_oversampled_zero_doppler,
    run_c_oversampled_zero_doppler_bin,
) = process_single_rx_sar_with_oversampled_range_fft(
    sar_radc_stack=run_c_sar_radc,
    rx_channel=oversampled_sar_rx_channel,
    range_fft_size=oversampled_range_fft_size,
    range_window="hann",
    doppler_window="hann",
)


# Both runs must use the same centered zero-Doppler convention.
assert (
    run_b_oversampled_zero_doppler_bin
    == run_c_oversampled_zero_doppler_bin
), "Run B and Run C zero-Doppler bins do not match."


# Verify repeated-frame phase stability on the denser range
# grid before coherent averaging.
run_b_oversampled_frame_coherence = (
    calculate_single_rx_frame_coherence(
        run_b_oversampled_zero_doppler
    )
)

run_c_oversampled_frame_coherence = (
    calculate_single_rx_frame_coherence(
        run_c_oversampled_zero_doppler
    )
)


# Coherently average the 20 repeated frames at each rail
# position. Averaging is performed on complex data so that the
# phase required for SAR focusing is retained.
run_b_oversampled_zero_doppler_mean = np.mean(
    run_b_oversampled_zero_doppler,
    axis=1,
)

run_c_oversampled_zero_doppler_mean = np.mean(
    run_c_oversampled_zero_doppler,
    axis=1,
)


# Reuse the Run B-to-Run C complex alignment coefficients
# estimated in Section 3.2.
#
# One coefficient exists for every rail position and RX channel.
# The coefficient corrects the matched Run B background's
# overall complex gain and phase before subtraction.
selected_background_alignment_coefficients = (
    sar_background_alignment_coefficients[
        :,
        oversampled_sar_rx_channel,
    ]
)


# Apply the existing alignment to every oversampled range bin
# at its corresponding rail position.
run_b_oversampled_aligned_mean = (
    run_b_oversampled_zero_doppler_mean
    * selected_background_alignment_coefficients[
        :,
        np.newaxis,
    ]
)


# Subtract the aligned matched background while the data remain
# complex.
#
# Output shape:
#   (17 aperture positions, 1024 oversampled range bins)
run_c_oversampled_background_subtracted = (
    run_c_oversampled_zero_doppler_mean
    - run_b_oversampled_aligned_mean
)


# The physical unambiguous range remains approximately 6 m.
# Only the numerical grid spacing changes from 6/128 m to
# 6/1024 m.
oversampled_range_bin_spacing_m = (
    sar_nominal_maximum_range_m
    / oversampled_range_fft_size
)

oversampled_range_bin_index = np.arange(
    oversampled_range_fft_size
)

# Express the oversampled bin coordinate in equivalent
# original 128-point-bin units. For example, oversampled bin
# 176 corresponds to original-bin coordinate 22.
equivalent_original_range_bin = (
    oversampled_range_bin_index
    / range_oversampling_factor
)


# Combine residual power noncoherently over all rail positions
# to locate the target-associated peak on the denser range grid.
oversampled_residual_range_power = np.sum(
    np.abs(
        run_c_oversampled_background_subtracted
    ) ** 2,
    axis=0,
)


# Search only near the previously identified q = 22 response.
#
# This prevents a distant residual or near-range coupling term
# from being selected as the sphere-associated reference.
oversampled_target_search_mask = (
    (equivalent_original_range_bin >= 19)
    & (equivalent_original_range_bin <= 26)
)

oversampled_target_search_indices = np.flatnonzero(
    oversampled_target_search_mask
)

oversampled_target_reference_bin = int(
    oversampled_target_search_indices[
        np.argmax(
            oversampled_residual_range_power[
                oversampled_target_search_indices
            ]
        )
    ]
)

oversampled_target_equivalent_original_bin = (
    oversampled_target_reference_bin
    / range_oversampling_factor
)


# Normalize the oversampled residual range-power profile to the
# strongest response inside the target-search region.
oversampled_residual_range_power_db = (
    10
    * np.log10(
        np.maximum(
            oversampled_residual_range_power,
            np.finfo(float).tiny,
        )
        / np.maximum(
            np.max(
                oversampled_residual_range_power[
                    oversampled_target_search_indices
                ]
            ),
            np.finfo(float).tiny,
        )
    )
)


# Use the same physical registration point as the first image.
#
# The detected oversampled reference bin is assigned to the
# same physical sphere reference range near 0.84 m.
oversampled_registration_range_m = (
    sar_registration_range_m
)


# Repeat backprojection with the densely sampled complex range
# profiles.
#
# The image grid, aperture geometry, phase sign, and uniform
# aperture weights are identical to the 128-point result.
(
    run_c_oversampled_backprojection_image,
    run_c_oversampled_noncoherent_image,
    run_c_oversampled_focusing_coherence,
    run_c_oversampled_valid_support_count,
    run_c_oversampled_cross_range_grid_m,
    run_c_oversampled_downrange_grid_m,
) = backproject_single_rx(
    aperture_range_profiles=(
        run_c_oversampled_background_subtracted
    ),
    aperture_positions_m=(
        selected_rx_aperture_positions_m
    ),
    image_cross_range_m=(
        sar_image_cross_range_m
    ),
    image_downrange_m=(
        sar_image_downrange_m
    ),
    carrier_frequency_hz=(
        carrier_frequency_hz
    ),
    measured_phase_sign=(
        backprojection_measured_phase_sign
    ),
    reference_range_bin=(
        oversampled_target_reference_bin
    ),
    registration_range_m=(
        oversampled_registration_range_m
    ),
    range_bin_spacing_m=(
        oversampled_range_bin_spacing_m
    ),
    reference_aperture_index=(
        CENTER_POSITION_INDEX
    ),
    aperture_weights=(
        sar_uniform_aperture_weights
    ),
)


# Confirm that every displayed pixel still uses all 17
# aperture positions.
assert np.all(
    run_c_oversampled_valid_support_count
    == selected_rx_aperture_profiles.shape[0]
), (
    "At least one oversampled image pixel does not have "
    "complete aperture support."
)


# Convert the oversampled coherent image into normalized power.
run_c_oversampled_backprojection_db = normalized_power_db(
    run_c_oversampled_backprojection_image,
    minimum_db=sar_image_display_floor_db,
)

run_c_oversampled_image_power = (
    np.abs(
        run_c_oversampled_backprojection_image
    ) ** 2
)


# Locate the strongest pixel in the oversampled-range image.
(
    oversampled_peak_downrange_index,
    oversampled_peak_cross_range_index,
) = np.unravel_index(
    np.argmax(
        run_c_oversampled_image_power
    ),
    run_c_oversampled_image_power.shape,
)

oversampled_peak_cross_range_m = (
    sar_image_cross_range_m[
        oversampled_peak_cross_range_index
    ]
)

oversampled_peak_downrange_m = (
    sar_image_downrange_m[
        oversampled_peak_downrange_index
    ]
)

oversampled_peak_focusing_coherence = (
    run_c_oversampled_focusing_coherence[
        oversampled_peak_downrange_index,
        oversampled_peak_cross_range_index,
    ]
)


# Determine the oversampled and original-equivalent range-bin
# coordinates used by the new focused peak at the center
# aperture position.
oversampled_peak_center_range_m = np.sqrt(
    (
        oversampled_peak_cross_range_m
        - selected_center_aperture_position_m
    ) ** 2
    + oversampled_peak_downrange_m ** 2
)

oversampled_peak_center_range_bin = (
    oversampled_target_reference_bin
    + (
        oversampled_peak_center_range_m
        - oversampled_registration_range_m
    )
    / oversampled_range_bin_spacing_m
)

oversampled_peak_equivalent_original_bin = (
    oversampled_peak_center_range_bin
    / range_oversampling_factor
)


# Extract cross-range and downrange profiles through the new
# focused peak.
oversampled_cross_range_power_profile = (
    run_c_oversampled_image_power[
        oversampled_peak_downrange_index,
        :,
    ]
)

oversampled_downrange_power_profile = (
    run_c_oversampled_image_power[
        :,
        oversampled_peak_cross_range_index,
    ]
)

oversampled_cross_range_width_result = (
    measure_contiguous_minus_3db_width(
        coordinate=sar_image_cross_range_m,
        power_profile=(
            oversampled_cross_range_power_profile
        ),
    )
)

oversampled_downrange_width_result = (
    measure_contiguous_minus_3db_width(
        coordinate=sar_image_downrange_m,
        power_profile=(
            oversampled_downrange_power_profile
        ),
    )
)


# Create a direct numerical comparison between the original
# sparse range grid and the eight-times-oversampled grid.
sar_range_oversampling_comparison = pd.DataFrame(
    [
        {
            "processing_path": (
                "128-point range FFT"
            ),
            "range_fft_size": 128,
            "range_grid_spacing_m": (
                sar_nominal_range_bin_spacing_m
            ),
            "target_equivalent_original_bin": (
                sar_peak_center_range_bin
            ),
            "peak_cross_range_m": (
                sar_peak_cross_range_m
            ),
            "peak_downrange_m": (
                sar_peak_downrange_m
            ),
            "peak_focusing_coherence": (
                sar_peak_focusing_coherence
            ),
            "minus3db_cross_range_width_m": (
                sar_cross_range_width_result[
                    "width"
                ]
            ),
            "minus3db_downrange_width_m": (
                sar_downrange_width_result[
                    "width"
                ]
            ),
        },
        {
            "processing_path": (
                "1024-point zero-padded range FFT"
            ),
            "range_fft_size": (
                oversampled_range_fft_size
            ),
            "range_grid_spacing_m": (
                oversampled_range_bin_spacing_m
            ),
            "target_equivalent_original_bin": (
                oversampled_peak_equivalent_original_bin
            ),
            "peak_cross_range_m": (
                oversampled_peak_cross_range_m
            ),
            "peak_downrange_m": (
                oversampled_peak_downrange_m
            ),
            "peak_focusing_coherence": (
                oversampled_peak_focusing_coherence
            ),
            "minus3db_cross_range_width_m": (
                oversampled_cross_range_width_result[
                    "width"
                ]
            ),
            "minus3db_downrange_width_m": (
                oversampled_downrange_width_result[
                    "width"
                ]
            ),
        },
    ]
)


print(
    "Oversampled range FFT size:",
    oversampled_range_fft_size,
)

print(
    "Oversampled range-grid spacing:",
    f"{oversampled_range_bin_spacing_m:.6f} m",
)

print(
    "Detected target oversampled bin:",
    oversampled_target_reference_bin,
)

print(
    "Equivalent original-bin coordinate:",
    f"{oversampled_target_equivalent_original_bin:.3f}",
)

print(
    "Oversampled-image peak:",
    f"x = {oversampled_peak_cross_range_m:.4f} m, "
    f"z = {oversampled_peak_downrange_m:.4f} m",
)

print(
    "Oversampled peak focusing coherence:",
    f"{oversampled_peak_focusing_coherence:.4f}",
)


# ----------------------------------------------------------
# Plot the densely sampled residual range response.
# ----------------------------------------------------------

fig, axis = plt.subplots(
    1,
    1,
    figsize=(11, 5),
    constrained_layout=True,
)

axis.plot(
    equivalent_original_range_bin,
    oversampled_residual_range_power_db,
    color="black",
    linewidth=1.5,
)

axis.axvline(
    run_c_observed_range_bin,
    color="red",
    linestyle=":",
    linewidth=1.5,
    label="Original q = 22",
)

axis.axvline(
    oversampled_target_equivalent_original_bin,
    color="cyan",
    linestyle="-.",
    linewidth=1.5,
    label=(
        "Oversampled peak = "
        f"{oversampled_target_equivalent_original_bin:.3f}"
    ),
)

axis.set_xlim(
    17,
    30,
)

axis.set_ylim(
    -50,
    2,
)

axis.set_title(
    "Run C Oversampled Background-Subtracted "
    "Range Response"
)

axis.set_xlabel(
    "Equivalent original range-bin coordinate"
)

axis.set_ylabel(
    "Power relative to target-region maximum (dB)"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()

plt.show()


# ----------------------------------------------------------
# Compare the original and oversampled SAR images.
# ----------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 6),
    constrained_layout=True,
)

original_image_handle = axes[0].imshow(
    run_c_sar_backprojection_db,
    origin="lower",
    extent=image_extent_cm,
    aspect="equal",
    cmap="viridis",
    vmin=sar_image_display_floor_db,
    vmax=0,
)

axes[0].set_title(
    "128-Point Range FFT"
)

axes[0].set_xlabel(
    "Cross-range, $x$ (cm)"
)

axes[0].set_ylabel(
    "Downrange, $z$ (cm)"
)

fig.colorbar(
    original_image_handle,
    ax=axes[0],
    label="Normalized power (dB)",
)


oversampled_image_handle = axes[1].imshow(
    run_c_oversampled_backprojection_db,
    origin="lower",
    extent=image_extent_cm,
    aspect="equal",
    cmap="viridis",
    vmin=sar_image_display_floor_db,
    vmax=0,
)

axes[1].set_title(
    "1024-Point Zero-Padded Range FFT"
)

axes[1].set_xlabel(
    "Cross-range, $x$ (cm)"
)

axes[1].set_ylabel(
    "Downrange, $z$ (cm)"
)

fig.colorbar(
    oversampled_image_handle,
    ax=axes[1],
    label="Normalized power (dB)",
)


for axis in axes:
    axis.plot(
        0,
        84,
        marker="+",
        color="red",
        markersize=14,
        markeredgewidth=2,
        label="Registration point",
    )

axes[0].plot(
    sar_peak_cross_range_m * 100,
    sar_peak_downrange_m * 100,
    marker="x",
    color="cyan",
    markersize=10,
    markeredgewidth=2,
    label="Original peak",
)

axes[1].plot(
    oversampled_peak_cross_range_m * 100,
    oversampled_peak_downrange_m * 100,
    marker="x",
    color="cyan",
    markersize=10,
    markeredgewidth=2,
    label="Oversampled peak",
)

axes[1].legend(
    loc="upper right",
)

fig.suptitle(
    "Effect of Range-FFT Oversampling on SAR Backprojection",
    fontsize=14,
)

plt.show()


# ----------------------------------------------------------
# Compare one-dimensional focused profiles.
# ----------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5),
    constrained_layout=True,
)

axes[0].plot(
    sar_image_cross_range_m * 100,
    sar_cross_range_width_result[
        "power_profile_db"
    ],
    linewidth=1.4,
    label="128-point range FFT",
)

axes[0].plot(
    sar_image_cross_range_m * 100,
    oversampled_cross_range_width_result[
        "power_profile_db"
    ],
    linewidth=1.4,
    label="1024-point range FFT",
)

axes[0].axhline(
    -3,
    color="gray",
    linestyle="--",
    linewidth=1,
)

axes[0].set_title(
    "Cross-Range Profiles Through Each Peak"
)

axes[0].set_xlabel(
    "Cross-range, $x$ (cm)"
)

axes[0].set_ylabel(
    "Normalized power (dB)"
)

axes[0].set_ylim(
    -40,
    2,
)

axes[0].grid(
    True,
    alpha=0.3,
)

axes[0].legend()


axes[1].plot(
    sar_image_downrange_m,
    sar_downrange_width_result[
        "power_profile_db"
    ],
    linewidth=1.4,
    label="128-point range FFT",
)

axes[1].plot(
    sar_image_downrange_m,
    oversampled_downrange_width_result[
        "power_profile_db"
    ],
    linewidth=1.4,
    label="1024-point range FFT",
)

axes[1].axhline(
    -3,
    color="gray",
    linestyle="--",
    linewidth=1,
)

axes[1].set_title(
    "Downrange Profiles Through Each Peak"
)

axes[1].set_xlabel(
    "Downrange, $z$ (m)"
)

axes[1].set_ylabel(
    "Normalized power (dB)"
)

axes[1].set_ylim(
    -40,
    2,
)

axes[1].grid(
    True,
    alpha=0.3,
)

axes[1].legend()

plt.show()


display(
    sar_range_oversampling_comparison
)

### 3.5.3 — Results and Interpretation

The eight-times-oversampled processing produced a 1024-point complex range FFT with nominal grid spacing

$$\Delta R_{\mathrm{grid}}=\frac{6\ \text{m}}{1024}=0.005859375\ \text{m}.$$

This denser grid does not represent improved physical range resolution. It samples the existing Hann-windowed range response more accurately.

<div style="font-size:2.0em; font-weight:bold;">
What was compared
</div>

This section did not compare a Run B image against a Run C image.

Both image-formation paths used the same target-present Run C scene after matched Run B background subtraction:

- the original path used a 128-point range FFT;
- the oversampled path used a 1024-point zero-padded range FFT.

Run B was reprocessed only because its complex background samples must occupy the same range grid as Run C before subtraction.

The two residual datasets were

$$G_{\mathrm{128}}=\overline{C}_{\mathrm{128}}-\alpha\overline{B}_{\mathrm{128}},$$

and

$$G_{\mathrm{1024}}=\overline{C}_{\mathrm{1024}}-\alpha\overline{B}_{\mathrm{1024}}.$$

The physical scene, selected RX channel, aperture coordinates, background-alignment coefficients, phase convention, image grid, and uniform aperture weights remained unchanged.

<div style="font-size:2.0em; font-weight:bold;">
Oversampled target range response
</div>

The strongest target-associated response occurs at oversampled range bin

$$q_{\mathrm{1024}}=176.$$

Expressed in original 128-point-bin coordinates,

$$\frac{176}{8}=22.000.$$

The oversampled processing therefore confirms that the sphere-associated response is centered at $q=22$. Zero-padding did not move the target or reveal a different range peak.

The oversampled range-power profile shows a smooth main response centered at $q=22$. The response extends continuously through fractional range-bin coordinates that were not directly represented by the original 128-point FFT.

This smooth response demonstrates why linear interpolation between only the original complex bins could not reliably represent the range main lobe.

<div style="font-size:2.0em; font-weight:bold;">
Backprojection comparison
</div>

The measured image results are:

| Measurement | 128-point FFT | 1024-point zero-padded FFT |
|---|---:|---:|
| Range-grid spacing | $46.875$ mm | $5.859$ mm |
| Equivalent target bin | $22.001$ | $22.001$ |
| Peak cross-range | $-0.7$ cm | $-0.6$ cm |
| Peak downrange | $0.840$ m | $0.840$ m |
| Peak focusing coherence | $0.9607$ | $0.9521$ |
| Cross-range $-3$ dB width | $1.765$ cm | $1.720$ cm |
| Downrange $-3$ dB width | $1.846$ cm | $7.049$ cm |

The target range, cross-range location, cross-range width, and focusing coherence remain nearly unchanged. This stability confirms that the synthetic-aperture phase focusing is not an artifact of the original range interpolation.

The focused cross-range peak remains slightly toward the negative side of the rail:

$$x_{\mathrm{peak}}\approx-0.6\ \text{cm}.$$

The small 1 mm difference between the two peak estimates is equal to one cross-range image-grid increment and is not significant.

<div style="font-size:2.0em; font-weight:bold;">
Cross-range stability
</div>

The original and oversampled cross-range profiles are nearly identical.

The oversampled result has measured width

$$\Delta x_{-3\mathrm{dB}}\approx1.72\ \text{cm}.$$

For an ideal uniformly weighted synthetic aperture, the approximate full $-3$ dB width is

$$\Delta x_{-3\mathrm{dB}}\approx0.443\frac{\lambda R}{L}\approx1.83\ \text{cm}.$$

The measured result remains close to this theoretical value.

This agreement confirms that the approximately 10 cm synthetic aperture is producing physically plausible cross-range focusing. Range-FFT zero-padding does not materially alter that focusing because cross-range resolution is determined primarily by wavelength, target range, and synthetic-aperture length.

<div style="font-size:2.0em; font-weight:bold;">
Why the downrange response became longer
</div>

The original 128-point processing reported downrange width

$$\Delta z_{-3\mathrm{dB}}\approx1.85\ \text{cm}.$$

This was substantially smaller than the nominal range-bin spacing of $4.6875$ cm and therefore was not physically credible as the measured range resolution.

The original backprojection linearly interpolated between sparsely sampled complex FFT bins. If neighboring complex bins pointed in different phase directions, their linear combination could pass near zero. This produced artificial nulls immediately above and below the center range bin, making the central response appear falsely narrow and point-like.

The 1024-point FFT samples the same complex range response at eight times as many frequency locations. Its downrange profile is smooth and has measured width

$$\Delta z_{-3\mathrm{dB}}\approx7.05\ \text{cm}.$$

This is approximately

$$\frac{7.05\ \text{cm}}{4.6875\ \text{cm}}\approx1.50$$

original range bins, which is physically reasonable for a Hann-windowed range response.

The oversampled image therefore appears longer in downrange because it reveals the actual range point-spread function instead of preserving artificial cancellation between sparse complex bins.

<div style="font-size:2.0em; font-weight:bold;">
Effect of the Hann range window
</div>

The Hann window reduces the contribution of fast-time samples near the beginning and end of the measured chirp record. This suppresses range sidelobes but also reduces the effective fast-time aperture and broadens the range main lobe.

The window coefficients multiply the measured fast-time samples before the range FFT:

$$x_w[i]=w_R[i]x[i].$$

This multiplication does not change the signal phase model. It changes the amplitude contribution of each fast-time sample and therefore changes the shape of the range point-spread function.

The observed approximately 7 cm downrange extent is therefore partly a deliberate consequence of selecting the Hann range window.

A rectangular window would produce a narrower central range response but stronger range sidelobes. A Hamming window may provide an intermediate main-lobe width with strong sidelobe suppression. A Blackman window would suppress sidelobes further but produce an even broader range response.

Reducing the main-lobe width through window selection is possible, but it is not free. Narrowing the main lobe generally increases sidelobe levels and can make weak nearby scatterers harder to distinguish from a strong target’s sidelobes.

A true improvement in physical range resolution would require greater transmitted FMCW bandwidth rather than a different window or larger zero-padded FFT.

<div style="font-size:2.0em; font-weight:bold;">
Focusing coherence
</div>

The peak focusing coherence changed only slightly:

$$0.9607\longrightarrow0.9521.$$

Both values indicate excellent coherent alignment across the 17 aperture positions.

The small reduction is not evidence of degraded focusing. The oversampled FFT and denser range interpolation provide a different and more accurate estimate of the complex range samples used by the coherent sum.

The focused peak remains bright, spatially stable, and strongly coherent.

<div style="font-size:2.0em; font-weight:bold;">
Visual interpretation
</div>

The 128-point image appears more point-like because artificial interpolation nulls separate the central response from adjacent downrange samples.

The 1024-point image appears vertically elongated because it displays the continuous Hann-windowed range response. Its cross-range concentration remains narrow.

The expected point-spread response is therefore anisotropic:

- narrow in cross-range because of the 10 cm synthetic aperture;
- broader in downrange because of the finite FMCW bandwidth and Hann range window.

Displaying the images with a $-25$ dB floor makes the dominant response easier to see by hiding weaker sidelobes and residual structure. This changes only the visualization and does not alter the image data or measured widths.

<div style="font-size:2.0em; font-weight:bold;">
Overall conclusion
</div>

The zero-padded range FFT resolved the principal uncertainty in the first reconstruction.

It demonstrated that:

1. the sphere-associated range response remains centered at equivalent bin $q=22$;
2. the focused cross-range position remains near $-0.6$ cm;
3. the approximately $1.7$ cm cross-range focus is stable;
4. the original approximately $1.85$ cm downrange width was an interpolation artifact;
5. the more credible Hann-windowed downrange width is approximately $7.05$ cm;
6. zero-padding improves numerical representation but does not improve physical range resolution.

The 1024-point zero-padded range processing should therefore replace the 128-point linear-interpolation path as the baseline for subsequent SAR image comparisons.

## 3.6 — Compare Oversampled Range Windows for SAR

Section 3.5 established that the 1024-point zero-padded range FFT provides a more accurate numerical representation of the complex range response.

The approximately 7 cm downrange width is partly caused by the Hann range window. Windowing creates a deliberate tradeoff:

- tapering the fast-time samples reduces range sidelobes;
- stronger tapering broadens the range main lobe;
- weaker tapering narrows the main lobe but increases sidelobes.

This section compares four range windows:

1. rectangular;
2. Hamming;
3. Hann;
4. Blackman.

All other processing choices remain fixed:

- RX channel 3;
- 1024-point range FFT;
- Hann Doppler window;
- coherent averaging over 20 frames;
- matched complex Run B background subtraction;
- uniform synthetic-aperture weighting;
- the measured $+1$ propagation-phase convention;
- the same physical image grid;
- local registration of each target peak to approximately $0.84$ m.

<div style="font-size:2.0em; font-weight:bold;">
Ideal window response versus measured scene response
</div>

Two different comparisons are made.

First, the ideal spectral response of each window is calculated directly from the window coefficients. This isolates the theoretical main-lobe and sidelobe properties of the window itself.

Second, each window is applied to the measured Run B and Run C RADC data and used to form a complete SAR image.

The measured target is an extended sphere in a multipath environment. Therefore, features outside the measured main lobe cannot automatically be identified as pure window sidelobes. They may also contain multipath, residual clutter, or additional sphere-associated scattering.

<div style="font-size:2.0em; font-weight:bold;">
Why these four windows were selected
</div>

There is no universally optimal range window. Window selection is a tradeoff among main-lobe width, sidelobe level, sidelobe decay, equivalent noise bandwidth, and coherent gain.

Rectangular, Hamming, Hann, and Blackman windows are compared first because they provide a clear progression from no taper to strong taper:

- rectangular establishes the narrowest-main-lobe baseline;
- Hann represents the current processing choice;
- Hamming provides an alternative balance with a lower first sidelobe;
- Blackman demonstrates the effect of stronger sidelobe suppression and greater main-lobe broadening.

Parameterized windows such as Kaiser and Taylor can provide more precise control of the tradeoff. They will be considered after the baseline comparison if none of the four standard windows provides an appropriate balance for the measured SAR data.

<div style="font-size:2.0em; font-weight:bold;">
Expected tradeoff
</div>

The rectangular window retains all fast-time samples with equal amplitude:

$$w_R[i]=1.$$

It provides the narrowest ideal range main lobe but has relatively high sidelobes.

The Hann, Hamming, and Blackman windows reduce the amplitudes of samples near the ends of the fast-time record. This lowers sidelobes but reduces the effective fast-time aperture.

The expected qualitative ordering is:

| Window | Main-lobe width | Sidelobe suppression |
|---|---|---|
| Rectangular | Narrowest | Lowest |
| Hamming | Intermediate | Strong |
| Hann | Wider | Strong |
| Blackman | Widest | Strongest |

The goal is not simply to select the narrowest image. The preferred window should provide an appropriate balance among:

- downrange width;
- range sidelobes;
- residual-scene structure;
- target focusing coherence;
- cross-range stability.

### 3.6.1 — Range-Window Comparison Functions

The following functions:

- calculate the ideal zero-padded spectral response of a window;
- estimate its ideal $-3$ dB width and peak sidelobe level;
- estimate a separate matched Run B-to-Run C complex alignment for every rail position;
- perform complex background subtraction on the oversampled single-RX data.

A separate background alignment is estimated for each range window because changing the fast-time weights changes the complex range profiles used in the least-squares alignment.

In [ ]:
def characterize_ideal_range_window(
    window_name,
    measured_sample_count,
    fft_size,
):
    """
    Calculate ideal spectral characteristics for one
    fast-time window.

    The result is expressed in original FFT-bin units so it can
    be compared with the measured 128-sample range response.

    Returns:
        frequency_offset_bins
        normalized_power_db
        characteristics
    """

    # Construct the requested window over the 128 measured
    # fast-time samples.
    window_coefficients = create_window(
        window_name,
        measured_sample_count,
    )

    # Calculate a densely sampled spectrum of the window itself.
    #
    # The FFT of the window is the ideal range response that
    # would be produced by a tone exactly centered on a range
    # bin, apart from a shift to the target's actual bin.
    ideal_complex_response = np.fft.fftshift(
        np.fft.fft(
            window_coefficients,
            n=fft_size,
        )
    )

    ideal_power = (
        np.abs(
            ideal_complex_response
        ) ** 2
    )

    ideal_power /= np.maximum(
        np.max(
            ideal_power
        ),
        np.finfo(float).tiny,
    )

    ideal_power_db = (
        10
        * np.log10(
            np.maximum(
                ideal_power,
                np.finfo(float).tiny,
            )
        )
    )

    # Express the dense FFT-frequency axis in original
    # 128-point-bin units.
    #
    # A coordinate of +1 means one original range bin above
    # the response center.
    frequency_offset_bins = (
        np.fft.fftshift(
            np.fft.fftfreq(
                fft_size
            )
        )
        * measured_sample_count
    )

    # Measure the ideal contiguous -3 dB width.
    ideal_width_result = (
        measure_contiguous_minus_3db_width(
            coordinate=frequency_offset_bins,
            power_profile=ideal_power,
        )
    )

    peak_index = int(
        np.argmax(
            ideal_power
        )
    )

    # Locate local minima on both sides of the central peak.
    #
    # These nearest minima approximate the boundaries of the
    # ideal window's central main lobe.
    local_minimum_indices = (
        np.flatnonzero(
            (
                ideal_power[1:-1]
                <= ideal_power[:-2]
            )
            & (
                ideal_power[1:-1]
                < ideal_power[2:]
            )
        )
        + 1
    )

    left_minimum_candidates = (
        local_minimum_indices[
            local_minimum_indices
            < peak_index
        ]
    )

    right_minimum_candidates = (
        local_minimum_indices[
            local_minimum_indices
            > peak_index
        ]
    )

    if (
        left_minimum_candidates.size == 0
        or right_minimum_candidates.size == 0
    ):
        raise ValueError(
            f"Could not identify main-lobe minima for "
            f"{window_name}."
        )

    left_mainlobe_minimum_index = (
        left_minimum_candidates[-1]
    )

    right_mainlobe_minimum_index = (
        right_minimum_candidates[0]
    )

    # Everything outside the first pair of minima is treated
    # as ideal sidelobe response.
    ideal_sidelobe_mask = np.ones(
        ideal_power.shape,
        dtype=bool,
    )

    ideal_sidelobe_mask[
        left_mainlobe_minimum_index:
        right_mainlobe_minimum_index + 1
    ] = False

    ideal_peak_sidelobe_level_db = np.max(
        ideal_power_db[
            ideal_sidelobe_mask
        ]
    )

    ideal_null_to_null_width_bins = (
        frequency_offset_bins[
            right_mainlobe_minimum_index
        ]
        - frequency_offset_bins[
            left_mainlobe_minimum_index
        ]
    )

    # Coherent gain describes the reduction in an exactly
    # bin-centered tone caused by the window's average value.
    coherent_gain = (
        np.sum(
            window_coefficients
        )
        / measured_sample_count
    )

    # Equivalent noise bandwidth measures the amount of noise
    # admitted by the window relative to its coherent gain.
    equivalent_noise_bandwidth_bins = (
        measured_sample_count
        * np.sum(
            window_coefficients ** 2
        )
        / np.maximum(
            np.sum(
                window_coefficients
            ) ** 2,
            np.finfo(float).tiny,
        )
    )

    characteristics = {
        "window": window_name.capitalize(),
        "coherent_gain": coherent_gain,
        "equivalent_noise_bandwidth_bins": (
            equivalent_noise_bandwidth_bins
        ),
        "ideal_minus3db_width_bins": (
            ideal_width_result[
                "width"
            ]
        ),
        "ideal_null_to_null_width_bins": (
            ideal_null_to_null_width_bins
        ),
        "ideal_peak_sidelobe_level_db": (
            ideal_peak_sidelobe_level_db
        ),
    }

    return (
        frequency_offset_bins,
        ideal_power_db,
        characteristics,
    )


def align_and_subtract_single_rx_background(
    target_mean,
    background_mean,
    target_coherence,
    background_coherence,
    reference_range_mask,
    minimum_reference_level_db=-35.0,
):
    """
    Estimate and subtract an aligned single-RX background
    independently at every rail position.

    Inputs have shape:
        (aperture position, oversampled range bin)

    Returns:
        aligned_background
        background_subtracted
        alignment_coefficients
        alignment_results_table
    """

    if target_mean.shape != background_mean.shape:
        raise ValueError(
            "Target and background means must have "
            "identical shapes."
        )

    if target_mean.ndim != 2:
        raise ValueError(
            "Input means must have shape "
            "(position, range bin)."
        )

    (
        number_of_positions,
        number_of_range_bins,
    ) = target_mean.shape

    if reference_range_mask.shape != (
        number_of_range_bins,
    ):
        raise ValueError(
            "reference_range_mask has the wrong shape."
        )

    minimum_reference_level_linear = (
        10 ** (
            minimum_reference_level_db / 20
        )
    )

    alignment_coefficients = np.empty(
        number_of_positions,
        dtype=np.complex128,
    )

    aligned_background = np.empty_like(
        background_mean,
        dtype=np.complex128,
    )

    background_subtracted = np.empty_like(
        target_mean,
        dtype=np.complex128,
    )

    alignment_results = []

    for position_index in range(
        number_of_positions
    ):
        target_profile = target_mean[
            position_index
        ]

        background_profile = background_mean[
            position_index
        ]

        # Establish separate reference-region peak magnitudes
        # for Run B and Run C at this rail position.
        #
        # The threshold prevents weak, noise-dominated samples
        # from controlling the complex alignment.
        target_reference_peak = np.max(
            np.abs(
                target_profile[
                    reference_range_mask
                ]
            )
        )

        background_reference_peak = np.max(
            np.abs(
                background_profile[
                    reference_range_mask
                ]
            )
        )

        target_is_strong = (
            np.abs(
                target_profile
            )
            >= (
                minimum_reference_level_linear
                * target_reference_peak
            )
        )

        background_is_strong = (
            np.abs(
                background_profile
            )
            >= (
                minimum_reference_level_linear
                * background_reference_peak
            )
        )

        valid_reference_mask = (
            reference_range_mask
            & target_is_strong
            & background_is_strong
        )

        target_reference = target_profile[
            valid_reference_mask
        ]

        background_reference = background_profile[
            valid_reference_mask
        ]

        if target_reference.size < 10:
            raise ValueError(
                "Too few valid alignment samples at "
                f"position {position_index}."
            )

        # Estimate alpha such that the aligned Run B profile
        # alpha*B best matches Run C over the reference region.
        alignment_numerator = np.sum(
            target_reference
            * np.conj(
                background_reference
            )
        )

        alignment_denominator = np.sum(
            np.abs(
                background_reference
            ) ** 2
        )

        alignment_coefficient = (
            alignment_numerator
            / np.maximum(
                alignment_denominator,
                np.finfo(float).tiny,
            )
        )

        alignment_coefficients[
            position_index
        ] = alignment_coefficient

        aligned_profile = (
            alignment_coefficient
            * background_profile
        )

        residual_profile = (
            target_profile
            - aligned_profile
        )

        aligned_background[
            position_index
        ] = aligned_profile

        background_subtracted[
            position_index
        ] = residual_profile

        # Measure how closely the two complex reference
        # profiles match after allowing one complex scale.
        complex_correlation = (
            np.abs(
                alignment_numerator
            )
            / np.maximum(
                np.sqrt(
                    np.sum(
                        np.abs(
                            target_reference
                        ) ** 2
                    )
                    * alignment_denominator
                ),
                np.finfo(float).tiny,
            )
        )

        residual_reference = residual_profile[
            valid_reference_mask
        ]

        reference_residual_reduction_db = (
            10
            * np.log10(
                np.maximum(
                    np.sum(
                        np.abs(
                            target_reference
                        ) ** 2
                    ),
                    np.finfo(float).tiny,
                )
                / np.maximum(
                    np.sum(
                        np.abs(
                            residual_reference
                        ) ** 2
                    ),
                    np.finfo(float).tiny,
                )
            )
        )

        alignment_results.append(
            {
                "position_index": position_index,
                "reference_sample_count": (
                    target_reference.size
                ),
                "alignment_magnitude_db": (
                    20
                    * np.log10(
                        np.maximum(
                            np.abs(
                                alignment_coefficient
                            ),
                            np.finfo(float).tiny,
                        )
                    )
                ),
                "alignment_phase_deg": (
                    np.rad2deg(
                        np.angle(
                            alignment_coefficient
                        )
                    )
                ),
                "complex_correlation": (
                    complex_correlation
                ),
                "median_target_frame_coherence": (
                    np.median(
                        target_coherence[
                            position_index,
                            valid_reference_mask,
                        ]
                    )
                ),
                "median_background_frame_coherence": (
                    np.median(
                        background_coherence[
                            position_index,
                            valid_reference_mask,
                        ]
                    )
                ),
                "reference_residual_reduction_db": (
                    reference_residual_reduction_db
                ),
            }
        )

    alignment_results_table = pd.DataFrame(
        alignment_results
    )

    return (
        aligned_background,
        background_subtracted,
        alignment_coefficients,
        alignment_results_table,
    )

### 3.6.2 — Process and Compare the Four Range Windows

Each range window is applied independently to both Run B and Run C.

For every window, the code:

1. calculates 1024-point range profiles;
2. verifies repeated-frame coherence;
3. coherently averages the 20 frames at each rail position;
4. estimates a window-specific Run B-to-Run C alignment;
5. performs complex background subtraction;
6. identifies the target-associated oversampled range peak;
7. registers that peak to the known approximately $0.84$ m reference;
8. forms a single-RX SAR image;
9. measures its downrange and cross-range widths;
10. records its focusing coherence.

The measured images are normalized independently for shape comparison. This allows their main-lobe and residual structures to be compared, but it does not compare their absolute processing gains.

The ideal-window plots provide the clean theoretical sidelobe comparison. The measured sphere plots contain the combined effects of windowing, extended-target scattering, multipath, and background-subtraction residuals.

In [ ]:
# Define the range windows to compare.
#
# Their order proceeds approximately from the least tapered
# window to the most strongly tapered window.
sar_range_windows_to_compare = [
    "rectangular",
    "hamming",
    "hann",
    "blackman",
]


# Use the same 1024-point FFT established in Section 3.5.
range_window_comparison_fft_size = (
    oversampled_range_fft_size
)

range_window_comparison_factor = (
    range_oversampling_factor
)

range_window_comparison_spacing_m = (
    sar_nominal_maximum_range_m
    / range_window_comparison_fft_size
)


# Express the 1024-point axis in equivalent original
# 128-point-bin coordinates.
range_window_oversampled_bin_index = np.arange(
    range_window_comparison_fft_size
)

range_window_equivalent_original_bin = (
    range_window_oversampled_bin_index
    / range_window_comparison_factor
)


# Define the background-alignment reference region.
#
# Use the same original-bin limits as Section 3.2:
#   - retain equivalent bins 1 through 50;
#   - exclude bins 15 through 26, which contain the
#     expected and observed sphere-associated regions.
range_window_reference_mask = (
    (
        range_window_equivalent_original_bin
        >= 1
    )
    & (
        range_window_equivalent_original_bin
        < 51
    )
    & ~(
        (
            range_window_equivalent_original_bin
            >= 15
        )
        & (
            range_window_equivalent_original_bin
            < 27
        )
    )
)


# Define the target-search interval around the previously
# identified q = 22 response.
range_window_target_search_mask = (
    (
        range_window_equivalent_original_bin
        >= 19
    )
    & (
        range_window_equivalent_original_bin
        <= 26
    )
)

range_window_target_search_indices = np.flatnonzero(
    range_window_target_search_mask
)


# Calculate the ideal response of every window before
# processing the measured radar data.
ideal_range_window_results = {}
ideal_range_window_characteristics = []

for range_window_name in (
    sar_range_windows_to_compare
):
    (
        ideal_frequency_offset_bins,
        ideal_window_power_db,
        ideal_window_characteristics,
    ) = characterize_ideal_range_window(
        window_name=range_window_name,
        measured_sample_count=(
            run_c_sar_radc.shape[2]
        ),
        fft_size=(
            range_window_comparison_fft_size
        ),
    )

    ideal_range_window_results[
        range_window_name
    ] = {
        "frequency_offset_bins": (
            ideal_frequency_offset_bins
        ),
        "power_db": ideal_window_power_db,
    }

    ideal_range_window_characteristics.append(
        ideal_window_characteristics
    )


ideal_range_window_table = pd.DataFrame(
    ideal_range_window_characteristics
)


# Store the measured result arrays and summary metrics for
# every tested range window.
sar_range_window_results = {}
sar_range_window_metrics = []


for range_window_name in (
    sar_range_windows_to_compare
):
    print(
        "Processing range window:",
        range_window_name,
    )

    # Recompute Run B using this range window.
    (
        run_b_window_zero_doppler,
        run_b_window_zero_doppler_bin,
    ) = process_single_rx_sar_with_oversampled_range_fft(
        sar_radc_stack=run_b_sar_radc,
        rx_channel=(
            selected_backprojection_rx_channel
        ),
        range_fft_size=(
            range_window_comparison_fft_size
        ),
        range_window=range_window_name,
        doppler_window="hann",
    )

    # Recompute Run C with the same range and Doppler windows.
    (
        run_c_window_zero_doppler,
        run_c_window_zero_doppler_bin,
    ) = process_single_rx_sar_with_oversampled_range_fft(
        sar_radc_stack=run_c_sar_radc,
        rx_channel=(
            selected_backprojection_rx_channel
        ),
        range_fft_size=(
            range_window_comparison_fft_size
        ),
        range_window=range_window_name,
        doppler_window="hann",
    )

    assert (
        run_b_window_zero_doppler_bin
        == run_c_window_zero_doppler_bin
    ), (
        "Run B and Run C zero-Doppler bins do not match "
        f"for {range_window_name}."
    )

    # Verify repeated-frame phase stability before averaging.
    run_b_window_frame_coherence = (
        calculate_single_rx_frame_coherence(
            run_b_window_zero_doppler
        )
    )

    run_c_window_frame_coherence = (
        calculate_single_rx_frame_coherence(
            run_c_window_zero_doppler
        )
    )

    # Coherently average repeated frames separately at every
    # rail position.
    run_b_window_mean = np.mean(
        run_b_window_zero_doppler,
        axis=1,
    )

    run_c_window_mean = np.mean(
        run_c_window_zero_doppler,
        axis=1,
    )

    # Estimate a separate matched-background alignment for
    # this range window, then subtract Run B from Run C while
    # retaining complex phase.
    (
        run_b_window_aligned,
        run_c_window_residual,
        window_alignment_coefficients,
        window_alignment_table,
    ) = align_and_subtract_single_rx_background(
        target_mean=run_c_window_mean,
        background_mean=run_b_window_mean,
        target_coherence=(
            run_c_window_frame_coherence
        ),
        background_coherence=(
            run_b_window_frame_coherence
        ),
        reference_range_mask=(
            range_window_reference_mask
        ),
        minimum_reference_level_db=-35.0,
    )

    # Combine residual power over all rail positions to locate
    # the sphere-associated peak on the oversampled grid.
    window_residual_range_power = np.sum(
        np.abs(
            run_c_window_residual
        ) ** 2,
        axis=0,
    )

    window_target_reference_bin = int(
        range_window_target_search_indices[
            np.argmax(
                window_residual_range_power[
                    range_window_target_search_indices
                ]
            )
        ]
    )

    window_target_equivalent_original_bin = (
        window_target_reference_bin
        / range_window_comparison_factor
    )

    # Normalize the measured residual range profile to the
    # strongest response within the target-search interval.
    window_residual_range_power_db = (
        10
        * np.log10(
            np.maximum(
                window_residual_range_power,
                np.finfo(float).tiny,
            )
            / np.maximum(
                np.max(
                    window_residual_range_power[
                        range_window_target_search_indices
                    ]
                ),
                np.finfo(float).tiny,
            )
        )
    )

    # Measure the target-region range response before SAR
    # focusing. The coordinate remains in original-bin units.
    measured_range_width_result = (
        measure_contiguous_minus_3db_width(
            coordinate=(
                range_window_equivalent_original_bin[
                    range_window_target_search_mask
                ]
            ),
            power_profile=(
                window_residual_range_power[
                    range_window_target_search_mask
                ]
            ),
        )
    )

    measured_unfocused_range_width_m = (
        measured_range_width_result[
            "width"
        ]
        * sar_nominal_range_bin_spacing_m
    )

    # Register this window's measured target peak to the same
    # known physical sphere reference range.
    #
    # This prevents a small sub-bin peak shift caused by the
    # window from appearing as an artificial physical-range
    # displacement between images.
    window_registration_range_m = (
        sar_registration_range_m
    )

    # Form the SAR image with every processing choice other
    # than the range window held fixed.
    (
        window_backprojection_image,
        window_noncoherent_image,
        window_focusing_coherence,
        window_valid_support_count,
        _,
        _,
    ) = backproject_single_rx(
        aperture_range_profiles=(
            run_c_window_residual
        ),
        aperture_positions_m=(
            selected_rx_aperture_positions_m
        ),
        image_cross_range_m=(
            sar_image_cross_range_m
        ),
        image_downrange_m=(
            sar_image_downrange_m
        ),
        carrier_frequency_hz=(
            carrier_frequency_hz
        ),
        measured_phase_sign=(
            backprojection_measured_phase_sign
        ),
        reference_range_bin=(
            window_target_reference_bin
        ),
        registration_range_m=(
            window_registration_range_m
        ),
        range_bin_spacing_m=(
            range_window_comparison_spacing_m
        ),
        reference_aperture_index=(
            CENTER_POSITION_INDEX
        ),
        aperture_weights=(
            sar_uniform_aperture_weights
        ),
    )

    assert np.all(
        window_valid_support_count
        == selected_rx_aperture_profiles.shape[0]
    ), (
        "Incomplete aperture support for "
        f"{range_window_name}."
    )

    window_backprojection_power = (
        np.abs(
            window_backprojection_image
        ) ** 2
    )

    window_backprojection_db = normalized_power_db(
        window_backprojection_image,
        minimum_db=-60.0,
    )

    # Locate the strongest coherently focused image pixel.
    (
        window_peak_downrange_index,
        window_peak_cross_range_index,
    ) = np.unravel_index(
        np.argmax(
            window_backprojection_power
        ),
        window_backprojection_power.shape,
    )

    window_peak_cross_range_m = (
        sar_image_cross_range_m[
            window_peak_cross_range_index
        ]
    )

    window_peak_downrange_m = (
        sar_image_downrange_m[
            window_peak_downrange_index
        ]
    )

    window_peak_focusing_coherence = (
        window_focusing_coherence[
            window_peak_downrange_index,
            window_peak_cross_range_index,
        ]
    )

    # Measure cross-range and downrange profiles through the
    # strongest focused image pixel.
    window_cross_range_power_profile = (
        window_backprojection_power[
            window_peak_downrange_index,
            :,
        ]
    )

    window_downrange_power_profile = (
        window_backprojection_power[
            :,
            window_peak_cross_range_index,
        ]
    )

    window_cross_range_width_result = (
        measure_contiguous_minus_3db_width(
            coordinate=sar_image_cross_range_m,
            power_profile=(
                window_cross_range_power_profile
            ),
        )
    )

    window_downrange_width_result = (
        measure_contiguous_minus_3db_width(
            coordinate=sar_image_downrange_m,
            power_profile=(
                window_downrange_power_profile
            ),
        )
    )

    # Save the complete arrays needed for plotting and later
    # selection of the preferred range window.
    sar_range_window_results[
        range_window_name
    ] = {
        "residual": run_c_window_residual,
        "residual_range_power_db": (
            window_residual_range_power_db
        ),
        "target_reference_bin": (
            window_target_reference_bin
        ),
        "target_equivalent_original_bin": (
            window_target_equivalent_original_bin
        ),
        "backprojection_image": (
            window_backprojection_image
        ),
        "backprojection_db": (
            window_backprojection_db
        ),
        "focusing_coherence": (
            window_focusing_coherence
        ),
        "peak_cross_range_m": (
            window_peak_cross_range_m
        ),
        "peak_downrange_m": (
            window_peak_downrange_m
        ),
        "peak_focusing_coherence": (
            window_peak_focusing_coherence
        ),
        "cross_range_width_result": (
            window_cross_range_width_result
        ),
        "downrange_width_result": (
            window_downrange_width_result
        ),
        "alignment_table": (
            window_alignment_table
        ),
    }

    ideal_characteristics_row = (
        ideal_range_window_table[
            ideal_range_window_table[
                "window"
            ]
            == range_window_name.capitalize()
        ].iloc[0]
    )

    # Collect numerical comparison metrics.
    sar_range_window_metrics.append(
        {
            "window": (
                range_window_name.capitalize()
            ),
            "target_equivalent_original_bin": (
                window_target_equivalent_original_bin
            ),
            "ideal_minus3db_width_bins": (
                ideal_characteristics_row[
                    "ideal_minus3db_width_bins"
                ]
            ),
            "ideal_peak_sidelobe_level_db": (
                ideal_characteristics_row[
                    "ideal_peak_sidelobe_level_db"
                ]
            ),
            "equivalent_noise_bandwidth_bins": (
                ideal_characteristics_row[
                    "equivalent_noise_bandwidth_bins"
                ]
            ),
            "measured_unfocused_range_width_m": (
                measured_unfocused_range_width_m
            ),
            "image_peak_cross_range_m": (
                window_peak_cross_range_m
            ),
            "image_peak_downrange_m": (
                window_peak_downrange_m
            ),
            "image_peak_focusing_coherence": (
                window_peak_focusing_coherence
            ),
            "image_minus3db_cross_range_width_m": (
                window_cross_range_width_result[
                    "width"
                ]
            ),
            "image_minus3db_downrange_width_m": (
                window_downrange_width_result[
                    "width"
                ]
            ),
            "median_background_correlation": (
                window_alignment_table[
                    "complex_correlation"
                ].median()
            ),
            "median_background_reduction_db": (
                window_alignment_table[
                    "reference_residual_reduction_db"
                ].median()
            ),
        }
    )


sar_range_window_metrics_table = pd.DataFrame(
    sar_range_window_metrics
)


# ----------------------------------------------------------
# Plot the ideal range-window responses.
# ----------------------------------------------------------

fig, axis = plt.subplots(
    1,
    1,
    figsize=(11, 6),
    constrained_layout=True,
)

for range_window_name in (
    sar_range_windows_to_compare
):
    ideal_result = ideal_range_window_results[
        range_window_name
    ]

    axis.plot(
        ideal_result[
            "frequency_offset_bins"
        ],
        ideal_result[
            "power_db"
        ],
        linewidth=1.4,
        label=range_window_name.capitalize(),
    )

axis.set_xlim(
    -8,
    8,
)

axis.set_ylim(
    -80,
    2,
)

axis.set_title(
    "Ideal Oversampled Range-Window Responses"
)

axis.set_xlabel(
    "Offset from response center "
    "(original range-bin units)"
)

axis.set_ylabel(
    "Normalized power (dB)"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()

plt.show()


# ----------------------------------------------------------
# Plot the measured background-subtracted range responses.
# ----------------------------------------------------------

fig, axis = plt.subplots(
    1,
    1,
    figsize=(11, 6),
    constrained_layout=True,
)

for range_window_name in (
    sar_range_windows_to_compare
):
    axis.plot(
        range_window_equivalent_original_bin,
        sar_range_window_results[
            range_window_name
        ][
            "residual_range_power_db"
        ],
        linewidth=1.4,
        label=range_window_name.capitalize(),
    )

axis.axvline(
    22,
    color="gray",
    linestyle="--",
    linewidth=1,
    label="Original q = 22",
)

axis.set_xlim(
    17,
    30,
)

axis.set_ylim(
    -50,
    2,
)

axis.set_title(
    "Measured Background-Subtracted Range Responses"
)

axis.set_xlabel(
    "Equivalent original range-bin coordinate"
)

axis.set_ylabel(
    "Power relative to target-region maximum (dB)"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()

plt.show()


# ----------------------------------------------------------
# Compare the four coherently focused SAR images.
# ----------------------------------------------------------

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 11),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

range_window_image_display_floor_db = -25.0

range_window_image_handle = None

for axis, range_window_name in zip(
    axes.flat,
    sar_range_windows_to_compare,
):
    range_window_image_handle = axis.imshow(
        sar_range_window_results[
            range_window_name
        ][
            "backprojection_db"
        ],
        origin="lower",
        extent=image_extent_cm,
        aspect="equal",
        cmap="viridis",
        vmin=(
            range_window_image_display_floor_db
        ),
        vmax=0,
    )

    axis.plot(
        0,
        84,
        marker="+",
        color="red",
        markersize=14,
        markeredgewidth=2,
        label="Registration point",
    )

    axis.plot(
        sar_range_window_results[
            range_window_name
        ][
            "peak_cross_range_m"
        ]
        * 100,
        sar_range_window_results[
            range_window_name
        ][
            "peak_downrange_m"
        ]
        * 100,
        marker="x",
        color="cyan",
        markersize=10,
        markeredgewidth=2,
        label="Focused peak",
    )

    axis.set_title(
        range_window_name.capitalize()
    )

    axis.set_xlabel(
        "Cross-range, $x$ (cm)"
    )

    axis.set_ylabel(
        "Downrange, $z$ (cm)"
    )

axes[0, 0].legend(
    loc="upper right",
)

fig.colorbar(
    range_window_image_handle,
    ax=axes.ravel().tolist(),
    label="Normalized power (dB)",
)

fig.suptitle(
    "Run C SAR Range-Window Comparison",
    fontsize=15,
)

plt.show()


# ----------------------------------------------------------
# Compare profiles through each focused image peak.
# ----------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
    constrained_layout=True,
)

for range_window_name in (
    sar_range_windows_to_compare
):
    result = sar_range_window_results[
        range_window_name
    ]

    axes[0].plot(
        sar_image_cross_range_m * 100,
        result[
            "cross_range_width_result"
        ][
            "power_profile_db"
        ],
        linewidth=1.3,
        label=range_window_name.capitalize(),
    )

    axes[1].plot(
        sar_image_downrange_m,
        result[
            "downrange_width_result"
        ][
            "power_profile_db"
        ],
        linewidth=1.3,
        label=range_window_name.capitalize(),
    )

axes[0].axhline(
    -3,
    color="gray",
    linestyle="--",
    linewidth=1,
)

axes[0].set_title(
    "Cross-Range Profiles Through Each Peak"
)

axes[0].set_xlabel(
    "Cross-range, $x$ (cm)"
)

axes[0].set_ylabel(
    "Normalized power (dB)"
)

axes[0].set_ylim(
    -40,
    2,
)

axes[0].grid(
    True,
    alpha=0.3,
)

axes[0].legend()


axes[1].axhline(
    -3,
    color="gray",
    linestyle="--",
    linewidth=1,
)

axes[1].set_title(
    "Downrange Profiles Through Each Peak"
)

axes[1].set_xlabel(
    "Downrange, $z$ (m)"
)

axes[1].set_ylabel(
    "Normalized power (dB)"
)

axes[1].set_ylim(
    -50,
    2,
)

axes[1].grid(
    True,
    alpha=0.3,
)

axes[1].legend()

plt.show()


display(
    ideal_range_window_table
)

display(
    sar_range_window_metrics_table
)

### 3.6.3 — Results and Interpretation

The four-window comparison produced the expected range main-lobe and sidelobe tradeoff while leaving the synthetic-aperture cross-range focusing nearly unchanged.

<div style="font-size:2.0em; font-weight:bold;">
Ideal range-window behavior
</div>

The ideal zero-padded window responses are:

| Window | Ideal $-3$ dB width | Null-to-null width | Peak sidelobe | Equivalent noise bandwidth |
|---|---:|---:|---:|---:|
| Rectangular | $0.873$ bins | $2$ bins | $-13.4$ dB | $1.000$ bins |
| Hamming | $1.303$ bins | $4$ bins | $-42.6$ dB | $1.371$ bins |
| Hann | $1.445$ bins | $4$ bins | $-31.5$ dB | $1.512$ bins |
| Blackman | $1.649$ bins | $6$ bins | $-58.3$ dB | $1.740$ bins |

The rectangular window has the narrowest central response and smallest equivalent noise bandwidth, but its first sidelobe is only approximately 13.4 dB below the main peak.

The three tapered windows broaden the main lobe while reducing ideal sidelobes. Blackman provides the strongest ideal sidelobe suppression but also has the widest main lobe and largest equivalent noise bandwidth.

Hamming provides a narrower ideal main lobe than Hann while producing a lower first sidelobe. Hann has a faster asymptotic sidelobe decay than Hamming, but its first sidelobe is higher.

<div style="font-size:2.0em; font-weight:bold;">
Measured target range responses
</div>

The measured background-subtracted target responses remain centered near equivalent original range bin $q=22$.

The detected peak coordinates are:

| Window | Equivalent original target bin |
|---|---:|
| Rectangular | $21.875$ |
| Hamming | $22.000$ |
| Hann | $22.000$ |
| Blackman | $22.000$ |

The rectangular peak shifts by only $0.125$ original bins, corresponding to one 1024-point range-grid interval. This small change results from the measured asymmetric range spectrum and different spectral weighting. It is not evidence of a significant target-range change.

The rectangular response is visibly narrower near $q=22$, but it contains stronger oscillations on both sides of the target. These oscillations are consistent with its high ideal sidelobes.

The Hamming, Hann, and Blackman responses are smoother. Beyond approximately $q=24$, however, the three tapered-window curves become very similar. This suggests that much of the residual energy at larger range bins is not controlled solely by the range window.

Those features may contain:

- sphere-associated multipath;
- additional scattering contributions;
- residual background mismatch;
- other scene structure.

If the structures were purely window sidelobes, they would decrease much more strongly as the taper changed from Hamming to Hann to Blackman.

<div style="font-size:2.0em; font-weight:bold;">
Measured downrange widths
</div>

The measured unfocused and focused downrange widths follow the expected ordering:

| Window | Unfocused range width | Focused-image downrange width |
|---|---:|---:|
| Rectangular | $4.54$ cm | $4.48$ cm |
| Hamming | $6.69$ cm | $6.53$ cm |
| Hann | $7.30$ cm | $7.05$ cm |
| Blackman | $8.36$ cm | $8.04$ cm |

The similarity between each unfocused range width and its focused-image downrange width confirms that downrange extent is controlled primarily by the fast-time range response.

Synthetic-aperture focusing does not materially narrow this response because the mechanical aperture provides cross-range spatial bandwidth, not additional FMCW range bandwidth.

The rectangular result is the shortest vertically, but the associated range sidelobes produce multiple separated responses above and below the principal target location.

Blackman provides the broadest response. Its stronger ideal sidelobe suppression does not produce a correspondingly cleaner measured image, indicating that the remaining structures are not dominated solely by ideal window sidelobes.

Hamming provides an intermediate downrange width of approximately

$$\Delta z_{-3\mathrm{dB}}\approx6.53\ \text{cm}.$$

This is approximately 0.52 cm narrower than Hann while retaining strong ideal first-sidelobe suppression.

<div style="font-size:2.0em; font-weight:bold;">
Cross-range stability
</div>

The measured cross-range widths are nearly identical:

| Window | Cross-range $-3$ dB width |
|---|---:|
| Rectangular | $1.7245$ cm |
| Hamming | $1.7204$ cm |
| Hann | $1.7195$ cm |
| Blackman | $1.7213$ cm |

This confirms that changing the fast-time range window does not materially change synthetic-aperture cross-range resolution.

The range window weights fast-time samples within each chirp. Cross-range focusing is instead determined primarily by:

- wavelength;
- target range;
- synthetic-aperture length;
- rail-position sampling;
- synthetic-aperture weighting.

The repeated horizontal lobes also remain nearly unchanged among the four images. These features are therefore associated primarily with the synthetic aperture, multipath, or scene response rather than the selected fast-time range window.

<div style="font-size:2.0em; font-weight:bold;">
Focusing coherence and background matching
</div>

The peak focusing coherences are:

| Window | Peak focusing coherence |
|---|---:|
| Rectangular | $0.9493$ |
| Hamming | $0.9520$ |
| Hann | $0.9521$ |
| Blackman | $0.9513$ |

All four values are close to 0.95. The target phase therefore focuses consistently regardless of the selected range window.

The median complex Run B-to-Run C background correlations are also approximately $0.999$ for every window. Changing the range window does not degrade matched complex background subtraction.

<div style="font-size:2.0em; font-weight:bold;">
Range-window selection
</div>

Rectangular weighting provides the narrowest measured downrange response, but it also produces the strongest range oscillations and most visibly separated vertical sidelobe structure. It is useful as a resolution baseline but is not the preferred final choice for this scene.

Blackman provides the strongest ideal sidelobe suppression, but its approximately 8.04 cm downrange width is the broadest of the tested windows. The measured image does not become sufficiently cleaner to justify this additional broadening.

Hann remains a valid conservative choice, but Hamming provides:

- a narrower ideal and measured main lobe;
- a lower ideal first sidelobe;
- lower equivalent noise bandwidth;
- nearly identical focusing coherence;
- nearly identical background correlation;
- no visible loss of cross-range focus.

Hamming is therefore selected as the preferred range window for subsequent Run C SAR processing.

The selected baseline is now:

- 1024-point zero-padded range FFT;
- Hamming range window;
- Hann Doppler window;
- coherent frame averaging;
- matched complex Run B background subtraction;
- RX channel 3;
- uniform synthetic-aperture weighting;
- measured $+1$ propagation-phase convention;
- complex backprojection.

This selection is specific to the present measured scene and processing objective. It does not imply that Hamming is universally optimal for every radar measurement.

## 3.7 — Compare Synthetic-Aperture Weighting

Section 3.6 selected the Hamming fast-time range window. That window controls the vertical downrange point-spread function.

This section addresses the remaining horizontal cross-range lobes by applying weights across the 17 synthetic-aperture positions.

The two operations act along different data dimensions:

| Weighting operation | Data dimension | Primary image effect |
|---|---|---|
| Range window | Fast-time sample $i$ | Downrange main lobe and range sidelobes |
| Aperture window | Rail-position index $m$ | Cross-range main lobe and cross-range sidelobes |

For a candidate image pixel, uniformly weighted backprojection is

$$I_{\mathrm{uniform}}(x,z)=\frac{1}{M}\sum_{m=0}^{M-1}\widetilde{G}_m(x,z).$$

A weighted synthetic aperture instead uses

$$I_w(x,z)=\sum_{m=0}^{M-1}a_m\widetilde{G}_m(x,z),$$

where the real nonnegative weights satisfy

$$\sum_m a_m=1.$$

<div style="font-size:2.0em; font-weight:bold;">
Why aperture weighting reduces sidelobes
</div>

The edge positions of a uniformly weighted aperture contribute as strongly as the center positions. The abrupt beginning and end of the finite aperture produce cross-range sidelobes.

A tapered aperture window reduces the amplitudes of measurements near the aperture edges. This creates a smoother effective aperture and lowers sidelobes.

The cost is a reduction in effective aperture length. Because approximate cross-range resolution is inversely proportional to aperture length,

$$\delta_x\propto\frac{1}{L_{\mathrm{effective}}},$$

stronger tapering produces a wider cross-range main lobe.

Aperture tapering also increases the noise penalty relative to uniform weighting because fewer measurements contribute with their full strength.

<div style="font-size:2.0em; font-weight:bold;">
Windows compared
</div>

Three aperture-weighting choices are tested:

1. uniform;
2. Hamming;
3. Hann.

Uniform weighting provides the narrowest cross-range main lobe and greatest sensitivity, but it has the highest ideal sidelobes.

Hamming and Hann weighting reduce the edge-position contributions. Hamming retains small nonzero edge weights, while Hann reduces the first and final positions to zero.

The measured target amplitudes are not normalized across rail positions. The aperture coefficients multiply the actual complex measurements, preserving their naturally measured amplitude variation.

<div style="font-size:2.0em; font-weight:bold;">
Processing held fixed
</div>

Every aperture comparison uses:

- the selected Hamming fast-time range window;
- a 1024-point zero-padded range FFT;
- Hann Doppler weighting;
- coherent averaging of 20 frames;
- matched complex Run B background subtraction;
- RX channel 3;
- the measured $+1$ propagation-phase convention;
- the same target range registration;
- the same physical image grid.

No RADC reprocessing is required because the Hamming range-window residual from Section 3.6 is reused. Only the weights applied during coherent aperture summation are changed.

The ideal point-target aperture response is also calculated for the actual 17-position rail geometry. This separates the theoretical window response from the measured sphere, multipath, and residual-scene structure.

### 3.7.1 — Synthetic-Aperture Weighting Functions

The following functions:

- construct uniform, Hamming, and Hann aperture weights;
- simulate the ideal near-field cross-range response for the measured 17-position geometry;
- measure the main-lobe width;
- identify the largest response outside the central main lobe.

The measured out-of-main-lobe response is not automatically a pure sidelobe measurement. It can also contain multipath, extended-target scattering, and background-subtraction residuals.

In [ ]:
def create_synthetic_aperture_weights(
    window_name,
    number_of_positions,
):
    """
    Create normalized nonnegative amplitude weights for the
    synthetic-aperture positions.

    The returned weights sum to one so that changing the window
    does not introduce an arbitrary coherent-image scale.
    """

    normalized_name = window_name.lower()

    if normalized_name == "uniform":
        raw_weights = np.ones(
            number_of_positions,
            dtype=float,
        )

    elif normalized_name == "hamming":
        raw_weights = np.hamming(
            number_of_positions
        )

    elif normalized_name == "hann":
        raw_weights = np.hanning(
            number_of_positions
        )

    else:
        raise ValueError(
            "Unsupported aperture window. Choose from "
            "'uniform', 'hamming', or 'hann'."
        )

    if not np.any(
        raw_weights > 0
    ):
        raise ValueError(
            "The aperture window contains no positive weights."
        )

    # Normalize the coefficients to unit sum.
    #
    # This preserves image shape and relative coherent gain
    # comparisons without allowing a window with a larger
    # coefficient sum to appear artificially brighter.
    normalized_weights = (
        raw_weights
        / np.sum(
            raw_weights
        )
    )

    return normalized_weights


def characterize_mainlobe_and_outside_response(
    coordinate,
    power_profile,
):
    """
    Characterize one normalized power profile.

    The nearest local minima around the strongest peak define
    the central main-lobe boundaries. The largest value outside
    those boundaries is reported as an out-of-main-lobe level.

    For ideal point-target simulations, this is the ideal peak
    sidelobe level.

    For measured data, it may also contain multipath, clutter,
    or additional scattering centers.
    """

    coordinate = np.asarray(
        coordinate,
        dtype=float,
    )

    power_profile = np.asarray(
        power_profile,
        dtype=float,
    )

    if coordinate.shape != power_profile.shape:
        raise ValueError(
            "coordinate and power_profile must match."
        )

    normalized_power = (
        power_profile
        / np.maximum(
            np.max(
                power_profile
            ),
            np.finfo(float).tiny,
        )
    )

    normalized_power_db = (
        10
        * np.log10(
            np.maximum(
                normalized_power,
                np.finfo(float).tiny,
            )
        )
    )

    peak_index = int(
        np.argmax(
            normalized_power
        )
    )

    # Measure the contiguous -3 dB width surrounding the
    # strongest peak.
    minus3db_result = (
        measure_contiguous_minus_3db_width(
            coordinate=coordinate,
            power_profile=normalized_power,
        )
    )

    # Find local minima that can serve as the boundaries of
    # the central main lobe.
    local_minimum_indices = (
        np.flatnonzero(
            (
                normalized_power[1:-1]
                <= normalized_power[:-2]
            )
            & (
                normalized_power[1:-1]
                < normalized_power[2:]
            )
        )
        + 1
    )

    left_minimum_candidates = (
        local_minimum_indices[
            local_minimum_indices
            < peak_index
        ]
    )

    right_minimum_candidates = (
        local_minimum_indices[
            local_minimum_indices
            > peak_index
        ]
    )

    if (
        left_minimum_candidates.size == 0
        or right_minimum_candidates.size == 0
    ):
        left_mainlobe_minimum_index = None
        right_mainlobe_minimum_index = None
        null_to_null_width = np.nan
        maximum_outside_mainlobe_db = np.nan

    else:
        left_mainlobe_minimum_index = (
            left_minimum_candidates[-1]
        )

        right_mainlobe_minimum_index = (
            right_minimum_candidates[0]
        )

        null_to_null_width = (
            coordinate[
                right_mainlobe_minimum_index
            ]
            - coordinate[
                left_mainlobe_minimum_index
            ]
        )

        # Exclude the region between the first minima and find
        # the largest remaining response.
        outside_mainlobe_mask = np.ones(
            normalized_power.shape,
            dtype=bool,
        )

        outside_mainlobe_mask[
            left_mainlobe_minimum_index:
            right_mainlobe_minimum_index + 1
        ] = False

        maximum_outside_mainlobe_db = np.max(
            normalized_power_db[
                outside_mainlobe_mask
            ]
        )

    return {
        "peak_index": peak_index,
        "peak_coordinate": coordinate[
            peak_index
        ],
        "minus3db_width": (
            minus3db_result[
                "width"
            ]
        ),
        "null_to_null_width": (
            null_to_null_width
        ),
        "maximum_outside_mainlobe_db": (
            maximum_outside_mainlobe_db
        ),
        "power_profile_db": (
            normalized_power_db
        ),
        "left_mainlobe_minimum_index": (
            left_mainlobe_minimum_index
        ),
        "right_mainlobe_minimum_index": (
            right_mainlobe_minimum_index
        ),
    }


def simulate_ideal_near_field_aperture_response(
    aperture_positions_m,
    aperture_weights,
    cross_range_coordinates_m,
    target_cross_range_m,
    target_downrange_m,
    carrier_frequency_hz,
    measured_phase_sign,
    reference_aperture_index,
):
    """
    Simulate the ideal cross-range response of one point target
    using the actual synthetic-aperture positions.

    The simulated point target has unit magnitude and exactly
    follows the assumed geometric phase model. Therefore, any
    sidelobes are produced by the finite sampled aperture and
    its selected weighting, not by noise, multipath, or clutter.
    """

    aperture_positions_m = np.asarray(
        aperture_positions_m,
        dtype=float,
    )

    aperture_weights = np.asarray(
        aperture_weights,
        dtype=float,
    )

    if aperture_positions_m.shape != (
        aperture_weights.shape
    ):
        raise ValueError(
            "Aperture positions and weights must match."
        )

    # Normalize again for safety so that the simulated response
    # is independent of the raw coefficient sum.
    aperture_weights = (
        aperture_weights
        / np.sum(
            aperture_weights
        )
    )

    speed_of_light_m_per_s = 299_792_458.0

    wavelength_m = (
        speed_of_light_m_per_s
        / carrier_frequency_hz
    )

    # Calculate the distance from every aperture position to
    # the true simulated point target.
    true_target_range_m = np.sqrt(
        (
            target_cross_range_m
            - aperture_positions_m
        ) ** 2
        + target_downrange_m ** 2
    )

    true_center_range_m = (
        true_target_range_m[
            reference_aperture_index
        ]
    )

    # Generate the complex measurements that an ideal point
    # target would produce across the aperture.
    ideal_target_samples = np.exp(
        1j
        * measured_phase_sign
        * 4
        * np.pi
        / wavelength_m
        * (
            true_target_range_m
            - true_center_range_m
        )
    )

    # Calculate the predicted distance from every aperture
    # position to every trial cross-range coordinate.
    #
    # Shape:
    #   (aperture position, trial cross-range coordinate)
    candidate_range_m = np.sqrt(
        (
            cross_range_coordinates_m[
                np.newaxis,
                :
            ]
            - aperture_positions_m[
                :,
                np.newaxis,
            ]
        ) ** 2
        + target_downrange_m ** 2
    )

    candidate_center_range_m = (
        candidate_range_m[
            reference_aperture_index,
            :
        ]
    )

    # Construct the phase correction associated with every
    # trial cross-range location.
    candidate_phase_correction = np.exp(
        -1j
        * measured_phase_sign
        * 4
        * np.pi
        / wavelength_m
        * (
            candidate_range_m
            - candidate_center_range_m[
                np.newaxis,
                :
            ]
        )
    )

    # Focus the ideal target at every trial cross-range
    # coordinate.
    #
    # The response peaks when the candidate coordinate matches
    # the true target coordinate.
    ideal_focused_response = np.sum(
        aperture_weights[
            :,
            np.newaxis,
        ]
        * ideal_target_samples[
            :,
            np.newaxis,
        ]
        * candidate_phase_correction,
        axis=0,
    )

    ideal_power_response = (
        np.abs(
            ideal_focused_response
        ) ** 2
    )

    return ideal_power_response

### 3.7.2 — Process and Compare Synthetic-Aperture Windows

The selected Hamming range-window residual from Section 3.6 is reused for every comparison.

For each synthetic-aperture window, the code:

1. constructs 17 normalized rail-position weights;
2. simulates the ideal point-target cross-range response;
3. forms a measured SAR image;
4. locates the focused image peak;
5. measures cross-range and downrange $-3$ dB widths;
6. measures the largest response outside the central cross-range main lobe;
7. calculates peak focusing coherence;
8. calculates the relative white-noise penalty.

The images are independently normalized to compare their spatial shapes. Their displayed brightness does not compare absolute processing gain.

In [ ]:
# Lock the selected range-processing result from Section 3.6.
#
# All aperture-window comparisons use the same Hamming
# fast-time range response. Only the weights across mechanical
# rail positions are changed.
selected_sar_range_window = "hamming"

selected_hamming_range_result = (
    sar_range_window_results[
        selected_sar_range_window
    ]
)

selected_hamming_aperture_profiles = (
    selected_hamming_range_result[
        "residual"
    ]
)

selected_hamming_target_reference_bin = (
    selected_hamming_range_result[
        "target_reference_bin"
    ]
)


# Define the synthetic-aperture windows to compare.
sar_aperture_windows_to_compare = [
    "uniform",
    "hamming",
    "hann",
]


# Use a finely sampled cross-range axis to characterize the
# ideal aperture responses independently of the 1 mm image grid.
ideal_aperture_cross_range_m = np.linspace(
    -0.10,
    0.10,
    4001,
)


# Store complete arrays and compact numerical metrics.
sar_aperture_window_results = {}
sar_aperture_window_metrics = []


for aperture_window_name in (
    sar_aperture_windows_to_compare
):
    print(
        "Processing aperture window:",
        aperture_window_name,
    )

    # Construct normalized weights across the 17 rail
    # positions.
    aperture_weights = (
        create_synthetic_aperture_weights(
            window_name=(
                aperture_window_name
            ),
            number_of_positions=(
                selected_hamming_aperture_profiles.shape[0]
            ),
        )
    )

    # Calculate the white-noise penalty relative to uniform
    # weighting.
    #
    # For weights that sum to one, uniform weighting minimizes
    # sum(a_m^2). Tapered windows increase this value because
    # fewer positions contribute with their full strength.
    aperture_noise_penalty_db = (
        10
        * np.log10(
            selected_hamming_aperture_profiles.shape[0]
            * np.sum(
                aperture_weights ** 2
            )
        )
    )

    # Simulate the ideal point-target cross-range response for
    # the actual rail geometry and selected weights.
    ideal_aperture_power = (
        simulate_ideal_near_field_aperture_response(
            aperture_positions_m=(
                selected_rx_aperture_positions_m
            ),
            aperture_weights=(
                aperture_weights
            ),
            cross_range_coordinates_m=(
                ideal_aperture_cross_range_m
            ),
            target_cross_range_m=0.0,
            target_downrange_m=0.84,
            carrier_frequency_hz=(
                carrier_frequency_hz
            ),
            measured_phase_sign=(
                backprojection_measured_phase_sign
            ),
            reference_aperture_index=(
                CENTER_POSITION_INDEX
            ),
        )
    )

    ideal_aperture_characteristics = (
        characterize_mainlobe_and_outside_response(
            coordinate=(
                ideal_aperture_cross_range_m
            ),
            power_profile=(
                ideal_aperture_power
            ),
        )
    )

    # Form the measured SAR image using the selected aperture
    # weights. All range processing and phase conventions are
    # held fixed.
    (
        aperture_window_image,
        aperture_window_noncoherent_image,
        aperture_window_focusing_coherence,
        aperture_window_valid_support_count,
        _,
        _,
    ) = backproject_single_rx(
        aperture_range_profiles=(
            selected_hamming_aperture_profiles
        ),
        aperture_positions_m=(
            selected_rx_aperture_positions_m
        ),
        image_cross_range_m=(
            sar_image_cross_range_m
        ),
        image_downrange_m=(
            sar_image_downrange_m
        ),
        carrier_frequency_hz=(
            carrier_frequency_hz
        ),
        measured_phase_sign=(
            backprojection_measured_phase_sign
        ),
        reference_range_bin=(
            selected_hamming_target_reference_bin
        ),
        registration_range_m=(
            sar_registration_range_m
        ),
        range_bin_spacing_m=(
            range_window_comparison_spacing_m
        ),
        reference_aperture_index=(
            CENTER_POSITION_INDEX
        ),
        aperture_weights=(
            aperture_weights
        ),
    )

    # Every candidate pixel should use all 17 aperture
    # positions, even though tapered windows reduce the edge
    # positions' amplitudes.
    assert np.all(
        aperture_window_valid_support_count
        == selected_hamming_aperture_profiles.shape[0]
    ), (
        "Incomplete aperture support for "
        f"{aperture_window_name}."
    )

    aperture_window_image_power = (
        np.abs(
            aperture_window_image
        ) ** 2
    )

    aperture_window_image_db = (
        normalized_power_db(
            aperture_window_image,
            minimum_db=-60.0,
        )
    )

    # Locate the largest coherently focused image pixel.
    (
        aperture_window_peak_downrange_index,
        aperture_window_peak_cross_range_index,
    ) = np.unravel_index(
        np.argmax(
            aperture_window_image_power
        ),
        aperture_window_image_power.shape,
    )

    aperture_window_peak_cross_range_m = (
        sar_image_cross_range_m[
            aperture_window_peak_cross_range_index
        ]
    )

    aperture_window_peak_downrange_m = (
        sar_image_downrange_m[
            aperture_window_peak_downrange_index
        ]
    )

    aperture_window_peak_coherence = (
        aperture_window_focusing_coherence[
            aperture_window_peak_downrange_index,
            aperture_window_peak_cross_range_index,
        ]
    )

    # Extract cross-range and downrange profiles through the
    # measured image peak.
    measured_cross_range_power = (
        aperture_window_image_power[
            aperture_window_peak_downrange_index,
            :,
        ]
    )

    measured_downrange_power = (
        aperture_window_image_power[
            :,
            aperture_window_peak_cross_range_index,
        ]
    )

    measured_cross_range_characteristics = (
        characterize_mainlobe_and_outside_response(
            coordinate=sar_image_cross_range_m,
            power_profile=(
                measured_cross_range_power
            ),
        )
    )

    measured_downrange_width_result = (
        measure_contiguous_minus_3db_width(
            coordinate=sar_image_downrange_m,
            power_profile=(
                measured_downrange_power
            ),
        )
    )

    # Save complete results for plotting and later selection.
    sar_aperture_window_results[
        aperture_window_name
    ] = {
        "weights": aperture_weights,
        "noise_penalty_db": (
            aperture_noise_penalty_db
        ),
        "ideal_power": (
            ideal_aperture_power
        ),
        "ideal_characteristics": (
            ideal_aperture_characteristics
        ),
        "image": aperture_window_image,
        "image_db": aperture_window_image_db,
        "focusing_coherence": (
            aperture_window_focusing_coherence
        ),
        "peak_cross_range_m": (
            aperture_window_peak_cross_range_m
        ),
        "peak_downrange_m": (
            aperture_window_peak_downrange_m
        ),
        "peak_coherence": (
            aperture_window_peak_coherence
        ),
        "cross_range_characteristics": (
            measured_cross_range_characteristics
        ),
        "downrange_width_result": (
            measured_downrange_width_result
        ),
    }

    sar_aperture_window_metrics.append(
        {
            "aperture_window": (
                aperture_window_name.capitalize()
            ),
            "noise_penalty_db": (
                aperture_noise_penalty_db
            ),
            "ideal_minus3db_width_m": (
                ideal_aperture_characteristics[
                    "minus3db_width"
                ]
            ),
            "ideal_peak_sidelobe_level_db": (
                ideal_aperture_characteristics[
                    "maximum_outside_mainlobe_db"
                ]
            ),
            "measured_peak_cross_range_m": (
                aperture_window_peak_cross_range_m
            ),
            "measured_peak_downrange_m": (
                aperture_window_peak_downrange_m
            ),
            "measured_peak_coherence": (
                aperture_window_peak_coherence
            ),
            "measured_minus3db_cross_range_width_m": (
                measured_cross_range_characteristics[
                    "minus3db_width"
                ]
            ),
            "measured_maximum_outside_mainlobe_db": (
                measured_cross_range_characteristics[
                    "maximum_outside_mainlobe_db"
                ]
            ),
            "measured_minus3db_downrange_width_m": (
                measured_downrange_width_result[
                    "width"
                ]
            ),
        }
    )


sar_aperture_window_metrics_table = pd.DataFrame(
    sar_aperture_window_metrics
)


# ----------------------------------------------------------
# Plot the aperture-weight coefficients.
# ----------------------------------------------------------

fig, axis = plt.subplots(
    1,
    1,
    figsize=(11, 5),
    constrained_layout=True,
)

for aperture_window_name in (
    sar_aperture_windows_to_compare
):
    aperture_weights = (
        sar_aperture_window_results[
            aperture_window_name
        ][
            "weights"
        ]
    )

    # Normalize to each window's maximum for a direct visual
    # comparison of taper shape.
    axis.plot(
        rail_positions_m * 100,
        aperture_weights
        / np.max(
            aperture_weights
        ),
        marker="o",
        linewidth=1.3,
        label=(
            aperture_window_name.capitalize()
        ),
    )

axis.set_title(
    "Normalized Synthetic-Aperture Weights"
)

axis.set_xlabel(
    "Rail position, $x_m$ (cm)"
)

axis.set_ylabel(
    "Amplitude coefficient relative to window maximum"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()

plt.show()


# ----------------------------------------------------------
# Plot the ideal point-target cross-range responses.
# ----------------------------------------------------------

fig, axis = plt.subplots(
    1,
    1,
    figsize=(11, 6),
    constrained_layout=True,
)

for aperture_window_name in (
    sar_aperture_windows_to_compare
):
    ideal_power = (
        sar_aperture_window_results[
            aperture_window_name
        ][
            "ideal_power"
        ]
    )

    ideal_power_db = (
        10
        * np.log10(
            np.maximum(
                ideal_power,
                np.finfo(float).tiny,
            )
            / np.maximum(
                np.max(
                    ideal_power
                ),
                np.finfo(float).tiny,
            )
        )
    )

    axis.plot(
        ideal_aperture_cross_range_m * 100,
        ideal_power_db,
        linewidth=1.4,
        label=(
            aperture_window_name.capitalize()
        ),
    )

axis.set_xlim(
    -10,
    10,
)

axis.set_ylim(
    -60,
    2,
)

axis.set_title(
    "Ideal Near-Field Synthetic-Aperture Responses"
)

axis.set_xlabel(
    "Cross-range, $x$ (cm)"
)

axis.set_ylabel(
    "Normalized power (dB)"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()

plt.show()


# ----------------------------------------------------------
# Compare the measured coherently focused SAR images.
# ----------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 6),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

aperture_image_display_floor_db = -25.0

aperture_image_handle = None

for axis, aperture_window_name in zip(
    axes,
    sar_aperture_windows_to_compare,
):
    aperture_image_handle = axis.imshow(
        sar_aperture_window_results[
            aperture_window_name
        ][
            "image_db"
        ],
        origin="lower",
        extent=image_extent_cm,
        aspect="equal",
        cmap="viridis",
        vmin=aperture_image_display_floor_db,
        vmax=0,
    )

    axis.plot(
        0,
        84,
        marker="+",
        color="red",
        markersize=14,
        markeredgewidth=2,
        label="Registration point",
    )

    axis.plot(
        sar_aperture_window_results[
            aperture_window_name
        ][
            "peak_cross_range_m"
        ]
        * 100,
        sar_aperture_window_results[
            aperture_window_name
        ][
            "peak_downrange_m"
        ]
        * 100,
        marker="x",
        color="cyan",
        markersize=10,
        markeredgewidth=2,
        label="Focused peak",
    )

    axis.set_title(
        aperture_window_name.capitalize()
    )

    axis.set_xlabel(
        "Cross-range, $x$ (cm)"
    )

    axis.set_ylabel(
        "Downrange, $z$ (cm)"
    )

axes[0].legend(
    loc="upper right",
)

fig.colorbar(
    aperture_image_handle,
    ax=axes,
    label="Normalized power (dB)",
)

fig.suptitle(
    "Run C Synthetic-Aperture Weighting Comparison",
    fontsize=15,
)

plt.show()


# ----------------------------------------------------------
# Compare profiles through each measured image peak.
# ----------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
    constrained_layout=True,
)

for aperture_window_name in (
    sar_aperture_windows_to_compare
):
    result = sar_aperture_window_results[
        aperture_window_name
    ]

    axes[0].plot(
        sar_image_cross_range_m * 100,
        result[
            "cross_range_characteristics"
        ][
            "power_profile_db"
        ],
        linewidth=1.4,
        label=(
            aperture_window_name.capitalize()
        ),
    )

    axes[1].plot(
        sar_image_downrange_m,
        result[
            "downrange_width_result"
        ][
            "power_profile_db"
        ],
        linewidth=1.4,
        label=(
            aperture_window_name.capitalize()
        ),
    )

axes[0].axhline(
    -3,
    color="gray",
    linestyle="--",
    linewidth=1,
)

axes[0].set_title(
    "Cross-Range Profiles Through Each Peak"
)

axes[0].set_xlabel(
    "Cross-range, $x$ (cm)"
)

axes[0].set_ylabel(
    "Normalized power (dB)"
)

axes[0].set_ylim(
    -50,
    2,
)

axes[0].grid(
    True,
    alpha=0.3,
)

axes[0].legend()


axes[1].axhline(
    -3,
    color="gray",
    linestyle="--",
    linewidth=1,
)

axes[1].set_title(
    "Downrange Profiles Through Each Peak"
)

axes[1].set_xlabel(
    "Downrange, $z$ (m)"
)

axes[1].set_ylabel(
    "Normalized power (dB)"
)

axes[1].set_ylim(
    -50,
    2,
)

axes[1].grid(
    True,
    alpha=0.3,
)

axes[1].legend()

plt.show()


display(
    sar_aperture_window_metrics_table
)

### 3.7.3 — Results and Interpretation

Synthetic-aperture weighting produced a substantial reduction in the measured horizontal cross-range lobes while leaving the downrange response nearly unchanged.

<div style="font-size:2.0em; font-weight:bold;">
Aperture-weight shapes
</div>

Uniform weighting assigns equal amplitude to every rail position:

$$a_m=\frac{1}{17}.$$

Hamming and Hann weighting reduce the contribution of positions near the ends of the aperture.

Hamming retains small nonzero edge weights, while Hann reduces the first and final aperture positions to zero. Both tapers emphasize measurements near the aperture center and reduce the abrupt boundaries responsible for uniform-aperture sidelobes.

The aperture weights multiply the measured complex samples. They do not modify or recalibrate the measured phase.

<div style="font-size:2.0em; font-weight:bold;">
Ideal point-target aperture responses
</div>

The ideal responses for the measured 17-position geometry are:

| Aperture window | Ideal $-3$ dB width | Ideal peak sidelobe | Noise penalty |
|---|---:|---:|---:|
| Uniform | $1.723$ cm | $-13.13$ dB | $0.00$ dB |
| Hamming | $2.630$ cm | $-39.71$ dB | $1.53$ dB |
| Hann | $2.971$ cm | $-31.36$ dB | $2.02$ dB |

Uniform weighting provides the narrowest ideal cross-range main lobe but produces the highest sidelobes.

Hamming broadens the ideal $-3$ dB response by approximately

$$\frac{2.630}{1.723}\approx1.53,$$

or 53%, while reducing the ideal first sidelobe by approximately 26.6 dB.

Hann produces the widest ideal response and a greater noise penalty. Its first sidelobe is higher than Hamming’s, although its distant sidelobes decay differently.

<div style="font-size:2.0em; font-weight:bold;">
Measured SAR images
</div>

The measured results are:

| Aperture window | Peak $x$ | Peak $z$ | Peak coherence | Cross-range width | Maximum outside main lobe | Downrange width |
|---|---:|---:|---:|---:|---:|---:|
| Uniform | $-0.6$ cm | $84.0$ cm | $0.9520$ | $1.720$ cm | $-10.89$ dB | $6.528$ cm |
| Hamming | $-0.6$ cm | $83.4$ cm | $0.9752$ | $2.559$ cm | $-20.60$ dB | $6.504$ cm |
| Hann | $-0.6$ cm | $83.4$ cm | $0.9832$ | $2.825$ cm | $-19.50$ dB | $6.499$ cm |

The uniform image contains multiple distinct horizontal lobes on both sides of the central response. Hamming and Hann tapering substantially reduce these structures and produce a cleaner dominant target region.

The measured maximum response outside the central main lobe changes from

$$-10.89\ \text{dB}$$

with uniform weighting to

$$-20.60\ \text{dB}$$

with Hamming weighting. This is an improvement of approximately

$$9.7\ \text{dB}.$$

The measured improvement is smaller than the ideal-window prediction because the measured out-of-main-lobe response is not composed solely of ideal aperture sidelobes. It may also contain:

- sphere-associated multipath;
- extended-target scattering;
- background-subtraction residuals;
- antenna phase-center modeling error;
- other scene structure.

Nevertheless, the substantial change with aperture weighting confirms that a major part of the original horizontal lobe structure was produced by the finite uniformly weighted synthetic aperture.

<div style="font-size:2.0em; font-weight:bold;">
Cross-range tradeoff
</div>

Hamming weighting increases the measured cross-range $-3$ dB width from

$$1.720\ \text{cm}$$

to

$$2.559\ \text{cm}.$$

This broadening is the cost of reducing the synthetic-aperture sidelobes. Tapering reduces the effective aperture length because the measurements near the rail endpoints contribute less strongly.

Hann broadens the response further to approximately

$$2.825\ \text{cm}.$$

The measured cross-range widths closely follow the corresponding ideal point-target predictions. This agreement demonstrates that the main-lobe broadening is an expected aperture-window effect rather than a processing failure.

<div style="font-size:2.0em; font-weight:bold;">
Downrange stability
</div>

The measured downrange widths remain approximately constant:

$$6.50\ \text{cm}\leq\Delta z_{-3\mathrm{dB}}\leq6.53\ \text{cm}.$$

This confirms that synthetic-aperture weighting acts primarily along cross-range.

The downrange response remains controlled by:

- FMCW bandwidth;
- Hamming fast-time range window;
- target range structure;
- range interpolation.

Changing weights across rail position does not change the fast-time bandwidth and therefore does not materially change range resolution.

<div style="font-size:2.0em; font-weight:bold;">
Peak-position stability
</div>

All three images place the focused cross-range peak at approximately

$$x_{\mathrm{peak}}=-0.6\ \text{cm}.$$

This stability confirms that aperture tapering changes the point-spread-function shape without materially shifting the target’s cross-range location.

The tapered images place the downrange peak at approximately $83.4$ cm rather than $84.0$ cm. This difference is approximately 6 mm, or one 1024-point range-grid interval.

Because the complete Hamming-windowed range response is approximately 6.5 cm wide and the absolute range axis uses a provisional one-point registration, this single-grid-step change is not physically significant.

<div style="font-size:2.0em; font-weight:bold;">
Focusing-coherence increase
</div>

The peak focusing coherence increases from

$$0.9520$$

with uniform weighting to

$$0.9752$$

with Hamming and

$$0.9832$$

with Hann.

This occurs because the tapered windows reduce the contributions of edge-aperture samples that have greater residual phase or model mismatch.

The increased coherence does not mean that tapered weighting adds information. It means the retained weighted samples align more efficiently. The improvement is obtained at the cost of:

- a broader cross-range main lobe;
- reduced effective aperture length;
- increased white-noise penalty.

Therefore, coherence alone should not determine the selected window.

<div style="font-size:2.0em; font-weight:bold;">
Aperture-window selection
</div>

Uniform weighting provides the greatest cross-range resolution and no additional white-noise penalty, but its approximately $-10.9$ dB measured out-of-main-lobe response produces visually prominent horizontal lobes.

Hann provides the highest focusing coherence, but it also produces:

- the widest measured cross-range response;
- the largest noise penalty;
- slightly worse measured out-of-main-lobe suppression than Hamming.

Hamming provides the preferred balance:

- approximately $9.7$ dB improvement in measured out-of-main-lobe response;
- approximately $2.56$ cm measured cross-range width;
- $1.53$ dB noise penalty;
- $0.975$ peak focusing coherence;
- stable target location;
- unchanged downrange width.

Hamming is therefore selected as the preferred synthetic-aperture window for the final processing pipeline.

<div style="font-size:2.0em; font-weight:bold;">
Selected Run C processing configuration
</div>

The selected baseline Run C SAR pipeline is:

1. decode complex V-MD3 RADC data;
2. select RX channel 3;
3. apply a Hamming fast-time range window;
4. calculate a 1024-point zero-padded range FFT;
5. apply a Hann slow-time Doppler window;
6. extract zero Doppler;
7. coherently average the 20 repeated frames at each rail position;
8. estimate matched complex Run B background alignment independently at every position;
9. subtract the aligned Run B background;
10. register the measured target-associated range response locally;
11. apply Hamming synthetic-aperture weighting;
12. interpolate the oversampled complex range profiles;
13. apply geometric backprojection phase compensation;
14. coherently sum the 17 rail positions;
15. display normalized image power and calculate focusing metrics.

The resulting image prioritizes sidelobe suppression and interpretability while retaining approximately 2.6 cm measured cross-range width and 6.5 cm measured downrange width.

## 3.8 — Final Run C Processing Comparison

This section summarizes the evolution from conventional V-MD3 processing to the selected Run C SAR reconstruction.

The four panels represent different stages:

1. a conventional four-RX range–azimuth heatmap formed at the center rail position using minimal rectangular-window processing;
2. the background-subtracted but unfocused synthetic-aperture data;
3. the first coherent backprojection result using the original 128-point range FFT and uniform synthetic-aperture weighting;
4. the selected final Run C result using a Hamming range window, a 1024-point zero-padded range FFT, and Hamming synthetic-aperture weighting.

The first two panels are diagnostic data products rather than SAR images. Only the last two panels have been geometrically focused into cross-range and downrange coordinates.

All four panels are normalized independently and displayed with the same decibel floor. Therefore, the figure compares spatial structure and artifact suppression, not absolute received power.

### 3.8.1 — Comparison Functions

The functions in this subsection prepare the four processing stages for a consistent side-by-side comparison.

They perform four tasks:

1. locate the saved intermediate products from the earlier Run C subsections;
2. reconstruct a minimally processed, conventional four-RX range–azimuth heatmap at the center rail position;
3. extract the first and final SAR images from their stored result dictionaries;
4. plot all four stages using a common decibel range.

The conventional heatmap uses only the four physical RX channels. It does not combine measurements from different rail positions and is therefore not a SAR image.

The unfocused synthetic-aperture panel retains the complex target response at every rail position. Its displayed magnitude may look orderly because the magnitude operation hides the position-dependent phase that must be corrected during focusing.

The final two panels are spatial images produced by coherent SAR backprojection.

In [ ]:
def get_existing_notebook_variable(*candidate_names):
    """
    Return the first existing notebook variable from a list of names.

    This helper is used only for the final summary figure. It lets the
    comparison cell find results produced in earlier subsections even if
    a result variable was given a slightly different descriptive name.
    """

    notebook_namespace = globals()

    for candidate_name in candidate_names:
        if candidate_name in notebook_namespace:
            return (
                notebook_namespace[candidate_name],
                candidate_name,
            )

    raise NameError(
        "None of the expected notebook variables exist: "
        + ", ".join(candidate_names)
    )


def extract_sar_image_db(result):
    """
    Extract an already-normalized SAR image from a result dictionary.

    Earlier backprojection sections stored the displayed image in a
    dictionary. This helper accepts the most likely key names and returns
    the two-dimensional decibel image.
    """

    if not isinstance(result, dict):
        image_db = np.asarray(result)

    else:
        possible_image_keys = (
            "image_db",
            "sar_image_db",
            "normalized_image_db",
        )

        image_db = None

        for image_key in possible_image_keys:
            if image_key in result:
                image_db = np.asarray(result[image_key])
                break

        if image_db is None:
            raise KeyError(
                "The SAR result does not contain any of these image keys: "
                + ", ".join(possible_image_keys)
            )

    if image_db.ndim != 2:
        raise ValueError(
            "The extracted SAR image must be two-dimensional."
        )

    return image_db


def form_minimal_center_position_range_azimuth(
    run_c_radc,
    center_position_index,
    azimuth_angle_deg,
    rx_positions_m,
    carrier_frequency_hz,
):
    """
    Form an original-style four-RX heatmap from the center rail position.

    This deliberately uses rectangular range and Doppler windows. It
    therefore resembles the minimally processed range–azimuth heatmap
    produced near the beginning of the notebook.

    The 20 frames are coherently averaged before beamforming. This reduces
    random noise but does not introduce synthetic-aperture processing:
    only the four physical RX elements are used to estimate angle.
    """

    run_c_radc = np.asarray(run_c_radc)

    if run_c_radc.ndim != 5:
        raise ValueError(
            "run_c_radc must have shape "
            "(position, frame, fast time, chirp, RX)."
        )

    center_position_radc = run_c_radc[
        center_position_index
    ]

    center_zero_doppler_frames = []

    for radc_frame in center_position_radc:

        # Perform the minimally processed rectangular-window
        # range–Doppler transform for one captured frame.
        range_doppler_cube = compute_radc_range_doppler(
            radc_frame=radc_frame,
            range_window="rectangular",
            doppler_window="rectangular",
            remove_fast_time_mean=False,
            center_doppler=True,
        )

        # Retain the stationary-scene Doppler bin. The sphere and most
        # laboratory clutter should appear near zero Doppler because the
        # radar was stationary while each frame was recorded.
        zero_doppler_profile, _ = extract_zero_doppler(
            range_doppler_cube,
            doppler_axis=1,
        )

        center_zero_doppler_frames.append(
            zero_doppler_profile
        )

    center_zero_doppler_frames = np.stack(
        center_zero_doppler_frames,
        axis=0,
    )

    # Coherently average complex values across the repeated frames.
    # This is still physical four-RX processing, not SAR focusing.
    center_zero_doppler_mean = np.mean(
        center_zero_doppler_frames,
        axis=0,
    )

    # Beamform across only the four physical RX channels. The result
    # estimates arrival angle independently at every range bin.
    range_azimuth_map = conventional_range_azimuth(
        range_rx_data=center_zero_doppler_mean,
        azimuth_angle_deg=azimuth_angle_deg,
        rx_positions_m=rx_positions_m,
        carrier_frequency_hz=carrier_frequency_hz,
    )

    return (
        center_zero_doppler_mean,
        range_azimuth_map,
    )


def plot_run_c_processing_evolution(
    range_azimuth_map,
    azimuth_angle_deg,
    unfocused_residual_cube,
    rail_position_m,
    selected_rx_channel,
    first_sar_image_db,
    final_sar_image_db,
    cross_range_grid_m,
    downrange_grid_m,
    observed_range_bin=22,
    registration_cross_range_m=0.0,
    registration_downrange_m=0.84,
    range_bin_limit=50,
    display_floor_db=-25.0,
):
    """
    Plot four stages of the Run C processing evolution.

    Panels 1 and 2 retain range-bin coordinates and are diagnostic views.
    Panels 3 and 4 are focused SAR images in physical spatial coordinates.
    """

    range_azimuth_map = np.asarray(
        range_azimuth_map
    )

    unfocused_residual_cube = np.asarray(
        unfocused_residual_cube
    )

    rail_position_m = np.asarray(
        rail_position_m
    )

    cross_range_grid_m = np.asarray(
        cross_range_grid_m
    )

    downrange_grid_m = np.asarray(
        downrange_grid_m
    )

    if unfocused_residual_cube.ndim != 3:
        raise ValueError(
            "unfocused_residual_cube must have shape "
            "(rail position, range bin, RX)."
        )

    # Keep the phase-preserving residual for the selected physical RX.
    # The plot displays its magnitude, although the underlying array is
    # still complex and contains the phase history required for focusing.
    unfocused_selected_rx = (
        unfocused_residual_cube[
            :,
            :range_bin_limit,
            selected_rx_channel,
        ].T
    )

    range_azimuth_display = normalized_magnitude_db(
        range_azimuth_map[
            :range_bin_limit,
            :
        ],
        minimum_db=display_floor_db,
    )

    unfocused_display = normalized_magnitude_db(
        unfocused_selected_rx,
        minimum_db=display_floor_db,
    )

    first_sar_display = np.maximum(
        np.asarray(first_sar_image_db),
        display_floor_db,
    )

    final_sar_display = np.maximum(
        np.asarray(final_sar_image_db),
        display_floor_db,
    )

    first_peak_index = np.unravel_index(
        np.argmax(first_sar_display),
        first_sar_display.shape,
    )

    final_peak_index = np.unravel_index(
        np.argmax(final_sar_display),
        final_sar_display.shape,
    )

    first_peak_x_m = cross_range_grid_m[
        first_peak_index[1]
    ]

    first_peak_z_m = downrange_grid_m[
        first_peak_index[0]
    ]

    final_peak_x_m = cross_range_grid_m[
        final_peak_index[1]
    ]

    final_peak_z_m = downrange_grid_m[
        final_peak_index[0]
    ]

    figure, axes = plt.subplots(
        1,
        4,
        figsize=(23, 6.5),
        constrained_layout=True,
    )

    figure.suptitle(
        "Run C Processing Evolution: Conventional Radar to Focused SAR",
        fontsize=16,
    )

    common_image_arguments = {
        "origin": "lower",
        "cmap": "viridis",
        "vmin": display_floor_db,
        "vmax": 0.0,
    }

    # Panel 1: conventional four-RX beamforming at one rail position.
    image_0 = axes[0].imshow(
        range_azimuth_display,
        extent=[
            azimuth_angle_deg[0],
            azimuth_angle_deg[-1],
            -0.5,
            range_bin_limit - 0.5,
        ],
        aspect="auto",
        **common_image_arguments,
    )

    axes[0].axhline(
        observed_range_bin,
        color="cyan",
        linestyle="-.",
        linewidth=1.5,
        label=f"Observed q = {observed_range_bin}",
    )

    axes[0].axvline(
        0.0,
        color="white",
        linestyle="--",
        linewidth=1.0,
        alpha=0.8,
        label="Boresight",
    )

    axes[0].set_title(
        "1. Conventional Four-RX Heatmap\n"
        "Center rail position, rectangular windows"
    )

    axes[0].set_xlabel(
        "Azimuth angle (degrees)"
    )

    axes[0].set_ylabel(
        "Range-bin index, $q$"
    )

    axes[0].legend(
        loc="upper right",
        fontsize=8,
    )

    # Panel 2: target data over the mechanical rail before focusing.
    axes[1].imshow(
        unfocused_display,
        extent=[
            100 * rail_position_m[0],
            100 * rail_position_m[-1],
            -0.5,
            range_bin_limit - 0.5,
        ],
        aspect="auto",
        **common_image_arguments,
    )

    axes[1].axhline(
        observed_range_bin,
        color="cyan",
        linestyle="-.",
        linewidth=1.5,
        label=f"Observed q = {observed_range_bin}",
    )

    axes[1].axvline(
        0.0,
        color="white",
        linestyle="--",
        linewidth=1.0,
        alpha=0.8,
        label="Aperture center",
    )

    axes[1].set_title(
        "2. Unfocused Synthetic-Aperture Data\n"
        f"Complex residual magnitude, RX {selected_rx_channel}"
    )

    axes[1].set_xlabel(
        "Rail position, $x_m$ (cm)"
    )

    axes[1].set_ylabel(
        "Range-bin index, $q$"
    )

    axes[1].legend(
        loc="upper right",
        fontsize=8,
    )

    sar_extent_cm = [
        100 * cross_range_grid_m[0],
        100 * cross_range_grid_m[-1],
        100 * downrange_grid_m[0],
        100 * downrange_grid_m[-1],
    ]

    # Panel 3: the first backprojection attempt.
    axes[2].imshow(
        first_sar_display,
        extent=sar_extent_cm,
        aspect="equal",
        **common_image_arguments,
    )

    axes[2].plot(
        100 * registration_cross_range_m,
        100 * registration_downrange_m,
        marker="+",
        color="red",
        markersize=14,
        markeredgewidth=2,
        label="Registration point",
    )

    axes[2].plot(
        100 * first_peak_x_m,
        100 * first_peak_z_m,
        marker="x",
        color="cyan",
        markersize=11,
        markeredgewidth=2,
        label="Focused peak",
    )

    axes[2].set_title(
        "3. First SAR Backprojection\n"
        "128-point range FFT, uniform aperture"
    )

    axes[2].set_xlabel(
        "Cross-range, $x$ (cm)"
    )

    axes[2].set_ylabel(
        "Downrange, $z$ (cm)"
    )

    axes[2].legend(
        loc="upper right",
        fontsize=8,
    )

    # Panel 4: the selected Run C processing configuration.
    axes[3].imshow(
        final_sar_display,
        extent=sar_extent_cm,
        aspect="equal",
        **common_image_arguments,
    )

    axes[3].plot(
        100 * registration_cross_range_m,
        100 * registration_downrange_m,
        marker="+",
        color="red",
        markersize=14,
        markeredgewidth=2,
        label="Registration point",
    )

    axes[3].plot(
        100 * final_peak_x_m,
        100 * final_peak_z_m,
        marker="x",
        color="cyan",
        markersize=11,
        markeredgewidth=2,
        label="Focused peak",
    )

    axes[3].set_title(
        "4. Final Run C SAR Image\n"
        "Hamming range and aperture windows"
    )

    axes[3].set_xlabel(
        "Cross-range, $x$ (cm)"
    )

    axes[3].set_ylabel(
        "Downrange, $z$ (cm)"
    )

    axes[3].legend(
        loc="upper right",
        fontsize=8,
    )

    figure.colorbar(
        image_0,
        ax=axes,
        shrink=0.83,
        pad=0.02,
        label="Independently normalized level (dB)",
    )

    comparison_metrics = pd.DataFrame(
        {
            "stage": [
                "First 128-point SAR",
                "Final Hamming SAR",
            ],
            "peak_cross_range_m": [
                first_peak_x_m,
                final_peak_x_m,
            ],
            "peak_downrange_m": [
                first_peak_z_m,
                final_peak_z_m,
            ],
        }
    )

    return figure, comparison_metrics

### 3.8.2 — Form and Compare the Run C Processing Stages

This subsection forms the four-panel processing comparison.

The stages are:

1. **Conventional four-RX processing:** rectangular range and Doppler windows are applied to the Run C data at the center rail position. The repeated frames are coherently averaged, and conventional beamforming is performed across the four physical RX channels.

2. **Unfocused synthetic-aperture data:** the complex background-subtracted range response is displayed over all 17 rail positions for the selected RX channel. No geometric phase correction or coherent aperture summation has yet been performed.

3. **First SAR reconstruction:** the initial backprojection result uses the original 128-point range FFT and uniform synthetic-aperture weighting. This image demonstrates that geometric phase compensation can localize the target, but it also contains strong sidelobes and coarse-range interpolation artifacts.

4. **Final Run C reconstruction:** the selected processing path uses a Hamming range window, a 1024-point zero-padded range FFT, and Hamming synthetic-aperture weighting. These choices provide denser range interpolation and lower range and cross-range sidelobes, at the cost of broader main lobes.

All panels are independently normalized to their own maxima and use the same display floor. Their colors therefore compare image structure and artifact suppression, not absolute signal strength or processing gain.

In [ ]:
# Use the exact intermediate products generated in the earlier sections.
#
# Section 3.2:
# Complex background-subtracted data before SAR focusing.
run_c_unfocused_residual = (
    run_c_sar_background_subtracted
)

# Section 3.4:
# First 128-point, uniformly weighted SAR backprojection.
first_run_c_sar_image_db = (
    run_c_sar_backprojection_db
)

# Section 3.7:
# Final selected result: Hamming range window, oversampled range FFT,
# and Hamming synthetic-aperture weighting.
final_run_c_sar_image_db = (
    sar_aperture_window_results[
        "hamming"
    ]["image_db"]
)


# The backprojection function stored the spatial coordinates as 2-D
# meshgrids. Extract their one-dimensional x and z coordinate axes for
# plotting and locating the image peaks.
comparison_cross_range_grid_m = (
    run_c_sar_cross_range_grid_m[0, :]
)

comparison_downrange_grid_m = (
    run_c_sar_downrange_grid_m[:, 0]
)


print(
    "Unfocused Run C data shape:",
    run_c_unfocused_residual.shape,
)

print(
    "First SAR image shape:",
    first_run_c_sar_image_db.shape,
)

print(
    "Final SAR image shape:",
    final_run_c_sar_image_db.shape,
)


# Reconstruct an original-style range–azimuth heatmap using Run C at
# the center rail position. Rectangular range and Doppler windows are
# used to represent the minimally processed starting point.
(
    run_c_minimal_center_profile,
    run_c_minimal_range_azimuth_map,
) = form_minimal_center_position_range_azimuth(
    run_c_radc=run_c_sar_radc,
    center_position_index=8,
    azimuth_angle_deg=azimuth_angle_deg,
    rx_positions_m=rx_positions_m,
    carrier_frequency_hz=carrier_frequency_hz,
)


# Form the four-panel processing-evolution figure.
run_c_evolution_figure, run_c_evolution_metrics = (
    plot_run_c_processing_evolution(
        range_azimuth_map=(
            run_c_minimal_range_azimuth_map
        ),
        azimuth_angle_deg=azimuth_angle_deg,
        unfocused_residual_cube=(
            run_c_unfocused_residual
        ),
        rail_position_m=(
            sar_position_table[
                "rail_position_m"
            ].to_numpy()
        ),
        selected_rx_channel=3,
        first_sar_image_db=(
            first_run_c_sar_image_db
        ),
        final_sar_image_db=(
            final_run_c_sar_image_db
        ),
        cross_range_grid_m=(
            comparison_cross_range_grid_m
        ),
        downrange_grid_m=(
            comparison_downrange_grid_m
        ),
        observed_range_bin=22,
        registration_cross_range_m=0.0,
        registration_downrange_m=0.84,
        range_bin_limit=50,
        display_floor_db=-25.0,
    )
)

plt.show()

display(run_c_evolution_metrics)

### 3.8.3 — Results and Interpretation

<div style="font-size:2.0em; font-weight:bold;">Conventional four-RX heatmap</div>

The first panel is a conventional range–azimuth heatmap formed from only the four physical V-MD3 RX channels at the center rail position. It is not a synthetic-aperture image.

The four-element physical array can estimate a broad arrival angle, but its short aperture provides limited angular resolution. A compact target can therefore appear as a wide angular response rather than a sharply localized cross-range object.

This panel also retains the original range-bin coordinate. The target-associated response remains near $q=22$ because no absolute range correction is applied merely by forming the heatmap.

<div style="font-size:2.0em; font-weight:bold;">Unfocused synthetic-aperture data</div>

The second panel shows the complex background-subtracted RX data as a function of rail position and range bin before backprojection. Each column is the range response measured from one radar position.

The target return is visible near $q=22$ over much of the rail. It has not yet been converted into a spatially localized target because the propagation phase associated with each radar position has not been compensated.

This magnitude view does not necessarily look like a random or jumbled mess. The information that prevents direct coherent addition is mainly contained in the complex phase, and magnitude plotting hides that phase. The important observation is that the target remains distributed across rail position rather than focused into one cross-range and downrange location.

<div style="font-size:2.0em; font-weight:bold;">First coherent SAR reconstruction</div>

The third panel applies backprojection for the first time. For every candidate image pixel, the algorithm:

1. calculates the predicted radar-to-pixel distance at every rail position;
2. samples the corresponding complex range response;
3. removes the predicted two-way propagation phase;
4. coherently adds the phase-corrected measurements.

A pixel near the true scattering location receives contributions that align in phase and add constructively. Incorrect pixels retain phase disagreement and add less strongly.

The first reconstruction successfully localizes the sphere-associated return near the measured scene location. However, the original 128-point range grid requires coarse complex interpolation, and uniform synthetic-aperture weighting retains relatively strong sidelobes. The resulting compact-looking downrange structure was partly an interpolation artifact rather than genuine range resolution.

<div style="font-size:2.0em; font-weight:bold;">Final selected Run C reconstruction</div>

The fourth panel uses the selected Run C processing configuration:

- Hamming fast-time range weighting;
- a 1024-point zero-padded range FFT;
- Hann Doppler weighting;
- coherent frame integration;
- matched complex Run B background subtraction;
- Hamming synthetic-aperture weighting;
- complex range interpolation and coherent backprojection.

The denser range grid reduces the misleading interpolation structure seen in the first reconstruction. Hamming range weighting suppresses range sidelobes, while Hamming synthetic-aperture weighting suppresses cross-range sidelobes. These improvements broaden the main response in exchange for a cleaner image.

The final bright region should be interpreted as the measured point-spread response of the sphere's dominant scattering center, not as a resolved optical outline of the sphere. Its size is controlled by bandwidth, aperture length, windowing, propagation geometry, and the target's scattering behavior.

All panels are normalized independently. Consequently, equal colors do not indicate equal absolute received power or processing gain.

This is the frozen candidate Run C processing pipeline. The next step is to apply the same choices to Run D without retuning them. Successful localization of Run D at its independently shifted cross-range position will provide a stronger validation than further optimization on Run C.

<div style="font-size:2.0em; font-weight:bold;">Range registration</div>

The focused image appears near $0.84\ \text{m}$ because a one-point local range registration was introduced during backprojection. The observed target bin was assigned to the independently measured target distance:

$$
q_{\text{ref}}=22
\quad\longleftrightarrow\quad
R_{\text{reg}}=0.84\ \text{m}.
$$

This registration removes the observed local range offset when constructing the SAR coordinate grid. It does not explain or correct the underlying V-MD3 timing or range-calibration error, and it does not independently validate the radar's absolute range scale.

The final peak near $0.834\ \text{m}$ differs from the registration distance by approximately one sample of the 1024-point zero-padded range grid. This small difference is consistent with discrete sampling and changes in the windowed, background-subtracted response.

# Current Cell Development Debug Below